# TACO / Atik14082026 — FULL FIXED v5

**Study:** robust real-time litter detection on TACO  
**Task:** TACO-1 one-class litter detection  
**Protocols:** random image split + batch-disjoint split  
**Models:** YOLOv8s, YOLO11s, YOLO26s, RT-DETR-L  
**Seeds:** 42, 123, 2026  
**Primary metric:** COCO mAP@[0.50:0.95]

## v5 correction — EXIF-aware geometry

The official final TACO release is accepted only when it has:
- 1500 image records
- 4784 annotations
- 60 categories
- all images resolvable from the official Zenodo archive

JPEG dimensions are validated in an EXIF-aware way, matching Ultralytics' own
image-verification behavior. Every canonical image is then loaded with OpenCV
to confirm that the model-visible geometry exactly matches the COCO width/height.

If an image fails the OpenCV geometry check, v5 physically applies EXIF
orientation to a local derived copy and verifies it again. Training stops if
any image still disagrees.

Persistent output:
`/content/drive/MyDrive/Atik14082026/TACO_Pipeline_v5`

Local workspace:
`/content/taco_v5`

In [ ]:
# STAGE 0 — INSTALL FIXED DEPENDENCIES
import sys, subprocess

PACKAGES = [
    "ultralytics==8.4.111",
    "pycocotools>=2.0.8",
    "imagehash>=4.3.2",
    "scikit-learn>=1.5",
    "pandas>=2.2",
    "numpy>=1.26",
    "scipy>=1.13",
    "Pillow>=10.3",
    "PyYAML>=6.0",
    "tqdm>=4.66",
    "matplotlib>=3.8",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
print("Dependencies installed.")

In [ ]:
# STAGE 1 — GPU CHECK + DRIVE + PROJECT TREE
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import json, os, sys, platform, subprocess, hashlib, shutil

BASE = Path("/content/drive/MyDrive/Atik14082026")
ROOT = BASE / "TACO_Pipeline_v5"

# Local (fast) workspace. This disappears after a runtime reset and is rebuilt.
LOCAL = Path("/content/taco_v5")
LOCAL_ZIP = Path("/content/TACO.zip")
LOCAL_EXTRACT = LOCAL / "zenodo"
LOCAL_REPO = LOCAL / "TACO_repo_master"
LOCAL_DATASET = LOCAL / "dataset"

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}
for p in [BASE, ROOT, LOCAL, *DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

import torch, ultralytics, numpy as np, pandas as pd

gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()

ENV = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_query": gpu_query,
    "ultralytics": ultralytics.__version__,
}
(DIRS["manifests"] / "environment.json").write_text(
    json.dumps(ENV, indent=2), encoding="utf-8"
)

print(json.dumps(ENV, indent=2))
assert torch.cuda.is_available(), "GPU runtime required: Runtime > Change runtime type > GPU."
print("\nProject:", ROOT)
print("Local workspace:", LOCAL)

## STAGE 2 — Version-safe official TACO acquisition

Data archive:
- Zenodo DOI: `10.5281/zenodo.3587843`
- File: `TACO.zip`
- MD5: `e9149407d883e21a8d224feef8210920`

Annotation resolution:
- official GitHub `pedropro/TACO` master is cloned;
- the exact master commit is recorded;
- any COCO-style annotation JSON inside the Zenodo archive is also inspected;
- candidates are tested against the extracted images;
- only a candidate with **1500 images + 4784 annotations + 100% image dimension agreement** is accepted.

This makes the pipeline robust to the historical `v0.1`/final-release mismatch.

In [ ]:
# STAGE 2A — DOWNLOAD OFFICIAL ZENODO ARCHIVE WITH RESUME + MD5 RETRY
from pathlib import Path
import subprocess, hashlib, os

from pathlib import Path

BASE = Path("/content/drive/MyDrive/Atik14082026")
ROOT = BASE / "TACO_Pipeline_v5"

LOCAL = Path("/content/taco_v5")
LOCAL_ZIP = Path("/content/TACO.zip")
LOCAL_EXTRACT = LOCAL / "zenodo"
LOCAL_REPO = LOCAL / "TACO_repo_master"
LOCAL_DATASET = LOCAL / "dataset"

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in [BASE, ROOT, LOCAL, *DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

ZENODO_URL = "https://zenodo.org/records/3587843/files/TACO.zip?download=1"
EXPECTED_MD5 = "e9149407d883e21a8d224feef8210920"

def md5_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def download_archive():
    LOCAL_ZIP.parent.mkdir(parents=True, exist_ok=True)
    # -c resumes a partial file. Retries make transient Zenodo failures less disruptive.
    subprocess.check_call([
        "wget",
        "-c",
        "--tries=10",
        "--timeout=60",
        "--retry-connrefused",
        "--show-progress",
        "-O", str(LOCAL_ZIP),
        ZENODO_URL
    ])

# Always let wget inspect/resume the current file.
print("Downloading/resuming official TACO.zip...")
download_archive()

actual = md5_file(LOCAL_ZIP)
print("Expected MD5:", EXPECTED_MD5)
print("Actual MD5  :", actual)

# If a pre-existing partial/corrupt file confused resume, retry once from zero.
if actual != EXPECTED_MD5:
    print("\nMD5 mismatch. Deleting local archive and retrying once from zero...")
    LOCAL_ZIP.unlink(missing_ok=True)
    download_archive()
    actual = md5_file(LOCAL_ZIP)
    print("Retry MD5   :", actual)

if actual != EXPECTED_MD5:
    raise RuntimeError(
        "Official TACO.zip MD5 still mismatches after a clean retry. "
        "Do not continue; this indicates a corrupted/incomplete download."
    )

print("\nZENODO ARCHIVE MD5 PASSED.")

In [ ]:
# STAGE 2B — EXTRACT ARCHIVE + CLONE OFFICIAL MASTER + RECORD EXACT COMMIT
import subprocess, shutil, json, hashlib
from pathlib import Path

from pathlib import Path

BASE = Path("/content/drive/MyDrive/Atik14082026")
ROOT = BASE / "TACO_Pipeline_v5"

LOCAL = Path("/content/taco_v5")
LOCAL_ZIP = Path("/content/TACO.zip")
LOCAL_EXTRACT = LOCAL / "zenodo"
LOCAL_REPO = LOCAL / "TACO_repo_master"
LOCAL_DATASET = LOCAL / "dataset"

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in [BASE, ROOT, LOCAL, *DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

if not LOCAL_ZIP.exists():
    raise FileNotFoundError(
        f"{LOCAL_ZIP} not found. Run STAGE 2A first."
    )

EXTRACT_MARKER = LOCAL_EXTRACT / ".extract_complete"

if not EXTRACT_MARKER.exists():
    if LOCAL_EXTRACT.exists():
        shutil.rmtree(LOCAL_EXTRACT)
    LOCAL_EXTRACT.mkdir(parents=True, exist_ok=True)
    print("Extracting official TACO.zip...")
    subprocess.check_call([
        "unzip", "-q",
        str(LOCAL_ZIP),
        "-d", str(LOCAL_EXTRACT)
    ])
    EXTRACT_MARKER.write_text("ok\n", encoding="utf-8")
else:
    print("Archive already extracted in this runtime.")

if not (LOCAL_REPO / ".git").exists():
    if LOCAL_REPO.exists():
        shutil.rmtree(LOCAL_REPO)
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/pedropro/TACO.git",
        str(LOCAL_REPO)
    ])
else:
    print("Official TACO master repository already present.")

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(LOCAL_REPO), "rev-parse", "HEAD"],
    text=True
).strip()

MASTER_JSON = LOCAL_REPO / "data" / "annotations.json"
assert MASTER_JSON.exists(), f"Official master annotations not found: {MASTER_JSON}"

master = json.loads(MASTER_JSON.read_text(encoding="utf-8"))

print("Official GitHub commit :", GIT_COMMIT)
print("Master image records   :", len(master.get("images", [])))
print("Master annotations     :", len(master.get("annotations", [])))
print("Master categories      :", len(master.get("categories", [])))

exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
archive_images = [
    p for p in LOCAL_EXTRACT.rglob("*")
    if p.is_file() and p.suffix.lower() in exts
]
archive_jsons = [
    p for p in LOCAL_EXTRACT.rglob("*.json")
    if p.is_file()
]

print("Image-like files in archive :", len(archive_images))
print("JSON files in archive       :", len(archive_jsons))

In [ ]:
# STAGE 2C — AUTO-RESOLVE THE ANNOTATION SOURCE THAT ACTUALLY MATCHES THE ZENODO IMAGES
import json, os, hashlib, subprocess
from collections import defaultdict
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd

from pathlib import Path

BASE = Path("/content/drive/MyDrive/Atik14082026")
ROOT = BASE / "TACO_Pipeline_v5"

LOCAL = Path("/content/taco_v5")
LOCAL_ZIP = Path("/content/TACO.zip")
LOCAL_EXTRACT = LOCAL / "zenodo"
LOCAL_REPO = LOCAL / "TACO_repo_master"
LOCAL_DATASET = LOCAL / "dataset"

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in [BASE, ROOT, LOCAL, *DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

if not LOCAL_EXTRACT.exists():
    raise FileNotFoundError(
        "Extracted Zenodo dataset not found. Run STAGE 2B first."
    )

MASTER_JSON = LOCAL_REPO / "data" / "annotations.json"
if not MASTER_JSON.exists():
    raise FileNotFoundError(
        "Official GitHub master annotations not found. Run STAGE 2B first."
    )

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(LOCAL_REPO), "rev-parse", "HEAD"],
    text=True
).strip()

EXPECTED_IMAGES = 1500
EXPECTED_ANNOTATIONS = 4784
EXPECTED_CATEGORIES = 60

import cv2
from PIL import Image, ImageOps
from ultralytics.data.utils import exif_size

# ------------------------------------------------------------
# 1. Index Zenodo images
# ------------------------------------------------------------
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
archive_images = [
    p for p in LOCAL_EXTRACT.rglob("*")
    if p.is_file() and p.suffix.lower() in exts
]

by_tail2 = defaultdict(list)
by_tail3 = defaultdict(list)
by_name = defaultdict(list)
geom_cache = {}

def image_geometry(path):
    key = str(path)
    if key in geom_cache:
        return geom_cache[key]
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            raw = tuple(im.size)
            corrected = tuple(exif_size(im))
            orientation = None
            try:
                orientation = im.getexif().get(274, None)
            except Exception:
                pass
        g = {
            "raw": raw,
            "exif": corrected,
            "orientation": orientation,
            "corrupt": False,
        }
    except Exception as e:
        g = {
            "raw": None, "exif": None, "orientation": None,
            "corrupt": True, "error": str(e),
        }
    geom_cache[key] = g
    return g

for p in tqdm(archive_images, desc="Indexing Zenodo images"):
    parts = p.parts
    if len(parts) >= 2:
        by_tail2["/".join(parts[-2:])].append(p)
    if len(parts) >= 3:
        by_tail3["/".join(parts[-3:])].append(p)
    by_name[p.name].append(p)

print("Image-like files in archive:", len(archive_images))

# ------------------------------------------------------------
# 2. Collect COCO-style annotation candidates
# ------------------------------------------------------------
candidate_paths = [MASTER_JSON]
for p in LOCAL_EXTRACT.rglob("*.json"):
    if "unofficial" in p.name.lower() or "unofficial" in str(p.parent).lower():
        continue
    if p not in candidate_paths:
        candidate_paths.append(p)

def load_coco_candidate(path):
    try:
        obj = json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None
    if not isinstance(obj, dict):
        return None
    if not all(k in obj for k in ("images","annotations","categories")):
        return None
    if not isinstance(obj["images"], list) or not isinstance(obj["annotations"], list):
        return None
    return obj

def candidate_pool(imrec):
    rel = Path(imrec["file_name"])
    pools = []
    if len(rel.parts) >= 3:
        pools.extend(by_tail3.get("/".join(rel.parts[-3:]), []))
    if len(rel.parts) >= 2:
        pools.extend(by_tail2.get("/".join(rel.parts[-2:]), []))
    pools.extend(by_name.get(rel.name, []))
    out, seen = [], set()
    for p in pools:
        sp = str(p)
        if sp not in seen:
            seen.add(sp)
            out.append(p)
    return out

def resolve_record(imrec):
    rel = Path(imrec["file_name"])
    expected = (int(imrec["width"]), int(imrec["height"]))
    candidates = candidate_pool(imrec)

    if not candidates:
        return None, "missing"

    raw_good, exif_good, corrupt = [], [], []
    for p in candidates:
        g = image_geometry(p)
        if g["corrupt"]:
            corrupt.append(p)
            continue
        if g["raw"] == expected:
            raw_good.append(p)
        if g["exif"] == expected:
            exif_good.append(p)

    good = exif_good

    if len(good) == 1:
        p = good[0]
        return p, ("raw_match" if p in raw_good else "exif_rotation_match")

    if len(good) > 1:
        batch = rel.parts[0] if len(rel.parts) > 1 else None
        preferred = [p for p in good if batch and batch in p.parts]
        p = preferred[0] if len(preferred) == 1 else sorted(good, key=lambda x: str(x))[0]
        return p, ("duplicate_path_raw_match" if p in raw_good else "duplicate_path_exif_match")

    if corrupt and len(corrupt) == len(candidates):
        return None, "corrupt"

    reversed_raw = [
        p for p in candidates
        if (not image_geometry(p)["corrupt"])
        and image_geometry(p)["raw"] == (expected[1], expected[0])
    ]
    if reversed_raw:
        return None, "raw_reversed_but_exif_not_matching"

    return None, "dimension_mismatch"

# ------------------------------------------------------------
# 3. Score candidates with EXIF-aware geometry
# ------------------------------------------------------------
reports = []
candidate_objects = {}
candidate_maps = {}
candidate_modes = {}

for path in candidate_paths:
    obj = load_coco_candidate(path)
    if obj is None:
        continue

    source = "github_master" if Path(path) == MASTER_JSON else "zenodo_archive_json"
    counts = defaultdict(int)
    resolved = {}
    modes = {}

    for imrec in tqdm(
        obj["images"],
        desc=f"Checking {source}:{Path(path).name} ({len(obj['images'])} images)",
        leave=False
    ):
        p, status = resolve_record(imrec)
        counts[status] += 1
        if p is not None:
            resolved[int(imrec["id"])] = str(p)
            modes[int(imrec["id"])] = status

    n_img = len(obj["images"])
    n_ann = len(obj["annotations"])
    n_cat = len(obj["categories"])
    published = (
        n_img == EXPECTED_IMAGES
        and n_ann == EXPECTED_ANNOTATIONS
        and n_cat == EXPECTED_CATEGORIES
    )
    full = len(resolved) == n_img

    rec = {
        "source": source,
        "path": str(path),
        "images": n_img,
        "annotations": n_ann,
        "categories": n_cat,
        "resolved": len(resolved),
        "raw_match": counts["raw_match"],
        "exif_rotation_match": counts["exif_rotation_match"],
        "duplicate_path_raw_match": counts["duplicate_path_raw_match"],
        "duplicate_path_exif_match": counts["duplicate_path_exif_match"],
        "missing": counts["missing"],
        "dimension_mismatch": counts["dimension_mismatch"],
        "raw_reversed_but_exif_not_matching": counts["raw_reversed_but_exif_not_matching"],
        "corrupt": counts["corrupt"],
        "published_counts_match": published,
        "full_exif_geometry_match": full,
        "eligible": bool(published and full),
    }
    reports.append(rec)
    candidate_objects[str(path)] = obj
    candidate_maps[str(path)] = resolved
    candidate_modes[str(path)] = modes

report_df = pd.DataFrame(reports)
report_df.to_csv(
    DIRS["audit"] / "annotation_source_candidates_exif_aware.csv",
    index=False
)
display(report_df)

eligible = report_df[report_df["eligible"] == True].copy()
if len(eligible) == 0:
    raise RuntimeError(
        "No final-release candidate achieved 1500 images / 4784 annotations / "
        "60 categories with 100% EXIF-aware geometry matching."
    )

eligible["priority"] = eligible["source"].map({
    "zenodo_archive_json": 0,
    "github_master": 1,
}).fillna(9)
chosen = eligible.sort_values(["priority","path"]).iloc[0]

SELECTED_SOURCE = str(chosen["source"])
SELECTED_ORIGINAL_JSON = Path(chosen["path"])
selected_coco = candidate_objects[str(SELECTED_ORIGINAL_JSON)]
selected_resolved = candidate_maps[str(SELECTED_ORIGINAL_JSON)]
selected_modes = candidate_modes[str(SELECTED_ORIGINAL_JSON)]

PERSISTED_JSON = DIRS["metadata"] / "TACO_SELECTED_OFFICIAL_annotations.json"
PERSISTED_JSON.write_text(
    json.dumps(selected_coco, ensure_ascii=False),
    encoding="utf-8"
)
OFFICIAL_JSON = PERSISTED_JSON
coco = selected_coco

# ------------------------------------------------------------
# 4. Build canonical image tree + verify model-visible geometry
# ------------------------------------------------------------
CANON_IMAGES = LOCAL_DATASET / "images"
CANON_IMAGES.mkdir(parents=True, exist_ok=True)

opencv_direct_ok = 0
normalized_files = []
opencv_failures = []

for imrec in tqdm(
    coco["images"],
    desc="Validating model-visible geometry"
):
    iid = int(imrec["id"])
    src = Path(selected_resolved[iid])
    dst = CANON_IMAGES / imrec["file_name"]
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists() or dst.is_symlink():
        dst.unlink()

    try:
        os.link(src, dst)
    except OSError:
        os.symlink(src, dst)

    expected = (int(imrec["width"]), int(imrec["height"]))
    arr = cv2.imread(str(dst), cv2.IMREAD_COLOR)

    if arr is not None and (arr.shape[1], arr.shape[0]) == expected:
        opencv_direct_ok += 1
        continue

    # Deterministic fallback: physically apply EXIF orientation locally.
    if dst.exists() or dst.is_symlink():
        dst.unlink()

    try:
        with Image.open(src) as pil:
            fixed = ImageOps.exif_transpose(pil).convert("RGB")
            if fixed.size != expected:
                raise ValueError(
                    f"EXIF-transposed size {fixed.size} != expected {expected}"
                )
            fixed.save(dst, "JPEG", quality=100, subsampling=0)

        arr = cv2.imread(str(dst), cv2.IMREAD_COLOR)
        if arr is None or (arr.shape[1], arr.shape[0]) != expected:
            raise ValueError(
                f"OpenCV size after normalization="
                f"{None if arr is None else (arr.shape[1], arr.shape[0])}, "
                f"expected={expected}"
            )

        normalized_files.append({
            "image_id": iid,
            "file_name": imrec["file_name"],
            "source": str(src),
            "resolution_mode": selected_modes.get(iid),
        })

    except Exception as e:
        opencv_failures.append({
            "image_id": iid,
            "file_name": imrec["file_name"],
            "source": str(src),
            "expected": list(expected),
            "resolution_mode": selected_modes.get(iid),
            "error": str(e),
        })

pd.DataFrame(
    normalized_files,
    columns=["image_id","file_name","source","resolution_mode"]
).to_csv(
    DIRS["audit"] / "physically_exif_normalized_images.csv",
    index=False
)

(DIRS["audit"] / "model_visible_geometry_failures.json").write_text(
    json.dumps(opencv_failures, indent=2),
    encoding="utf-8"
)

if opencv_failures:
    raise RuntimeError(
        f"{len(opencv_failures)} images still disagree with COCO geometry "
        "after EXIF normalization. Do not continue."
    )

assert len(coco["images"]) == 1500
assert len(coco["annotations"]) == 4784
assert len(coco["categories"]) == 60

print("\n" + "="*70)
print("OFFICIAL TACO SOURCE + EXIF GEOMETRY RESOLVED")
print("="*70)
print("Selected source             :", SELECTED_SOURCE)
print("Images                      :", len(coco["images"]))
print("Annotations                 :", len(coco["annotations"]))
print("Categories                  :", len(coco["categories"]))
print("EXIF-aware records resolved :", len(selected_resolved), "/", len(coco["images"]))
print("OpenCV direct geometry OK   :", opencv_direct_ok)
print("Physically EXIF-normalized  :", len(normalized_files))
print("Model-visible failures      :", len(opencv_failures))
print("="*70)
print("SOURCE/IMAGE/EXIF RECONCILIATION PASSED.")

In [ ]:
# STAGE 2D — FREEZE EXACT RAW-SOURCE MANIFEST
import hashlib, json, subprocess
from pathlib import Path

from pathlib import Path

BASE = Path("/content/drive/MyDrive/Atik14082026")
ROOT = BASE / "TACO_Pipeline_v5"

LOCAL = Path("/content/taco_v5")
LOCAL_ZIP = Path("/content/TACO.zip")
LOCAL_EXTRACT = LOCAL / "zenodo"
LOCAL_REPO = LOCAL / "TACO_repo_master"
LOCAL_DATASET = LOCAL / "dataset"

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in [BASE, ROOT, LOCAL, *DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

required = ["OFFICIAL_JSON", "SELECTED_SOURCE", "SELECTED_ORIGINAL_JSON", "coco"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run STAGE 2C immediately before STAGE 2D. "
        f"Missing variables: {missing}"
    )

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(LOCAL_REPO), "rev-parse", "HEAD"],
    text=True
).strip()

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

manifest = {
    "dataset": "TACO: Trash Annotations in Context",
    "zenodo_doi": "10.5281/zenodo.3587843",
    "zenodo_file": "TACO.zip",
    "zenodo_md5": "e9149407d883e21a8d224feef8210920",
    "official_repo": "https://github.com/pedropro/TACO",
    "github_master_commit_at_run": GIT_COMMIT,
    "selected_annotation_source_type": SELECTED_SOURCE,
    "selected_annotation_original_path": str(SELECTED_ORIGINAL_JSON),
    "persisted_annotation_path": str(OFFICIAL_JSON),
    "selected_annotations_sha256": sha256_file(OFFICIAL_JSON),
    "image_records": len(coco["images"]),
    "annotations": len(coco["annotations"]),
    "categories": len(coco["categories"]),
    "expected_published_release_counts": {
        "images": 1500,
        "annotations": 4784
    },
    "annotation_policy": "official final-release candidate only; unofficial annotations excluded",
    "geometry_policy": "EXIF-aware geometry + OpenCV model-visible verification",
    "opencv_direct_geometry_ok": opencv_direct_ok,
    "physically_exif_normalized_count": len(normalized_files),
}
p = DIRS["manifests"] / "raw_source_manifest.json"
p.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

## STAGE 3 — Strict dataset audit

This stage checks:

- annotation geometry,
- missing/invalid category/image references,
- exact SHA-256 duplicate images,
- perceptual-hash near-duplicate candidates,
- object-size distribution using standard COCO area bins,
- TACO batch distribution.

Near-duplicate pHash matches are **screening candidates only** and are not automatically removed.

In [ ]:
# STAGE 3A — ANNOTATION + IMAGE HASH AUDIT
# FIXED VERSION:
# - Official TACO annotations are preserved as source data.
# - Bounding boxes extending slightly outside image boundaries are clipped.
# - Every repair is logged.
# - Severe/malformed annotations still stop the pipeline.
# - Cleaned COCO annotations are persisted and used downstream.

import json
import hashlib
import numpy as np
import pandas as pd
import imagehash
import cv2

from pathlib import Path
from PIL import Image, ImageOps
from collections import defaultdict, Counter
from tqdm.auto import tqdm


# ============================================================
# PATHS
# ============================================================

BASE = Path("/content/drive/MyDrive/Atik14082026")
ROOT = BASE / "TACO_Pipeline_v5"

LOCAL = Path("/content/taco_v5")
LOCAL_DATASET = LOCAL / "dataset"
CANON_IMAGES = LOCAL_DATASET / "images"

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOAD OFFICIAL ANNOTATIONS SELECTED IN STAGE 2C
# ============================================================

SOURCE_JSON = (
    DIRS["metadata"] /
    "TACO_SELECTED_OFFICIAL_annotations.json"
)

if not SOURCE_JSON.exists():
    raise FileNotFoundError(
        "TACO_SELECTED_OFFICIAL_annotations.json not found. "
        "Run STAGE 2C first."
    )

coco_original = json.loads(
    SOURCE_JSON.read_text(encoding="utf-8")
)

# Deep copy so the original annotation source remains unchanged.
coco = json.loads(json.dumps(coco_original))

images = {
    int(im["id"]): im
    for im in coco["images"]
}

categories = {
    int(c["id"]): c["name"]
    for c in coco["categories"]
}


# ============================================================
# VALIDATE + REPAIR BOUNDING BOXES
# ============================================================

anns_by_img = defaultdict(list)

repair_log = []
severe_invalid = []

clean_annotations = []

for a in coco["annotations"]:

    ann = dict(a)

    ann_id = int(ann["id"])
    iid = int(ann["image_id"])
    cid = int(ann["category_id"])

    # --------------------------------------------------------
    # Referential integrity
    # --------------------------------------------------------

    if iid not in images:

        severe_invalid.append({
            "ann_id": ann_id,
            "image_id": iid,
            "reason": "missing_image_record"
        })

        continue

    if cid not in categories:

        severe_invalid.append({
            "ann_id": ann_id,
            "image_id": iid,
            "reason": "missing_category_record"
        })

        continue


    im = images[iid]

    W = float(im["width"])
    H = float(im["height"])


    # --------------------------------------------------------
    # Basic bbox integrity
    # --------------------------------------------------------

    try:

        x, y, w, h = map(
            float,
            ann["bbox"]
        )

    except Exception:

        severe_invalid.append({
            "ann_id": ann_id,
            "image_id": iid,
            "reason": "malformed_bbox",
            "bbox": ann.get("bbox")
        })

        continue


    if not all(
        np.isfinite([x, y, w, h])
    ):

        severe_invalid.append({
            "ann_id": ann_id,
            "image_id": iid,
            "reason": "nonfinite_bbox",
            "bbox": ann["bbox"]
        })

        continue


    if w <= 0 or h <= 0:

        severe_invalid.append({
            "ann_id": ann_id,
            "image_id": iid,
            "reason": "nonpositive_bbox",
            "bbox": ann["bbox"]
        })

        continue


    # --------------------------------------------------------
    # Clip bounding box to valid image region
    #
    # COCO bbox:
    # [x_min, y_min, width, height]
    # --------------------------------------------------------

    x1 = x
    y1 = y
    x2 = x + w
    y2 = y + h

    clipped_x1 = max(0.0, min(x1, W))
    clipped_y1 = max(0.0, min(y1, H))

    clipped_x2 = max(
        clipped_x1,
        min(x2, W)
    )

    clipped_y2 = max(
        clipped_y1,
        min(y2, H)
    )

    clipped_w = clipped_x2 - clipped_x1
    clipped_h = clipped_y2 - clipped_y1


    # --------------------------------------------------------
    # Reject annotation if clipping destroys bbox
    # --------------------------------------------------------

    if clipped_w <= 0 or clipped_h <= 0:

        severe_invalid.append({
            "ann_id": ann_id,
            "image_id": iid,
            "reason": "bbox_outside_image_after_clipping",
            "bbox": ann["bbox"],
            "image_size": [W, H]
        })

        continue


    original_area = w * h
    clipped_area = clipped_w * clipped_h

    retained_fraction = (
        clipped_area / original_area
        if original_area > 0
        else 0.0
    )


    # --------------------------------------------------------
    # If more than 5% of bbox would disappear, do NOT
    # silently repair it.
    # --------------------------------------------------------

    if retained_fraction < 0.95:

        severe_invalid.append({
            "ann_id": ann_id,
            "image_id": iid,
            "reason": "large_bbox_outside_image",
            "bbox_original": [x, y, w, h],
            "bbox_clipped": [
                clipped_x1,
                clipped_y1,
                clipped_w,
                clipped_h
            ],
            "retained_fraction": retained_fraction,
            "image_size": [W, H]
        })

        continue


    # --------------------------------------------------------
    # Detect whether annotation was actually modified
    # --------------------------------------------------------

    changed = not np.allclose(
        [x, y, w, h],
        [
            clipped_x1,
            clipped_y1,
            clipped_w,
            clipped_h
        ],
        atol=1e-8
    )


    if changed:

        repair_log.append({
            "ann_id": ann_id,
            "image_id": iid,
            "file_name": im["file_name"],

            "original_bbox": [
                x,
                y,
                w,
                h
            ],

            "repaired_bbox": [
                clipped_x1,
                clipped_y1,
                clipped_w,
                clipped_h
            ],

            "image_width": W,
            "image_height": H,

            "original_area": original_area,
            "repaired_area": clipped_area,

            "retained_fraction":
                retained_fraction
        })


    # --------------------------------------------------------
    # Save corrected bbox
    # --------------------------------------------------------

    ann["bbox"] = [
        float(clipped_x1),
        float(clipped_y1),
        float(clipped_w),
        float(clipped_h)
    ]

    # Detection experiment uses bbox geometry.
    # Keep COCO area consistent with the repaired bbox.
    ann["area"] = float(
        clipped_w *
        clipped_h
    )

    ann["iscrowd"] = int(
        ann.get(
            "iscrowd",
            0
        )
    )

    clean_annotations.append(
        ann
    )

    anns_by_img[iid].append(
        ann
    )


# ============================================================
# SEVERE ERROR GATE
# ============================================================

severe_path = (
    DIRS["audit"] /
    "severe_invalid_annotations.json"
)

severe_path.write_text(
    json.dumps(
        severe_invalid,
        indent=2
    ),
    encoding="utf-8"
)


if severe_invalid:

    print(
        "\nSEVERE INVALID ANNOTATIONS:"
    )

    print(
        json.dumps(
            severe_invalid,
            indent=2
        )
    )

    raise RuntimeError(
        f"{len(severe_invalid)} annotations cannot be "
        "safely repaired. Inspect "
        "01_audit/severe_invalid_annotations.json."
    )


# ============================================================
# SAVE REPAIR LOG
# ============================================================

repair_df = pd.DataFrame(
    repair_log
)

repair_df.to_csv(
    DIRS["audit"] /
    "bbox_repair_log.csv",
    index=False
)


# ============================================================
# CREATE CLEANED COCO ANNOTATION FILE
# ============================================================

coco["annotations"] = clean_annotations

CLEAN_JSON = (
    DIRS["metadata"] /
    "TACO_OFFICIAL_CLEANED_annotations.json"
)

CLEAN_JSON.write_text(
    json.dumps(
        coco,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# IMPORTANT:
# All following stages use cleaned annotations.
OFFICIAL_JSON = CLEAN_JSON


# ============================================================
# SHA-256 + pHash IMAGE AUDIT
# ============================================================

def file_sha256(
    path,
    chunk=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for b in iter(
            lambda:
            f.read(chunk),
            b""
        ):

            h.update(b)

    return h.hexdigest()


rows = []

sha_groups = defaultdict(list)

phashes = {}


for iid, im in tqdm(
    images.items(),
    desc="Hashing/auditing images"
):

    p = (
        CANON_IMAGES /
        im["file_name"]
    )


    if not p.exists():

        raise FileNotFoundError(
            f"Canonical image missing: {p}"
        )


    # --------------------------------------------------------
    # EXIF-aware perceptual hash
    # --------------------------------------------------------

    with Image.open(p) as pil:

        visual = ImageOps.exif_transpose(
            pil
        ).convert("RGB")

        ph = imagehash.phash(
            visual
        )


    # --------------------------------------------------------
    # Verify the geometry actually seen by OpenCV/model
    # --------------------------------------------------------

    cv_im = cv2.imread(
        str(p),
        cv2.IMREAD_COLOR
    )

    if cv_im is None:

        raise RuntimeError(
            f"OpenCV could not read: {p}"
        )


    cv_size = (
        int(cv_im.shape[1]),
        int(cv_im.shape[0])
    )

    expected_size = (
        int(im["width"]),
        int(im["height"])
    )


    if cv_size != expected_size:

        raise RuntimeError(
            f"Geometry mismatch: image_id={iid}, "
            f"OpenCV={cv_size}, "
            f"COCO={expected_size}"
        )


    sha = file_sha256(
        p
    )


    sha_groups[sha].append(
        iid
    )

    phashes[iid] = ph


    anns = anns_by_img.get(
        iid,
        []
    )


    areas = [

        float(a["bbox"][2]) *
        float(a["bbox"][3])

        for a in anns
    ]


    rows.append({

        "image_id":
            iid,

        "file_name":
            im["file_name"],

        "batch":
            Path(
                im["file_name"]
            ).parts[0],

        "width":
            int(im["width"]),

        "height":
            int(im["height"]),

        "n_objects":
            len(anns),

        "sha256":
            sha,

        "phash":
            str(ph),

        "small_objects":
            sum(
                a < 32 ** 2
                for a in areas
            ),

        "medium_objects":
            sum(
                32 ** 2 <= a < 96 ** 2
                for a in areas
            ),

        "large_objects":
            sum(
                a >= 96 ** 2
                for a in areas
            )
    })


img_df = pd.DataFrame(
    rows
)


img_df.to_csv(
    DIRS["audit"] /
    "image_audit.csv",
    index=False
)


# ============================================================
# EXACT DUPLICATES
# ============================================================

exact_dups = {

    h: ids

    for h, ids
    in sha_groups.items()

    if len(ids) > 1
}


(
    DIRS["audit"] /
    "exact_duplicate_groups.json"
).write_text(

    json.dumps(
        exact_dups,
        indent=2
    ),

    encoding="utf-8"
)


# ============================================================
# pHash NEAR-DUPLICATES
# ============================================================

items = list(
    phashes.items()
)

near_pairs = []


for i in tqdm(
    range(len(items)),
    desc="pHash near-duplicate screening"
):

    iid1, h1 = items[i]

    for j in range(
        i + 1,
        len(items)
    ):

        iid2, h2 = items[j]

        distance = int(
            h1 - h2
        )

        if distance <= 4:

            near_pairs.append({

                "image_id_1":
                    iid1,

                "image_id_2":
                    iid2,

                "phash_distance":
                    distance
            })


pd.DataFrame(
    near_pairs,
    columns=[
        "image_id_1",
        "image_id_2",
        "phash_distance"
    ]
).to_csv(

    DIRS["audit"] /
    "near_duplicate_candidates_phash_le4.csv",

    index=False
)


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

cat_counts = Counter(

    categories[
        int(a["category_id"])
    ]

    for a in clean_annotations
)


pd.DataFrame(

    [
        {
            "category": k,
            "annotations": v
        }

        for k, v
        in cat_counts.items()
    ]

).sort_values(

    "annotations",
    ascending=False

).to_csv(

    DIRS["audit"] /
    "official_category_distribution.csv",

    index=False
)


# ============================================================
# FINAL AUDIT SUMMARY
# ============================================================

audit_summary = {

    "images":
        len(images),

    "annotations_original":
        len(coco_original["annotations"]),

    "annotations_cleaned":
        len(clean_annotations),

    "categories":
        len(categories),

    "batches":
        int(
            img_df["batch"].nunique()
        ),

    "images_without_annotations":
        int(
            (
                img_df["n_objects"]
                == 0
            ).sum()
        ),

    "bbox_repairs":
        len(repair_log),

    "severe_invalid_annotations":
        len(severe_invalid),

    "exact_duplicate_groups":
        len(exact_dups),

    "near_duplicate_candidate_pairs_phash_le4":
        len(near_pairs),

    "small_objects_bbox_area":
        int(
            img_df[
                "small_objects"
            ].sum()
        ),

    "medium_objects_bbox_area":
        int(
            img_df[
                "medium_objects"
            ].sum()
        ),

    "large_objects_bbox_area":
        int(
            img_df[
                "large_objects"
            ].sum()
        )
}


(
    DIRS["audit"] /
    "dataset_audit_summary.json"
).write_text(

    json.dumps(
        audit_summary,
        indent=2
    ),

    encoding="utf-8"
)


print(
    json.dumps(
        audit_summary,
        indent=2
    )
)


# ============================================================
# EXACT DUPLICATE GATE
# ============================================================

if exact_dups:

    raise RuntimeError(
        "Exact duplicate images detected. "
        "Inspect exact_duplicate_groups.json."
    )


print(
    "\nCORE AUDIT PASSED."
)

print(
    f"Bounding boxes repaired safely: "
    f"{len(repair_log)}"
)

print(
    "Clean annotation file:"
)

print(
    OFFICIAL_JSON
)

In [ ]:
# STAGE 3B — CREATE CONTACT SHEETS FOR pHash NEAR-DUPLICATE CANDIDATES
# This is a visual review aid only. It does not delete data.
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw

pairs_path = DIRS["audit"] / "near_duplicate_candidates_phash_le4.csv"
pairs = pd.read_csv(pairs_path)
out = DIRS["audit"] / "near_duplicate_contact_sheets"
out.mkdir(parents=True, exist_ok=True)

if len(pairs) == 0:
    print("No pHash<=4 near-duplicate candidates.")
else:
    for idx, row in pairs.head(250).iterrows():
        panels = []
        for col in ["image_id_1", "image_id_2"]:
            iid = int(row[col])
            rec = images[iid]
            im = Image.open(CANON_IMAGES / rec["file_name"]).convert("RGB")
            im.thumbnail((500, 340))
            panel = Image.new("RGB", (520, 390), "white")
            panel.paste(im, ((520-im.width)//2, 8))
            d = ImageDraw.Draw(panel)
            d.text((8, 355), f"id={iid} | {rec['file_name']}", fill="black")
            panels.append(panel)
        sheet = Image.new("RGB", (1040, 390), "white")
        sheet.paste(panels[0], (0,0))
        sheet.paste(panels[1], (520,0))
        sheet.save(out / f"pair_{idx:04d}_d{int(row['phash_distance'])}.jpg", quality=90)
    print(f"Saved {min(len(pairs),250)} contact sheets to:\n{out}")

## STAGE 4 — Freeze two split protocols and build local TACO-1 YOLO dataset

### Protocol A: `random`
Deterministic image-level split (70/15/15), stratified approximately by object count.

### Protocol B: `batch_disjoint`
Entire TACO `batch_*` groups are assigned to only one split. A deterministic search balances image counts and small/medium/large object counts.

The split JSON files are saved to Drive. On later Colab sessions, the same frozen splits are **reused**, never regenerated.

In [ ]:
# STAGE 4A — CREATE OR REUSE FROZEN SPLITS
import json, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

SPLIT_SEED = 20260814
RANDOM_SPLIT_PATH = DIRS["splits"] / "random_split.json"
BATCH_SPLIT_PATH = DIRS["splits"] / "batch_disjoint_split.json"

img_df = pd.read_csv(DIRS["audit"] / "image_audit.csv")
all_ids = set(img_df["image_id"].astype(int))

def validate_split(split_dict, name):
    sets = {k:set(map(int,v)) for k,v in split_dict.items()}
    assert set(sets) == {"train","val","test"}
    assert sets["train"].isdisjoint(sets["val"])
    assert sets["train"].isdisjoint(sets["test"])
    assert sets["val"].isdisjoint(sets["test"])
    assert sets["train"] | sets["val"] | sets["test"] == all_ids

    rows = []
    for s, ids_ in sets.items():
        d = img_df[img_df.image_id.isin(ids_)]
        rows.append({
            "protocol": name,
            "split": s,
            "images": len(d),
            "objects": int(d.n_objects.sum()),
            "small": int(d.small_objects.sum()),
            "medium": int(d.medium_objects.sum()),
            "large": int(d.large_objects.sum()),
            "batches": int(d.batch.nunique()),
        })
    return pd.DataFrame(rows)

if RANDOM_SPLIT_PATH.exists() and BATCH_SPLIT_PATH.exists():
    print("Frozen split files already exist; reusing them.")
    random_split = json.loads(RANDOM_SPLIT_PATH.read_text())
    batch_split = json.loads(BATCH_SPLIT_PATH.read_text())
else:
    # ---------- Random protocol ----------
    tmp = img_df.copy()
    strata = tmp["n_objects"].clip(upper=8).astype(str)
    freq = strata.value_counts()
    strata = strata.where(strata.map(freq) >= 4, "rare")

    train_ids, temp_ids = train_test_split(
        tmp["image_id"].astype(int).to_numpy(),
        test_size=0.30,
        random_state=SPLIT_SEED,
        stratify=strata
    )
    temp = tmp[tmp.image_id.isin(temp_ids)].copy()
    tstr = temp["n_objects"].clip(upper=8).astype(str)
    freq2 = tstr.value_counts()
    tstr = tstr.where(tstr.map(freq2) >= 2, "rare")
    try:
        val_ids, test_ids = train_test_split(
            temp["image_id"].astype(int).to_numpy(),
            test_size=0.50,
            random_state=SPLIT_SEED,
            stratify=tstr
        )
    except ValueError:
        val_ids, test_ids = train_test_split(
            temp["image_id"].astype(int).to_numpy(),
            test_size=0.50,
            random_state=SPLIT_SEED
        )

    random_split = {
        "train": sorted(map(int, train_ids)),
        "val": sorted(map(int, val_ids)),
        "test": sorted(map(int, test_ids)),
    }

    # ---------- Batch-disjoint protocol ----------
    gs = img_df.groupby("batch").agg(
        images=("image_id","size"),
        objects=("n_objects","sum"),
        small=("small_objects","sum"),
        medium=("medium_objects","sum"),
        large=("large_objects","sum"),
    ).reset_index()

    target_ratio = {"train":0.70, "val":0.15, "test":0.15}
    totals = gs[["images","objects","small","medium","large"]].sum().to_dict()
    target = {
        s:{k:target_ratio[s]*totals[k] for k in totals}
        for s in target_ratio
    }

    rng = np.random.default_rng(SPLIT_SEED)
    groups = gs["batch"].tolist()

    def assignment_score(assign):
        score = 0.0
        for s in ["train","val","test"]:
            sub = gs[gs["batch"].map(assign) == s]
            if len(sub) == 0:
                return float("inf")
            for k in ["images","objects","small","medium","large"]:
                got = float(sub[k].sum())
                tgt = max(float(target[s][k]), 1.0)
                weight = 3.0 if k == "images" else 1.0
                score += weight * ((got - tgt)/tgt)**2
        return score

    best_assign, best_score = None, float("inf")
    for _ in range(100000):
        assign = {}
        for g in groups:
            r = rng.random()
            assign[g] = "train" if r < 0.70 else ("val" if r < 0.85 else "test")
        sc = assignment_score(assign)
        if sc < best_score:
            best_score, best_assign = sc, assign.copy()

    assert best_assign is not None
    batch_split = {}
    for s in ["train","val","test"]:
        batch_split[s] = sorted(
            img_df.loc[img_df["batch"].map(best_assign) == s, "image_id"]
            .astype(int).tolist()
        )

    RANDOM_SPLIT_PATH.write_text(json.dumps(random_split, indent=2), encoding="utf-8")
    BATCH_SPLIT_PATH.write_text(json.dumps(batch_split, indent=2), encoding="utf-8")

    pd.DataFrame(
        [{"batch":b, "split":best_assign[b]} for b in sorted(best_assign)]
    ).to_csv(DIRS["splits"] / "batch_disjoint_group_assignment.csv", index=False)

# Validate both every time.
summary = pd.concat([
    validate_split(random_split, "random"),
    validate_split(batch_split, "batch_disjoint")
], ignore_index=True)

# Explicit batch leakage test for primary protocol.
for a,b in [("train","val"),("train","test"),("val","test")]:
    ga = set(img_df[img_df.image_id.isin(batch_split[a])]["batch"])
    gb = set(img_df[img_df.image_id.isin(batch_split[b])]["batch"])
    assert ga.isdisjoint(gb), f"Batch leakage {a}-{b}: {ga & gb}"

summary.to_csv(DIRS["splits"] / "split_summary.csv", index=False)
display(summary)

In [ ]:
# STAGE 4B — BUILD TACO-1 YOLO LABELS + COCO GT SUBSETS
import json, yaml, hashlib
from pathlib import Path

LABELS = LOCAL_DATASET / "labels"
LABELS.mkdir(parents=True, exist_ok=True)

coco = json.loads(OFFICIAL_JSON.read_text(encoding="utf-8"))
im_map = {int(im["id"]): im for im in coco["images"]}
anns_by_img = {}
for a in coco["annotations"]:
    anns_by_img.setdefault(int(a["image_id"]), []).append(a)

# All 60 official categories -> class 0 = litter.
for iid, im in im_map.items():
    rel = Path(im["file_name"])
    label_path = LABELS / rel.with_suffix(".txt")
    label_path.parent.mkdir(parents=True, exist_ok=True)

    W, H = float(im["width"]), float(im["height"])
    lines = []
    for a in anns_by_img.get(iid, []):
        x, y, w, h = map(float, a["bbox"])
        xc, yc, wn, hn = (x+w/2)/W, (y+h/2)/H, w/W, h/H
        vals = [xc,yc,wn,hn]
        if not all(0 <= v <= 1 for v in vals):
            raise ValueError(f"Invalid normalized bbox: image={iid}, ann={a.get('id')}, {vals}")
        lines.append(f"0 {xc:.10f} {yc:.10f} {wn:.10f} {hn:.10f}")
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

def make_coco_subset(ids, out_json):
    ids = set(map(int, ids))
    anns = []
    ann_id = 1
    for a in coco["annotations"]:
        if int(a["image_id"]) in ids:
            b = dict(a)
            b["id"] = ann_id
            b["category_id"] = 1
            b["iscrowd"] = int(b.get("iscrowd", 0))
            if "area" not in b:
                b["area"] = float(b["bbox"][2]) * float(b["bbox"][3])
            anns.append(b)
            ann_id += 1

    subset = {
        "info": coco.get("info", {}),
        "licenses": coco.get("licenses", []),
        "images": [im for im in coco["images"] if int(im["id"]) in ids],
        "annotations": anns,
        "categories": [{"id":1, "name":"litter", "supercategory":"litter"}],
    }
    Path(out_json).write_text(json.dumps(subset), encoding="utf-8")

protocol_splits = {
    "random": json.loads(RANDOM_SPLIT_PATH.read_text()),
    "batch_disjoint": json.loads(BATCH_SPLIT_PATH.read_text()),
}

for protocol, split in protocol_splits.items():
    pdir = LOCAL_DATASET / protocol
    pdir.mkdir(parents=True, exist_ok=True)

    for s in ["train","val","test"]:
        txt = pdir / f"{s}.txt"
        image_paths = [str(CANON_IMAGES / im_map[int(i)]["file_name"]) for i in split[s]]
        txt.write_text("\n".join(image_paths) + "\n", encoding="utf-8")
        make_coco_subset(split[s], pdir / f"{s}_coco_taco1.json")

    data_yaml = {
        "path": str(LOCAL_DATASET),
        "train": str(pdir / "train.txt"),
        "val": str(pdir / "val.txt"),
        "test": str(pdir / "test.txt"),
        "names": {0:"litter"},
    }
    (pdir / "data.yaml").write_text(
        yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8"
    )

print("Local YOLO dataset built:", LOCAL_DATASET)

In [ ]:
# STAGE 4C — FREEZE SPLIT HASHES / DATASET MANIFEST
import hashlib, json

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

freeze = {
    "split_seed": 20260814,
    "task": "TACO-1 one-class litter detection",
    "protocols": {},
}
for protocol in ["random","batch_disjoint"]:
    pdir = LOCAL_DATASET / protocol
    freeze["protocols"][protocol] = {
        "split_json_sha256": sha256_file(
            RANDOM_SPLIT_PATH if protocol=="random" else BATCH_SPLIT_PATH
        ),
        "train_txt_sha256": sha256_file(pdir/"train.txt"),
        "val_txt_sha256": sha256_file(pdir/"val.txt"),
        "test_txt_sha256": sha256_file(pdir/"test.txt"),
        "train_gt_sha256": sha256_file(pdir/"train_coco_taco1.json"),
        "val_gt_sha256": sha256_file(pdir/"val_coco_taco1.json"),
        "test_gt_sha256": sha256_file(pdir/"test_coco_taco1.json"),
    }

frozen_path = DIRS["manifests"] / "splits_frozen.json"
frozen_path.write_text(json.dumps(freeze, indent=2), encoding="utf-8")
print(json.dumps(freeze, indent=2))

## STAGE 5 — Model preflight and frozen training configuration

The same image size, epoch budget, augmentation policy and batch size are used for all models.  
`batch=4` is intentionally conservative so RT-DETR-L can run on common Colab GPUs such as T4/L4 as well as A100.

Only `best.pt` and `last.pt` are retained; periodic checkpoints are disabled to avoid excessive Drive storage.

In [ ]:
# STAGE 5A — MODEL WEIGHT PREFLIGHT
from ultralytics import YOLO, RTDETR
import gc, torch

MODEL_SPECS = {
    "yolov8s": ("YOLO", "yolov8s.pt"),
    "yolo11s": ("YOLO", "yolo11s.pt"),
    "yolo26s": ("YOLO", "yolo26s.pt"),
    "rtdetr_l": ("RTDETR", "rtdetr-l.pt"),
}

for key, (family, weights) in MODEL_SPECS.items():
    print(f"\nChecking {key}: {weights}")
    model = RTDETR(weights) if family == "RTDETR" else YOLO(weights)
    model.info()
    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\nALL MODEL WEIGHTS LOADED SUCCESSFULLY.")

In [ ]:
# STAGE 5B — FREEZE TRAINING CONFIGURATION
# H100 CONFIGURATION

from pathlib import Path
import json

# ============================================================
# PATHS
# ============================================================

BASE = Path("/content/drive/MyDrive/Atik14082026")
ROOT = BASE / "TACO_Pipeline_v5"

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)


# ============================================================
# MODEL DEFINITIONS
# ============================================================

MODEL_SPECS = {
    "yolov8s": [
        "YOLO",
        "yolov8s.pt"
    ],

    "yolo11s": [
        "YOLO",
        "yolo11s.pt"
    ],

    "yolo26s": [
        "YOLO",
        "yolo26s.pt"
    ],

    "rtdetr_l": [
        "RTDETR",
        "rtdetr-l.pt"
    ],
}


# ============================================================
# FROZEN EXPERIMENTAL CONFIGURATION
# ============================================================

CONFIG = {

    # --------------------------------------------------------
    # Input
    # --------------------------------------------------------

    "imgsz": 640,

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    "epochs": 100,
    "patience": 20,

    # H100:
    # 16 is conservative and suitable for all four models.
    "batch": 16,

    "workers": 8,

    "amp": True,
    "deterministic": True,

    "cache": False,
    "pretrained": True,

    "optimizer": "auto",

    # --------------------------------------------------------
    # Augmentation
    # --------------------------------------------------------

    "close_mosaic": 10,

    "mosaic": 0.5,
    "mixup": 0.0,
    "copy_paste": 0.0,

    "degrees": 5.0,
    "translate": 0.05,
    "scale": 0.15,

    "fliplr": 0.5,
    "flipud": 0.0,

    "hsv_h": 0.015,
    "hsv_s": 0.4,
    "hsv_v": 0.3,

    # --------------------------------------------------------
    # Repeated experiments
    # --------------------------------------------------------

    "seeds": [
        42,
        123,
        2026
    ],

    # --------------------------------------------------------
    # Evaluation protocols
    # --------------------------------------------------------

    "protocols": [
        "random",
        "batch_disjoint"
    ],

    # --------------------------------------------------------
    # Models
    # --------------------------------------------------------

    "models": MODEL_SPECS,

    # --------------------------------------------------------
    # Primary outcome
    # --------------------------------------------------------

    "primary_metric":
        "COCO mAP@[0.50:0.95]",

    # --------------------------------------------------------
    # Threshold policy
    # --------------------------------------------------------

    "threshold_policy":
        (
            "Confidence threshold selected only on validation "
            "by maximum F1 at IoU=0.50; locked test threshold "
            "is never optimized on test data."
        ),
}


# ============================================================
# SAVE / FREEZE CONFIGURATION
# ============================================================

CONFIG_PATH = (
    DIRS["manifests"] /
    "training_config_frozen.json"
)


if CONFIG_PATH.exists():

    existing = json.loads(
        CONFIG_PATH.read_text(
            encoding="utf-8"
        )
    )

    if existing != CONFIG:

        print(
            "A training configuration already exists:"
        )

        print(
            CONFIG_PATH
        )

        print(
            "\nExisting configuration:"
        )

        print(
            json.dumps(
                existing,
                indent=2
            )
        )

        raise RuntimeError(
            "Existing frozen configuration differs from "
            "the current configuration. Do not overwrite "
            "a frozen experimental design."
        )

    else:

        print(
            "Existing frozen training configuration "
            "matches the current design."
        )

else:

    CONFIG_PATH.write_text(
        json.dumps(
            CONFIG,
            indent=2
        ),
        encoding="utf-8"
    )

    print(
        "Training configuration frozen."
    )


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 65)
print("FROZEN TRAINING CONFIGURATION")
print("=" * 65)

print(
    json.dumps(
        CONFIG,
        indent=2
    )
)

print("=" * 65)

print(
    "\nSaved to:"
)

print(
    CONFIG_PATH
)


# ============================================================
# FINAL CHECK
# ============================================================

assert CONFIG_PATH.exists()

print(
    "\nSTAGE 5B PASSED."
)

print(
    "Experimental design:"
)

print(
    f"{len(CONFIG['protocols'])} protocols × "
    f"{len(CONFIG['models'])} models × "
    f"{len(CONFIG['seeds'])} seeds = "
    f"{len(CONFIG['protocols']) * len(CONFIG['models']) * len(CONFIG['seeds'])} training runs"
)

## STAGE 6 — Training + validation only

Total full design:

`2 protocols × 4 models × 3 seeds = 24 training runs`

The training loop is resumable:
- completed run with `best.pt` + `results.csv` → skipped;
- interrupted run with `last.pt` → resumed;
- missing run → started from pretrained weights.

Do **not** use test data in this stage.

In [ ]:
# STAGE 6A — TRAIN / RESUME ALL 24 RUNS
import json, gc, torch
from pathlib import Path
from ultralytics import YOLO, RTDETR

CONFIG = json.loads((DIRS["manifests"]/"training_config_frozen.json").read_text())

def build_model(model_key, checkpoint=None):
    family, default_weights = CONFIG["models"][model_key]
    src = str(checkpoint) if checkpoint is not None else default_weights
    return RTDETR(src) if family == "RTDETR" else YOLO(src)

for protocol in CONFIG["protocols"]:
    data_yaml = LOCAL_DATASET / protocol / "data.yaml"

    for model_key in CONFIG["models"]:
        for seed in CONFIG["seeds"]:
            run_name = f"{model_key}_seed{seed}"
            run_dir = DIRS["runs"] / protocol / run_name
            best_pt = run_dir / "weights" / "best.pt"
            last_pt = run_dir / "weights" / "last.pt"

            if best_pt.exists() and (run_dir/"results.csv").exists():
                print(f"[SKIP COMPLETE] {protocol}/{run_name}")
                continue

            if last_pt.exists():
                print(f"[RESUME] {protocol}/{run_name}")
                model = build_model(model_key, last_pt)
                model.train(resume=True)
            else:
                print(f"[START] {protocol}/{run_name}")
                model = build_model(model_key)
                model.train(
                    data=str(data_yaml),
                    imgsz=CONFIG["imgsz"],
                    epochs=CONFIG["epochs"],
                    patience=CONFIG["patience"],
                    batch=CONFIG["batch"],
                    workers=CONFIG["workers"],
                    amp=CONFIG["amp"],
                    deterministic=CONFIG["deterministic"],
                    seed=seed,
                    cache=CONFIG["cache"],
                    pretrained=CONFIG["pretrained"],
                    optimizer=CONFIG["optimizer"],
                    close_mosaic=CONFIG["close_mosaic"],
                    mosaic=CONFIG["mosaic"],
                    mixup=CONFIG["mixup"],
                    copy_paste=CONFIG["copy_paste"],
                    degrees=CONFIG["degrees"],
                    translate=CONFIG["translate"],
                    scale=CONFIG["scale"],
                    fliplr=CONFIG["fliplr"],
                    flipud=CONFIG["flipud"],
                    hsv_h=CONFIG["hsv_h"],
                    hsv_s=CONFIG["hsv_s"],
                    hsv_v=CONFIG["hsv_v"],
                    device=0,
                    project=str(DIRS["runs"]/protocol),
                    name=run_name,
                    exist_ok=True,
                    save=True,
                    save_period=-1,
                    plots=True,
                    verbose=True,
                )

            del model
            gc.collect()
            torch.cuda.empty_cache()

print("TRAINING LOOP FINISHED.")

In [ ]:
# ============================================================
# STAGE 6B — VERIFY ALL TRAINING RUNS
# FIXED / SELF-CONTAINED VERSION
# ============================================================

from pathlib import Path
import pandas as pd
import json
import os


# ============================================================
# 1. FIXED PROJECT PATH
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

RUNS_DIR = ROOT / "03_runs"

CONFIG_PATH = (
    ROOT /
    "07_manifests" /
    "training_config_frozen.json"
)

VALIDATION_DIR = (
    ROOT /
    "04_validation"
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. BASIC PATH CHECK
# ============================================================

print("=" * 70)
print("PROJECT PATH CHECK")
print("=" * 70)

print("ROOT:")
print(ROOT)

print("\nROOT exists:")
print(ROOT.exists())

print("\nRUNS directory:")
print(RUNS_DIR)

print("\nRUNS exists:")
print(RUNS_DIR.exists())

print("\nCONFIG:")
print(CONFIG_PATH)

print("\nCONFIG exists:")
print(CONFIG_PATH.exists())

print("=" * 70)


if not ROOT.exists():
    raise FileNotFoundError(
        f"Project root does not exist:\n{ROOT}"
    )

if not RUNS_DIR.exists():
    raise FileNotFoundError(
        f"Training runs directory does not exist:\n{RUNS_DIR}"
    )

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Frozen config does not exist:\n{CONFIG_PATH}"
    )


# ============================================================
# 3. SHOW ACTUAL CONTENT OF 03_runs
# ============================================================

print("\nActual folders found under 03_runs:\n")

for p in sorted(RUNS_DIR.rglob("best.pt")):
    print(p)

found_best_files = list(
    RUNS_DIR.rglob("best.pt")
)

print(
    f"\nTotal best.pt files physically found: "
    f"{len(found_best_files)}"
)


# ============================================================
# 4. LOAD FROZEN CONFIG
# ============================================================

CONFIG = json.loads(
    CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

print("\nProtocols:")
print(CONFIG["protocols"])

print("\nModels:")
print(list(CONFIG["models"].keys()))

print("\nSeeds:")
print(CONFIG["seeds"])


# ============================================================
# 5. VERIFY EXPECTED 24 RUNS
# ============================================================

rows = []

for protocol in CONFIG["protocols"]:

    for model in CONFIG["models"].keys():

        for seed in CONFIG["seeds"]:

            run_dir = (
                RUNS_DIR /
                protocol /
                f"{model}_seed{seed}"
            )

            weights_dir = (
                run_dir /
                "weights"
            )

            best_pt = (
                weights_dir /
                "best.pt"
            )

            last_pt = (
                weights_dir /
                "last.pt"
            )

            results_csv = (
                run_dir /
                "results.csv"
            )

            rows.append({

                "protocol":
                    protocol,

                "model":
                    model,

                "seed":
                    int(seed),

                "run_dir_exists":
                    run_dir.exists(),

                "weights_dir_exists":
                    weights_dir.exists(),

                "best_pt":
                    best_pt.exists(),

                "last_pt":
                    last_pt.exists(),

                "results_csv":
                    results_csv.exists(),

                "best_pt_path":
                    str(best_pt)
            })


status = pd.DataFrame(
    rows
)


# ============================================================
# 6. DISPLAY
# ============================================================

display(
    status[
        [
            "protocol",
            "model",
            "seed",
            "run_dir_exists",
            "best_pt",
            "last_pt",
            "results_csv"
        ]
    ]
)


# ============================================================
# 7. SAVE
# ============================================================

STATUS_FILE = (
    VALIDATION_DIR /
    "training_completion_status.csv"
)

status.to_csv(
    STATUS_FILE,
    index=False
)


# ============================================================
# 8. SUMMARY
# ============================================================

expected = len(status)

run_dirs = int(
    status["run_dir_exists"].sum()
)

best_count = int(
    status["best_pt"].sum()
)

last_count = int(
    status["last_pt"].sum()
)

results_count = int(
    status["results_csv"].sum()
)


print("\n" + "=" * 70)
print("TRAINING COMPLETION SUMMARY")
print("=" * 70)

print(
    f"Expected runs       : {expected}"
)

print(
    f"Run folders found   : {run_dirs}/{expected}"
)

print(
    f"best.pt found       : {best_count}/{expected}"
)

print(
    f"last.pt found       : {last_count}/{expected}"
)

print(
    f"results.csv found   : {results_count}/{expected}"
)

print("=" * 70)


# ============================================================
# 9. SHOW MISSING RUNS
# ============================================================

missing = status[
    ~status["best_pt"]
].copy()


if len(missing) > 0:

    print(
        "\nMISSING best.pt RUNS:"
    )

    display(
        missing[
            [
                "protocol",
                "model",
                "seed",
                "run_dir_exists",
                "weights_dir_exists",
                "best_pt",
                "last_pt",
                "results_csv",
                "best_pt_path"
            ]
        ]
    )

    print(
        "\nIMPORTANT:"
    )

    print(
        "Do NOT restart training yet."
    )

    print(
        "First compare the paths above with the "
        "physically discovered best.pt files printed "
        "at the beginning of this cell."
    )

else:

    print(
        "\nALL 24 TRAINING RUNS ARE PRESENT."
    )

    print(
        "\nSTAGE 6B PASSED."
    )

    print(
        "\nNext step: STAGE 6C."
    )


print(
    "\nStatus saved to:"
)

print(
    STATUS_FILE
)

In [ ]:
# ============================================================
# FIND LOST TRAINING OUTPUTS
# Search Drive + local Colab runtime
# ============================================================

from pathlib import Path
import os
import subprocess


print("=" * 80)
print("1. GOOGLE DRIVE MOUNT CHECK")
print("=" * 80)

print("\n/content/drive exists:", Path("/content/drive").exists())
print("/content/drive/MyDrive exists:", Path("/content/drive/MyDrive").exists())

print("\nMount information:")
os.system("mount | grep '/content/drive' || true")

print("\nDisk information:")
os.system("df -h /content/drive 2>/dev/null || true")


# ============================================================
# 2. CHECK EXPECTED PROJECT
# ============================================================

expected = Path(
    "/content/drive/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

print("\n" + "=" * 80)
print("2. EXPECTED PROJECT")
print("=" * 80)

print("Expected project:", expected)
print("Exists:", expected.exists())

if expected.exists():

    print("\nTop-level contents:")

    for p in sorted(expected.iterdir()):
        print(
            "DIR " if p.is_dir() else "FILE",
            p
        )


# ============================================================
# 3. SEARCH ALL best.pt FILES ON GOOGLE DRIVE
# ============================================================

print("\n" + "=" * 80)
print("3. SEARCHING GOOGLE DRIVE FOR best.pt")
print("=" * 80)

drive_root = Path("/content/drive")

drive_best = []

if drive_root.exists():

    try:
        drive_best = list(
            drive_root.rglob("best.pt")
        )
    except Exception as e:
        print("Drive search error:", e)


print(
    f"\nbest.pt files found anywhere on Drive: "
    f"{len(drive_best)}"
)

for p in drive_best:
    print(p)


# ============================================================
# 4. SEARCH GOOGLE DRIVE FOR results.csv
# ============================================================

print("\n" + "=" * 80)
print("4. SEARCHING GOOGLE DRIVE FOR TRAINING results.csv")
print("=" * 80)

drive_results = []

if drive_root.exists():

    try:

        for p in drive_root.rglob("results.csv"):

            s = str(p).lower()

            if any(
                x in s
                for x in [
                    "yolov8s",
                    "yolo11s",
                    "yolo26s",
                    "rtdetr"
                ]
            ):

                drive_results.append(p)

    except Exception as e:

        print(
            "Drive results search error:",
            e
        )


print(
    f"\nTraining results.csv files found on Drive: "
    f"{len(drive_results)}"
)

for p in drive_results:
    print(p)


# ============================================================
# 5. SEARCH LOCAL COLAB RUNTIME
# ============================================================

print("\n" + "=" * 80)
print("5. SEARCHING LOCAL /content FOR best.pt")
print("=" * 80)

local_best = []

content_root = Path("/content")

try:

    for p in content_root.rglob("best.pt"):

        # Avoid duplicating Drive results
        if not str(p).startswith(
            "/content/drive/"
        ):

            local_best.append(p)

except Exception as e:

    print(
        "Local search error:",
        e
    )


print(
    f"\nbest.pt files found in local runtime: "
    f"{len(local_best)}"
)

for p in local_best:
    print(p)


# ============================================================
# 6. FIND EVERY TACO PIPELINE DIRECTORY
# ============================================================

print("\n" + "=" * 80)
print("6. TACO PIPELINE DIRECTORIES ON DRIVE")
print("=" * 80)

pipeline_dirs = []

if Path("/content/drive/MyDrive").exists():

    try:

        for p in Path(
            "/content/drive/MyDrive"
        ).rglob("TACO_Pipeline*"):

            if p.is_dir():
                pipeline_dirs.append(p)

    except Exception as e:

        print(
            "Pipeline search error:",
            e
        )


for p in pipeline_dirs:

    n_best = 0

    try:
        n_best = len(
            list(p.rglob("best.pt"))
        )
    except:
        pass

    print(
        f"{p}   |   best.pt = {n_best}"
    )


# ============================================================
# 7. FIND Atik14082026 DIRECTORIES
# ============================================================

print("\n" + "=" * 80)
print("7. ALL Atik14082026 DIRECTORIES")
print("=" * 80)

try:

    for p in Path(
        "/content/drive"
    ).rglob("Atik14082026"):

        if p.is_dir():
            print(p)

except Exception as e:

    print(
        "Search error:",
        e
    )


# ============================================================
# 8. SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("SEARCH SUMMARY")
print("=" * 80)

print(
    "Drive best.pt :",
    len(drive_best)
)

print(
    "Local best.pt :",
    len(local_best)
)

print(
    "Drive training results.csv :",
    len(drive_results)
)

print(
    "TACO pipeline directories :",
    len(pipeline_dirs)
)

print("=" * 80)


if len(drive_best) == 0 and len(local_best) == 0:

    print(
        "\nWARNING: No training checkpoints were found "
        "in the currently mounted Drive or current runtime."
    )

    print(
        "Do NOT start training again yet. "
        "First send this complete output for diagnosis."
    )

else:

    print(
        "\nCHECKPOINTS FOUND."
    )

    print(
        "Send this output before moving/copying anything."
    )

In [ ]:
from pathlib import Path

REAL_PROJECT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

RUNS = REAL_PROJECT / "03_runs"

print("=" * 70)
print("FAST CHECK — TACO TRAINING OUTPUTS")
print("=" * 70)

print("RUNS:", RUNS)
print("Exists:", RUNS.exists())

# Sadece proje içindeki checkpointleri ara
best_files = list(RUNS.glob("*/*/weights/best.pt"))
last_files = list(RUNS.glob("*/*/weights/last.pt"))
results_files = list(RUNS.glob("*/*/results.csv"))

print("\nbest.pt   :", len(best_files))
print("last.pt   :", len(last_files))
print("results.csv:", len(results_files))

print("\nFOUND best.pt FILES:")
for p in sorted(best_files):
    print(p)

print("\n" + "=" * 70)

if len(best_files) == 24:
    print("SUCCESS: 24/24 best.pt FOUND.")
    print("DO NOT RETRAIN.")
elif len(best_files) > 0:
    print(f"PARTIAL: {len(best_files)}/24 best.pt found.")
    print("DO NOT RETRAIN YET.")
else:
    print("0/24 best.pt found inside the real project.")

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

In [ ]:
# ============================================================
# DATASET RECOVERY CHECK
# ============================================================

from pathlib import Path

DRIVE_ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026"
)

PROJECT = (
    DRIVE_ROOT /
    "TACO_Pipeline_v5"
)

print("=" * 75)
print("DATASET RECOVERY CHECK")
print("=" * 75)

# ------------------------------------------------------------
# 1. Existing local dataset
# ------------------------------------------------------------

LOCAL_DATASET = Path(
    "/content/taco_v5/dataset"
)

print("\nLocal dataset:")
print(LOCAL_DATASET)
print("Exists:", LOCAL_DATASET.exists())


# ------------------------------------------------------------
# 2. Split files saved on Drive
# ------------------------------------------------------------

SPLITS = PROJECT / "02_splits"

print("\n" + "=" * 75)
print("FILES IN 02_splits")
print("=" * 75)

if SPLITS.exists():
    for p in sorted(SPLITS.rglob("*")):
        if p.is_file():
            print(p.relative_to(PROJECT))
else:
    print("02_splits does not exist.")


# ------------------------------------------------------------
# 3. Search for previously persisted dataset copies
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SEARCHING FOR data.yaml")
print("=" * 75)

yaml_files = list(
    DRIVE_ROOT.rglob("data.yaml")
)

for p in yaml_files:
    print(p)

print("\ndata.yaml count:", len(yaml_files))


# ------------------------------------------------------------
# 4. Search for COCO validation files
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SEARCHING FOR val_coco_taco1.json")
print("=" * 75)

val_jsons = list(
    DRIVE_ROOT.rglob("val_coco_taco1.json")
)

for p in val_jsons:
    print(p)

print(
    "\nval_coco_taco1.json count:",
    len(val_jsons)
)


# ------------------------------------------------------------
# 5. Search for original TACO ZIP
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SEARCHING FOR TACO ZIP")
print("=" * 75)

zip_files = [
    p
    for p in DRIVE_ROOT.rglob("*.zip")
    if "taco" in p.name.lower()
]

for p in zip_files:
    print(
        p,
        f"({p.stat().st_size / 1024**3:.2f} GB)"
    )

print("\nTACO ZIP count:", len(zip_files))


# ------------------------------------------------------------
# 6. Search metadata needed for reconstruction
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("METADATA FILES")
print("=" * 75)

META = PROJECT / "08_metadata"

if META.exists():
    for p in sorted(META.iterdir()):
        if p.is_file():
            print(p.name)


print("\n" + "=" * 75)
print("CHECK FINISHED")
print("=" * 75)

In [ ]:
# ============================================================
# RECOVERY — REBUILD EXACT LOCAL TACO DATASET AFTER RUNTIME RESET
# NO NEW SPLITS
# NO RETRAINING
# USES THE REAL GOOGLE DRIVE
# ============================================================

from pathlib import Path
from collections import defaultdict
from PIL import Image, ImageOps
from tqdm.auto import tqdm

import os
import cv2
import json
import yaml
import shutil
import hashlib
import subprocess
import pandas as pd

from ultralytics.data.utils import exif_size


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

LOCAL = Path("/content/taco_v5")
LOCAL_ZIP = Path("/content/TACO.zip")
LOCAL_EXTRACT = LOCAL / "zenodo"
LOCAL_DATASET = LOCAL / "dataset"

CANON_IMAGES = LOCAL_DATASET / "images"
LABELS = LOCAL_DATASET / "labels"

SELECTED_JSON = (
    DIRS["metadata"] /
    "TACO_SELECTED_OFFICIAL_annotations.json"
)

CLEAN_JSON = (
    DIRS["metadata"] /
    "TACO_OFFICIAL_CLEANED_annotations.json"
)

RANDOM_SPLIT_PATH = (
    DIRS["splits"] /
    "random_split.json"
)

BATCH_SPLIT_PATH = (
    DIRS["splits"] /
    "batch_disjoint_split.json"
)

FROZEN_MANIFEST = (
    DIRS["manifests"] /
    "splits_frozen.json"
)

IMAGE_AUDIT = (
    DIRS["audit"] /
    "image_audit.csv"
)

for p in [LOCAL, *DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. REQUIRED PERSISTENT FILES
# ============================================================

required = [
    SELECTED_JSON,
    CLEAN_JSON,
    RANDOM_SPLIT_PATH,
    BATCH_SPLIT_PATH,
]

for p in required:
    if not p.exists():
        raise FileNotFoundError(
            f"Required persistent file is missing:\n{p}"
        )

print("Persistent scientific files found.")


# ============================================================
# 3. LOAD AND VERIFY ANNOTATIONS
# ============================================================

selected_coco = json.loads(
    SELECTED_JSON.read_text(
        encoding="utf-8"
    )
)

clean_coco = json.loads(
    CLEAN_JSON.read_text(
        encoding="utf-8"
    )
)

print("\nSelected official annotation:")
print(
    len(selected_coco["images"]),
    "images |",
    len(selected_coco["annotations"]),
    "annotations |",
    len(selected_coco["categories"]),
    "categories"
)

print("\nCleaned annotation:")
print(
    len(clean_coco["images"]),
    "images |",
    len(clean_coco["annotations"]),
    "annotations |",
    len(clean_coco["categories"]),
    "categories"
)

assert len(selected_coco["images"]) == 1500
assert len(selected_coco["annotations"]) == 4784
assert len(selected_coco["categories"]) == 60

assert len(clean_coco["images"]) == 1500
assert len(clean_coco["annotations"]) == 4784
assert len(clean_coco["categories"]) == 60

selected_ids = {
    int(x["id"])
    for x in selected_coco["images"]
}

clean_ids = {
    int(x["id"])
    for x in clean_coco["images"]
}

assert selected_ids == clean_ids

print("\nANNOTATION CHECK PASSED.")


# ============================================================
# 4. DOWNLOAD OFFICIAL ZENODO TACO.zip
# ============================================================

ZENODO_URL = (
    "https://zenodo.org/records/3587843/"
    "files/TACO.zip?download=1"
)

EXPECTED_MD5 = (
    "e9149407d883e21a8d224feef8210920"
)


def md5_file(path, chunk=8 * 1024 * 1024):

    h = hashlib.md5()

    with open(path, "rb") as f:

        for b in iter(
            lambda: f.read(chunk),
            b""
        ):
            h.update(b)

    return h.hexdigest()


def sha256_file(path, chunk=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for b in iter(
            lambda: f.read(chunk),
            b""
        ):
            h.update(b)

    return h.hexdigest()


if (
    not LOCAL_ZIP.exists()
    or md5_file(LOCAL_ZIP) != EXPECTED_MD5
):

    print(
        "\nDownloading/resuming official TACO.zip..."
    )

    subprocess.check_call([
        "wget",
        "-c",
        "--tries=10",
        "--timeout=60",
        "--retry-connrefused",
        "--show-progress",
        "-O",
        str(LOCAL_ZIP),
        ZENODO_URL
    ])


actual_md5 = md5_file(
    LOCAL_ZIP
)

print("\nExpected MD5:", EXPECTED_MD5)
print("Actual MD5  :", actual_md5)

if actual_md5 != EXPECTED_MD5:

    print(
        "\nMD5 mismatch. Performing clean re-download..."
    )

    LOCAL_ZIP.unlink(
        missing_ok=True
    )

    subprocess.check_call([
        "wget",
        "--tries=10",
        "--timeout=60",
        "--retry-connrefused",
        "--show-progress",
        "-O",
        str(LOCAL_ZIP),
        ZENODO_URL
    ])

    actual_md5 = md5_file(
        LOCAL_ZIP
    )


if actual_md5 != EXPECTED_MD5:

    raise RuntimeError(
        "TACO.zip MD5 mismatch. "
        "Do not continue."
    )

print("\nZENODO ARCHIVE MD5 PASSED.")


# ============================================================
# 5. EXTRACT
# ============================================================

EXTRACT_MARKER = (
    LOCAL_EXTRACT /
    ".extract_complete"
)

if not EXTRACT_MARKER.exists():

    if LOCAL_EXTRACT.exists():
        shutil.rmtree(
            LOCAL_EXTRACT
        )

    LOCAL_EXTRACT.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        "\nExtracting official TACO.zip..."
    )

    subprocess.check_call([
        "unzip",
        "-q",
        str(LOCAL_ZIP),
        "-d",
        str(LOCAL_EXTRACT)
    ])

    EXTRACT_MARKER.write_text(
        "ok\n",
        encoding="utf-8"
    )

else:

    print(
        "\nArchive already extracted."
    )


# ============================================================
# 6. INDEX ARCHIVE IMAGES
# ============================================================

EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

archive_images = [
    p
    for p in LOCAL_EXTRACT.rglob("*")
    if (
        p.is_file()
        and p.suffix.lower() in EXTS
    )
]

print(
    "\nImage-like files in archive:",
    len(archive_images)
)

by_tail2 = defaultdict(list)
by_tail3 = defaultdict(list)
by_name = defaultdict(list)

for p in archive_images:

    parts = p.parts

    if len(parts) >= 2:
        by_tail2[
            "/".join(parts[-2:])
        ].append(p)

    if len(parts) >= 3:
        by_tail3[
            "/".join(parts[-3:])
        ].append(p)

    by_name[p.name].append(p)


# ============================================================
# 7. EXIF-AWARE SOURCE RESOLUTION
# ============================================================

geometry_cache = {}


def geometry(path):

    key = str(path)

    if key in geometry_cache:
        return geometry_cache[key]

    try:

        with Image.open(path) as im:
            im.verify()

        with Image.open(path) as im:

            raw = tuple(
                im.size
            )

            corrected = tuple(
                exif_size(im)
            )

        result = {
            "raw": raw,
            "exif": corrected,
            "corrupt": False
        }

    except Exception as e:

        result = {
            "raw": None,
            "exif": None,
            "corrupt": True,
            "error": str(e)
        }

    geometry_cache[key] = result

    return result


def candidate_pool(imrec):

    rel = Path(
        imrec["file_name"]
    )

    candidates = []

    if len(rel.parts) >= 3:

        candidates.extend(
            by_tail3.get(
                "/".join(
                    rel.parts[-3:]
                ),
                []
            )
        )

    if len(rel.parts) >= 2:

        candidates.extend(
            by_tail2.get(
                "/".join(
                    rel.parts[-2:]
                ),
                []
            )
        )

    candidates.extend(
        by_name.get(
            rel.name,
            []
        )
    )

    out = []
    seen = set()

    for p in candidates:

        s = str(p)

        if s not in seen:

            seen.add(s)
            out.append(p)

    return out


def resolve_image(imrec):

    rel = Path(
        imrec["file_name"]
    )

    expected = (
        int(imrec["width"]),
        int(imrec["height"])
    )

    candidates = candidate_pool(
        imrec
    )

    if not candidates:
        return None

    good = []

    for p in candidates:

        g = geometry(p)

        if (
            not g["corrupt"]
            and g["exif"] == expected
        ):
            good.append(p)

    if len(good) == 1:
        return good[0]

    if len(good) > 1:

        batch = (
            rel.parts[0]
            if len(rel.parts) > 1
            else None
        )

        preferred = [
            p
            for p in good
            if (
                batch is not None
                and batch in p.parts
            )
        ]

        if len(preferred) == 1:
            return preferred[0]

        return sorted(
            good,
            key=lambda x: str(x)
        )[0]

    return None


resolved = {}

for imrec in tqdm(
    selected_coco["images"],
    desc="Resolving official TACO images"
):

    src = resolve_image(
        imrec
    )

    if src is None:

        raise RuntimeError(
            "Could not resolve image_id="
            f"{imrec['id']} "
            f"file={imrec['file_name']}"
        )

    resolved[
        int(imrec["id"])
    ] = src


print(
    "\nResolved:",
    len(resolved),
    "/ 1500"
)

assert len(resolved) == 1500


# ============================================================
# 8. REBUILD CANONICAL IMAGE TREE
# SAME MODEL-VISIBLE GEOMETRY POLICY AS v5
# ============================================================

if LOCAL_DATASET.exists():

    shutil.rmtree(
        LOCAL_DATASET
    )

CANON_IMAGES.mkdir(
    parents=True,
    exist_ok=True
)

direct_ok = 0
normalized = 0
geometry_failures = []


for imrec in tqdm(
    selected_coco["images"],
    desc="Rebuilding canonical images"
):

    iid = int(
        imrec["id"]
    )

    src = resolved[iid]

    dst = (
        CANON_IMAGES /
        imrec["file_name"]
    )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    expected = (
        int(imrec["width"]),
        int(imrec["height"])
    )

    # --------------------------------------------------------
    # First reproduce v5 direct-image behavior
    # --------------------------------------------------------

    try:
        os.link(
            src,
            dst
        )

    except OSError:
        os.symlink(
            src,
            dst
        )


    arr = cv2.imread(
        str(dst),
        cv2.IMREAD_COLOR
    )


    if (
        arr is not None
        and (
            arr.shape[1],
            arr.shape[0]
        ) == expected
    ):

        direct_ok += 1
        continue


    # --------------------------------------------------------
    # Same v5 deterministic EXIF fallback
    # --------------------------------------------------------

    if (
        dst.exists()
        or dst.is_symlink()
    ):
        dst.unlink()


    try:

        with Image.open(src) as pil:

            fixed = (
                ImageOps
                .exif_transpose(pil)
                .convert("RGB")
            )

            if fixed.size != expected:

                raise ValueError(
                    f"EXIF size={fixed.size}, "
                    f"expected={expected}"
                )

            fixed.save(
                dst,
                "JPEG",
                quality=100,
                subsampling=0
            )


        arr = cv2.imread(
            str(dst),
            cv2.IMREAD_COLOR
        )

        if (
            arr is None
            or (
                arr.shape[1],
                arr.shape[0]
            ) != expected
        ):

            raise ValueError(
                "OpenCV geometry mismatch "
                "after EXIF normalization."
            )

        normalized += 1


    except Exception as e:

        geometry_failures.append({
            "image_id": iid,
            "file_name":
                imrec["file_name"],
            "error":
                str(e)
        })


print(
    "\nDirect geometry OK :",
    direct_ok
)

print(
    "EXIF normalized    :",
    normalized
)

print(
    "Geometry failures  :",
    len(geometry_failures)
)


if geometry_failures:

    print(
        json.dumps(
            geometry_failures[:20],
            indent=2
        )
    )

    raise RuntimeError(
        "Canonical image reconstruction failed."
    )


assert (
    direct_ok + normalized
) == 1500


# ============================================================
# 9. STRICT SHA-256 RECOVERY CHECK
# ============================================================

if IMAGE_AUDIT.exists():

    audit_df = pd.read_csv(
        IMAGE_AUDIT
    )

    hash_rows = []

    for r in tqdm(
        audit_df.itertuples(),
        total=len(audit_df),
        desc="Comparing reconstructed image hashes"
    ):

        p = (
            CANON_IMAGES /
            r.file_name
        )

        current_hash = (
            sha256_file(p)
        )

        hash_rows.append({
            "image_id":
                int(r.image_id),

            "file_name":
                r.file_name,

            "previous_sha256":
                str(r.sha256),

            "current_sha256":
                current_hash,

            "match":
                current_hash
                == str(r.sha256)
        })


    hash_df = pd.DataFrame(
        hash_rows
    )

    matched = int(
        hash_df["match"].sum()
    )

    print(
        "\nOriginal canonical-image SHA-256 matches:",
        f"{matched}/{len(hash_df)}"
    )


    if matched != len(hash_df):

        display(
            hash_df[
                ~hash_df["match"]
            ].head(20)
        )

        raise RuntimeError(
            "Reconstructed canonical images "
            "do not exactly match the images "
            "used during the original experiment. "
            "Do not run validation yet."
        )

else:

    print(
        "\nWARNING: image_audit.csv not found; "
        "exact prior-image SHA verification skipped."
    )


# ============================================================
# 10. LOAD FROZEN SPLITS — NEVER REGENERATE
# ============================================================

random_split = json.loads(
    RANDOM_SPLIT_PATH.read_text(
        encoding="utf-8"
    )
)

batch_split = json.loads(
    BATCH_SPLIT_PATH.read_text(
        encoding="utf-8"
    )
)


def validate_frozen_split(
    split,
    name
):

    assert set(split.keys()) == {
        "train",
        "val",
        "test"
    }

    sets = {
        k: set(
            map(int, v)
        )
        for k, v in split.items()
    }

    assert sets["train"].isdisjoint(
        sets["val"]
    )

    assert sets["train"].isdisjoint(
        sets["test"]
    )

    assert sets["val"].isdisjoint(
        sets["test"]
    )

    union = (
        sets["train"]
        | sets["val"]
        | sets["test"]
    )

    assert union == clean_ids

    print(
        f"{name:16s}: "
        f"train={len(sets['train'])}, "
        f"val={len(sets['val'])}, "
        f"test={len(sets['test'])}"
    )


print(
    "\nFROZEN SPLIT COUNTS"
)

validate_frozen_split(
    random_split,
    "random"
)

validate_frozen_split(
    batch_split,
    "batch_disjoint"
)


# ============================================================
# 11. REBUILD TACO-1 YOLO LABELS
# ============================================================

LABELS.mkdir(
    parents=True,
    exist_ok=True
)

coco = clean_coco

im_map = {
    int(im["id"]): im
    for im in coco["images"]
}

anns_by_img = defaultdict(list)

for a in coco["annotations"]:

    anns_by_img[
        int(a["image_id"])
    ].append(a)


for iid, im in tqdm(
    im_map.items(),
    desc="Rebuilding TACO-1 labels"
):

    rel = Path(
        im["file_name"]
    )

    label_path = (
        LABELS /
        rel.with_suffix(".txt")
    )

    label_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    W = float(
        im["width"]
    )

    H = float(
        im["height"]
    )

    lines = []

    for a in anns_by_img.get(
        iid,
        []
    ):

        x, y, w, h = map(
            float,
            a["bbox"]
        )

        xc = (
            x + w / 2
        ) / W

        yc = (
            y + h / 2
        ) / H

        wn = w / W
        hn = h / H

        vals = [
            xc,
            yc,
            wn,
            hn
        ]

        if not all(
            0 <= v <= 1
            for v in vals
        ):

            raise ValueError(
                f"Invalid normalized bbox: "
                f"image={iid}, "
                f"ann={a.get('id')}, "
                f"{vals}"
            )

        lines.append(
            f"0 "
            f"{xc:.10f} "
            f"{yc:.10f} "
            f"{wn:.10f} "
            f"{hn:.10f}"
        )


    label_path.write_text(
        "\n".join(lines)
        + (
            "\n"
            if lines
            else ""
        ),
        encoding="utf-8"
    )


# ============================================================
# 12. REBUILD EXACT COCO TACO-1 SUBSETS
# ============================================================

def make_coco_subset(
    ids,
    out_json
):

    ids = set(
        map(int, ids)
    )

    anns = []

    ann_id = 1

    for a in coco["annotations"]:

        if int(
            a["image_id"]
        ) not in ids:

            continue

        b = dict(a)

        b["id"] = ann_id

        b["category_id"] = 1

        b["iscrowd"] = int(
            b.get(
                "iscrowd",
                0
            )
        )

        if "area" not in b:

            b["area"] = (
                float(
                    b["bbox"][2]
                )
                *
                float(
                    b["bbox"][3]
                )
            )

        anns.append(
            b
        )

        ann_id += 1


    subset = {

        "info":
            coco.get(
                "info",
                {}
            ),

        "licenses":
            coco.get(
                "licenses",
                []
            ),

        "images": [
            im
            for im in coco["images"]
            if int(im["id"]) in ids
        ],

        "annotations":
            anns,

        "categories": [{
            "id": 1,
            "name": "litter",
            "supercategory": "litter"
        }]
    }


    Path(
        out_json
    ).write_text(
        json.dumps(
            subset
        ),
        encoding="utf-8"
    )


# ============================================================
# 13. REBUILD protocol folders
# ============================================================

protocol_splits = {

    "random":
        random_split,

    "batch_disjoint":
        batch_split
}


for protocol, split in protocol_splits.items():

    pdir = (
        LOCAL_DATASET /
        protocol
    )

    pdir.mkdir(
        parents=True,
        exist_ok=True
    )


    for s in [
        "train",
        "val",
        "test"
    ]:

        txt = (
            pdir /
            f"{s}.txt"
        )

        image_paths = [

            str(
                CANON_IMAGES /
                im_map[int(i)][
                    "file_name"
                ]
            )

            for i in split[s]
        ]


        txt.write_text(
            "\n".join(
                image_paths
            )
            + "\n",
            encoding="utf-8"
        )


        make_coco_subset(
            split[s],
            pdir /
            f"{s}_coco_taco1.json"
        )


    data_yaml = {

        "path":
            str(
                LOCAL_DATASET
            ),

        "train":
            str(
                pdir /
                "train.txt"
            ),

        "val":
            str(
                pdir /
                "val.txt"
            ),

        "test":
            str(
                pdir /
                "test.txt"
            ),

        "names":
            {
                0:
                    "litter"
            }
    }


    (
        pdir /
        "data.yaml"
    ).write_text(

        yaml.safe_dump(
            data_yaml,
            sort_keys=False
        ),

        encoding="utf-8"
    )


# ============================================================
# 14. VERIFY REBUILT DATASET
# ============================================================

for protocol in [
    "random",
    "batch_disjoint"
]:

    pdir = (
        LOCAL_DATASET /
        protocol
    )

    for f in [

        "train.txt",
        "val.txt",
        "test.txt",

        "train_coco_taco1.json",
        "val_coco_taco1.json",
        "test_coco_taco1.json",

        "data.yaml"
    ]:

        p = pdir / f

        if not p.exists():

            raise FileNotFoundError(
                f"Reconstruction missing: {p}"
            )


print(
    "\nLOCAL TACO-1 FILE STRUCTURE PASSED."
)


# ============================================================
# 15. COMPARE WITH ORIGINAL SPLITS_FROZEN MANIFEST
# ============================================================

if FROZEN_MANIFEST.exists():

    frozen = json.loads(
        FROZEN_MANIFEST.read_text(
            encoding="utf-8"
        )
    )

    comparison = []


    for protocol in [
        "random",
        "batch_disjoint"
    ]:

        pdir = (
            LOCAL_DATASET /
            protocol
        )

        old = (
            frozen["protocols"][
                protocol
            ]
        )


        current = {

            "split_json_sha256":
                sha256_file(
                    RANDOM_SPLIT_PATH
                    if protocol == "random"
                    else BATCH_SPLIT_PATH
                ),

            "train_txt_sha256":
                sha256_file(
                    pdir /
                    "train.txt"
                ),

            "val_txt_sha256":
                sha256_file(
                    pdir /
                    "val.txt"
                ),

            "test_txt_sha256":
                sha256_file(
                    pdir /
                    "test.txt"
                ),

            "train_gt_sha256":
                sha256_file(
                    pdir /
                    "train_coco_taco1.json"
                ),

            "val_gt_sha256":
                sha256_file(
                    pdir /
                    "val_coco_taco1.json"
                ),

            "test_gt_sha256":
                sha256_file(
                    pdir /
                    "test_coco_taco1.json"
                ),
        }


        for key, value in current.items():

            comparison.append({

                "protocol":
                    protocol,

                "file_hash":
                    key,

                "original":
                    old.get(key),

                "reconstructed":
                    value,

                "match":
                    old.get(key)
                    == value
            })


    comparison_df = pd.DataFrame(
        comparison
    )

    display(
        comparison_df
    )

    nmatch = int(
        comparison_df[
            "match"
        ].sum()
    )

    ntotal = len(
        comparison_df
    )

    print(
        "\nFrozen dataset hashes matched:",
        f"{nmatch}/{ntotal}"
    )


    if not comparison_df[
        "match"
    ].all():

        raise RuntimeError(
            "Reconstructed split/data files "
            "do not exactly match the frozen "
            "original experiment. "
            "Do not run validation."
        )


else:

    print(
        "\nWARNING: splits_frozen.json "
        "not found. Hash comparison skipped."
    )


# ============================================================
# 16. RELOAD CONFIG FOR NEXT STAGES
# ============================================================

CONFIG_PATH = (
    DIRS["manifests"] /
    "training_config_frozen.json"
)

if not CONFIG_PATH.exists():

    raise FileNotFoundError(
        CONFIG_PATH
    )


CONFIG = json.loads(
    CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# FINAL
# ============================================================

print(
    "\n" + "=" * 75
)

print(
    "LOCAL DATASET RECOVERY PASSED"
)

print(
    "=" * 75
)

print(
    "Canonical images : 1500"
)

print(
    "Local dataset    :",
    LOCAL_DATASET
)

print(
    "Random split     : REUSED, NOT REGENERATED"
)

print(
    "Batch-disjoint   : REUSED, NOT REGENERATED"
)

print(
    "Training models  : PRESERVED; NO RETRAINING"
)

print(
    "=" * 75
)

In [ ]:
# ============================================================
# STAGE 6C PATCH
# FIX: Ultralytics may return synthetic paths such as image0.jpg
#
# We therefore DO NOT obtain COCO image_id from r.path.
# Instead:
#   1. map the original input paths to official COCO image IDs
#      BEFORE prediction,
#   2. use the preserved prediction order,
#   3. verify that exactly one result is returned per image.
#
# This does NOT touch test data.
# ============================================================

from pathlib import Path
from collections import defaultdict

import json
import numpy as np
import pandas as pd
import torch


# ============================================================
# 1. GT PATH MAPS
# ============================================================

def gt_path_maps(gt_json):

    gt = json.loads(
        Path(gt_json).read_text(
            encoding="utf-8"
        )
    )

    # Preferred mapping:
    # batch_x/filename.jpg -> official COCO image_id
    tail2 = {}

    # Fallback:
    # filename.jpg -> [possible image_ids]
    names = defaultdict(list)

    gt_by_id = {}

    for im in gt["images"]:

        iid = int(im["id"])

        rel = Path(
            im["file_name"]
        )

        key = "/".join(
            rel.parts[-2:]
        )

        if key in tail2:

            raise RuntimeError(
                "Duplicate last-two-component path "
                f"in COCO GT: {key}"
            )

        tail2[key] = iid

        names[
            rel.name
        ].append(iid)

        gt_by_id[iid] = im

    return (
        gt,
        tail2,
        names,
        gt_by_id
    )


# ============================================================
# 2. MAP ORIGINAL INPUT PATH
# ============================================================

def map_input_path_to_image_id(
    path,
    tail2,
    names
):

    p = Path(path)

    # Example:
    # /content/taco_v5/dataset/images/batch_3/000012.jpg
    # ->
    # batch_3/000012.jpg

    if len(p.parts) >= 2:

        key = "/".join(
            p.parts[-2:]
        )

        if key in tail2:
            return int(
                tail2[key]
            )


    # Fallback only if basename is unique
    candidates = names.get(
        p.name,
        []
    )

    if len(candidates) == 1:

        return int(
            candidates[0]
        )


    raise KeyError(
        "Could not uniquely map ORIGINAL input path "
        f"to COCO image_id:\n{path}\n"
        f"basename candidates={candidates}"
    )


# ============================================================
# 3. FIXED predict_coco
# ============================================================

@torch.inference_mode()
def predict_coco(
    model,
    image_paths,
    gt_json,
    imgsz=640,
    conf=0.001,
    max_det=300
):

    (
        gt,
        tail2,
        names,
        gt_by_id
    ) = gt_path_maps(
        gt_json
    )


    # --------------------------------------------------------
    # Normalize original input path list
    # --------------------------------------------------------

    image_paths = [
        str(Path(p))
        for p in image_paths
    ]


    if len(image_paths) == 0:

        raise RuntimeError(
            "Prediction input list is empty."
        )


    # --------------------------------------------------------
    # IMPORTANT:
    # Determine official image IDs BEFORE model.predict().
    #
    # We never depend on Result.path.
    # --------------------------------------------------------

    input_records = []

    for p in image_paths:

        if not Path(p).exists():

            raise FileNotFoundError(
                f"Validation image does not exist:\n{p}"
            )


        iid = map_input_path_to_image_id(
            p,
            tail2,
            names
        )


        input_records.append({
            "path": p,
            "image_id": int(iid)
        })


    # --------------------------------------------------------
    # Every validation image should appear exactly once
    # --------------------------------------------------------

    input_ids = [
        x["image_id"]
        for x in input_records
    ]


    if len(set(input_ids)) != len(input_ids):

        duplicate_ids = [

            iid

            for iid in set(input_ids)

            if input_ids.count(iid) > 1
        ]

        raise RuntimeError(
            "Duplicate COCO image IDs in validation "
            f"input list: {duplicate_ids}"
        )


    gt_ids = {
        int(im["id"])
        for im in gt["images"]
    }


    if set(input_ids) != gt_ids:

        missing = sorted(
            gt_ids - set(input_ids)
        )

        extra = sorted(
            set(input_ids) - gt_ids
        )

        raise RuntimeError(
            "Validation input list and validation COCO GT "
            "do not contain exactly the same image IDs.\n"
            f"Missing IDs: {missing[:20]}\n"
            f"Extra IDs: {extra[:20]}"
        )


    print(
        f"Pre-mapped validation inputs: "
        f"{len(input_records)}/{len(gt_ids)}"
    )


    # --------------------------------------------------------
    # MODEL PREDICTION
    # --------------------------------------------------------

    preds = []
    speeds = []


    stream = model.predict(

        source=image_paths,

        imgsz=imgsz,

        conf=conf,

        max_det=max_det,

        device=0,

        batch=1,

        stream=True,

        verbose=False
    )


    n_results = 0


    for idx, r in enumerate(stream):


        # ----------------------------------------------------
        # Protect against unexpected result count
        # ----------------------------------------------------

        if idx >= len(input_records):

            raise RuntimeError(
                "Ultralytics returned more prediction "
                "results than input images."
            )


        # ----------------------------------------------------
        # CRITICAL FIX
        #
        # Use ORIGINAL ordered source path rather than r.path.
        #
        # r.path may be:
        # image0.jpg
        # image1.jpg
        # ...
        # ----------------------------------------------------

        source_record = input_records[idx]

        source_path = source_record[
            "path"
        ]

        iid = int(
            source_record[
                "image_id"
            ]
        )


        # ----------------------------------------------------
        # Geometry sanity check
        # ----------------------------------------------------

        gt_im = gt_by_id[iid]

        expected_w = int(
            gt_im["width"]
        )

        expected_h = int(
            gt_im["height"]
        )


        if hasattr(
            r,
            "orig_shape"
        ) and r.orig_shape is not None:

            result_h = int(
                r.orig_shape[0]
            )

            result_w = int(
                r.orig_shape[1]
            )


            if (
                result_w != expected_w
                or
                result_h != expected_h
            ):

                raise RuntimeError(
                    "Prediction/COCO geometry mismatch.\n"
                    f"Input: {source_path}\n"
                    f"image_id: {iid}\n"
                    f"Result geometry: "
                    f"{result_w}x{result_h}\n"
                    f"COCO geometry: "
                    f"{expected_w}x{expected_h}"
                )


        # ----------------------------------------------------
        # SPEED
        # ----------------------------------------------------

        speeds.append({

            "image_id":
                iid,

            "source_path":
                source_path,

            # Keep r.path for auditing only.
            # It is NOT used for ID mapping.
            "ultralytics_result_path":
                str(
                    getattr(
                        r,
                        "path",
                        ""
                    )
                ),

            "preprocess_ms":
                float(
                    r.speed.get(
                        "preprocess",
                        np.nan
                    )
                ),

            "inference_ms":
                float(
                    r.speed.get(
                        "inference",
                        np.nan
                    )
                ),

            "postprocess_ms":
                float(
                    r.speed.get(
                        "postprocess",
                        np.nan
                    )
                ),
        })


        # ----------------------------------------------------
        # NO DETECTIONS
        # ----------------------------------------------------

        if (
            r.boxes is None
            or
            len(r.boxes) == 0
        ):

            n_results += 1
            continue


        # ----------------------------------------------------
        # DETECTIONS
        # ----------------------------------------------------

        xyxy = (
            r.boxes
            .xyxy
            .detach()
            .cpu()
            .numpy()
        )

        confs = (
            r.boxes
            .conf
            .detach()
            .cpu()
            .numpy()
        )


        if len(xyxy) != len(confs):

            raise RuntimeError(
                f"Box/confidence count mismatch "
                f"for image_id={iid}"
            )


        for box, score in zip(
            xyxy,
            confs
        ):

            x1, y1, x2, y2 = map(
                float,
                box
            )


            w = x2 - x1
            h = y2 - y1


            # Ignore impossible zero/negative predictions.
            if w <= 0 or h <= 0:
                continue


            preds.append({

                "image_id":
                    iid,

                "category_id":
                    1,

                "bbox": [
                    x1,
                    y1,
                    w,
                    h
                ],

                "score":
                    float(score)
            })


        n_results += 1


    # ========================================================
    # 4. RESULT COUNT CHECK
    # ========================================================

    if n_results != len(
        input_records
    ):

        raise RuntimeError(
            "Ultralytics prediction result count mismatch.\n"
            f"Inputs : {len(input_records)}\n"
            f"Results: {n_results}"
        )


    # --------------------------------------------------------
    # Speed table should have exactly one row/image
    # --------------------------------------------------------

    speed_df = pd.DataFrame(
        speeds
    )


    if len(speed_df) != len(
        input_records
    ):

        raise RuntimeError(
            "Speed table does not contain exactly "
            "one row per validation image."
        )


    if speed_df[
        "image_id"
    ].duplicated().any():

        raise RuntimeError(
            "Duplicate image_id detected in speed table."
        )


    print(
        f"Prediction results mapped: "
        f"{n_results}/{len(input_records)}"
    )

    print(
        f"COCO detections produced: "
        f"{len(preds)}"
    )


    return (
        preds,
        speed_df
    )


print("=" * 70)
print("STAGE 6C PATCH LOADED")
print("=" * 70)

print(
    "COCO image IDs are now mapped from the "
    "ORIGINAL validation input paths."
)

print(
    "Ultralytics Result.path is retained only "
    "for auditing and is NOT used for ID mapping."
)

print(
    "TEST DATA HAS NOT BEEN ACCESSED."
)

print("=" * 70)

In [ ]:
# ============================================================
# STAGE 6D — VALIDATION-ONLY PREDICTIONS + THRESHOLD FREEZE
# FIXED FOR REAL GOOGLE DRIVE
#
# IMPORTANT:
# - VALIDATION ONLY
# - TEST DATA IS NOT USED
# - THRESHOLDS ARE SELECTED ONLY FROM VALIDATION DATA
# - ALL 24 best.pt FILES MUST EXIST
# - STAGE 6C MUST HAVE BEEN RUN FIRST
# ============================================================

from pathlib import Path
import json
import gc
import torch
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

LOCAL_DATASET = Path(
    "/content/taco_v5/dataset"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 2. SAFETY — LOCKED TEST MUST STILL BE CLOSED
# ============================================================

LOCKED_TEST_GATE = (
    DIRS["manifests"] /
    "LOCKED_TEST_OPENED.json"
)

if LOCKED_TEST_GATE.exists():

    raise RuntimeError(
        "LOCKED TEST HAS ALREADY BEEN OPENED.\n"
        "Do not rerun validation threshold selection after "
        "opening the locked test."
    )


# ============================================================
# 3. LOAD FROZEN TRAINING CONFIG
# ============================================================

CONFIG_PATH = (
    DIRS["manifests"] /
    "training_config_frozen.json"
)

if not CONFIG_PATH.exists():

    raise FileNotFoundError(
        f"Frozen training configuration missing:\n"
        f"{CONFIG_PATH}"
    )


CONFIG = json.loads(
    CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)


print("=" * 75)
print("STAGE 6D — VALIDATION-ONLY EVALUATION")
print("=" * 75)

print(
    "Protocols:",
    CONFIG["protocols"]
)

print(
    "Models:",
    list(CONFIG["models"].keys())
)

print(
    "Seeds:",
    CONFIG["seeds"]
)

print(
    "Image size:",
    CONFIG["imgsz"]
)


# ============================================================
# 4. VERIFY LOCAL DATASET
# ============================================================

if not LOCAL_DATASET.exists():

    raise FileNotFoundError(
        f"Recovered local dataset missing:\n"
        f"{LOCAL_DATASET}"
    )


for protocol in CONFIG["protocols"]:

    pdir = (
        LOCAL_DATASET /
        protocol
    )

    required_files = [
        pdir / "val.txt",
        pdir / "val_coco_taco1.json",
    ]

    for f in required_files:

        if not f.exists():

            raise FileNotFoundError(
                f"Required validation file missing:\n{f}"
            )


print(
    "\nLOCAL VALIDATION DATASET CHECK PASSED."
)


# ============================================================
# 5. VERIFY STAGE 6C FUNCTIONS
# ============================================================

REQUIRED_HELPERS = [
    "load_best_model",
    "predict_coco",
    "coco_metrics",
    "select_val_threshold",
    "detection_counts",
]

missing_helpers = [
    name
    for name in REQUIRED_HELPERS
    if name not in globals()
]


if missing_helpers:

    raise RuntimeError(
        "STAGE 6C has not been loaded in this runtime.\n"
        "Missing helper functions:\n"
        + ", ".join(missing_helpers)
        + "\n\nRun STAGE 6C first, then rerun STAGE 6D."
    )


print(
    "STAGE 6C HELPER CHECK PASSED."
)


# ============================================================
# 6. VERIFY ALL 24 TRAINED MODELS
# ============================================================

missing_models = []

expected_runs = (
    len(CONFIG["protocols"])
    *
    len(CONFIG["models"])
    *
    len(CONFIG["seeds"])
)


for protocol in CONFIG["protocols"]:

    for model_key in CONFIG["models"]:

        for seed in CONFIG["seeds"]:

            best_pt = (
                DIRS["runs"]
                /
                protocol
                /
                f"{model_key}_seed{seed}"
                /
                "weights"
                /
                "best.pt"
            )

            if not best_pt.exists():

                missing_models.append(
                    str(best_pt)
                )


if missing_models:

    print(
        "\nMissing best.pt files:"
    )

    for p in missing_models:
        print(p)

    raise RuntimeError(
        f"{len(missing_models)} trained model checkpoints "
        "are missing. Do not continue."
    )


print(
    f"TRAINED MODEL CHECK PASSED: "
    f"{expected_runs}/{expected_runs} best.pt"
)


# ============================================================
# 7. VERIFY VALIDATION SPLITS
# ============================================================

for protocol in CONFIG["protocols"]:

    pdir = (
        LOCAL_DATASET /
        protocol
    )

    val_txt = (
        pdir /
        "val.txt"
    )

    gt_json = (
        pdir /
        "val_coco_taco1.json"
    )

    val_paths = [
        x.strip()
        for x in val_txt.read_text(
            encoding="utf-8"
        ).splitlines()
        if x.strip()
    ]

    gt = json.loads(
        gt_json.read_text(
            encoding="utf-8"
        )
    )

    n_gt_images = len(
        gt["images"]
    )

    if len(val_paths) != n_gt_images:

        raise RuntimeError(
            f"{protocol}: val.txt has {len(val_paths)} images "
            f"but validation COCO JSON has {n_gt_images} images."
        )

    missing_images = [
        p
        for p in val_paths
        if not Path(p).exists()
    ]

    if missing_images:

        raise FileNotFoundError(
            f"{protocol}: {len(missing_images)} validation "
            "images are missing."
        )

    print(
        f"{protocol:16s}: "
        f"{len(val_paths)} validation images"
    )


print(
    "\nVALIDATION SPLIT CHECK PASSED."
)


# ============================================================
# 8. VALIDATION EVALUATION
# ============================================================

summary_rows = []

run_counter = 0


for protocol in CONFIG["protocols"]:

    pdir = (
        LOCAL_DATASET /
        protocol
    )

    val_paths = [
        x.strip()
        for x in (
            pdir /
            "val.txt"
        ).read_text(
            encoding="utf-8"
        ).splitlines()
        if x.strip()
    ]

    gt_json = (
        pdir /
        "val_coco_taco1.json"
    )


    for model_key in CONFIG["models"]:

        for seed in CONFIG["seeds"]:

            run_counter += 1

            run_name = (
                f"{protocol}/"
                f"{model_key}_seed{seed}"
            )


            print(
                "\n"
                + "=" * 75
            )

            print(
                f"[{run_counter}/{expected_runs}] "
                f"VALIDATION: {run_name}"
            )

            print(
                "=" * 75
            )


            # ------------------------------------------------
            # TRAINED MODEL
            # ------------------------------------------------

            run_dir = (
                DIRS["runs"]
                /
                protocol
                /
                f"{model_key}_seed{seed}"
            )

            best_pt = (
                run_dir /
                "weights" /
                "best.pt"
            )


            if not best_pt.exists():

                raise FileNotFoundError(
                    f"Missing best.pt:\n{best_pt}"
                )


            # ------------------------------------------------
            # OUTPUT DIRECTORY
            # ------------------------------------------------

            out = (
                DIRS["validation"]
                /
                protocol
                /
                f"{model_key}_seed{seed}"
            )

            out.mkdir(
                parents=True,
                exist_ok=True
            )


            pred_path = (
                out /
                "val_predictions_coco.json"
            )

            speed_path = (
                out /
                "val_speed.csv"
            )

            threshold_grid_path = (
                out /
                "val_threshold_grid.csv"
            )

            per_image_path = (
                out /
                "val_per_image_metrics.csv"
            )

            selected_threshold_path = (
                out /
                "val_selected_threshold.json"
            )


            # ------------------------------------------------
            # PREDICTIONS
            #
            # conf=0.001 is intentionally low.
            # Threshold selection happens afterwards using
            # validation only.
            # ------------------------------------------------

            model = None

            if pred_path.exists():

                print(
                    "Existing validation predictions found; "
                    "reusing them."
                )

                preds = json.loads(
                    pred_path.read_text(
                        encoding="utf-8"
                    )
                )

                if speed_path.exists():

                    speed = pd.read_csv(
                        speed_path
                    )

                else:

                    speed = pd.DataFrame()


            else:

                print(
                    "Generating validation predictions..."
                )

                model = load_best_model(
                    model_key,
                    best_pt
                )

                preds, speed = predict_coco(
                    model,
                    val_paths,
                    gt_json,
                    imgsz=CONFIG["imgsz"],
                    conf=0.001,
                    max_det=300
                )


                pred_path.write_text(
                    json.dumps(
                        preds,
                        ensure_ascii=False
                    ),
                    encoding="utf-8"
                )


                speed.to_csv(
                    speed_path,
                    index=False
                )


            # ------------------------------------------------
            # BASIC PREDICTION SANITY CHECK
            # ------------------------------------------------

            gt_obj = json.loads(
                gt_json.read_text(
                    encoding="utf-8"
                )
            )

            valid_image_ids = {
                int(im["id"])
                for im in gt_obj["images"]
            }


            prediction_image_ids = {
                int(p["image_id"])
                for p in preds
            }


            invalid_prediction_ids = (
                prediction_image_ids
                -
                valid_image_ids
            )


            if invalid_prediction_ids:

                raise RuntimeError(
                    f"{run_name}: predictions contain "
                    "image IDs outside validation split:\n"
                    f"{sorted(invalid_prediction_ids)}"
                )


            # ------------------------------------------------
            # COCO METRICS
            #
            # Primary:
            # mAP@[0.50:0.95]
            #
            # Secondary:
            # AP50, AP75,
            # AP-small/medium/large,
            # AR metrics
            # ------------------------------------------------

            print(
                "\nCOCO evaluation:"
            )

            metrics = coco_metrics(
                gt_json,
                preds
            )


            # ------------------------------------------------
            # VALIDATION-ONLY THRESHOLD SELECTION
            #
            # Grid:
            # confidence 0.05 ... 0.95
            #
            # Criterion:
            # maximum GLOBAL F1 at IoU=0.50
            # ------------------------------------------------

            thr, grid = select_val_threshold(
                gt_json,
                preds
            )


            grid.to_csv(
                threshold_grid_path,
                index=False
            )


            # ------------------------------------------------
            # METRICS AT SELECTED VALIDATION THRESHOLD
            # ------------------------------------------------

            counts, per_img = detection_counts(
                gt_json,
                preds,
                thr,
                0.50
            )


            per_img.to_csv(
                per_image_path,
                index=False
            )


            # ------------------------------------------------
            # SAVE RUN-SPECIFIC THRESHOLD
            # ------------------------------------------------

            selected_threshold_payload = {

                "selected_threshold":
                    float(thr),

                "selection_split":
                    "validation",

                "criterion":
                    "maximum global F1 at IoU=0.50",

                "prediction_confidence_floor":
                    0.001,

                "threshold_grid_min":
                    0.05,

                "threshold_grid_max":
                    0.95,

                "threshold_grid_step":
                    0.01,

                "matching_iou":
                    0.50
            }


            selected_threshold_path.write_text(
                json.dumps(
                    selected_threshold_payload,
                    indent=2
                ),
                encoding="utf-8"
            )


            # ------------------------------------------------
            # SUMMARY ROW
            # ------------------------------------------------

            summary_rows.append({

                "protocol":
                    protocol,

                "model":
                    model_key,

                "seed":
                    int(seed),

                **metrics,

                "selected_threshold":
                    float(thr),

                "val_precision":
                    float(
                        counts["precision"]
                    ),

                "val_recall":
                    float(
                        counts["recall"]
                    ),

                "val_f1":
                    float(
                        counts["f1"]
                    ),
            })


            # ------------------------------------------------
            # RUN SUMMARY
            # ------------------------------------------------

            print(
                "\nSelected validation threshold:",
                round(
                    float(thr),
                    4
                )
            )

            print(
                "Validation Precision:",
                round(
                    float(
                        counts["precision"]
                    ),
                    4
                )
            )

            print(
                "Validation Recall   :",
                round(
                    float(
                        counts["recall"]
                    ),
                    4
                )
            )

            print(
                "Validation F1       :",
                round(
                    float(
                        counts["f1"]
                    ),
                    4
                )
            )

            print(
                "mAP50-95            :",
                round(
                    float(
                        metrics["mAP50_95"]
                    ),
                    4
                )
            )


            # ------------------------------------------------
            # MEMORY CLEANUP
            # ------------------------------------------------

            if model is not None:
                del model

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


# ============================================================
# 9. CREATE COMPLETE VALIDATION SUMMARY
# ============================================================

val_summary = pd.DataFrame(
    summary_rows
)


if len(val_summary) != expected_runs:

    raise RuntimeError(
        f"Validation summary contains "
        f"{len(val_summary)} rows; "
        f"expected {expected_runs}."
    )


# Confirm unique protocol/model/seed combinations
n_unique = (
    val_summary[
        [
            "protocol",
            "model",
            "seed"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


if n_unique != expected_runs:

    raise RuntimeError(
        "Validation summary contains duplicate "
        "protocol/model/seed combinations."
    )


SUMMARY_PATH = (
    DIRS["validation"] /
    "validation_summary_all_runs.csv"
)


val_summary.to_csv(
    SUMMARY_PATH,
    index=False
)


print(
    "\n"
    + "=" * 75
)

print(
    "VALIDATION SUMMARY — ALL 24 RUNS"
)

print(
    "=" * 75
)


display(
    val_summary
)


# ============================================================
# 10. BUILD FROZEN THRESHOLD MANIFEST
# ============================================================

threshold_manifest = {

    (
        f"{r.protocol}/"
        f"{r.model}/"
        f"seed{int(r.seed)}"
    ):
        float(
            r.selected_threshold
        )

    for r in val_summary.itertuples()
}


if len(threshold_manifest) != expected_runs:

    raise RuntimeError(
        f"Threshold manifest contains "
        f"{len(threshold_manifest)} entries; "
        f"expected {expected_runs}."
    )


THRESHOLD_MANIFEST_PATH = (
    DIRS["manifests"] /
    "validation_thresholds_frozen.json"
)


# ============================================================
# 11. FREEZE SAFELY
#
# If a previous frozen manifest exists, do not silently
# replace it with different thresholds.
# ============================================================

if THRESHOLD_MANIFEST_PATH.exists():

    previous_manifest = json.loads(
        THRESHOLD_MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )


    if previous_manifest != threshold_manifest:

        raise RuntimeError(
            "A different validation_thresholds_frozen.json "
            "already exists.\n"
            "It will NOT be overwritten automatically.\n"
            "Inspect the previous validation results before "
            "continuing."
        )

    print(
        "\nExisting frozen threshold manifest is "
        "identical to the current validation results."
    )

else:

    THRESHOLD_MANIFEST_PATH.write_text(
        json.dumps(
            threshold_manifest,
            indent=2
        ),
        encoding="utf-8"
    )


# ============================================================
# 12. FINAL VERIFICATION
# ============================================================

frozen_check = json.loads(
    THRESHOLD_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


if len(frozen_check) != expected_runs:

    raise RuntimeError(
        "Frozen threshold manifest is incomplete."
    )


print(
    "\n"
    + "=" * 75
)

print(
    "STAGE 6D PASSED"
)

print(
    "=" * 75
)

print(
    f"Validation runs completed : "
    f"{len(val_summary)}/{expected_runs}"
)

print(
    f"Thresholds frozen         : "
    f"{len(frozen_check)}/{expected_runs}"
)

print(
    "\nValidation summary:"
)

print(
    SUMMARY_PATH
)

print(
    "\nFrozen threshold manifest:"
)

print(
    THRESHOLD_MANIFEST_PATH
)

print(
    "\nVALIDATION THRESHOLDS FROZEN."
)

print(
    "LOCKED TEST REMAINS CLOSED."
)

print(
    "=" * 75
)

In [ ]:
# ============================================================
# STAGE 6E — VALIDATION AGGREGATION + SEED STABILITY ANALYSIS
#
# PURPOSE:
# - Aggregate 3-seed validation results as mean ± SD
# - Quantify seed stability
# - Compare random vs batch-disjoint protocols descriptively
# - Produce manuscript-ready validation tables
#
# IMPORTANT:
# - VALIDATION ONLY
# - LOCKED TEST REMAINS CLOSED
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 2. SAFETY CHECK — TEST MUST STILL BE CLOSED
# ============================================================

LOCKED_TEST_GATE = (
    DIRS["manifests"] /
    "LOCKED_TEST_OPENED.json"
)

if LOCKED_TEST_GATE.exists():
    raise RuntimeError(
        "Locked test has already been opened. "
        "Validation aggregation should have been finalized first."
    )


# ============================================================
# 3. LOAD VALIDATION SUMMARY
# ============================================================

SUMMARY_PATH = (
    DIRS["validation"] /
    "validation_summary_all_runs.csv"
)

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Validation summary not found:\n{SUMMARY_PATH}\n"
        "Run STAGE 6D first."
    )

df = pd.read_csv(
    SUMMARY_PATH
)


# ============================================================
# 4. REQUIRED STRUCTURE CHECK
# ============================================================

required_columns = [
    "protocol",
    "model",
    "seed",
    "mAP50_95",
    "AP50",
    "AP75",
    "AP_small",
    "AP_medium",
    "AP_large",
    "AR1",
    "AR10",
    "AR100",
    "AR_small",
    "AR_medium",
    "AR_large",
    "selected_threshold",
    "val_precision",
    "val_recall",
    "val_f1",
]

missing_columns = [
    c
    for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise RuntimeError(
        "Validation summary is missing columns:\n"
        + ", ".join(missing_columns)
    )


if len(df) != 24:
    raise RuntimeError(
        f"Expected 24 validation runs, found {len(df)}."
    )


if (
    df[
        ["protocol", "model", "seed"]
    ]
    .drop_duplicates()
    .shape[0]
    != 24
):
    raise RuntimeError(
        "Duplicate protocol/model/seed combinations detected."
    )


expected_seeds = {42, 123, 2026}

for (protocol, model), g in df.groupby(
    ["protocol", "model"]
):
    seeds = set(
        g["seed"].astype(int)
    )

    if seeds != expected_seeds:
        raise RuntimeError(
            f"{protocol}/{model}: "
            f"unexpected seeds {sorted(seeds)}"
        )


print("=" * 80)
print("STAGE 6E — VALIDATION AGGREGATION")
print("=" * 80)

print(
    f"Validation runs loaded : {len(df)}/24"
)

print(
    "Seed sets verified     : 42, 123, 2026"
)


# ============================================================
# 5. NUMERIC METRICS TO AGGREGATE
# ============================================================

metrics = [
    "mAP50_95",
    "AP50",
    "AP75",
    "AP_small",
    "AP_medium",
    "AP_large",
    "AR1",
    "AR10",
    "AR100",
    "AR_small",
    "AR_medium",
    "AR_large",
    "selected_threshold",
    "val_precision",
    "val_recall",
    "val_f1",
]


# ============================================================
# 6. MEAN ± SD OVER THREE SEEDS
# ============================================================

rows = []

for (protocol, model), g in df.groupby(
    ["protocol", "model"],
    sort=False
):

    row = {
        "protocol": protocol,
        "model": model,
        "n_seeds": len(g),
    }

    for metric in metrics:

        values = g[
            metric
        ].astype(float)

        row[
            f"{metric}_mean"
        ] = float(
            values.mean()
        )

        row[
            f"{metric}_sd"
        ] = float(
            values.std(
                ddof=1
            )
        )

        row[
            f"{metric}_min"
        ] = float(
            values.min()
        )

        row[
            f"{metric}_max"
        ] = float(
            values.max()
        )

    rows.append(
        row
    )


agg = pd.DataFrame(
    rows
)


# ============================================================
# 7. SEED STABILITY INDICATORS
# ============================================================

agg[
    "mAP50_95_range"
] = (
    agg["mAP50_95_max"]
    -
    agg["mAP50_95_min"]
)


agg[
    "val_f1_range"
] = (
    agg["val_f1_max"]
    -
    agg["val_f1_min"]
)


# Coefficient of variation (%)
agg[
    "mAP50_95_cv_percent"
] = (
    100.0
    *
    agg["mAP50_95_sd"]
    /
    agg["mAP50_95_mean"]
)


# ============================================================
# 8. SAVE FULL AGGREGATED TABLE
# ============================================================

FULL_AGG_PATH = (
    DIRS["validation"] /
    "validation_aggregate_mean_sd.csv"
)

agg.to_csv(
    FULL_AGG_PATH,
    index=False
)


# ============================================================
# 9. MANUSCRIPT-READY COMPACT TABLE
# ============================================================

compact = agg[
    [
        "protocol",
        "model",

        "mAP50_95_mean",
        "mAP50_95_sd",

        "AP50_mean",
        "AP50_sd",

        "AP75_mean",
        "AP75_sd",

        "AP_small_mean",
        "AP_small_sd",

        "AP_medium_mean",
        "AP_medium_sd",

        "AP_large_mean",
        "AP_large_sd",

        "val_precision_mean",
        "val_precision_sd",

        "val_recall_mean",
        "val_recall_sd",

        "val_f1_mean",
        "val_f1_sd",

        "mAP50_95_range",
        "mAP50_95_cv_percent",
    ]
].copy()


COMPACT_PATH = (
    DIRS["validation"] /
    "validation_manuscript_table.csv"
)

compact.to_csv(
    COMPACT_PATH,
    index=False
)


# ============================================================
# 10. FORMATTED MEAN ± SD TABLE
# ============================================================

formatted_rows = []

for r in agg.itertuples():

    formatted_rows.append({

        "protocol":
            r.protocol,

        "model":
            r.model,

        "mAP50-95":
            (
                f"{r.mAP50_95_mean:.4f} "
                f"± {r.mAP50_95_sd:.4f}"
            ),

        "AP50":
            (
                f"{r.AP50_mean:.4f} "
                f"± {r.AP50_sd:.4f}"
            ),

        "AP75":
            (
                f"{r.AP75_mean:.4f} "
                f"± {r.AP75_sd:.4f}"
            ),

        "AP-small":
            (
                f"{r.AP_small_mean:.4f} "
                f"± {r.AP_small_sd:.4f}"
            ),

        "AP-medium":
            (
                f"{r.AP_medium_mean:.4f} "
                f"± {r.AP_medium_sd:.4f}"
            ),

        "AP-large":
            (
                f"{r.AP_large_mean:.4f} "
                f"± {r.AP_large_sd:.4f}"
            ),

        "Precision":
            (
                f"{r.val_precision_mean:.4f} "
                f"± {r.val_precision_sd:.4f}"
            ),

        "Recall":
            (
                f"{r.val_recall_mean:.4f} "
                f"± {r.val_recall_sd:.4f}"
            ),

        "F1":
            (
                f"{r.val_f1_mean:.4f} "
                f"± {r.val_f1_sd:.4f}"
            ),

        "mAP50-95 range":
            f"{r.mAP50_95_range:.4f}",

        "mAP50-95 CV (%)":
            f"{r.mAP50_95_cv_percent:.2f}",
    })


formatted = pd.DataFrame(
    formatted_rows
)


FORMATTED_PATH = (
    DIRS["validation"] /
    "validation_mean_sd_formatted.csv"
)

formatted.to_csv(
    FORMATTED_PATH,
    index=False
)


# ============================================================
# 11. PROTOCOL DIFFERENCE
#
# Difference = batch-disjoint mean - random mean
#
# This is DESCRIPTIVE only because the validation image sets
# are different. Do not interpret as a paired causal effect.
# ============================================================

protocol_gap_rows = []

models = sorted(
    df["model"].unique()
)


for model in models:

    random_row = agg[
        (agg["protocol"] == "random")
        &
        (agg["model"] == model)
    ]

    batch_row = agg[
        (agg["protocol"] == "batch_disjoint")
        &
        (agg["model"] == model)
    ]


    if (
        len(random_row) != 1
        or
        len(batch_row) != 1
    ):
        raise RuntimeError(
            f"Protocol rows missing for {model}"
        )


    random_row = random_row.iloc[0]
    batch_row = batch_row.iloc[0]


    protocol_gap_rows.append({

        "model":
            model,

        "random_mAP50_95_mean":
            float(
                random_row[
                    "mAP50_95_mean"
                ]
            ),

        "batch_disjoint_mAP50_95_mean":
            float(
                batch_row[
                    "mAP50_95_mean"
                ]
            ),

        "batch_minus_random_mAP50_95":
            float(
                batch_row[
                    "mAP50_95_mean"
                ]
                -
                random_row[
                    "mAP50_95_mean"
                ]
            ),

        "random_val_f1_mean":
            float(
                random_row[
                    "val_f1_mean"
                ]
            ),

        "batch_disjoint_val_f1_mean":
            float(
                batch_row[
                    "val_f1_mean"
                ]
            ),

        "batch_minus_random_val_f1":
            float(
                batch_row[
                    "val_f1_mean"
                ]
                -
                random_row[
                    "val_f1_mean"
                ]
            ),
    })


protocol_gap = pd.DataFrame(
    protocol_gap_rows
)


PROTOCOL_GAP_PATH = (
    DIRS["validation"] /
    "validation_protocol_descriptive_gap.csv"
)

protocol_gap.to_csv(
    PROTOCOL_GAP_PATH,
    index=False
)


# ============================================================
# 12. RANK MODELS WITHIN EACH VALIDATION PROTOCOL
# ============================================================

ranking_rows = []

for protocol in [
    "random",
    "batch_disjoint"
]:

    temp = (
        agg[
            agg["protocol"]
            == protocol
        ]
        .sort_values(
            [
                "mAP50_95_mean",
                "mAP50_95_sd"
            ],
            ascending=[
                False,
                True
            ]
        )
        .copy()
    )

    temp[
        "validation_rank"
    ] = np.arange(
        1,
        len(temp) + 1
    )

    ranking_rows.append(
        temp[
            [
                "protocol",
                "model",
                "validation_rank",
                "mAP50_95_mean",
                "mAP50_95_sd",
                "val_f1_mean",
                "val_f1_sd",
                "mAP50_95_range",
                "mAP50_95_cv_percent",
            ]
        ]
    )


ranking = pd.concat(
    ranking_rows,
    ignore_index=True
)


RANKING_PATH = (
    DIRS["validation"] /
    "validation_model_ranking.csv"
)

ranking.to_csv(
    RANKING_PATH,
    index=False
)


# ============================================================
# 13. THRESHOLD MANIFEST VERIFICATION
# ============================================================

THRESHOLD_PATH = (
    DIRS["manifests"] /
    "validation_thresholds_frozen.json"
)

if not THRESHOLD_PATH.exists():

    raise FileNotFoundError(
        f"Frozen thresholds missing:\n"
        f"{THRESHOLD_PATH}"
    )


thresholds = json.loads(
    THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)


if len(thresholds) != 24:

    raise RuntimeError(
        f"Expected 24 frozen thresholds, "
        f"found {len(thresholds)}."
    )


# ============================================================
# 14. DISPLAY RESULTS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "VALIDATION MEAN ± SD — 3 SEEDS"
)

print(
    "=" * 80
)

display(
    formatted
)


print(
    "\n" + "=" * 80
)

print(
    "VALIDATION MODEL RANKING"
)

print(
    "=" * 80
)

display(
    ranking
)


print(
    "\n" + "=" * 80
)

print(
    "DESCRIPTIVE PROTOCOL DIFFERENCE"
)

print(
    "batch-disjoint mean minus random mean"
)

print(
    "=" * 80
)

display(
    protocol_gap
)


# ============================================================
# 15. SIMPLE STABILITY FLAGS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "SEED STABILITY CHECK"
)

print(
    "=" * 80
)


for r in agg.itertuples():

    if r.mAP50_95_sd < 0.005:

        label = "high stability"

    elif r.mAP50_95_sd < 0.015:

        label = "moderate stability"

    else:

        label = "high seed variability"


    print(
        f"{r.protocol:16s} "
        f"{r.model:10s} "
        f"mAP50-95="
        f"{r.mAP50_95_mean:.4f}"
        f"±{r.mAP50_95_sd:.4f} "
        f"| range={r.mAP50_95_range:.4f} "
        f"| {label}"
    )


# ============================================================
# 16. FINAL
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "STAGE 6E PASSED"
)

print(
    "=" * 80
)

print(
    "Validation aggregate table:"
)

print(
    FULL_AGG_PATH
)

print(
    "\nManuscript compact table:"
)

print(
    COMPACT_PATH
)

print(
    "\nFormatted mean ± SD table:"
)

print(
    FORMATTED_PATH
)

print(
    "\nProtocol descriptive gap:"
)

print(
    PROTOCOL_GAP_PATH
)

print(
    "\nValidation ranking:"
)

print(
    RANKING_PATH
)

print(
    "\nFrozen validation thresholds:"
)

print(
    f"{len(thresholds)}/24"
)

print(
    "\nLOCKED TEST REMAINS CLOSED."
)

print(
    "=" * 80
)

In [ ]:
# ============================================================
# STAGE 6F — FINAL VALIDATION DECISION GATE
#
# PURPOSE
# ------------------------------------------------------------
# Freeze all decisions BEFORE opening the locked test.
#
# Decisions:
# - Keep all 4 predefined models.
# - Keep all 3 predefined seeds.
# - Keep both predefined protocols.
# - Do NOT retrain based on validation outcomes.
# - Do NOT exclude RT-DETR-L because of seed variability.
# - Use validation-frozen confidence thresholds for
#   threshold-dependent test metrics.
# - Use low-confidence predictions for COCO AP evaluation.
#
# IMPORTANT:
# THIS CELL DOES NOT READ TEST IMAGES OR TEST ANNOTATIONS.
# ============================================================

from pathlib import Path
import json
import hashlib
import pandas as pd
from datetime import datetime, timezone


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}


# ============================================================
# 2. FILE HASH FUNCTION
# ============================================================

def sha256_file(path, chunk=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(chunk),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ============================================================
# 3. REQUIRED FROZEN FILES
# ============================================================

CONFIG_PATH = (
    DIRS["manifests"] /
    "training_config_frozen.json"
)

THRESHOLD_PATH = (
    DIRS["manifests"] /
    "validation_thresholds_frozen.json"
)

SPLIT_MANIFEST_PATH = (
    DIRS["manifests"] /
    "splits_frozen.json"
)

VAL_SUMMARY_PATH = (
    DIRS["validation"] /
    "validation_summary_all_runs.csv"
)

VAL_AGG_PATH = (
    DIRS["validation"] /
    "validation_aggregate_mean_sd.csv"
)

VAL_RANKING_PATH = (
    DIRS["validation"] /
    "validation_model_ranking.csv"
)


required = [
    CONFIG_PATH,
    THRESHOLD_PATH,
    SPLIT_MANIFEST_PATH,
    VAL_SUMMARY_PATH,
    VAL_AGG_PATH,
    VAL_RANKING_PATH,
]


for p in required:

    if not p.exists():

        raise FileNotFoundError(
            f"Required pre-test file missing:\n{p}"
        )


print("=" * 78)
print("STAGE 6F — FINAL VALIDATION DECISION GATE")
print("=" * 78)

print("\nRequired validation artifacts: PASSED")


# ============================================================
# 4. LOAD CONFIG
# ============================================================

CONFIG = json.loads(
    CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

thresholds = json.loads(
    THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)

val_summary = pd.read_csv(
    VAL_SUMMARY_PATH
)

val_agg = pd.read_csv(
    VAL_AGG_PATH
)


# ============================================================
# 5. STRUCTURAL CHECKS
# ============================================================

expected_protocols = [
    "random",
    "batch_disjoint"
]

expected_models = [
    "yolov8s",
    "yolo11s",
    "yolo26s",
    "rtdetr_l"
]

expected_seeds = [
    42,
    123,
    2026
]

expected_runs = 24


if CONFIG["protocols"] != expected_protocols:

    raise RuntimeError(
        f"Unexpected protocols: {CONFIG['protocols']}"
    )


if list(CONFIG["models"].keys()) != expected_models:

    raise RuntimeError(
        f"Unexpected models: {list(CONFIG['models'].keys())}"
    )


if list(CONFIG["seeds"]) != expected_seeds:

    raise RuntimeError(
        f"Unexpected seeds: {CONFIG['seeds']}"
    )


if len(val_summary) != expected_runs:

    raise RuntimeError(
        f"Expected {expected_runs} validation runs, "
        f"found {len(val_summary)}."
    )


if len(thresholds) != expected_runs:

    raise RuntimeError(
        f"Expected {expected_runs} frozen thresholds, "
        f"found {len(thresholds)}."
    )


print("Protocols              : 2/2")
print("Models                 : 4/4")
print("Seeds                  : 3/3")
print("Validation runs        : 24/24")
print("Frozen thresholds      : 24/24")


# ============================================================
# 6. VERIFY ALL 24 best.pt CHECKPOINTS
# ============================================================

checkpoint_rows = []


for protocol in expected_protocols:

    for model in expected_models:

        for seed in expected_seeds:

            pt = (
                DIRS["runs"] /
                protocol /
                f"{model}_seed{seed}" /
                "weights" /
                "best.pt"
            )


            if not pt.exists():

                raise FileNotFoundError(
                    f"Missing checkpoint:\n{pt}"
                )


            checkpoint_rows.append({
                "protocol": protocol,
                "model": model,
                "seed": seed,
                "checkpoint": str(pt),
                "size_bytes": int(
                    pt.stat().st_size
                ),
                "sha256": sha256_file(pt),
            })


checkpoint_df = pd.DataFrame(
    checkpoint_rows
)


if len(checkpoint_df) != 24:

    raise RuntimeError(
        "Checkpoint verification failed."
    )


CHECKPOINT_REGISTRY = (
    DIRS["manifests"] /
    "checkpoint_registry_pre_locked_test.csv"
)


checkpoint_df.to_csv(
    CHECKPOINT_REGISTRY,
    index=False
)


print("Checkpoints            : 24/24")
print("Checkpoint SHA-256     : generated")


# ============================================================
# 7. RECORD VALIDATION RESULTS USED FOR THE DECISION
# ============================================================

validation_snapshot = []


for r in val_agg.itertuples():

    validation_snapshot.append({

        "protocol":
            str(r.protocol),

        "model":
            str(r.model),

        "mAP50_95_mean":
            float(r.mAP50_95_mean),

        "mAP50_95_sd":
            float(r.mAP50_95_sd),

        "mAP50_95_min":
            float(r.mAP50_95_min),

        "mAP50_95_max":
            float(r.mAP50_95_max),

        "val_f1_mean":
            float(r.val_f1_mean),

        "val_f1_sd":
            float(r.val_f1_sd),
    })


# ============================================================
# 8. FORMAL PRE-TEST DECISION
# ============================================================

decision = {

    "stage":
        "pre_locked_test_validation_freeze",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "experiment_status":
        "validation_finalized_locked_test_not_opened",

    "protocols_retained":
        expected_protocols,

    "models_retained":
        expected_models,

    "seeds_retained":
        expected_seeds,

    "number_of_models":
        4,

    "number_of_seeds":
        3,

    "number_of_protocols":
        2,

    "total_test_evaluations_planned":
        24,

    "primary_metric":
        "COCO mAP@[0.50:0.95]",

    "secondary_coco_metrics": [
        "AP50",
        "AP75",
        "AP-small",
        "AP-medium",
        "AP-large",
        "AR1",
        "AR10",
        "AR100",
        "AR-small",
        "AR-medium",
        "AR-large"
    ],

    "threshold_dependent_metrics": [
        "precision",
        "recall",
        "F1",
        "TP",
        "FP",
        "FN"
    ],

    "confidence_threshold_policy":
        (
            "Each protocol/model/seed confidence threshold "
            "was selected exclusively on its validation split "
            "by maximum global F1 at IoU=0.50 and is frozen "
            "before locked-test evaluation."
        ),

    "coco_prediction_policy":
        (
            "Locked-test predictions for COCO AP metrics will "
            "be generated using the predefined low confidence "
            "floor (0.001), independent of the F1-selected "
            "operating threshold."
        ),

    "model_selection_policy":
        (
            "No model is excluded on the basis of validation "
            "performance. All four models defined before "
            "evaluation are retained for locked-test analysis."
        ),

    "rtdetr_decision":
        (
            "RT-DETR-L is retained despite higher seed "
            "variability. This variability is treated as an "
            "experimental result and is not used as a post-hoc "
            "exclusion criterion."
        ),

    "retraining_after_validation":
        False,

    "additional_seed_selection_after_validation":
        False,

    "hyperparameter_changes_after_validation":
        False,

    "threshold_changes_after_validation":
        False,

    "protocol_interpretation":
        (
            "Random and batch-disjoint validation sets contain "
            "different images. Differences between their "
            "validation means are descriptive and are not "
            "treated as paired causal effects."
        ),

    "validation_snapshot":
        validation_snapshot,

    "artifact_sha256": {

        "training_config_frozen.json":
            sha256_file(
                CONFIG_PATH
            ),

        "validation_thresholds_frozen.json":
            sha256_file(
                THRESHOLD_PATH
            ),

        "splits_frozen.json":
            sha256_file(
                SPLIT_MANIFEST_PATH
            ),

        "validation_summary_all_runs.csv":
            sha256_file(
                VAL_SUMMARY_PATH
            ),

        "validation_aggregate_mean_sd.csv":
            sha256_file(
                VAL_AGG_PATH
            ),

        "validation_model_ranking.csv":
            sha256_file(
                VAL_RANKING_PATH
            ),

        "checkpoint_registry_pre_locked_test.csv":
            sha256_file(
                CHECKPOINT_REGISTRY
            ),
    }
}


# ============================================================
# 9. WRITE IMMUTABLE DECISION MANIFEST
# ============================================================

DECISION_PATH = (
    DIRS["manifests"] /
    "PRE_LOCKED_TEST_DECISION.json"
)


serialized = json.dumps(
    decision,
    indent=2,
    ensure_ascii=False
)


if DECISION_PATH.exists():

    old = json.loads(
        DECISION_PATH.read_text(
            encoding="utf-8"
        )
    )

    # Ignore timestamp for equivalence comparison
    old_cmp = dict(old)
    new_cmp = dict(decision)

    old_cmp.pop(
        "created_utc",
        None
    )

    new_cmp.pop(
        "created_utc",
        None
    )


    if old_cmp != new_cmp:

        raise RuntimeError(
            "A different PRE_LOCKED_TEST_DECISION.json "
            "already exists.\n"
            "It will NOT be overwritten."
        )


    print(
        "\nExisting pre-test decision manifest "
        "matches current experiment."
    )

else:

    DECISION_PATH.write_text(
        serialized,
        encoding="utf-8"
    )


# ============================================================
# 10. SAVE HUMAN-READABLE VALIDATION GATE
# ============================================================

GATE_TXT = (
    DIRS["manifests"] /
    "PRE_LOCKED_TEST_GATE.txt"
)


gate_text = """
TACO PIPELINE v5 — PRE-LOCKED-TEST GATE

VALIDATION STATUS
-----------------
24/24 validation evaluations completed.
24/24 confidence thresholds frozen.
4/4 models retained.
3/3 seeds retained.
2/2 evaluation protocols retained.

MODEL POLICY
------------
YOLOv8s: retained
YOLO11s: retained
YOLO26s: retained
RT-DETR-L: retained

RT-DETR-L shows greater between-seed variability,
but no post-hoc exclusion or retraining is permitted.

NO changes to:
- architecture selection
- hyperparameters
- seeds
- validation confidence thresholds
- data splits

PRIMARY LOCKED-TEST METRIC
--------------------------
COCO mAP@[0.50:0.95]

OPERATING-POINT METRICS
-----------------------
Precision, Recall, F1, TP, FP, and FN will use
the previously frozen validation-derived thresholds.

LOCKED TEST
-----------
Not opened at the time this decision was frozen.
"""


GATE_TXT.write_text(
    gate_text.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# 11. IMPORTANT — DO NOT CREATE TEST-OPENED MARKER HERE
# ============================================================

LOCKED_TEST_OPENED = (
    DIRS["manifests"] /
    "LOCKED_TEST_OPENED.json"
)


if LOCKED_TEST_OPENED.exists():

    raise RuntimeError(
        "LOCKED_TEST_OPENED.json already exists. "
        "The locked test may previously have been opened."
    )


# ============================================================
# 12. DISPLAY VALIDATION SNAPSHOT
# ============================================================

snapshot_df = pd.DataFrame(
    validation_snapshot
)

print(
    "\n" + "=" * 78
)

print(
    "FINAL PRE-TEST VALIDATION SNAPSHOT"
)

print(
    "=" * 78
)

display(
    snapshot_df[
        [
            "protocol",
            "model",
            "mAP50_95_mean",
            "mAP50_95_sd",
            "val_f1_mean",
            "val_f1_sd"
        ]
    ]
)


# ============================================================
# 13. FINAL
# ============================================================

print(
    "\n" + "=" * 78
)

print(
    "STAGE 6F PASSED"
)

print(
    "=" * 78
)

print(
    "Validation decisions frozen : YES"
)

print(
    "Models retained             : 4/4"
)

print(
    "Seeds retained              : 3/3"
)

print(
    "Checkpoints registered      : 24/24"
)

print(
    "Validation thresholds       : 24/24"
)

print(
    "Retraining permitted        : NO"
)

print(
    "Post-hoc model exclusion    : NO"
)

print(
    "Locked test opened          : NO"
)

print(
    "\nDecision manifest:"
)

print(
    DECISION_PATH
)

print(
    "\nCheckpoint registry:"
)

print(
    CHECKPOINT_REGISTRY
)

print(
    "\nPRE-TEST FREEZE COMPLETE."
)

print(
    "NEXT STAGE MAY OPEN THE LOCKED TEST."
)

print(
    "=" * 78
)

# STAGE 7 — LOCKED TEST

Do not run this stage until:

1. all 24 runs have completed,
2. the model set is final,
3. hyperparameters are final,
4. validation thresholds are frozen,
5. no further tuning decision will be made.

When ready, change:

`UNLOCK_LOCKED_TEST = False`

to

`UNLOCK_LOCKED_TEST = True`

and run **once**.

In [ ]:
# ============================================================
# STAGE 7A — OPEN LOCKED TEST + ONE-SHOT TEST EVALUATION
#
# SCIENTIFIC POLICY
# ------------------------------------------------------------
# - Validation is finalized.
# - All 4 models are retained.
# - All 3 seeds are retained.
# - Both protocols are retained.
# - NO threshold tuning on test.
# - NO model selection on test.
# - NO retraining after test access.
#
# COCO metrics:
#   predictions generated at conf=0.001
#
# Operating-point metrics:
#   use ONLY the validation-frozen confidence threshold
#   at IoU = 0.50
#
# Supports safe resume if runtime is interrupted.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import os
import gc
import json
import hashlib

import numpy as np
import pandas as pd
import torch


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

LOCAL_DATASET = Path(
    "/content/taco_v5/dataset"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 2. HASH FUNCTION
# ============================================================

def sha256_file(
    path,
    chunk=1024 * 1024
):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(chunk),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ============================================================
# 3. REQUIRED PRE-TEST ARTIFACTS
# ============================================================

CONFIG_PATH = (
    DIRS["manifests"] /
    "training_config_frozen.json"
)

THRESHOLD_PATH = (
    DIRS["manifests"] /
    "validation_thresholds_frozen.json"
)

SPLITS_FROZEN_PATH = (
    DIRS["manifests"] /
    "splits_frozen.json"
)

DECISION_PATH = (
    DIRS["manifests"] /
    "PRE_LOCKED_TEST_DECISION.json"
)

CHECKPOINT_REGISTRY_PATH = (
    DIRS["manifests"] /
    "checkpoint_registry_pre_locked_test.csv"
)


required_files = [
    CONFIG_PATH,
    THRESHOLD_PATH,
    SPLITS_FROZEN_PATH,
    DECISION_PATH,
    CHECKPOINT_REGISTRY_PATH,
]


for p in required_files:

    if not p.exists():

        raise FileNotFoundError(
            f"Required pre-test artifact missing:\n{p}"
        )


print("=" * 80)
print("STAGE 7A — LOCKED TEST EVALUATION")
print("=" * 80)

print(
    "\nPre-test artifacts found: PASSED"
)


# ============================================================
# 4. HELPER FUNCTION CHECK
# ============================================================

# These must come from STAGE 6C + the image-ID patch.
required_helpers = [
    "load_best_model",
    "predict_coco",
    "coco_metrics",
    "detection_counts",
]


missing_helpers = [
    name
    for name in required_helpers
    if name not in globals()
]


if missing_helpers:

    raise RuntimeError(
        "Required Stage 6C evaluation helpers are not loaded:\n"
        + ", ".join(missing_helpers)
        + "\n\n"
        "Run STAGE 6C and the STAGE 6C PATCH first. "
        "Do NOT run training again."
    )


print(
    "Evaluation helpers       : PASSED"
)


# ============================================================
# 5. LOAD FROZEN EXPERIMENT STATE
# ============================================================

CONFIG = json.loads(
    CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

thresholds = json.loads(
    THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)

decision = json.loads(
    DECISION_PATH.read_text(
        encoding="utf-8"
    )
)

splits_frozen = json.loads(
    SPLITS_FROZEN_PATH.read_text(
        encoding="utf-8"
    )
)


protocols = [
    "random",
    "batch_disjoint"
]

models = [
    "yolov8s",
    "yolo11s",
    "yolo26s",
    "rtdetr_l"
]

seeds = [
    42,
    123,
    2026
]

expected_runs = 24


# ============================================================
# 6. VERIFY FROZEN CONFIGURATION
# ============================================================

if CONFIG["protocols"] != protocols:

    raise RuntimeError(
        "Frozen protocol configuration changed."
    )


if list(
    CONFIG["models"].keys()
) != models:

    raise RuntimeError(
        "Frozen model configuration changed."
    )


if list(
    CONFIG["seeds"]
) != seeds:

    raise RuntimeError(
        "Frozen seed configuration changed."
    )


if len(thresholds) != 24:

    raise RuntimeError(
        f"Expected 24 frozen thresholds; "
        f"found {len(thresholds)}."
    )


print(
    "Frozen experiment design : PASSED"
)


# ============================================================
# 7. VERIFY PRE-TEST ARTIFACT HASHES
# ============================================================

artifact_hashes = decision.get(
    "artifact_sha256",
    {}
)


critical_hash_checks = {

    "training_config_frozen.json":
        CONFIG_PATH,

    "validation_thresholds_frozen.json":
        THRESHOLD_PATH,

    "splits_frozen.json":
        SPLITS_FROZEN_PATH,
}


for key, path in critical_hash_checks.items():

    expected_hash = artifact_hashes.get(
        key
    )

    if expected_hash is None:

        raise RuntimeError(
            f"Pre-test decision does not contain "
            f"SHA-256 for {key}."
        )


    current_hash = sha256_file(
        path
    )


    if current_hash != expected_hash:

        raise RuntimeError(
            f"PRE-TEST ARTIFACT CHANGED:\n"
            f"{key}\n"
            f"Expected: {expected_hash}\n"
            f"Current : {current_hash}\n\n"
            "Do not open the locked test."
        )


print(
    "Frozen artifact SHA-256   : PASSED"
)


# ============================================================
# 8. VERIFY ALL CHECKPOINTS AGAINST PRE-TEST REGISTRY
# ============================================================

checkpoint_registry = pd.read_csv(
    CHECKPOINT_REGISTRY_PATH
)


if len(checkpoint_registry) != 24:

    raise RuntimeError(
        "Pre-test checkpoint registry is incomplete."
    )


for r in checkpoint_registry.itertuples():

    pt = Path(
        r.checkpoint
    )


    if not pt.exists():

        raise FileNotFoundError(
            f"Checkpoint missing:\n{pt}"
        )


    current_hash = sha256_file(
        pt
    )


    if current_hash != str(
        r.sha256
    ):

        raise RuntimeError(
            "Checkpoint SHA-256 changed after "
            "validation freeze:\n"
            f"{pt}"
        )


print(
    "Checkpoint integrity      : 24/24 PASSED"
)


# ============================================================
# 9. VERIFY LOCAL DATASET EXISTS
# ============================================================

if not LOCAL_DATASET.exists():

    raise FileNotFoundError(
        f"Recovered local dataset missing:\n"
        f"{LOCAL_DATASET}"
    )


for protocol in protocols:

    pdir = (
        LOCAL_DATASET /
        protocol
    )

    for filename in [
        "test.txt",
        "test_coco_taco1.json"
    ]:

        p = (
            pdir /
            filename
        )

        if not p.exists():

            raise FileNotFoundError(
                f"Locked test file missing:\n{p}"
            )


print(
    "Local locked-test files   : PRESENT"
)


# ============================================================
# 10. OPEN LOCKED TEST
#
# Marker is written BEFORE evaluating or inspecting test
# performance.
#
# If runtime later crashes, this marker remains.
# Resuming the same frozen evaluation is permitted.
# ============================================================

OPEN_MARKER = (
    DIRS["manifests"] /
    "LOCKED_TEST_OPENED.json"
)

decision_sha256 = sha256_file(
    DECISION_PATH
)


if OPEN_MARKER.exists():

    opened = json.loads(
        OPEN_MARKER.read_text(
            encoding="utf-8"
        )
    )


    if (
        opened.get(
            "pre_locked_test_decision_sha256"
        )
        !=
        decision_sha256
    ):

        raise RuntimeError(
            "Locked test was previously opened under a "
            "different pre-test decision state."
        )


    print(
        "\nLOCKED TEST WAS PREVIOUSLY OPENED."
    )

    print(
        "Frozen state matches. Safe resume is permitted."
    )


else:

    opened = {

        "status":
            "LOCKED_TEST_OPENED",

        "opened_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "pre_locked_test_decision":
            str(
                DECISION_PATH
            ),

        "pre_locked_test_decision_sha256":
            decision_sha256,

        "policy":
            (
                "Locked test opened only after model, seed, "
                "protocol and validation-threshold decisions "
                "were frozen."
            ),

        "post_test_tuning_permitted":
            False,

        "retraining_permitted":
            False,

        "threshold_reselection_permitted":
            False,
    }


    OPEN_MARKER.write_text(
        json.dumps(
            opened,
            indent=2
        ),
        encoding="utf-8"
    )


    print(
        "\nLOCKED TEST STATUS CHANGED:"
    )

    print(
        "CLOSED -> OPENED"
    )


print(
    "Marker:",
    OPEN_MARKER
)


# ============================================================
# 11. VERIFY FROZEN TEST FILE HASHES
#
# Locked test is now formally opened.
# ============================================================

print(
    "\nVerifying recovered locked-test split hashes..."
)


for protocol in protocols:

    pdir = (
        LOCAL_DATASET /
        protocol
    )


    current_test_txt = (
        pdir /
        "test.txt"
    )

    current_test_gt = (
        pdir /
        "test_coco_taco1.json"
    )


    old = (
        splits_frozen[
            "protocols"
        ][
            protocol
        ]
    )


    current_txt_hash = sha256_file(
        current_test_txt
    )

    current_gt_hash = sha256_file(
        current_test_gt
    )


    if (
        current_txt_hash
        !=
        old["test_txt_sha256"]
    ):

        raise RuntimeError(
            f"{protocol}: locked test.txt "
            "does not match frozen experiment."
        )


    if (
        current_gt_hash
        !=
        old["test_gt_sha256"]
    ):

        raise RuntimeError(
            f"{protocol}: locked test COCO GT "
            "does not match frozen experiment."
        )


    print(
        f"{protocol:16s}: "
        f"test.txt + test GT SHA-256 PASSED"
    )


# ============================================================
# 12. VERIFY TEST IMAGE COUNTS
# ============================================================

test_info = {}


for protocol in protocols:

    pdir = (
        LOCAL_DATASET /
        protocol
    )

    test_paths = [

        x.strip()

        for x in (
            pdir /
            "test.txt"
        ).read_text(
            encoding="utf-8"
        ).splitlines()

        if x.strip()
    ]


    gt = json.loads(
        (
            pdir /
            "test_coco_taco1.json"
        ).read_text(
            encoding="utf-8"
        )
    )


    if (
        len(test_paths)
        !=
        len(gt["images"])
    ):

        raise RuntimeError(
            f"{protocol}: test image count mismatch."
        )


    missing_images = [
        p
        for p in test_paths
        if not Path(p).exists()
    ]


    if missing_images:

        raise FileNotFoundError(
            f"{protocol}: "
            f"{len(missing_images)} test images missing."
        )


    test_info[
        protocol
    ] = {

        "images":
            len(test_paths),

        "annotations":
            len(
                gt["annotations"]
            ),
    }


    print(
        f"{protocol:16s}: "
        f"{len(test_paths)} test images | "
        f"{len(gt['annotations'])} objects"
    )


# Expected image counts from frozen splits
if test_info["random"]["images"] != 225:

    raise RuntimeError(
        "Random test image count is not 225."
    )


if (
    test_info[
        "batch_disjoint"
    ][
        "images"
    ]
    != 200
):

    raise RuntimeError(
        "Batch-disjoint test image count is not 200."
    )


print(
    "\nLOCKED TEST SPLIT INTEGRITY: PASSED"
)


# ============================================================
# 13. SAFE FILE WRITERS
# ============================================================

def atomic_write_json(
    obj,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )


    tmp.write_text(
        json.dumps(
            obj,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )


    os.replace(
        tmp,
        path
    )


def atomic_write_csv(
    df,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )


    df.to_csv(
        tmp,
        index=False
    )


    os.replace(
        tmp,
        path
    )


# ============================================================
# 14. LOCKED TEST EVALUATION
# ============================================================

summary_rows = []

run_counter = 0


for protocol in protocols:

    pdir = (
        LOCAL_DATASET /
        protocol
    )


    test_paths = [

        x.strip()

        for x in (
            pdir /
            "test.txt"
        ).read_text(
            encoding="utf-8"
        ).splitlines()

        if x.strip()
    ]


    gt_json = (
        pdir /
        "test_coco_taco1.json"
    )


    gt_obj = json.loads(
        gt_json.read_text(
            encoding="utf-8"
        )
    )


    valid_test_ids = {
        int(im["id"])
        for im in gt_obj["images"]
    }


    for model_key in models:

        for seed in seeds:

            run_counter += 1


            run_label = (
                f"{protocol}/"
                f"{model_key}_seed{seed}"
            )


            print(
                "\n"
                + "=" * 80
            )

            print(
                f"[{run_counter}/{expected_runs}] "
                f"LOCKED TEST: {run_label}"
            )

            print(
                "=" * 80
            )


            # =================================================
            # CHECKPOINT
            # =================================================

            best_pt = (
                DIRS["runs"] /
                protocol /
                f"{model_key}_seed{seed}" /
                "weights" /
                "best.pt"
            )


            if not best_pt.exists():

                raise FileNotFoundError(
                    best_pt
                )


            # =================================================
            # FROZEN VALIDATION THRESHOLD
            # =================================================

            threshold_key = (
                f"{protocol}/"
                f"{model_key}/"
                f"seed{seed}"
            )


            if threshold_key not in thresholds:

                raise KeyError(
                    f"Frozen validation threshold "
                    f"missing: {threshold_key}"
                )


            frozen_threshold = float(
                thresholds[
                    threshold_key
                ]
            )


            print(
                "Frozen validation threshold:",
                frozen_threshold
            )


            # =================================================
            # OUTPUT PATHS
            # =================================================

            out = (
                DIRS["test"] /
                protocol /
                f"{model_key}_seed{seed}"
            )


            out.mkdir(
                parents=True,
                exist_ok=True
            )


            pred_path = (
                out /
                "test_predictions_coco.json"
            )

            speed_path = (
                out /
                "test_speed.csv"
            )

            per_image_path = (
                out /
                "test_per_image_metrics.csv"
            )

            counts_path = (
                out /
                "test_operating_point_counts.json"
            )

            run_summary_path = (
                out /
                "test_run_summary.json"
            )


            # =================================================
            # PREDICTION
            #
            # Reuse completed raw prediction artifact on resume.
            # Never tune threshold here.
            # =================================================

            model_obj = None


            if (
                pred_path.exists()
                and
                speed_path.exists()
            ):

                print(
                    "Existing completed test predictions "
                    "found; reusing for safe resume."
                )


                preds = json.loads(
                    pred_path.read_text(
                        encoding="utf-8"
                    )
                )


                speed = pd.read_csv(
                    speed_path
                )


            elif (
                pred_path.exists()
                !=
                speed_path.exists()
            ):

                raise RuntimeError(
                    "Partial test prediction artifact detected:\n"
                    f"{pred_path}\n"
                    f"{speed_path}\n\n"
                    "Do not silently rerun. "
                    "Inspect this run first."
                )


            else:

                print(
                    "Generating ONE-SHOT locked-test predictions..."
                )


                model_obj = load_best_model(
                    model_key,
                    best_pt
                )


                preds, speed = predict_coco(
                    model_obj,
                    test_paths,
                    gt_json,
                    imgsz=CONFIG["imgsz"],
                    conf=0.001,
                    max_det=300
                )


                # ---------------------------------------------
                # Prediction sanity check before persistence
                # ---------------------------------------------

                prediction_ids = {
                    int(x["image_id"])
                    for x in preds
                }


                invalid_ids = (
                    prediction_ids
                    -
                    valid_test_ids
                )


                if invalid_ids:

                    raise RuntimeError(
                        f"{run_label}: predictions contain "
                        f"invalid locked-test image IDs:\n"
                        f"{sorted(invalid_ids)}"
                    )


                if len(speed) != len(
                    test_paths
                ):

                    raise RuntimeError(
                        f"{run_label}: speed table does not "
                        "contain one row per test image."
                    )


                # Safe persistence
                atomic_write_json(
                    preds,
                    pred_path
                )

                atomic_write_csv(
                    speed,
                    speed_path
                )


            # =================================================
            # VERIFY SAVED PREDICTIONS
            # =================================================

            preds = json.loads(
                pred_path.read_text(
                    encoding="utf-8"
                )
            )


            prediction_ids = {
                int(p["image_id"])
                for p in preds
            }


            invalid_ids = (
                prediction_ids
                -
                valid_test_ids
            )


            if invalid_ids:

                raise RuntimeError(
                    f"{run_label}: saved predictions "
                    "contain invalid image IDs."
                )


            print(
                "Raw detections:",
                len(preds)
            )


            # =================================================
            # COCO METRICS
            #
            # Threshold-independent ranking evaluation
            # from low confidence floor predictions.
            # =================================================

            print(
                "\nCOCO locked-test evaluation:"
            )


            metrics = coco_metrics(
                gt_json,
                preds
            )


            # =================================================
            # OPERATING-POINT METRICS
            #
            # IMPORTANT:
            # threshold comes exclusively from validation.
            # No test optimization.
            # =================================================

            counts, per_img = detection_counts(
                gt_json,
                preds,
                frozen_threshold,
                0.50
            )


            atomic_write_csv(
                per_img,
                per_image_path
            )


            counts_payload = {

                "protocol":
                    protocol,

                "model":
                    model_key,

                "seed":
                    int(seed),

                "threshold":
                    frozen_threshold,

                "threshold_source":
                    "validation",

                "threshold_selection_rule":
                    "maximum global F1 on validation at IoU=0.50",

                "matching_iou":
                    0.50,

                "TP":
                    int(
                        counts.get(
                            "tp",
                            0
                        )
                    ),

                "FP":
                    int(
                        counts.get(
                            "fp",
                            0
                        )
                    ),

                "FN":
                    int(
                        counts.get(
                            "fn",
                            0
                        )
                    ),

                "precision":
                    float(
                        counts[
                            "precision"
                        ]
                    ),

                "recall":
                    float(
                        counts[
                            "recall"
                        ]
                    ),

                "f1":
                    float(
                        counts[
                            "f1"
                        ]
                    ),
            }


            atomic_write_json(
                counts_payload,
                counts_path
            )


            # =================================================
            # SPEED SUMMARY
            # =================================================

            speed = pd.read_csv(
                speed_path
            )


            inference_ms_mean = float(
                speed[
                    "inference_ms"
                ].mean()
            )


            preprocess_ms_mean = float(
                speed[
                    "preprocess_ms"
                ].mean()
            )


            postprocess_ms_mean = float(
                speed[
                    "postprocess_ms"
                ].mean()
            )


            total_ms_mean = float(
                (
                    speed[
                        "preprocess_ms"
                    ]
                    +
                    speed[
                        "inference_ms"
                    ]
                    +
                    speed[
                        "postprocess_ms"
                    ]
                ).mean()
            )


            fps_from_processing = (
                1000.0 /
                total_ms_mean
                if total_ms_mean > 0
                else np.nan
            )


            # =================================================
            # RUN SUMMARY
            # =================================================

            row = {

                "protocol":
                    protocol,

                "model":
                    model_key,

                "seed":
                    int(seed),

                **metrics,

                "frozen_threshold":
                    frozen_threshold,

                "test_precision":
                    float(
                        counts[
                            "precision"
                        ]
                    ),

                "test_recall":
                    float(
                        counts[
                            "recall"
                        ]
                    ),

                "test_f1":
                    float(
                        counts[
                            "f1"
                        ]
                    ),

                "TP":
                    int(
                        counts.get(
                            "tp",
                            0
                        )
                    ),

                "FP":
                    int(
                        counts.get(
                            "fp",
                            0
                        )
                    ),

                "FN":
                    int(
                        counts.get(
                            "fn",
                            0
                        )
                    ),

                "preprocess_ms_mean":
                    preprocess_ms_mean,

                "inference_ms_mean":
                    inference_ms_mean,

                "postprocess_ms_mean":
                    postprocess_ms_mean,

                "processing_ms_mean":
                    total_ms_mean,

                "processing_fps":
                    fps_from_processing,

                "n_test_images":
                    len(
                        test_paths
                    ),

                "n_test_gt_objects":
                    len(
                        gt_obj[
                            "annotations"
                        ]
                    ),

                "n_raw_predictions":
                    len(
                        preds
                    ),

                "checkpoint_sha256":
                    sha256_file(
                        best_pt
                    ),

                "predictions_sha256":
                    sha256_file(
                        pred_path
                    ),
            }


            summary_rows.append(
                row
            )


            atomic_write_json(
                row,
                run_summary_path
            )


            # =================================================
            # PRINT RUN RESULT
            # =================================================

            print(
                "\nTEST RESULT"
            )

            print(
                "mAP50-95 :",
                f"{metrics['mAP50_95']:.4f}"
            )

            print(
                "AP50      :",
                f"{metrics['AP50']:.4f}"
            )

            print(
                "AP75      :",
                f"{metrics['AP75']:.4f}"
            )

            print(
                "AP-small  :",
                f"{metrics['AP_small']:.4f}"
            )

            print(
                "AP-medium :",
                f"{metrics['AP_medium']:.4f}"
            )

            print(
                "AP-large  :",
                f"{metrics['AP_large']:.4f}"
            )

            print(
                "Threshold :",
                f"{frozen_threshold:.2f}",
                "(FROZEN FROM VALIDATION)"
            )

            print(
                "Precision :",
                f"{counts['precision']:.4f}"
            )

            print(
                "Recall    :",
                f"{counts['recall']:.4f}"
            )

            print(
                "F1        :",
                f"{counts['f1']:.4f}"
            )

            print(
                "TP / FP / FN:",
                counts.get(
                    "tp",
                    "?"
                ),
                "/",
                counts.get(
                    "fp",
                    "?"
                ),
                "/",
                counts.get(
                    "fn",
                    "?"
                )
            )


            # =================================================
            # MEMORY CLEANUP
            # =================================================

            if model_obj is not None:
                del model_obj

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


# ============================================================
# 15. COMPLETE SUMMARY
# ============================================================

test_summary = pd.DataFrame(
    summary_rows
)


if len(test_summary) != expected_runs:

    raise RuntimeError(
        f"Expected {expected_runs} locked-test results; "
        f"found {len(test_summary)}."
    )


n_unique = (
    test_summary[
        [
            "protocol",
            "model",
            "seed"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


if n_unique != expected_runs:

    raise RuntimeError(
        "Duplicate test protocol/model/seed "
        "combinations detected."
    )


TEST_SUMMARY_PATH = (
    DIRS["test"] /
    "locked_test_summary_all_runs.csv"
)


atomic_write_csv(
    test_summary,
    TEST_SUMMARY_PATH
)


# ============================================================
# 16. CHECK THAT FROZEN THRESHOLDS WERE USED EXACTLY
# ============================================================

for r in test_summary.itertuples():

    key = (
        f"{r.protocol}/"
        f"{r.model}/"
        f"seed{int(r.seed)}"
    )


    expected_thr = float(
        thresholds[
            key
        ]
    )


    if not np.isclose(
        float(
            r.frozen_threshold
        ),
        expected_thr,
        rtol=0,
        atol=1e-12
    ):

        raise RuntimeError(
            f"Threshold mismatch in test output: {key}"
        )


print(
    "\nFrozen threshold usage: 24/24 PASSED"
)


# ============================================================
# 17. CREATE LOCKED-TEST COMPLETION MANIFEST
# ============================================================

COMPLETE_MARKER = (
    DIRS["manifests"] /
    "LOCKED_TEST_COMPLETED.json"
)


completion = {

    "status":
        "LOCKED_TEST_COMPLETED",

    "completed_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "number_of_test_evaluations":
        24,

    "protocols":
        protocols,

    "models":
        models,

    "seeds":
        seeds,

    "primary_metric":
        "COCO mAP@[0.50:0.95]",

    "threshold_policy":
        (
            "All operating-point metrics used "
            "validation-frozen confidence thresholds. "
            "No test threshold optimization was performed."
        ),

    "post_test_model_selection":
        False,

    "post_test_retraining":
        False,

    "test_summary":
        str(
            TEST_SUMMARY_PATH
        ),

    "test_summary_sha256":
        sha256_file(
            TEST_SUMMARY_PATH
        ),

    "pre_locked_test_decision_sha256":
        decision_sha256,

    "threshold_manifest_sha256":
        sha256_file(
            THRESHOLD_PATH
        ),
}


if COMPLETE_MARKER.exists():

    previous = json.loads(
        COMPLETE_MARKER.read_text(
            encoding="utf-8"
        )
    )


    # Do not silently replace an already completed
    # locked-test record with a different result.
    if (
        previous.get(
            "test_summary_sha256"
        )
        !=
        completion[
            "test_summary_sha256"
        ]
    ):

        raise RuntimeError(
            "A different LOCKED_TEST_COMPLETED.json "
            "already exists. Do not overwrite it."
        )


else:

    atomic_write_json(
        completion,
        COMPLETE_MARKER
    )


# ============================================================
# 18. DISPLAY ALL 24 LOCKED-TEST RESULTS
# ============================================================

display_columns = [

    "protocol",
    "model",
    "seed",

    "mAP50_95",
    "AP50",
    "AP75",

    "AP_small",
    "AP_medium",
    "AP_large",

    "frozen_threshold",

    "test_precision",
    "test_recall",
    "test_f1",

    "TP",
    "FP",
    "FN",

    "inference_ms_mean",
]


print(
    "\n" + "=" * 80
)

print(
    "LOCKED TEST — ALL 24 RUNS"
)

print(
    "=" * 80
)


display(
    test_summary[
        display_columns
    ]
)


# ============================================================
# 19. FINAL
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "STAGE 7A PASSED"
)

print(
    "=" * 80
)

print(
    "Locked test opened       : YES"
)

print(
    "Locked test completed    : YES"
)

print(
    "Test evaluations         : 24/24"
)

print(
    "Frozen thresholds used   : 24/24"
)

print(
    "Test threshold tuning    : NO"
)

print(
    "Post-test retraining     : NO"
)

print(
    "\nTest summary:"
)

print(
    TEST_SUMMARY_PATH
)

print(
    "\nCompletion manifest:"
)

print(
    COMPLETE_MARKER
)

print(
    "\nLOCKED TEST EVALUATION COMPLETE."
)

print(
    "=" * 80
)

In [ ]:
# ============================================================
# STAGE 7B — FINAL LOCKED-TEST AGGREGATION
#             + GENERALIZATION
#             + OBJECT-SIZE
#             + SEED-STABILITY
#             + EFFICIENCY SUMMARY
#
# IMPORTANT
# ------------------------------------------------------------
# - Reads ONLY already-completed locked-test outputs.
# - Does NOT run inference.
# - Does NOT change thresholds.
# - Does NOT select/retrain models.
# - Does NOT optimize anything on the test set.
#
# Main reporting:
#   3-seed mean ± SD
#
# Protocol comparisons:
#   DESCRIPTIVE ONLY.
#   Random and batch-disjoint protocols use different
#   train/validation/test compositions.
#
# Statistical comparisons / bootstrap:
#   NOT performed here.
#   Planned for STAGE 7C.
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from datetime import datetime, timezone


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file(path, chunk=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(chunk),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def mean_sd_string(mean, sd, digits=4):

    return (
        f"{float(mean):.{digits}f} "
        f"± {float(sd):.{digits}f}"
    )


# ============================================================
# 3. REQUIRED FILES
# ============================================================

TEST_SUMMARY_PATH = (
    DIRS["test"] /
    "locked_test_summary_all_runs.csv"
)

TEST_COMPLETE_PATH = (
    DIRS["manifests"] /
    "LOCKED_TEST_COMPLETED.json"
)

TEST_OPEN_PATH = (
    DIRS["manifests"] /
    "LOCKED_TEST_OPENED.json"
)

THRESHOLD_PATH = (
    DIRS["manifests"] /
    "validation_thresholds_frozen.json"
)

VALIDATION_SUMMARY_PATH = (
    DIRS["validation"] /
    "validation_summary_all_runs.csv"
)

VALIDATION_AGG_PATH = (
    DIRS["validation"] /
    "validation_aggregate_mean_sd.csv"
)


required_files = [
    TEST_SUMMARY_PATH,
    TEST_COMPLETE_PATH,
    TEST_OPEN_PATH,
    THRESHOLD_PATH,
    VALIDATION_SUMMARY_PATH,
    VALIDATION_AGG_PATH,
]


for p in required_files:

    if not p.exists():

        raise FileNotFoundError(
            f"Required file missing:\n{p}"
        )


print("=" * 84)
print("STAGE 7B — FINAL LOCKED-TEST AGGREGATION")
print("=" * 84)

print(
    "\nRequired locked-test artifacts: PASSED"
)


# ============================================================
# 4. LOAD RESULTS
# ============================================================

test_df = pd.read_csv(
    TEST_SUMMARY_PATH
)

val_df = pd.read_csv(
    VALIDATION_SUMMARY_PATH
)

val_agg = pd.read_csv(
    VALIDATION_AGG_PATH
)

completion = json.loads(
    TEST_COMPLETE_PATH.read_text(
        encoding="utf-8"
    )
)

thresholds = json.loads(
    THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# 5. VERIFY LOCKED-TEST COMPLETION HASH
# ============================================================

expected_test_hash = completion.get(
    "test_summary_sha256"
)

current_test_hash = sha256_file(
    TEST_SUMMARY_PATH
)


if expected_test_hash is None:

    raise RuntimeError(
        "LOCKED_TEST_COMPLETED.json does not contain "
        "the test summary SHA-256."
    )


if current_test_hash != expected_test_hash:

    raise RuntimeError(
        "locked_test_summary_all_runs.csv changed after "
        "locked-test completion.\n"
        "Do not continue."
    )


print(
    "Locked-test summary SHA-256: PASSED"
)


# ============================================================
# 6. STRUCTURAL CHECKS
# ============================================================

protocols = [
    "random",
    "batch_disjoint"
]

models = [
    "yolov8s",
    "yolo11s",
    "yolo26s",
    "rtdetr_l"
]

seeds = [
    42,
    123,
    2026
]

expected_runs = 24


required_test_columns = [

    "protocol",
    "model",
    "seed",

    "mAP50_95",
    "AP50",
    "AP75",

    "AP_small",
    "AP_medium",
    "AP_large",

    "AR1",
    "AR10",
    "AR100",

    "AR_small",
    "AR_medium",
    "AR_large",

    "frozen_threshold",

    "test_precision",
    "test_recall",
    "test_f1",

    "TP",
    "FP",
    "FN",

    "preprocess_ms_mean",
    "inference_ms_mean",
    "postprocess_ms_mean",
    "processing_ms_mean",
    "processing_fps",

    "n_test_images",
    "n_test_gt_objects",
]


missing_columns = [

    c
    for c in required_test_columns
    if c not in test_df.columns
]


if missing_columns:

    raise RuntimeError(
        "Locked-test summary is missing columns:\n"
        + ", ".join(missing_columns)
    )


if len(test_df) != expected_runs:

    raise RuntimeError(
        f"Expected 24 locked-test rows, "
        f"found {len(test_df)}."
    )


n_unique = (
    test_df[
        [
            "protocol",
            "model",
            "seed"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


if n_unique != expected_runs:

    raise RuntimeError(
        "Duplicate protocol/model/seed combinations "
        "found in locked-test summary."
    )


for protocol in protocols:

    for model in models:

        subset = test_df[
            (test_df["protocol"] == protocol)
            &
            (test_df["model"] == model)
        ]

        found_seeds = set(
            subset["seed"].astype(int)
        )

        if found_seeds != set(seeds):

            raise RuntimeError(
                f"{protocol}/{model}: "
                f"unexpected seeds {sorted(found_seeds)}"
            )


print(
    "Locked-test rows           : 24/24"
)

print(
    "Unique model/protocol/seed : 24/24"
)

print(
    "Seed structure             : PASSED"
)


# ============================================================
# 7. VERIFY THAT FROZEN VALIDATION THRESHOLDS WERE USED
# ============================================================

threshold_errors = []


for r in test_df.itertuples():

    key = (
        f"{r.protocol}/"
        f"{r.model}/"
        f"seed{int(r.seed)}"
    )


    if key not in thresholds:

        threshold_errors.append(
            f"Missing threshold: {key}"
        )

        continue


    expected_thr = float(
        thresholds[key]
    )

    actual_thr = float(
        r.frozen_threshold
    )


    if not np.isclose(
        actual_thr,
        expected_thr,
        rtol=0,
        atol=1e-12
    ):

        threshold_errors.append(
            f"{key}: "
            f"expected={expected_thr}, "
            f"used={actual_thr}"
        )


if threshold_errors:

    raise RuntimeError(
        "Frozen threshold verification failed:\n"
        + "\n".join(
            threshold_errors
        )
    )


print(
    "Frozen thresholds used     : 24/24 PASSED"
)


# ============================================================
# 8. TEST SET SIZE CONSISTENCY
# ============================================================

expected_images = {
    "random": 225,
    "batch_disjoint": 200,
}

expected_objects = {
    "random": 826,
    "batch_disjoint": 707,
}


for protocol in protocols:

    p = test_df[
        test_df["protocol"] == protocol
    ]

    image_counts = set(
        p["n_test_images"].astype(int)
    )

    object_counts = set(
        p["n_test_gt_objects"].astype(int)
    )


    if image_counts != {
        expected_images[protocol]
    }:

        raise RuntimeError(
            f"{protocol}: unexpected test image counts "
            f"{image_counts}"
        )


    if object_counts != {
        expected_objects[protocol]
    }:

        raise RuntimeError(
            f"{protocol}: unexpected GT object counts "
            f"{object_counts}"
        )


print(
    "Locked-test sample counts  : PASSED"
)


# ============================================================
# 9. METRICS TO AGGREGATE
# ============================================================

metrics = [

    "mAP50_95",
    "AP50",
    "AP75",

    "AP_small",
    "AP_medium",
    "AP_large",

    "AR1",
    "AR10",
    "AR100",

    "AR_small",
    "AR_medium",
    "AR_large",

    "test_precision",
    "test_recall",
    "test_f1",

    "TP",
    "FP",
    "FN",

    "preprocess_ms_mean",
    "inference_ms_mean",
    "postprocess_ms_mean",
    "processing_ms_mean",
    "processing_fps",
]


# ============================================================
# 10. THREE-SEED MEAN ± SD
# ============================================================

aggregate_rows = []


for protocol in protocols:

    for model in models:

        g = test_df[
            (test_df["protocol"] == protocol)
            &
            (test_df["model"] == model)
        ].copy()


        if len(g) != 3:

            raise RuntimeError(
                f"{protocol}/{model}: "
                f"expected 3 seeds, found {len(g)}."
            )


        row = {

            "protocol":
                protocol,

            "model":
                model,

            "n_seeds":
                len(g),

            "n_test_images":
                int(
                    g[
                        "n_test_images"
                    ].iloc[0]
                ),

            "n_test_gt_objects":
                int(
                    g[
                        "n_test_gt_objects"
                    ].iloc[0]
                ),
        }


        for metric in metrics:

            values = (
                g[metric]
                .astype(float)
            )


            row[
                f"{metric}_mean"
            ] = float(
                values.mean()
            )


            row[
                f"{metric}_sd"
            ] = float(
                values.std(
                    ddof=1
                )
            )


            row[
                f"{metric}_min"
            ] = float(
                values.min()
            )


            row[
                f"{metric}_max"
            ] = float(
                values.max()
            )


        aggregate_rows.append(
            row
        )


agg = pd.DataFrame(
    aggregate_rows
)


# ============================================================
# 11. SEED-STABILITY VARIABLES
# ============================================================

agg[
    "mAP50_95_range"
] = (
    agg["mAP50_95_max"]
    -
    agg["mAP50_95_min"]
)


agg[
    "mAP50_95_cv_percent"
] = np.where(

    agg["mAP50_95_mean"] != 0,

    (
        100.0
        *
        agg["mAP50_95_sd"]
        /
        agg["mAP50_95_mean"]
    ),

    np.nan
)


agg[
    "test_f1_range"
] = (
    agg["test_f1_max"]
    -
    agg["test_f1_min"]
)


# ============================================================
# 12. OBJECT-SIZE GAPS
#
# Do not use fold ratios because AP-small is near zero
# in several runs; ratios would be unstable/misleading.
# ============================================================

agg[
    "AP_large_minus_small"
] = (
    agg["AP_large_mean"]
    -
    agg["AP_small_mean"]
)


agg[
    "AP_medium_minus_small"
] = (
    agg["AP_medium_mean"]
    -
    agg["AP_small_mean"]
)


agg[
    "AP_large_minus_medium"
] = (
    agg["AP_large_mean"]
    -
    agg["AP_medium_mean"]
)


# ============================================================
# 13. SAVE FULL AGGREGATE
# ============================================================

AGG_PATH = (
    DIRS["test"] /
    "locked_test_aggregate_mean_sd.csv"
)


agg.to_csv(
    AGG_PATH,
    index=False
)


# ============================================================
# 14. MANUSCRIPT COMPACT NUMERIC TABLE
# ============================================================

compact_columns = [

    "protocol",
    "model",
    "n_seeds",
    "n_test_images",
    "n_test_gt_objects",

    "mAP50_95_mean",
    "mAP50_95_sd",

    "AP50_mean",
    "AP50_sd",

    "AP75_mean",
    "AP75_sd",

    "AP_small_mean",
    "AP_small_sd",

    "AP_medium_mean",
    "AP_medium_sd",

    "AP_large_mean",
    "AP_large_sd",

    "test_precision_mean",
    "test_precision_sd",

    "test_recall_mean",
    "test_recall_sd",

    "test_f1_mean",
    "test_f1_sd",

    "mAP50_95_range",
    "mAP50_95_cv_percent",

    "inference_ms_mean_mean",
    "inference_ms_mean_sd",

    "processing_ms_mean_mean",
    "processing_ms_mean_sd",
]


compact = agg[
    compact_columns
].copy()


COMPACT_PATH = (
    DIRS["test"] /
    "locked_test_manuscript_table.csv"
)


compact.to_csv(
    COMPACT_PATH,
    index=False
)


# ============================================================
# 15. FORMATTED MEAN ± SD TABLE
# ============================================================

formatted_rows = []


for r in agg.itertuples():

    formatted_rows.append({

        "Protocol":
            r.protocol,

        "Model":
            r.model,

        "mAP50-95":
            mean_sd_string(
                r.mAP50_95_mean,
                r.mAP50_95_sd
            ),

        "AP50":
            mean_sd_string(
                r.AP50_mean,
                r.AP50_sd
            ),

        "AP75":
            mean_sd_string(
                r.AP75_mean,
                r.AP75_sd
            ),

        "AP-small":
            mean_sd_string(
                r.AP_small_mean,
                r.AP_small_sd
            ),

        "AP-medium":
            mean_sd_string(
                r.AP_medium_mean,
                r.AP_medium_sd
            ),

        "AP-large":
            mean_sd_string(
                r.AP_large_mean,
                r.AP_large_sd
            ),

        "Precision":
            mean_sd_string(
                r.test_precision_mean,
                r.test_precision_sd
            ),

        "Recall":
            mean_sd_string(
                r.test_recall_mean,
                r.test_recall_sd
            ),

        "F1":
            mean_sd_string(
                r.test_f1_mean,
                r.test_f1_sd
            ),

        "mAP50-95 range":
            f"{r.mAP50_95_range:.4f}",

        "mAP50-95 CV (%)":
            f"{r.mAP50_95_cv_percent:.2f}",

        "Inference (ms/image)":
            mean_sd_string(
                r.inference_ms_mean_mean,
                r.inference_ms_mean_sd
            ),
    })


formatted = pd.DataFrame(
    formatted_rows
)


FORMATTED_PATH = (
    DIRS["test"] /
    "locked_test_mean_sd_formatted.csv"
)


formatted.to_csv(
    FORMATTED_PATH,
    index=False
)


# ============================================================
# 16. DESCRIPTIVE MODEL RANKING
#
# IMPORTANT:
# Ranking is based only on mean mAP50-95.
# This is NOT a statistical significance claim.
# ============================================================

ranking_parts = []


for protocol in protocols:

    temp = (
        agg[
            agg["protocol"]
            == protocol
        ]
        .sort_values(
            [
                "mAP50_95_mean",
                "mAP50_95_sd"
            ],
            ascending=[
                False,
                True
            ]
        )
        .copy()
    )


    temp[
        "descriptive_rank"
    ] = np.arange(
        1,
        len(temp) + 1
    )


    ranking_parts.append(

        temp[
            [
                "protocol",
                "model",
                "descriptive_rank",

                "mAP50_95_mean",
                "mAP50_95_sd",
                "mAP50_95_range",
                "mAP50_95_cv_percent",

                "test_f1_mean",
                "test_f1_sd",

                "AP_small_mean",
                "AP_medium_mean",
                "AP_large_mean",
            ]
        ]
    )


ranking = pd.concat(
    ranking_parts,
    ignore_index=True
)


RANKING_PATH = (
    DIRS["test"] /
    "locked_test_model_ranking_descriptive.csv"
)


ranking.to_csv(
    RANKING_PATH,
    index=False
)


# ============================================================
# 17. RANDOM VS BATCH-DISJOINT
#
# CRITICAL INTERPRETATION:
# ------------------------------------------------------------
# These are NOT paired test sets.
# Training sets also differ by protocol.
#
# Therefore:
#   batch - random
#
# is a descriptive protocol-associated difference,
# NOT a causal estimate and NOT a paired comparison.
# ============================================================

protocol_gap_rows = []


for model in models:

    random_row = agg[
        (agg["protocol"] == "random")
        &
        (agg["model"] == model)
    ]


    batch_row = agg[
        (agg["protocol"] == "batch_disjoint")
        &
        (agg["model"] == model)
    ]


    if (
        len(random_row) != 1
        or
        len(batch_row) != 1
    ):

        raise RuntimeError(
            f"Could not uniquely retrieve "
            f"protocol rows for {model}."
        )


    rr = random_row.iloc[0]
    br = batch_row.iloc[0]


    protocol_gap_rows.append({

        "model":
            model,

        "random_mAP50_95_mean":
            float(
                rr["mAP50_95_mean"]
            ),

        "batch_disjoint_mAP50_95_mean":
            float(
                br["mAP50_95_mean"]
            ),

        "batch_minus_random_mAP50_95":
            float(
                br["mAP50_95_mean"]
                -
                rr["mAP50_95_mean"]
            ),

        "random_AP50_mean":
            float(
                rr["AP50_mean"]
            ),

        "batch_disjoint_AP50_mean":
            float(
                br["AP50_mean"]
            ),

        "batch_minus_random_AP50":
            float(
                br["AP50_mean"]
                -
                rr["AP50_mean"]
            ),

        "random_test_f1_mean":
            float(
                rr["test_f1_mean"]
            ),

        "batch_disjoint_test_f1_mean":
            float(
                br["test_f1_mean"]
            ),

        "batch_minus_random_test_f1":
            float(
                br["test_f1_mean"]
                -
                rr["test_f1_mean"]
            ),

        "interpretation":
            (
                "descriptive only; "
                "different protocol-specific "
                "training and test compositions"
            ),
    })


protocol_gap = pd.DataFrame(
    protocol_gap_rows
)


PROTOCOL_GAP_PATH = (
    DIRS["test"] /
    "locked_test_protocol_descriptive_gap.csv"
)


protocol_gap.to_csv(
    PROTOCOL_GAP_PATH,
    index=False
)


# ============================================================
# 18. VALIDATION -> TEST GENERALIZATION GAP
#
# Seed-matched descriptive difference:
#
#     test metric - validation metric
#
# Negative value = lower performance on locked test.
#
# This is descriptive because validation and test contain
# different images.
# ============================================================

merge_keys = [
    "protocol",
    "model",
    "seed"
]


val_needed = val_df[
    merge_keys
    +
    [
        "mAP50_95",
        "AP50",
        "AP75",
        "AP_small",
        "AP_medium",
        "AP_large",
        "val_precision",
        "val_recall",
        "val_f1",
    ]
].copy()


val_needed = val_needed.rename(
    columns={

        "mAP50_95":
            "val_mAP50_95",

        "AP50":
            "val_AP50",

        "AP75":
            "val_AP75",

        "AP_small":
            "val_AP_small",

        "AP_medium":
            "val_AP_medium",

        "AP_large":
            "val_AP_large",
    }
)


test_needed = test_df[
    merge_keys
    +
    [
        "mAP50_95",
        "AP50",
        "AP75",
        "AP_small",
        "AP_medium",
        "AP_large",
        "test_precision",
        "test_recall",
        "test_f1",
    ]
].copy()


test_needed = test_needed.rename(
    columns={

        "mAP50_95":
            "test_mAP50_95",

        "AP50":
            "test_AP50",

        "AP75":
            "test_AP75",

        "AP_small":
            "test_AP_small",

        "AP_medium":
            "test_AP_medium",

        "AP_large":
            "test_AP_large",
    }
)


generalization_seed = pd.merge(

    val_needed,
    test_needed,

    on=merge_keys,

    how="inner",

    validate="one_to_one"
)


if len(
    generalization_seed
) != 24:

    raise RuntimeError(
        "Validation-test seed matching failed."
    )


# ------------------------------------------------------------
# Test minus validation
# ------------------------------------------------------------

for metric in [
    "mAP50_95",
    "AP50",
    "AP75",
    "AP_small",
    "AP_medium",
    "AP_large",
]:

    generalization_seed[
        f"delta_{metric}_test_minus_val"
    ] = (
        generalization_seed[
            f"test_{metric}"
        ]
        -
        generalization_seed[
            f"val_{metric}"
        ]
    )


generalization_seed[
    "delta_precision_test_minus_val"
] = (
    generalization_seed[
        "test_precision"
    ]
    -
    generalization_seed[
        "val_precision"
    ]
)


generalization_seed[
    "delta_recall_test_minus_val"
] = (
    generalization_seed[
        "test_recall"
    ]
    -
    generalization_seed[
        "val_recall"
    ]
)


generalization_seed[
    "delta_f1_test_minus_val"
] = (
    generalization_seed[
        "test_f1"
    ]
    -
    generalization_seed[
        "val_f1"
    ]
)


GENERALIZATION_SEED_PATH = (
    DIRS["analysis"] /
    "validation_to_test_gap_seed_level.csv"
)


generalization_seed.to_csv(
    GENERALIZATION_SEED_PATH,
    index=False
)


# ============================================================
# 19. AGGREGATE VALIDATION -> TEST GAPS
# ============================================================

gap_metrics = [

    "delta_mAP50_95_test_minus_val",
    "delta_AP50_test_minus_val",
    "delta_AP75_test_minus_val",

    "delta_AP_small_test_minus_val",
    "delta_AP_medium_test_minus_val",
    "delta_AP_large_test_minus_val",

    "delta_precision_test_minus_val",
    "delta_recall_test_minus_val",
    "delta_f1_test_minus_val",
]


generalization_rows = []


for protocol in protocols:

    for model in models:

        g = generalization_seed[
            (generalization_seed["protocol"] == protocol)
            &
            (generalization_seed["model"] == model)
        ]


        row = {
            "protocol":
                protocol,

            "model":
                model,

            "n_seeds":
                len(g),
        }


        for metric in gap_metrics:

            values = (
                g[metric]
                .astype(float)
            )


            row[
                f"{metric}_mean"
            ] = float(
                values.mean()
            )


            row[
                f"{metric}_sd"
            ] = float(
                values.std(
                    ddof=1
                )
            )


        generalization_rows.append(
            row
        )


generalization = pd.DataFrame(
    generalization_rows
)


GENERALIZATION_PATH = (
    DIRS["analysis"] /
    "validation_to_test_gap_mean_sd.csv"
)


generalization.to_csv(
    GENERALIZATION_PATH,
    index=False
)


# ============================================================
# 20. OBJECT-SIZE ANALYSIS TABLE
# ============================================================

object_size = agg[
    [
        "protocol",
        "model",

        "AP_small_mean",
        "AP_small_sd",

        "AP_medium_mean",
        "AP_medium_sd",

        "AP_large_mean",
        "AP_large_sd",

        "AR_small_mean",
        "AR_small_sd",

        "AR_medium_mean",
        "AR_medium_sd",

        "AR_large_mean",
        "AR_large_sd",

        "AP_medium_minus_small",
        "AP_large_minus_small",
        "AP_large_minus_medium",
    ]
].copy()


OBJECT_SIZE_PATH = (
    DIRS["analysis"] /
    "locked_test_object_size_analysis.csv"
)


object_size.to_csv(
    OBJECT_SIZE_PATH,
    index=False
)


# ============================================================
# 21. SEED STABILITY TABLE
# ============================================================

stability = agg[
    [
        "protocol",
        "model",

        "mAP50_95_mean",
        "mAP50_95_sd",
        "mAP50_95_min",
        "mAP50_95_max",
        "mAP50_95_range",
        "mAP50_95_cv_percent",

        "test_f1_mean",
        "test_f1_sd",
        "test_f1_min",
        "test_f1_max",
        "test_f1_range",
    ]
].copy()


STABILITY_PATH = (
    DIRS["analysis"] /
    "locked_test_seed_stability.csv"
)


stability.to_csv(
    STABILITY_PATH,
    index=False
)


# ============================================================
# 22. EFFICIENCY SUMMARY
#
# Framework-reported processing times only.
# Not end-to-end deployment latency.
# ============================================================

efficiency = agg[
    [
        "protocol",
        "model",

        "preprocess_ms_mean_mean",
        "preprocess_ms_mean_sd",

        "inference_ms_mean_mean",
        "inference_ms_mean_sd",

        "postprocess_ms_mean_mean",
        "postprocess_ms_mean_sd",

        "processing_ms_mean_mean",
        "processing_ms_mean_sd",

        "processing_fps_mean",
        "processing_fps_sd",

        "mAP50_95_mean",
        "mAP50_95_sd",
    ]
].copy()


efficiency[
    "latency_definition"
] = (
    "Ultralytics framework-reported processing time; "
    "not end-to-end application latency"
)


EFFICIENCY_PATH = (
    DIRS["analysis"] /
    "locked_test_efficiency_summary.csv"
)


efficiency.to_csv(
    EFFICIENCY_PATH,
    index=False
)


# ============================================================
# 23. DISPLAY MAIN MANUSCRIPT TABLE
# ============================================================

print(
    "\n"
    + "=" * 84
)

print(
    "LOCKED TEST — 3-SEED MEAN ± SD"
)

print(
    "=" * 84
)


display(
    formatted
)


# ============================================================
# 24. DISPLAY DESCRIPTIVE RANKING
# ============================================================

print(
    "\n"
    + "=" * 84
)

print(
    "LOCKED-TEST DESCRIPTIVE MODEL RANKING"
)

print(
    "Ranking metric: mean COCO mAP@[0.50:0.95]"
)

print(
    "No statistical significance claim is made here."
)

print(
    "=" * 84
)


display(
    ranking
)


# ============================================================
# 25. DISPLAY PROTOCOL DIFFERENCES
# ============================================================

print(
    "\n"
    + "=" * 84
)

print(
    "RANDOM VS BATCH-DISJOINT — DESCRIPTIVE DIFFERENCE"
)

print(
    "Difference = batch-disjoint mean - random mean"
)

print(
    "WARNING: protocol-specific train/test sets differ."
)

print(
    "=" * 84
)


display(
    protocol_gap
)


# ============================================================
# 26. DISPLAY VALIDATION -> TEST GAP
# ============================================================

gap_display = generalization[
    [
        "protocol",
        "model",

        "delta_mAP50_95_test_minus_val_mean",
        "delta_mAP50_95_test_minus_val_sd",

        "delta_AP50_test_minus_val_mean",
        "delta_AP75_test_minus_val_mean",

        "delta_AP_small_test_minus_val_mean",
        "delta_AP_medium_test_minus_val_mean",
        "delta_AP_large_test_minus_val_mean",

        "delta_f1_test_minus_val_mean",
        "delta_f1_test_minus_val_sd",
    ]
].copy()


print(
    "\n"
    + "=" * 84
)

print(
    "VALIDATION -> LOCKED TEST GENERALIZATION GAP"
)

print(
    "Difference = test - validation"
)

print(
    "Negative values indicate lower locked-test performance."
)

print(
    "=" * 84
)


display(
    gap_display
)


# ============================================================
# 27. DISPLAY OBJECT-SIZE RESULTS
# ============================================================

print(
    "\n"
    + "=" * 84
)

print(
    "LOCKED-TEST OBJECT-SIZE ANALYSIS"
)

print(
    "=" * 84
)


display(
    object_size
)


# ============================================================
# 28. DISPLAY SEED STABILITY
# ============================================================

print(
    "\n"
    + "=" * 84
)

print(
    "LOCKED-TEST SEED STABILITY"
)

print(
    "=" * 84
)


display(
    stability
)


# ============================================================
# 29. DISPLAY EFFICIENCY
# ============================================================

print(
    "\n"
    + "=" * 84
)

print(
    "LOCKED-TEST EFFICIENCY SUMMARY"
)

print(
    "Times are framework-reported, not end-to-end latency."
)

print(
    "=" * 84
)


display(
    efficiency
)


# ============================================================
# 30. AUTOMATIC DESCRIPTIVE CHECKS
# ============================================================

print(
    "\n"
    + "=" * 84
)

print(
    "DESCRIPTIVE CHECKS"
)

print(
    "=" * 84
)


for protocol in protocols:

    temp = (
        agg[
            agg["protocol"]
            == protocol
        ]
        .sort_values(
            "mAP50_95_mean",
            ascending=False
        )
    )


    best = temp.iloc[0]


    print(
        f"\n{protocol}"
    )

    print(
        "Highest mean mAP50-95:",
        best["model"],
        f"{best['mAP50_95_mean']:.4f}"
        f" ± {best['mAP50_95_sd']:.4f}"
    )


    most_variable = temp.sort_values(
        "mAP50_95_sd",
        ascending=False
    ).iloc[0]


    print(
        "Largest seed SD:",
        most_variable["model"],
        f"{most_variable['mAP50_95_sd']:.4f}"
    )


    smallest_ap = temp[
        "AP_small_mean"
    ].max()

    print(
        "Best AP-small among models:",
        f"{smallest_ap:.4f}"
    )


# ============================================================
# 31. VERIFY EXPECTED PRIMARY DESCRIPTIVE WINNER
#
# This is NOT used to change experiments.
# It merely reports the completed locked-test results.
# ============================================================

for protocol in protocols:

    protocol_rows = agg[
        agg["protocol"]
        == protocol
    ]


    best_model = (
        protocol_rows
        .sort_values(
            "mAP50_95_mean",
            ascending=False
        )
        .iloc[0]["model"]
    )


    print(
        f"\n{protocol}: "
        f"descriptive top model = {best_model}"
    )


# ============================================================
# 32. OUTPUT HASH REGISTRY
# ============================================================

output_paths = {

    "locked_test_aggregate_mean_sd.csv":
        AGG_PATH,

    "locked_test_manuscript_table.csv":
        COMPACT_PATH,

    "locked_test_mean_sd_formatted.csv":
        FORMATTED_PATH,

    "locked_test_model_ranking_descriptive.csv":
        RANKING_PATH,

    "locked_test_protocol_descriptive_gap.csv":
        PROTOCOL_GAP_PATH,

    "validation_to_test_gap_seed_level.csv":
        GENERALIZATION_SEED_PATH,

    "validation_to_test_gap_mean_sd.csv":
        GENERALIZATION_PATH,

    "locked_test_object_size_analysis.csv":
        OBJECT_SIZE_PATH,

    "locked_test_seed_stability.csv":
        STABILITY_PATH,

    "locked_test_efficiency_summary.csv":
        EFFICIENCY_PATH,
}


output_hashes = {

    name:
        sha256_file(path)

    for name, path in output_paths.items()
}


# ============================================================
# 33. STAGE 7B MANIFEST
# ============================================================

STAGE_7B_MANIFEST = (
    DIRS["manifests"] /
    "STAGE_7B_FINAL_TEST_AGGREGATION.json"
)


manifest = {

    "stage":
        "7B",

    "status":
        "LOCKED_TEST_AGGREGATION_COMPLETED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_locked_test_summary":
        str(
            TEST_SUMMARY_PATH
        ),

    "source_locked_test_summary_sha256":
        current_test_hash,

    "number_of_test_runs":
        24,

    "number_of_protocols":
        2,

    "number_of_models":
        4,

    "number_of_seeds":
        3,

    "primary_metric":
        "COCO mAP@[0.50:0.95]",

    "aggregation":
        "mean and sample SD across seeds 42, 123, and 2026",

    "protocol_comparison_policy":
        (
            "Random versus batch-disjoint differences are "
            "descriptive only because protocol-specific "
            "training and test compositions differ."
        ),

    "generalization_gap_policy":
        (
            "Validation-to-test differences are descriptive "
            "test-minus-validation differences; validation "
            "and locked-test images are distinct."
        ),

    "object_size_policy":
        (
            "COCO AP-small, AP-medium, and AP-large are "
            "reported without post-test optimization."
        ),

    "statistical_significance_tested":
        False,

    "bootstrap_performed":
        False,

    "test_threshold_reselection":
        False,

    "post_test_retraining":
        False,

    "output_sha256":
        output_hashes,
}


if STAGE_7B_MANIFEST.exists():

    previous = json.loads(
        STAGE_7B_MANIFEST.read_text(
            encoding="utf-8"
        )
    )


    old_cmp = dict(
        previous
    )

    new_cmp = dict(
        manifest
    )


    old_cmp.pop(
        "created_utc",
        None
    )

    new_cmp.pop(
        "created_utc",
        None
    )


    if old_cmp != new_cmp:

        raise RuntimeError(
            "A different STAGE 7B manifest already exists. "
            "Do not overwrite it automatically."
        )


else:

    STAGE_7B_MANIFEST.write_text(

        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),

        encoding="utf-8"
    )


# ============================================================
# 34. FINAL
# ============================================================

print(
    "\n"
    + "=" * 84
)

print(
    "STAGE 7B PASSED"
)

print(
    "=" * 84
)

print(
    "Locked-test runs aggregated : 24/24"
)

print(
    "3-seed mean ± SD            : COMPLETE"
)

print(
    "Seed stability              : COMPLETE"
)

print(
    "Object-size analysis        : COMPLETE"
)

print(
    "Validation -> test gap      : COMPLETE"
)

print(
    "Protocol descriptive gap    : COMPLETE"
)

print(
    "Efficiency summary          : COMPLETE"
)

print(
    "Statistical bootstrap       : NOT YET"
)

print(
    "Test threshold tuning       : NO"
)

print(
    "Post-test retraining        : NO"
)


print(
    "\nPrimary output:"
)

print(
    FORMATTED_PATH
)


print(
    "\nStage manifest:"
)

print(
    STAGE_7B_MANIFEST
)


print(
    "\nNEXT: STAGE 7C — PRE-SPECIFIED BOOTSTRAP / "
    "UNCERTAINTY ANALYSIS."
)

print(
    "=" * 84
)

In [ ]:
# ============================================================
# STAGE 7C — PAIRED IMAGE-LEVEL BOOTSTRAP
#             FOR LOCKED-TEST GLOBAL F1
#
# PURPOSE
# ------------------------------------------------------------
# Secondary uncertainty analysis.
#
# - Resampling unit: IMAGE
# - Bootstrap replicates: 5000
# - Confidence interval: percentile 95%
# - Comparison: WITHIN PROTOCOL ONLY
# - Models are paired on exactly the same test images.
# - All 3 observed seeds are retained.
#
# For each bootstrap replicate:
#   1. Sample test images with replacement.
#   2. Recompute GLOBAL F1 from summed TP/FP/FN
#      separately for each of the 3 seeds.
#   3. Average the 3 seed-specific F1 values.
#
# This estimates image-sampling uncertainty conditional
# on the three observed trained seeds.
#
# IMPORTANT:
# - NO new inference
# - NO threshold tuning
# - NO model selection
# - NO random-vs-batch paired bootstrap
# - NO retraining
# ============================================================

from pathlib import Path
from itertools import combinations
from datetime import datetime, timezone

import json
import hashlib
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 2. BOOTSTRAP CONFIGURATION
# ============================================================

N_BOOT = 5000

BOOTSTRAP_RANDOM_SEED = 20260815

CI_LEVEL = 0.95

ALPHA = 1.0 - CI_LEVEL


protocols = [
    "random",
    "batch_disjoint"
]

models = [
    "yolov8s",
    "yolo11s",
    "yolo26s",
    "rtdetr_l"
]

seeds = [
    42,
    123,
    2026
]


# ============================================================
# 3. HELPERS
# ============================================================

def sha256_file(
    path,
    chunk=1024 * 1024
):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(chunk),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def global_f1(
    tp,
    fp,
    fn
):

    denom = (
        2.0 * tp
        +
        fp
        +
        fn
    )

    return np.divide(
        2.0 * tp,
        denom,
        out=np.zeros_like(
            denom,
            dtype=float
        ),
        where=denom > 0
    )


def normalize_per_image_table(
    df,
    source_name
):
    """
    Normalize image_id / TP / FP / FN columns.

    Stage 6C detection_counts() may use lowercase or
    uppercase count column names. This function accepts
    either but refuses ambiguous/missing structures.
    """

    lower_map = {
        str(c).lower():
            c
        for c in df.columns
    }


    required = [
        "image_id",
        "tp",
        "fp",
        "fn"
    ]


    missing = [
        c
        for c in required
        if c not in lower_map
    ]


    if missing:

        raise RuntimeError(
            f"{source_name} is missing required "
            f"per-image columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )


    out = pd.DataFrame({

        "image_id":
            pd.to_numeric(
                df[
                    lower_map[
                        "image_id"
                    ]
                ],
                errors="raise"
            ).astype(int),

        "tp":
            pd.to_numeric(
                df[
                    lower_map["tp"]
                ],
                errors="raise"
            ).astype(int),

        "fp":
            pd.to_numeric(
                df[
                    lower_map["fp"]
                ],
                errors="raise"
            ).astype(int),

        "fn":
            pd.to_numeric(
                df[
                    lower_map["fn"]
                ],
                errors="raise"
            ).astype(int),
    })


    if out[
        "image_id"
    ].duplicated().any():

        raise RuntimeError(
            f"Duplicate image_id values in:\n"
            f"{source_name}"
        )


    if (
        out[
            ["tp", "fp", "fn"]
        ] < 0
    ).any().any():

        raise RuntimeError(
            f"Negative TP/FP/FN count detected in:\n"
            f"{source_name}"
        )


    return (
        out
        .sort_values(
            "image_id"
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 4. REQUIRED ARTIFACTS
# ============================================================

TEST_SUMMARY_PATH = (
    DIRS["test"] /
    "locked_test_summary_all_runs.csv"
)

LOCKED_COMPLETE_PATH = (
    DIRS["manifests"] /
    "LOCKED_TEST_COMPLETED.json"
)

STAGE7B_PATH = (
    DIRS["manifests"] /
    "STAGE_7B_FINAL_TEST_AGGREGATION.json"
)


for p in [
    TEST_SUMMARY_PATH,
    LOCKED_COMPLETE_PATH,
    STAGE7B_PATH
]:

    if not p.exists():

        raise FileNotFoundError(
            f"Required Stage 7C input missing:\n{p}"
        )


print("=" * 86)
print("STAGE 7C — PAIRED IMAGE-LEVEL BOOTSTRAP")
print("=" * 86)


# ============================================================
# 5. VERIFY LOCKED TEST SUMMARY HAS NOT CHANGED
# ============================================================

test_df = pd.read_csv(
    TEST_SUMMARY_PATH
)

completion = json.loads(
    LOCKED_COMPLETE_PATH.read_text(
        encoding="utf-8"
    )
)


expected_hash = completion[
    "test_summary_sha256"
]

current_hash = sha256_file(
    TEST_SUMMARY_PATH
)


if current_hash != expected_hash:

    raise RuntimeError(
        "Locked-test summary changed after completion."
    )


if len(test_df) != 24:

    raise RuntimeError(
        f"Expected 24 test runs; found {len(test_df)}."
    )


print(
    "Locked-test summary integrity : PASSED"
)


# ============================================================
# 6. LOAD AND VERIFY ALL 24 PER-IMAGE TABLES
# ============================================================

per_run = {}

expected_counts = {
    "random": 225,
    "batch_disjoint": 200,
}


for protocol in protocols:

    for model in models:

        for seed in seeds:

            run_key = (
                protocol,
                model,
                seed
            )


            per_image_path = (
                DIRS["test"] /
                protocol /
                f"{model}_seed{seed}" /
                "test_per_image_metrics.csv"
            )


            if not per_image_path.exists():

                raise FileNotFoundError(
                    f"Missing per-image locked-test metrics:\n"
                    f"{per_image_path}"
                )


            raw = pd.read_csv(
                per_image_path
            )


            d = normalize_per_image_table(
                raw,
                str(per_image_path)
            )


            expected_n = (
                expected_counts[
                    protocol
                ]
            )


            if len(d) != expected_n:

                raise RuntimeError(
                    f"{protocol}/{model}/seed{seed}: "
                    f"expected {expected_n} per-image rows, "
                    f"found {len(d)}."
                )


            # -----------------------------------------------
            # Compare summed per-image counts against
            # Stage 7A global counts
            # -----------------------------------------------

            summary_row = test_df[
                (test_df["protocol"] == protocol)
                &
                (test_df["model"] == model)
                &
                (
                    test_df["seed"].astype(int)
                    == seed
                )
            ]


            if len(summary_row) != 1:

                raise RuntimeError(
                    f"Could not uniquely locate summary row "
                    f"for {protocol}/{model}/seed{seed}."
                )


            sr = summary_row.iloc[0]


            tp_sum = int(
                d["tp"].sum()
            )

            fp_sum = int(
                d["fp"].sum()
            )

            fn_sum = int(
                d["fn"].sum()
            )


            if tp_sum != int(sr["TP"]):

                raise RuntimeError(
                    f"TP mismatch: "
                    f"{protocol}/{model}/seed{seed}"
                )


            if fp_sum != int(sr["FP"]):

                raise RuntimeError(
                    f"FP mismatch: "
                    f"{protocol}/{model}/seed{seed}"
                )


            if fn_sum != int(sr["FN"]):

                raise RuntimeError(
                    f"FN mismatch: "
                    f"{protocol}/{model}/seed{seed}"
                )


            full_f1 = (
                2.0 * tp_sum
                /
                (
                    2.0 * tp_sum
                    +
                    fp_sum
                    +
                    fn_sum
                )
            )


            if not np.isclose(
                full_f1,
                float(
                    sr["test_f1"]
                ),
                rtol=0,
                atol=1e-10
            ):

                raise RuntimeError(
                    f"Recomputed F1 mismatch: "
                    f"{protocol}/{model}/seed{seed}\n"
                    f"Per-image counts -> {full_f1}\n"
                    f"Stage 7A          -> "
                    f"{float(sr['test_f1'])}"
                )


            per_run[
                run_key
            ] = d


print(
    "Per-image result integrity    : 24/24 PASSED"
)


# ============================================================
# 7. VERIFY EXACT IMAGE PAIRING WITHIN EACH PROTOCOL
# ============================================================

protocol_image_ids = {}


for protocol in protocols:

    reference_ids = None


    for model in models:

        for seed in seeds:

            ids = tuple(
                per_run[
                    (
                        protocol,
                        model,
                        seed
                    )
                ]["image_id"]
                .astype(int)
                .tolist()
            )


            if reference_ids is None:

                reference_ids = ids


            elif ids != reference_ids:

                raise RuntimeError(
                    "Per-image ordering/content differs "
                    "within protocol:\n"
                    f"{protocol}/{model}/seed{seed}"
                )


    protocol_image_ids[
        protocol
    ] = np.array(
        reference_ids,
        dtype=int
    )


    print(
        f"{protocol:16s}: "
        f"{len(reference_ids)} exactly paired images"
    )


print(
    "\nWithin-protocol pairing       : PASSED"
)


# ============================================================
# 8. VERIFY FULL-SAMPLE SEED-MEAN F1
#    AGAINST STAGE 7B
# ============================================================

stage7b_agg_path = (
    DIRS["test"] /
    "locked_test_aggregate_mean_sd.csv"
)


if not stage7b_agg_path.exists():

    raise FileNotFoundError(
        stage7b_agg_path
    )


stage7b_agg = pd.read_csv(
    stage7b_agg_path
)


point_estimates = {}


for protocol in protocols:

    for model in models:

        seed_f1 = []


        for seed in seeds:

            d = per_run[
                (
                    protocol,
                    model,
                    seed
                )
            ]


            tp = int(
                d["tp"].sum()
            )

            fp = int(
                d["fp"].sum()
            )

            fn = int(
                d["fn"].sum()
            )


            f1 = (
                2.0 * tp
                /
                (
                    2.0 * tp
                    +
                    fp
                    +
                    fn
                )
            )


            seed_f1.append(
                f1
            )


        point = float(
            np.mean(
                seed_f1
            )
        )


        official = stage7b_agg[
            (
                stage7b_agg[
                    "protocol"
                ] == protocol
            )
            &
            (
                stage7b_agg[
                    "model"
                ] == model
            )
        ]


        if len(official) != 1:

            raise RuntimeError(
                f"Missing Stage 7B aggregate row: "
                f"{protocol}/{model}"
            )


        official_f1 = float(
            official.iloc[0][
                "test_f1_mean"
            ]
        )


        if not np.isclose(
            point,
            official_f1,
            rtol=0,
            atol=1e-10
        ):

            raise RuntimeError(
                f"Stage 7B F1 mismatch: "
                f"{protocol}/{model}"
            )


        point_estimates[
            (
                protocol,
                model
            )
        ] = point


print(
    "Stage 7B point estimates     : PASSED"
)


# ============================================================
# 9. PAIRED IMAGE BOOTSTRAP
# ============================================================

rng_master = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)


bootstrap_cache = {}

model_summary_rows = []

distribution_frames = []


for protocol_index, protocol in enumerate(
    protocols
):

    image_ids = (
        protocol_image_ids[
            protocol
        ]
    )

    n_images = len(
        image_ids
    )


    # --------------------------------------------------------
    # One bootstrap index matrix per protocol.
    #
    # EXACT SAME bootstrap samples are used for every model
    # within the protocol -> paired comparison.
    # --------------------------------------------------------

    protocol_seed = int(
        rng_master.integers(
            0,
            2**32 - 1
        )
    )


    rng = np.random.default_rng(
        protocol_seed
    )


    bootstrap_indices = rng.integers(
        low=0,
        high=n_images,
        size=(
            N_BOOT,
            n_images
        ),
        endpoint=False
    )


    print(
        f"\nBootstrapping {protocol}: "
        f"{n_images} images × {N_BOOT} replicates"
    )


    for model in models:

        seed_bootstrap_f1 = []


        for seed in seeds:

            d = per_run[
                (
                    protocol,
                    model,
                    seed
                )
            ]


            # Tables already have identical image ordering.
            tp = d[
                "tp"
            ].to_numpy(
                dtype=np.int64
            )

            fp = d[
                "fp"
            ].to_numpy(
                dtype=np.int64
            )

            fn = d[
                "fn"
            ].to_numpy(
                dtype=np.int64
            )


            boot_tp = (
                tp[
                    bootstrap_indices
                ]
                .sum(
                    axis=1
                )
            )


            boot_fp = (
                fp[
                    bootstrap_indices
                ]
                .sum(
                    axis=1
                )
            )


            boot_fn = (
                fn[
                    bootstrap_indices
                ]
                .sum(
                    axis=1
                )
            )


            boot_f1 = global_f1(
                boot_tp,
                boot_fp,
                boot_fn
            )


            seed_bootstrap_f1.append(
                boot_f1
            )


        # ----------------------------------------------------
        # Average global F1 across the three fixed seeds
        # for each bootstrap-resampled image set.
        # ----------------------------------------------------

        seed_bootstrap_f1 = np.vstack(
            seed_bootstrap_f1
        )


        model_boot = np.mean(
            seed_bootstrap_f1,
            axis=0
        )


        bootstrap_cache[
            (
                protocol,
                model
            )
        ] = model_boot


        point = point_estimates[
            (
                protocol,
                model
            )
        ]


        ci_low = float(
            np.quantile(
                model_boot,
                ALPHA / 2.0
            )
        )


        ci_high = float(
            np.quantile(
                model_boot,
                1.0 - ALPHA / 2.0
            )
        )


        bootstrap_mean = float(
            np.mean(
                model_boot
            )
        )


        bootstrap_sd = float(
            np.std(
                model_boot,
                ddof=1
            )
        )


        model_summary_rows.append({

            "protocol":
                protocol,

            "model":
                model,

            "n_images":
                n_images,

            "n_observed_seeds":
                3,

            "bootstrap_replicates":
                N_BOOT,

            "point_estimate_seed_mean_global_f1":
                point,

            "bootstrap_mean":
                bootstrap_mean,

            "bootstrap_sd":
                bootstrap_sd,

            "bootstrap_ci_low_95":
                ci_low,

            "bootstrap_ci_high_95":
                ci_high,

            "ci_width":
                ci_high
                -
                ci_low,

            "resampling_unit":
                "image",

            "seed_policy":
                (
                    "three observed seeds fixed; "
                    "global F1 recomputed per seed "
                    "then averaged"
                ),
        })


        distribution_frames.append(

            pd.DataFrame({

                "protocol":
                    protocol,

                "model":
                    model,

                "bootstrap_id":
                    np.arange(
                        1,
                        N_BOOT + 1
                    ),

                "seed_mean_global_f1":
                    model_boot,
            })
        )


model_bootstrap = pd.DataFrame(
    model_summary_rows
)


bootstrap_distributions = pd.concat(
    distribution_frames,
    ignore_index=True
)


# ============================================================
# 10. ALL WITHIN-PROTOCOL PAIRWISE MODEL DIFFERENCES
#
# No pair is selected post hoc.
# All six model pairs are reported for each protocol.
#
# Difference = model_A - model_B
# ============================================================

model_pairs = list(
    combinations(
        models,
        2
    )
)


pairwise_rows = []


for protocol in protocols:

    for model_a, model_b in model_pairs:

        dist_a = bootstrap_cache[
            (
                protocol,
                model_a
            )
        ]


        dist_b = bootstrap_cache[
            (
                protocol,
                model_b
            )
        ]


        # Paired because same bootstrap image samples
        # were used for all models in this protocol.
        diff = (
            dist_a
            -
            dist_b
        )


        point_diff = (
            point_estimates[
                (
                    protocol,
                    model_a
                )
            ]
            -
            point_estimates[
                (
                    protocol,
                    model_b
                )
            ]
        )


        ci_low = float(
            np.quantile(
                diff,
                ALPHA / 2.0
            )
        )


        ci_high = float(
            np.quantile(
                diff,
                1.0 - ALPHA / 2.0
            )
        )


        pairwise_rows.append({

            "protocol":
                protocol,

            "model_A":
                model_a,

            "model_B":
                model_b,

            "difference_definition":
                "F1(A) - F1(B)",

            "point_difference":
                float(
                    point_diff
                ),

            "bootstrap_mean_difference":
                float(
                    np.mean(
                        diff
                    )
                ),

            "bootstrap_sd_difference":
                float(
                    np.std(
                        diff,
                        ddof=1
                    )
                ),

            "bootstrap_ci_low_95":
                ci_low,

            "bootstrap_ci_high_95":
                ci_high,

            "bootstrap_replicates":
                N_BOOT,

            "paired_on":
                "test image",

            "formal_p_value":
                "not calculated",

            "interpretation":
                (
                    "secondary paired-bootstrap "
                    "uncertainty interval; "
                    "within-protocol only"
                ),
        })


pairwise = pd.DataFrame(
    pairwise_rows
)


# ============================================================
# 11. FORMATTED MODEL CI TABLE
# ============================================================

formatted_model_rows = []


for r in model_bootstrap.itertuples():

    formatted_model_rows.append({

        "Protocol":
            r.protocol,

        "Model":
            r.model,

        "F1 point estimate":
            f"{r.point_estimate_seed_mean_global_f1:.4f}",

        "95% paired image-bootstrap CI":
            (
                f"[{r.bootstrap_ci_low_95:.4f}, "
                f"{r.bootstrap_ci_high_95:.4f}]"
            ),

        "Bootstrap SD":
            f"{r.bootstrap_sd:.4f}",

        "Images":
            r.n_images,

        "Replicates":
            r.bootstrap_replicates,
    })


formatted_model = pd.DataFrame(
    formatted_model_rows
)


# ============================================================
# 12. FORMATTED PAIRWISE TABLE
# ============================================================

formatted_pair_rows = []


for r in pairwise.itertuples():

    formatted_pair_rows.append({

        "Protocol":
            r.protocol,

        "Comparison":
            (
                f"{r.model_A} - "
                f"{r.model_B}"
            ),

        "F1 difference":
            f"{r.point_difference:.4f}",

        "95% paired bootstrap CI":
            (
                f"[{r.bootstrap_ci_low_95:.4f}, "
                f"{r.bootstrap_ci_high_95:.4f}]"
            ),
    })


formatted_pairwise = pd.DataFrame(
    formatted_pair_rows
)


# ============================================================
# 13. SAVE OUTPUTS
# ============================================================

MODEL_CI_PATH = (
    DIRS["analysis"] /
    "bootstrap_model_global_f1_ci.csv"
)

PAIRWISE_PATH = (
    DIRS["analysis"] /
    "bootstrap_pairwise_global_f1_differences.csv"
)

FORMATTED_MODEL_PATH = (
    DIRS["analysis"] /
    "bootstrap_model_global_f1_ci_formatted.csv"
)

FORMATTED_PAIRWISE_PATH = (
    DIRS["analysis"] /
    "bootstrap_pairwise_global_f1_formatted.csv"
)

DISTRIBUTIONS_PATH = (
    DIRS["analysis"] /
    "bootstrap_global_f1_distributions.csv.gz"
)


model_bootstrap.to_csv(
    MODEL_CI_PATH,
    index=False
)

pairwise.to_csv(
    PAIRWISE_PATH,
    index=False
)

formatted_model.to_csv(
    FORMATTED_MODEL_PATH,
    index=False
)

formatted_pairwise.to_csv(
    FORMATTED_PAIRWISE_PATH,
    index=False
)

bootstrap_distributions.to_csv(
    DISTRIBUTIONS_PATH,
    index=False,
    compression="gzip"
)


# ============================================================
# 14. DISPLAY MODEL-LEVEL BOOTSTRAP CIs
# ============================================================

print(
    "\n"
    + "=" * 86
)

print(
    "LOCKED-TEST GLOBAL F1 — "
    "PAIRED IMAGE-BOOTSTRAP 95% CI"
)

print(
    "Three observed seeds fixed and averaged."
)

print(
    "=" * 86
)


display(
    formatted_model
)


# ============================================================
# 15. DISPLAY ALL WITHIN-PROTOCOL PAIRWISE DIFFERENCES
# ============================================================

print(
    "\n"
    + "=" * 86
)

print(
    "WITHIN-PROTOCOL MODEL DIFFERENCES"
)

print(
    "Difference = F1(model A) - F1(model B)"
)

print(
    "All six model pairs reported; "
    "no post-hoc pair selection."
)

print(
    "=" * 86
)


display(
    formatted_pairwise
)


# ============================================================
# 16. SANITY CHECK
#
# Bootstrap means should be reasonably close to the
# full-sample point estimates.
# ============================================================

model_bootstrap[
    "bootstrap_minus_point"
] = (
    model_bootstrap[
        "bootstrap_mean"
    ]
    -
    model_bootstrap[
        "point_estimate_seed_mean_global_f1"
    ]
)


max_abs_bootstrap_bias = float(
    model_bootstrap[
        "bootstrap_minus_point"
    ]
    .abs()
    .max()
)


print(
    "\nMaximum |bootstrap mean - point estimate|:",
    f"{max_abs_bootstrap_bias:.6f}"
)


if max_abs_bootstrap_bias > 0.02:

    raise RuntimeError(
        "Unexpectedly large bootstrap mean deviation "
        "from the original point estimate. "
        "Inspect bootstrap implementation."
    )


# ============================================================
# 17. OUTPUT HASHES
# ============================================================

outputs = {

    "bootstrap_model_global_f1_ci.csv":
        MODEL_CI_PATH,

    "bootstrap_pairwise_global_f1_differences.csv":
        PAIRWISE_PATH,

    "bootstrap_model_global_f1_ci_formatted.csv":
        FORMATTED_MODEL_PATH,

    "bootstrap_pairwise_global_f1_formatted.csv":
        FORMATTED_PAIRWISE_PATH,

    "bootstrap_global_f1_distributions.csv.gz":
        DISTRIBUTIONS_PATH,
}


output_hashes = {

    name:
        sha256_file(
            path
        )

    for name, path in outputs.items()
}


# ============================================================
# 18. STAGE 7C MANIFEST
# ============================================================

MANIFEST_PATH = (
    DIRS["manifests"] /
    "STAGE_7C_BOOTSTRAP.json"
)


manifest = {

    "stage":
        "7C",

    "status":
        "PAIRED_IMAGE_BOOTSTRAP_COMPLETED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_locked_test_summary":
        str(
            TEST_SUMMARY_PATH
        ),

    "source_locked_test_summary_sha256":
        current_hash,

    "metric":
        "global F1 at IoU=0.50 using validation-frozen thresholds",

    "bootstrap_replicates":
        N_BOOT,

    "bootstrap_random_seed":
        BOOTSTRAP_RANDOM_SEED,

    "confidence_interval":
        "percentile 95%",

    "resampling_unit":
        "image",

    "within_protocol_pairing":
        True,

    "between_protocol_bootstrap":
        False,

    "seed_handling":
        (
            "The three observed trained seeds "
            "(42, 123, 2026) were fixed. "
            "For each bootstrap image sample, global F1 "
            "was recomputed separately for each seed "
            "and then averaged across seeds."
        ),

    "uncertainty_interpretation":
        (
            "Bootstrap intervals quantify image-sampling "
            "uncertainty conditional on the three observed "
            "trained seeds. Between-seed variability is "
            "reported separately using mean ± sample SD."
        ),

    "dependency_note":
        (
            "Image-level bootstrap does not model possible "
            "within-batch correlation and therefore should "
            "be interpreted as a secondary uncertainty "
            "analysis rather than population-level inference "
            "over unseen TACO batches."
        ),

    "pairwise_policy":
        (
            "All six model pairs were evaluated separately "
            "within each protocol. No cross-protocol paired "
            "comparison was performed."
        ),

    "formal_p_values":
        False,

    "multiple_testing_claim":
        False,

    "test_threshold_tuning":
        False,

    "post_test_retraining":
        False,

    "output_sha256":
        output_hashes,
}


if MANIFEST_PATH.exists():

    previous = json.loads(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )


    old_cmp = dict(
        previous
    )

    new_cmp = dict(
        manifest
    )

    old_cmp.pop(
        "created_utc",
        None
    )

    new_cmp.pop(
        "created_utc",
        None
    )


    if old_cmp != new_cmp:

        raise RuntimeError(
            "A different STAGE_7C_BOOTSTRAP.json "
            "already exists. "
            "It will NOT be overwritten."
        )


else:

    MANIFEST_PATH.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )


# ============================================================
# 19. FINAL
# ============================================================

print(
    "\n"
    + "=" * 86
)

print(
    "STAGE 7C PASSED"
)

print(
    "=" * 86
)

print(
    f"Bootstrap replicates        : {N_BOOT}"
)

print(
    "Resampling unit             : IMAGE"
)

print(
    "Within-protocol pairing     : YES"
)

print(
    "Cross-protocol pairing      : NO"
)

print(
    "Observed seeds retained     : 42, 123, 2026"
)

print(
    "Seed variability replaced   : NO"
)

print(
    "Formal p-values             : NO"
)

print(
    "Test threshold tuning       : NO"
)

print(
    "Post-test retraining        : NO"
)


print(
    "\nModel bootstrap CI:"
)

print(
    MODEL_CI_PATH
)


print(
    "\nPairwise model differences:"
)

print(
    PAIRWISE_PATH
)


print(
    "\nBootstrap distributions:"
)

print(
    DISTRIBUTIONS_PATH
)


print(
    "\nManifest:"
)

print(
    MANIFEST_PATH
)


print(
    "\nNEXT: ERROR TAXONOMY + QUALITATIVE FAILURE ANALYSIS."
)

print(
    "=" * 86
)

In [ ]:
# ============================================================
# STAGE 7C-FIX — VERIFY COMPLETED BOOTSTRAP
#                 + REPAIR MANIFEST SAFELY
#
# NO bootstrap recomputation
# NO inference
# NO threshold change
# NO retraining
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import json
import hashlib
import shutil
import pandas as pd
import numpy as np


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

ANALYSIS = ROOT / "06_analysis"
MANIFESTS = ROOT / "07_manifests"
TEST = ROOT / "05_locked_test"

MODEL_CI_PATH = (
    ANALYSIS /
    "bootstrap_model_global_f1_ci.csv"
)

PAIRWISE_PATH = (
    ANALYSIS /
    "bootstrap_pairwise_global_f1_differences.csv"
)

FORMATTED_MODEL_PATH = (
    ANALYSIS /
    "bootstrap_model_global_f1_ci_formatted.csv"
)

FORMATTED_PAIRWISE_PATH = (
    ANALYSIS /
    "bootstrap_pairwise_global_f1_formatted.csv"
)

DISTRIBUTIONS_PATH = (
    ANALYSIS /
    "bootstrap_global_f1_distributions.csv.gz"
)

TEST_SUMMARY_PATH = (
    TEST /
    "locked_test_summary_all_runs.csv"
)

OLD_MANIFEST_PATH = (
    MANIFESTS /
    "STAGE_7C_BOOTSTRAP.json"
)

FIXED_MANIFEST_PATH = (
    MANIFESTS /
    "STAGE_7C_BOOTSTRAP.json"
)


# ============================================================
# 2. HASH HELPERS
# ============================================================

def sha256_file(path, chunk=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(chunk),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def dataframe_content_hash(df):
    """
    Deterministic scientific-content hash.
    Independent of gzip container timestamp/metadata.
    """

    normalized = df.copy()

    # Preserve row and column order exactly
    csv_bytes = normalized.to_csv(
        index=False,
        lineterminator="\n"
    ).encode("utf-8")

    return hashlib.sha256(
        csv_bytes
    ).hexdigest()


# ============================================================
# 3. REQUIRED OUTPUTS
# ============================================================

required = [
    MODEL_CI_PATH,
    PAIRWISE_PATH,
    FORMATTED_MODEL_PATH,
    FORMATTED_PAIRWISE_PATH,
    DISTRIBUTIONS_PATH,
    TEST_SUMMARY_PATH,
]

for p in required:

    if not p.exists():

        raise FileNotFoundError(
            f"Required completed Stage 7C output missing:\n{p}"
        )


print("=" * 82)
print("STAGE 7C-FIX — BOOTSTRAP OUTPUT VERIFICATION")
print("=" * 82)


# ============================================================
# 4. LOAD BOOTSTRAP RESULTS
# ============================================================

model_ci = pd.read_csv(
    MODEL_CI_PATH
)

pairwise = pd.read_csv(
    PAIRWISE_PATH
)

dist = pd.read_csv(
    DISTRIBUTIONS_PATH,
    compression="gzip"
)


# ============================================================
# 5. STRUCTURAL CHECKS
# ============================================================

if len(model_ci) != 8:

    raise RuntimeError(
        f"Expected 8 model-level bootstrap rows; "
        f"found {len(model_ci)}."
    )


if len(pairwise) != 12:

    raise RuntimeError(
        f"Expected 12 within-protocol pairwise rows; "
        f"found {len(pairwise)}."
    )


expected_distribution_rows = (
    2       # protocols
    * 4     # models
    * 5000  # bootstrap replicates
)


if len(dist) != expected_distribution_rows:

    raise RuntimeError(
        f"Expected {expected_distribution_rows} "
        f"bootstrap distribution rows; "
        f"found {len(dist)}."
    )


print(
    "Model bootstrap rows       : 8/8"
)

print(
    "Pairwise comparisons       : 12/12"
)

print(
    "Bootstrap distribution rows:",
    f"{len(dist)}/{expected_distribution_rows}"
)


# ============================================================
# 6. CHECK MODEL/PROTOCOL STRUCTURE
# ============================================================

protocols = {
    "random",
    "batch_disjoint"
}

models = {
    "yolov8s",
    "yolo11s",
    "yolo26s",
    "rtdetr_l"
}


if set(
    model_ci["protocol"]
) != protocols:

    raise RuntimeError(
        "Unexpected protocols in bootstrap model table."
    )


if set(
    model_ci["model"]
) != models:

    raise RuntimeError(
        "Unexpected models in bootstrap model table."
    )


for protocol in protocols:

    p = model_ci[
        model_ci["protocol"]
        == protocol
    ]

    if len(p) != 4:

        raise RuntimeError(
            f"{protocol}: expected 4 models."
        )


print(
    "Model/protocol structure   : PASSED"
)


# ============================================================
# 7. VERIFY 5000 REPLICATES PER MODEL
# ============================================================

replicate_check = (
    dist
    .groupby(
        ["protocol", "model"]
    )["bootstrap_id"]
    .nunique()
)


if not (
    replicate_check == 5000
).all():

    print(
        replicate_check
    )

    raise RuntimeError(
        "Not every model has exactly 5000 "
        "bootstrap replicates."
    )


print(
    "Bootstrap replicates       : 5000/model PASSED"
)


# ============================================================
# 8. VERIFY CI ORDER
# ============================================================

if not (
    model_ci[
        "bootstrap_ci_low_95"
    ]
    <=
    model_ci[
        "point_estimate_seed_mean_global_f1"
    ]
).all():

    raise RuntimeError(
        "At least one F1 point estimate lies "
        "below its bootstrap CI."
    )


if not (
    model_ci[
        "point_estimate_seed_mean_global_f1"
    ]
    <=
    model_ci[
        "bootstrap_ci_high_95"
    ]
).all():

    raise RuntimeError(
        "At least one F1 point estimate lies "
        "above its bootstrap CI."
    )


print(
    "Bootstrap CI sanity        : PASSED"
)


# ============================================================
# 9. RECOMPUTE DISTRIBUTION SUMMARY
# ============================================================

verification_rows = []


for r in model_ci.itertuples():

    d = dist[
        (dist["protocol"] == r.protocol)
        &
        (dist["model"] == r.model)
    ][
        "seed_mean_global_f1"
    ].to_numpy(
        dtype=float
    )


    calc_mean = float(
        np.mean(d)
    )

    calc_sd = float(
        np.std(
            d,
            ddof=1
        )
    )

    calc_low = float(
        np.quantile(
            d,
            0.025
        )
    )

    calc_high = float(
        np.quantile(
            d,
            0.975
        )
    )


    verification_rows.append({

        "protocol":
            r.protocol,

        "model":
            r.model,

        "stored_mean":
            float(
                r.bootstrap_mean
            ),

        "recomputed_mean":
            calc_mean,

        "mean_match":
            np.isclose(
                float(r.bootstrap_mean),
                calc_mean,
                atol=1e-12,
                rtol=0
            ),

        "stored_sd":
            float(
                r.bootstrap_sd
            ),

        "recomputed_sd":
            calc_sd,

        "sd_match":
            np.isclose(
                float(r.bootstrap_sd),
                calc_sd,
                atol=1e-12,
                rtol=0
            ),

        "stored_ci_low":
            float(
                r.bootstrap_ci_low_95
            ),

        "recomputed_ci_low":
            calc_low,

        "ci_low_match":
            np.isclose(
                float(r.bootstrap_ci_low_95),
                calc_low,
                atol=1e-12,
                rtol=0
            ),

        "stored_ci_high":
            float(
                r.bootstrap_ci_high_95
            ),

        "recomputed_ci_high":
            calc_high,

        "ci_high_match":
            np.isclose(
                float(r.bootstrap_ci_high_95),
                calc_high,
                atol=1e-12,
                rtol=0
            ),
    })


verification = pd.DataFrame(
    verification_rows
)


display(
    verification
)


check_cols = [
    "mean_match",
    "sd_match",
    "ci_low_match",
    "ci_high_match",
]


if not verification[
    check_cols
].all().all():

    raise RuntimeError(
        "Bootstrap distribution does not reproduce "
        "the saved summary exactly."
    )


print(
    "\nBootstrap distribution -> CI table: PASSED"
)


# ============================================================
# 10. SCIENTIFIC CONTENT HASH OF GZIP DATA
#
# This is the important correction:
# hash decompressed dataframe content, not gzip container.
# ============================================================

distribution_content_hash = (
    dataframe_content_hash(
        dist
    )
)


print(
    "\nDistribution scientific-content SHA-256:"
)

print(
    distribution_content_hash
)


# ============================================================
# 11. BACK UP OLD MANIFEST
# ============================================================

if OLD_MANIFEST_PATH.exists():

    OLD_MANIFEST_BACKUP = (
        MANIFESTS /
        "STAGE_7C_BOOTSTRAP_previous_manifest.json"
    )


    if not OLD_MANIFEST_BACKUP.exists():

        shutil.copy2(
            OLD_MANIFEST_PATH,
            OLD_MANIFEST_BACKUP
        )


    print(
        "\nPrevious Stage 7C manifest preserved:"
    )

    print(
        OLD_MANIFEST_BACKUP
    )


# ============================================================
# 12. BUILD CORRECTED MANIFEST
# ============================================================

manifest = {

    "stage":
        "7C",

    "status":
        "PAIRED_IMAGE_BOOTSTRAP_COMPLETED",

    "manifest_revision":
        "7C-fix1",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_locked_test_summary":
        str(
            TEST_SUMMARY_PATH
        ),

    "source_locked_test_summary_sha256":
        sha256_file(
            TEST_SUMMARY_PATH
        ),

    "metric":
        (
            "global F1 at IoU=0.50 using "
            "validation-frozen thresholds"
        ),

    "bootstrap_replicates":
        5000,

    "bootstrap_random_seed":
        20260815,

    "confidence_interval":
        "percentile 95%",

    "resampling_unit":
        "image",

    "within_protocol_pairing":
        True,

    "between_protocol_bootstrap":
        False,

    "observed_seeds": [
        42,
        123,
        2026
    ],

    "seed_handling":
        (
            "Three observed trained seeds were held fixed. "
            "Within each image bootstrap replicate, global "
            "F1 was recomputed separately for each seed "
            "and then averaged across the three seeds."
        ),

    "uncertainty_interpretation":
        (
            "Bootstrap confidence intervals quantify "
            "image-sampling uncertainty conditional on "
            "the three observed trained seeds. "
            "Between-seed variability is reported "
            "separately as sample SD."
        ),

    "dependency_note":
        (
            "Image-level resampling does not explicitly "
            "model within-batch dependence. Results are "
            "therefore treated as secondary uncertainty "
            "analysis rather than population-level "
            "inference over unseen batches."
        ),

    "pairwise_policy":
        (
            "All six model pairs were evaluated within "
            "each protocol using identical bootstrap "
            "image samples. No cross-protocol paired "
            "bootstrap was performed."
        ),

    "formal_p_values":
        False,

    "multiple_testing_claim":
        False,

    "test_threshold_tuning":
        False,

    "post_test_retraining":
        False,

    "gzip_hash_note":
        (
            "The gzip container SHA-256 is not used as "
            "the scientific-content identity because gzip "
            "metadata may change across equivalent writes. "
            "A deterministic hash of the decompressed CSV "
            "content is stored instead."
        ),

    "output_sha256": {

        "bootstrap_model_global_f1_ci.csv":
            sha256_file(
                MODEL_CI_PATH
            ),

        "bootstrap_pairwise_global_f1_differences.csv":
            sha256_file(
                PAIRWISE_PATH
            ),

        "bootstrap_model_global_f1_ci_formatted.csv":
            sha256_file(
                FORMATTED_MODEL_PATH
            ),

        "bootstrap_pairwise_global_f1_formatted.csv":
            sha256_file(
                FORMATTED_PAIRWISE_PATH
            ),
    },

    "bootstrap_distribution_content_sha256":
        distribution_content_hash,

    "bootstrap_distribution_rows":
        len(dist),
}


# ============================================================
# 13. WRITE CORRECTED MANIFEST
# ============================================================

FIXED_MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ============================================================
# 14. VERIFY WRITTEN MANIFEST
# ============================================================

check_manifest = json.loads(
    FIXED_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    check_manifest[
        "bootstrap_distribution_content_sha256"
    ]
    !=
    distribution_content_hash
):

    raise RuntimeError(
        "Corrected manifest verification failed."
    )


# ============================================================
# 15. FINAL
# ============================================================

print(
    "\n"
    + "=" * 82
)

print(
    "STAGE 7C-FIX PASSED"
)

print(
    "=" * 82
)

print(
    "Bootstrap recomputed       : NO"
)

print(
    "Inference rerun            : NO"
)

print(
    "Model CI rows              : 8/8"
)

print(
    "Pairwise comparisons       : 12/12"
)

print(
    "Bootstrap replicates       : 5000/model"
)

print(
    "Distribution rows          : 40000/40000"
)

print(
    "Scientific content checked : YES"
)

print(
    "Manifest repaired          : YES"
)

print(
    "Threshold tuning           : NO"
)

print(
    "Retraining                 : NO"
)

print(
    "\nCorrected manifest:"
)

print(
    FIXED_MANIFEST_PATH
)

print(
    "\nSTAGE 7C IS NOW COMPLETE."
)

print(
    "=" * 82
)

## STAGE 8 — Analysis package

This stage is descriptive/statistical analysis of already-frozen locked-test outputs. It does not retune any model.

In [ ]:
# ============================================================
# STAGE 8A — LOCKED-TEST ERROR TAXONOMY
#
# PURPOSE
# ------------------------------------------------------------
# Analyse completed locked-test predictions WITHOUT:
# - inference
# - threshold tuning
# - retraining
# - model exclusion
#
# Error taxonomy at the frozen validation operating point:
#
# TP:
#   prediction matched to previously unmatched GT
#   at IoU >= 0.50
#
# Duplicate FP:
#   prediction overlaps GT at IoU >= 0.50,
#   but that GT has already been matched
#
# Localization FP:
#   unmatched prediction with
#   0.10 <= best IoU < 0.50
#
# Background-like FP:
#   unmatched prediction with best IoU < 0.10
#   OR prediction in image with no GT object
#
# FN:
#   unmatched GT after prediction matching
#
# Additionally:
# - TP/FN by COCO object size
# - per-image error counts
# - detailed error-event registry for later visualization
# - 3-seed mean ± SD summaries
#
# NOTE:
# "background-like" does NOT imply a manually labelled
# background class. It means the prediction has little/no
# overlap with annotated litter.
# ============================================================

from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone

import json
import hashlib
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

LOCAL_DATASET = Path(
    "/content/taco_v5/dataset"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 2. EXPERIMENT STRUCTURE
# ============================================================

protocols = [
    "random",
    "batch_disjoint"
]

models = [
    "yolov8s",
    "yolo11s",
    "yolo26s",
    "rtdetr_l"
]

seeds = [
    42,
    123,
    2026
]

IOU_TP = 0.50

IOU_LOCALIZATION_MIN = 0.10


# ============================================================
# 3. FILES
# ============================================================

TEST_SUMMARY_PATH = (
    DIRS["test"] /
    "locked_test_summary_all_runs.csv"
)

TEST_COMPLETE_PATH = (
    DIRS["manifests"] /
    "LOCKED_TEST_COMPLETED.json"
)

STAGE7B_PATH = (
    DIRS["manifests"] /
    "STAGE_7B_FINAL_TEST_AGGREGATION.json"
)

STAGE7C_PATH = (
    DIRS["manifests"] /
    "STAGE_7C_BOOTSTRAP.json"
)

THRESHOLD_PATH = (
    DIRS["manifests"] /
    "validation_thresholds_frozen.json"
)


required_files = [
    TEST_SUMMARY_PATH,
    TEST_COMPLETE_PATH,
    STAGE7B_PATH,
    STAGE7C_PATH,
    THRESHOLD_PATH,
]


for p in required_files:

    if not p.exists():

        raise FileNotFoundError(
            f"Required input missing:\n{p}"
        )


print("=" * 88)
print("STAGE 8A — LOCKED-TEST ERROR TAXONOMY")
print("=" * 88)


# ============================================================
# 4. HELPERS
# ============================================================

def sha256_file(
    path,
    chunk=1024 * 1024
):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(chunk),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def xywh_to_xyxy(b):

    x, y, w, h = map(
        float,
        b
    )

    return np.array(
        [
            x,
            y,
            x + w,
            y + h
        ],
        dtype=float
    )


def box_iou(a, b):

    x1 = max(
        a[0],
        b[0]
    )

    y1 = max(
        a[1],
        b[1]
    )

    x2 = min(
        a[2],
        b[2]
    )

    y2 = min(
        a[3],
        b[3]
    )


    inter = (
        max(
            0.0,
            x2 - x1
        )
        *
        max(
            0.0,
            y2 - y1
        )
    )


    aa = (
        max(
            0.0,
            a[2] - a[0]
        )
        *
        max(
            0.0,
            a[3] - a[1]
        )
    )


    bb = (
        max(
            0.0,
            b[2] - b[0]
        )
        *
        max(
            0.0,
            b[3] - b[1]
        )
    )


    union = (
        aa
        +
        bb
        -
        inter
    )


    return (
        inter / union
        if union > 0
        else 0.0
    )


def coco_size_class(area):
    """
    Standard COCO area definitions:
      small  : area < 32^2
      medium : 32^2 <= area < 96^2
      large  : area >= 96^2
    """

    area = float(
        area
    )


    if area < (32 ** 2):

        return "small"


    if area < (96 ** 2):

        return "medium"


    return "large"


def safe_ratio(
    numerator,
    denominator
):

    if denominator == 0:

        return np.nan


    return float(
        numerator
        /
        denominator
    )


# ============================================================
# 5. VERIFY TEST SUMMARY INTEGRITY
# ============================================================

test_df = pd.read_csv(
    TEST_SUMMARY_PATH
)

completion = json.loads(
    TEST_COMPLETE_PATH.read_text(
        encoding="utf-8"
    )
)

thresholds = json.loads(
    THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)


if len(test_df) != 24:

    raise RuntimeError(
        f"Expected 24 locked-test rows; "
        f"found {len(test_df)}."
    )


expected_test_hash = completion.get(
    "test_summary_sha256"
)

current_test_hash = sha256_file(
    TEST_SUMMARY_PATH
)


if (
    expected_test_hash
    != current_test_hash
):

    raise RuntimeError(
        "Locked-test summary has changed after completion."
    )


if len(thresholds) != 24:

    raise RuntimeError(
        "Frozen threshold manifest is incomplete."
    )


print(
    "Locked-test summary integrity : PASSED"
)

print(
    "Frozen thresholds              : 24/24"
)


# ============================================================
# 6. OUTPUT COLLECTIONS
# ============================================================

run_rows = []

per_image_rows = []

event_rows = []


# ============================================================
# 7. LOOP OVER ALL 24 LOCKED-TEST RUNS
# ============================================================

run_number = 0


for protocol in protocols:

    gt_path = (
        LOCAL_DATASET /
        protocol /
        "test_coco_taco1.json"
    )


    if not gt_path.exists():

        raise FileNotFoundError(
            gt_path
        )


    gt = json.loads(
        gt_path.read_text(
            encoding="utf-8"
        )
    )


    # --------------------------------------------------------
    # Image registry
    # --------------------------------------------------------

    images_by_id = {

        int(im["id"]):
            im

        for im in gt["images"]
    }


    # --------------------------------------------------------
    # Ground truth by image
    # --------------------------------------------------------

    gt_by_image = defaultdict(
        list
    )


    for ann in gt[
        "annotations"
    ]:

        iid = int(
            ann["image_id"]
        )


        area = float(
            ann.get(
                "area",
                float(
                    ann["bbox"][2]
                    *
                    ann["bbox"][3]
                )
            )
        )


        gt_by_image[
            iid
        ].append({

            "annotation_id":
                int(
                    ann.get(
                        "id",
                        -1
                    )
                ),

            "bbox_xywh":
                [
                    float(x)
                    for x in ann["bbox"]
                ],

            "bbox_xyxy":
                xywh_to_xyxy(
                    ann["bbox"]
                ),

            "area":
                area,

            "size":
                coco_size_class(
                    area
                ),
        })


    # ========================================================
    # MODEL / SEED
    # ========================================================

    for model in models:

        for seed in seeds:

            run_number += 1


            print(
                "\n"
                + "=" * 88
            )

            print(
                f"[{run_number}/24] "
                f"{protocol}/"
                f"{model}_seed{seed}"
            )

            print(
                "=" * 88
            )


            # ------------------------------------------------
            # Frozen threshold
            # ------------------------------------------------

            threshold_key = (
                f"{protocol}/"
                f"{model}/"
                f"seed{seed}"
            )


            if threshold_key not in thresholds:

                raise KeyError(
                    f"Frozen threshold missing: "
                    f"{threshold_key}"
                )


            threshold = float(
                thresholds[
                    threshold_key
                ]
            )


            # ------------------------------------------------
            # Prediction file
            # ------------------------------------------------

            pred_path = (
                DIRS["test"] /
                protocol /
                f"{model}_seed{seed}" /
                "test_predictions_coco.json"
            )


            if not pred_path.exists():

                raise FileNotFoundError(
                    pred_path
                )


            predictions = json.loads(
                pred_path.read_text(
                    encoding="utf-8"
                )
            )


            # ------------------------------------------------
            # Predictions at FROZEN operating threshold
            # ------------------------------------------------

            pred_by_image = defaultdict(
                list
            )


            for pred_index, p in enumerate(
                predictions
            ):

                score = float(
                    p["score"]
                )


                if score < threshold:

                    continue


                iid = int(
                    p["image_id"]
                )


                if iid not in images_by_id:

                    raise RuntimeError(
                        f"Prediction has invalid image_id "
                        f"{iid}: {pred_path}"
                    )


                pred_by_image[
                    iid
                ].append({

                    "prediction_index":
                        int(
                            pred_index
                        ),

                    "score":
                        score,

                    "bbox_xywh":
                        [
                            float(x)
                            for x in p["bbox"]
                        ],

                    "bbox_xyxy":
                        xywh_to_xyxy(
                            p["bbox"]
                        ),
                })


            # ------------------------------------------------
            # RUN COUNTERS
            # ------------------------------------------------

            TP = 0

            duplicate_fp = 0

            localization_fp = 0

            background_fp = 0

            FN = 0


            tp_small = 0
            tp_medium = 0
            tp_large = 0

            fn_small = 0
            fn_medium = 0
            fn_large = 0


            matched_iou_values = []


            # =================================================
            # IMAGE-BY-IMAGE MATCHING
            #
            # IMPORTANT:
            # This mirrors detection_counts() matching logic.
            # =================================================

            for iid in sorted(
                images_by_id.keys()
            ):

                image_info = (
                    images_by_id[
                        iid
                    ]
                )


                file_name = str(
                    image_info.get(
                        "file_name",
                        ""
                    )
                )


                gts = (
                    gt_by_image.get(
                        iid,
                        []
                    )
                )


                prs = sorted(
                    pred_by_image.get(
                        iid,
                        []
                    ),
                    key=lambda z:
                        -z["score"]
                )


                matched_gt = set()


                image_tp = 0

                image_dup = 0

                image_loc = 0

                image_bg = 0


                # ---------------------------------------------
                # Prediction classification
                # ---------------------------------------------

                for p in prs:

                    pb = p[
                        "bbox_xyxy"
                    ]


                    # -----------------------------------------
                    # No GT in image -> background-like FP
                    # -----------------------------------------

                    if not gts:

                        background_fp += 1

                        image_bg += 1


                        event_rows.append({

                            "protocol":
                                protocol,

                            "model":
                                model,

                            "seed":
                                seed,

                            "image_id":
                                iid,

                            "file_name":
                                file_name,

                            "event":
                                "background_fp",

                            "score":
                                p["score"],

                            "best_iou":
                                0.0,

                            "prediction_bbox_x":
                                p["bbox_xywh"][0],

                            "prediction_bbox_y":
                                p["bbox_xywh"][1],

                            "prediction_bbox_w":
                                p["bbox_xywh"][2],

                            "prediction_bbox_h":
                                p["bbox_xywh"][3],

                            "gt_annotation_id":
                                np.nan,

                            "gt_size":
                                None,
                        })


                        continue


                    # -----------------------------------------
                    # IoU against all GTs
                    # -----------------------------------------

                    ious = [

                        box_iou(
                            pb,
                            gt_item[
                                "bbox_xyxy"
                            ]
                        )

                        for gt_item in gts
                    ]


                    order = np.argsort(
                        ious
                    )[::-1]


                    assigned_gt = None

                    assigned_iou = None


                    # -----------------------------------------
                    # TP:
                    # best available unmatched GT >= 0.50
                    # -----------------------------------------

                    for gi in order:

                        gi = int(
                            gi
                        )


                        if (
                            ious[gi]
                            >= IOU_TP
                            and
                            gi not in matched_gt
                        ):

                            assigned_gt = gi

                            assigned_iou = float(
                                ious[gi]
                            )

                            matched_gt.add(
                                gi
                            )

                            break


                    if assigned_gt is not None:

                        TP += 1

                        image_tp += 1

                        matched_iou_values.append(
                            assigned_iou
                        )


                        gt_size = (
                            gts[
                                assigned_gt
                            ][
                                "size"
                            ]
                        )


                        if gt_size == "small":

                            tp_small += 1

                        elif gt_size == "medium":

                            tp_medium += 1

                        else:

                            tp_large += 1


                        event_rows.append({

                            "protocol":
                                protocol,

                            "model":
                                model,

                            "seed":
                                seed,

                            "image_id":
                                iid,

                            "file_name":
                                file_name,

                            "event":
                                "tp",

                            "score":
                                p["score"],

                            "best_iou":
                                assigned_iou,

                            "prediction_bbox_x":
                                p["bbox_xywh"][0],

                            "prediction_bbox_y":
                                p["bbox_xywh"][1],

                            "prediction_bbox_w":
                                p["bbox_xywh"][2],

                            "prediction_bbox_h":
                                p["bbox_xywh"][3],

                            "gt_annotation_id":
                                gts[
                                    assigned_gt
                                ][
                                    "annotation_id"
                                ],

                            "gt_size":
                                gt_size,
                        })


                        continue


                    # -----------------------------------------
                    # Prediction is unmatched.
                    # Classify FP.
                    # -----------------------------------------

                    best_iou = float(
                        max(
                            ious
                        )
                    )


                    best_gt_index = int(
                        np.argmax(
                            ious
                        )
                    )


                    best_gt = gts[
                        best_gt_index
                    ]


                    # -----------------------------------------
                    # Duplicate FP
                    # -----------------------------------------

                    if best_iou >= IOU_TP:

                        duplicate_fp += 1

                        image_dup += 1

                        event_type = (
                            "duplicate_fp"
                        )


                    # -----------------------------------------
                    # Localization error
                    # -----------------------------------------

                    elif (
                        best_iou
                        >=
                        IOU_LOCALIZATION_MIN
                    ):

                        localization_fp += 1

                        image_loc += 1

                        event_type = (
                            "localization_fp"
                        )


                    # -----------------------------------------
                    # Background-like FP
                    # -----------------------------------------

                    else:

                        background_fp += 1

                        image_bg += 1

                        event_type = (
                            "background_fp"
                        )


                    event_rows.append({

                        "protocol":
                            protocol,

                        "model":
                            model,

                        "seed":
                            seed,

                        "image_id":
                            iid,

                        "file_name":
                            file_name,

                        "event":
                            event_type,

                        "score":
                            p["score"],

                        "best_iou":
                            best_iou,

                        "prediction_bbox_x":
                            p["bbox_xywh"][0],

                        "prediction_bbox_y":
                            p["bbox_xywh"][1],

                        "prediction_bbox_w":
                            p["bbox_xywh"][2],

                        "prediction_bbox_h":
                            p["bbox_xywh"][3],

                        "gt_annotation_id":
                            best_gt[
                                "annotation_id"
                            ],

                        "gt_size":
                            best_gt[
                                "size"
                            ],
                    })


                # ---------------------------------------------
                # FALSE NEGATIVES
                # ---------------------------------------------

                unmatched_gt_indices = [

                    gi

                    for gi in range(
                        len(
                            gts
                        )
                    )

                    if gi not in matched_gt
                ]


                image_fn = len(
                    unmatched_gt_indices
                )


                FN += image_fn


                for gi in unmatched_gt_indices:

                    gt_item = gts[
                        gi
                    ]

                    size = gt_item[
                        "size"
                    ]


                    if size == "small":

                        fn_small += 1

                    elif size == "medium":

                        fn_medium += 1

                    else:

                        fn_large += 1


                    # -----------------------------------------
                    # Diagnostic max IoU among predictions
                    # active at frozen threshold
                    # -----------------------------------------

                    if prs:

                        max_active_iou = float(
                            max(
                                box_iou(
                                    gt_item[
                                        "bbox_xyxy"
                                    ],
                                    p[
                                        "bbox_xyxy"
                                    ]
                                )

                                for p in prs
                            )
                        )

                    else:

                        max_active_iou = 0.0


                    event_rows.append({

                        "protocol":
                            protocol,

                        "model":
                            model,

                        "seed":
                            seed,

                        "image_id":
                            iid,

                        "file_name":
                            file_name,

                        "event":
                            "fn",

                        "score":
                            np.nan,

                        "best_iou":
                            max_active_iou,

                        "prediction_bbox_x":
                            np.nan,

                        "prediction_bbox_y":
                            np.nan,

                        "prediction_bbox_w":
                            np.nan,

                        "prediction_bbox_h":
                            np.nan,

                        "gt_annotation_id":
                            gt_item[
                                "annotation_id"
                            ],

                        "gt_size":
                            size,
                    })


                # ---------------------------------------------
                # Per-image record
                # ---------------------------------------------

                per_image_rows.append({

                    "protocol":
                        protocol,

                    "model":
                        model,

                    "seed":
                        seed,

                    "image_id":
                        iid,

                    "file_name":
                        file_name,

                    "n_gt":
                        len(
                            gts
                        ),

                    "n_predictions_at_frozen_threshold":
                        len(
                            prs
                        ),

                    "tp":
                        image_tp,

                    "duplicate_fp":
                        image_dup,

                    "localization_fp":
                        image_loc,

                    "background_fp":
                        image_bg,

                    "fp":
                        (
                            image_dup
                            +
                            image_loc
                            +
                            image_bg
                        ),

                    "fn":
                        image_fn,

                    "total_errors":
                        (
                            image_dup
                            +
                            image_loc
                            +
                            image_bg
                            +
                            image_fn
                        ),
                })


            # =================================================
            # 8. RUN TOTALS
            # =================================================

            FP = (
                duplicate_fp
                +
                localization_fp
                +
                background_fp
            )


            n_gt = len(
                gt["annotations"]
            )


            # ------------------------------------------------
            # Compare with Stage 7A summary
            # ------------------------------------------------

            sr = test_df[
                (
                    test_df[
                        "protocol"
                    ] == protocol
                )
                &
                (
                    test_df[
                        "model"
                    ] == model
                )
                &
                (
                    test_df[
                        "seed"
                    ].astype(int) == seed
                )
            ]


            if len(sr) != 1:

                raise RuntimeError(
                    f"Missing Stage 7A summary row: "
                    f"{protocol}/{model}/seed{seed}"
                )


            sr = sr.iloc[0]


            expected_tp = int(
                sr["TP"]
            )

            expected_fp = int(
                sr["FP"]
            )

            expected_fn = int(
                sr["FN"]
            )


            if (
                TP != expected_tp
                or
                FP != expected_fp
                or
                FN != expected_fn
            ):

                raise RuntimeError(
                    "\nERROR TAXONOMY DOES NOT REPRODUCE "
                    "STAGE 7A COUNTS\n"
                    f"Run: {protocol}/{model}/seed{seed}\n"
                    f"Stage 8A: TP={TP}, FP={FP}, FN={FN}\n"
                    f"Stage 7A: TP={expected_tp}, "
                    f"FP={expected_fp}, FN={expected_fn}\n"
                    "STOP: matching logic must not be changed."
                )


            # ------------------------------------------------
            # GT accounting check
            # ------------------------------------------------

            if (
                TP
                +
                FN
                !=
                n_gt
            ):

                raise RuntimeError(
                    f"GT accounting failed: "
                    f"{protocol}/{model}/seed{seed}"
                )


            # ------------------------------------------------
            # FP accounting check
            # ------------------------------------------------

            if (
                duplicate_fp
                +
                localization_fp
                +
                background_fp
                !=
                FP
            ):

                raise RuntimeError(
                    f"FP taxonomy accounting failed: "
                    f"{protocol}/{model}/seed{seed}"
                )


            # ------------------------------------------------
            # Size accounting
            # ------------------------------------------------

            if (
                tp_small
                +
                tp_medium
                +
                tp_large
                !=
                TP
            ):

                raise RuntimeError(
                    f"TP size accounting failed: "
                    f"{protocol}/{model}/seed{seed}"
                )


            if (
                fn_small
                +
                fn_medium
                +
                fn_large
                !=
                FN
            ):

                raise RuntimeError(
                    f"FN size accounting failed: "
                    f"{protocol}/{model}/seed{seed}"
                )


            # ------------------------------------------------
            # Mean IoU among matched TPs
            # ------------------------------------------------

            mean_tp_iou = (
                float(
                    np.mean(
                        matched_iou_values
                    )
                )
                if matched_iou_values
                else np.nan
            )


            # ------------------------------------------------
            # Run summary
            # ------------------------------------------------

            run_rows.append({

                "protocol":
                    protocol,

                "model":
                    model,

                "seed":
                    seed,

                "frozen_threshold":
                    threshold,

                "n_test_images":
                    len(
                        gt["images"]
                    ),

                "n_gt_objects":
                    n_gt,

                "TP":
                    TP,

                "FP":
                    FP,

                "FN":
                    FN,

                "duplicate_fp":
                    duplicate_fp,

                "localization_fp":
                    localization_fp,

                "background_fp":
                    background_fp,

                "duplicate_share_of_fp":
                    safe_ratio(
                        duplicate_fp,
                        FP
                    ),

                "localization_share_of_fp":
                    safe_ratio(
                        localization_fp,
                        FP
                    ),

                "background_share_of_fp":
                    safe_ratio(
                        background_fp,
                        FP
                    ),

                "fn_rate":
                    safe_ratio(
                        FN,
                        n_gt
                    ),

                "tp_rate_gt":
                    safe_ratio(
                        TP,
                        n_gt
                    ),

                "mean_tp_iou":
                    mean_tp_iou,

                "TP_small":
                    tp_small,

                "TP_medium":
                    tp_medium,

                "TP_large":
                    tp_large,

                "FN_small":
                    fn_small,

                "FN_medium":
                    fn_medium,

                "FN_large":
                    fn_large,

                "small_fn_rate":
                    safe_ratio(
                        fn_small,
                        (
                            tp_small
                            +
                            fn_small
                        )
                    ),

                "medium_fn_rate":
                    safe_ratio(
                        fn_medium,
                        (
                            tp_medium
                            +
                            fn_medium
                        )
                    ),

                "large_fn_rate":
                    safe_ratio(
                        fn_large,
                        (
                            tp_large
                            +
                            fn_large
                        )
                    ),
            })


            print(
                f"Threshold     : {threshold:.2f}"
            )

            print(
                f"TP / FP / FN : "
                f"{TP} / {FP} / {FN}"
            )

            print(
                "FP taxonomy   : "
                f"duplicate={duplicate_fp}, "
                f"localization={localization_fp}, "
                f"background-like={background_fp}"
            )

            print(
                "FN by size    : "
                f"small={fn_small}, "
                f"medium={fn_medium}, "
                f"large={fn_large}"
            )

            print(
                "Stage 7A count reproduction: PASSED"
            )


# ============================================================
# 9. CREATE DATAFRAMES
# ============================================================

run_taxonomy = pd.DataFrame(
    run_rows
)

per_image = pd.DataFrame(
    per_image_rows
)

events = pd.DataFrame(
    event_rows
)


if len(
    run_taxonomy
) != 24:

    raise RuntimeError(
        "Expected 24 run taxonomy rows."
    )


print(
    "\n"
    + "=" * 88
)

print(
    "ALL 24 RUNS REPRODUCED EXACTLY"
)

print(
    "=" * 88
)


# ============================================================
# 10. AGGREGATE ERROR TAXONOMY ACROSS 3 SEEDS
#
# Each seed remains one experimental replicate.
# We report mean ± sample SD across seeds.
# ============================================================

aggregate_metrics = [

    "TP",
    "FP",
    "FN",

    "duplicate_fp",
    "localization_fp",
    "background_fp",

    "duplicate_share_of_fp",
    "localization_share_of_fp",
    "background_share_of_fp",

    "fn_rate",
    "tp_rate_gt",
    "mean_tp_iou",

    "TP_small",
    "TP_medium",
    "TP_large",

    "FN_small",
    "FN_medium",
    "FN_large",

    "small_fn_rate",
    "medium_fn_rate",
    "large_fn_rate",
]


aggregate_rows = []


for protocol in protocols:

    for model in models:

        g = run_taxonomy[
            (
                run_taxonomy[
                    "protocol"
                ] == protocol
            )
            &
            (
                run_taxonomy[
                    "model"
                ] == model
            )
        ]


        if len(g) != 3:

            raise RuntimeError(
                f"Expected three seeds: "
                f"{protocol}/{model}"
            )


        row = {

            "protocol":
                protocol,

            "model":
                model,

            "n_seeds":
                3,
        }


        for metric in aggregate_metrics:

            values = pd.to_numeric(
                g[metric],
                errors="coerce"
            )


            row[
                f"{metric}_mean"
            ] = float(
                values.mean()
            )


            row[
                f"{metric}_sd"
            ] = float(
                values.std(
                    ddof=1
                )
            )


        aggregate_rows.append(
            row
        )


aggregate = pd.DataFrame(
    aggregate_rows
)


# ============================================================
# 11. FORMATTED MANUSCRIPT ERROR TABLE
# ============================================================

def fmt(
    mean,
    sd,
    digits=2
):

    return (
        f"{mean:.{digits}f} "
        f"± {sd:.{digits}f}"
    )


formatted_rows = []


for r in aggregate.itertuples():

    formatted_rows.append({

        "Protocol":
            r.protocol,

        "Model":
            r.model,

        "TP":
            fmt(
                r.TP_mean,
                r.TP_sd
            ),

        "FP":
            fmt(
                r.FP_mean,
                r.FP_sd
            ),

        "FN":
            fmt(
                r.FN_mean,
                r.FN_sd
            ),

        "Duplicate FP":
            fmt(
                r.duplicate_fp_mean,
                r.duplicate_fp_sd
            ),

        "Localization FP":
            fmt(
                r.localization_fp_mean,
                r.localization_fp_sd
            ),

        "Background-like FP":
            fmt(
                r.background_fp_mean,
                r.background_fp_sd
            ),

        "Duplicate % of FP":
            fmt(
                100
                *
                r.duplicate_share_of_fp_mean,

                100
                *
                r.duplicate_share_of_fp_sd
            ),

        "Localization % of FP":
            fmt(
                100
                *
                r.localization_share_of_fp_mean,

                100
                *
                r.localization_share_of_fp_sd
            ),

        "Background-like % of FP":
            fmt(
                100
                *
                r.background_share_of_fp_mean,

                100
                *
                r.background_share_of_fp_sd
            ),

        "FN rate (%)":
            fmt(
                100
                *
                r.fn_rate_mean,

                100
                *
                r.fn_rate_sd
            ),

        "Small FN rate (%)":
            fmt(
                100
                *
                r.small_fn_rate_mean,

                100
                *
                r.small_fn_rate_sd
            ),

        "Medium FN rate (%)":
            fmt(
                100
                *
                r.medium_fn_rate_mean,

                100
                *
                r.medium_fn_rate_sd
            ),

        "Large FN rate (%)":
            fmt(
                100
                *
                r.large_fn_rate_mean,

                100
                *
                r.large_fn_rate_sd
            ),

        "Mean TP IoU":
            fmt(
                r.mean_tp_iou_mean,
                r.mean_tp_iou_sd,
                digits=4
            ),
    })


formatted = pd.DataFrame(
    formatted_rows
)


# ============================================================
# 12. IMAGE-LEVEL QUALITATIVE CANDIDATE TABLES
#
# No image is manually chosen here.
# Candidates are ranked deterministically from error counts.
# ============================================================

candidate_parts = []


for protocol in protocols:

    for model in models:

        for seed in seeds:

            g = per_image[
                (
                    per_image[
                        "protocol"
                    ] == protocol
                )
                &
                (
                    per_image[
                        "model"
                    ] == model
                )
                &
                (
                    per_image[
                        "seed"
                    ] == seed
                )
            ].copy()


            candidate_specs = {

                "highest_false_negative":
                    "fn",

                "highest_false_positive":
                    "fp",

                "highest_localization_error":
                    "localization_fp",

                "highest_duplicate_error":
                    "duplicate_fp",

                "highest_background_like_fp":
                    "background_fp",

                "highest_total_error":
                    "total_errors",
            }


            for candidate_type, metric in candidate_specs.items():

                top = (
                    g.sort_values(
                        [
                            metric,
                            "total_errors",
                            "image_id"
                        ],
                        ascending=[
                            False,
                            False,
                            True
                        ]
                    )
                    .head(
                        10
                    )
                    .copy()
                )


                top[
                    "candidate_type"
                ] = candidate_type


                top[
                    "ranking_metric"
                ] = metric


                top[
                    "ranking_value"
                ] = top[
                    metric
                ]


                candidate_parts.append(
                    top
                )


qualitative_candidates = pd.concat(
    candidate_parts,
    ignore_index=True
)


# ============================================================
# 13. CONSENSUS DIFFICULTY BY IMAGE
#
# For each protocol/model, average error counts across seeds.
# Useful for finding images that are consistently difficult,
# rather than difficult for only one seed.
# ============================================================

consensus = (

    per_image
    .groupby(
        [
            "protocol",
            "model",
            "image_id",
            "file_name"
        ],
        as_index=False
    )
    .agg(

        mean_tp=(
            "tp",
            "mean"
        ),

        mean_fp=(
            "fp",
            "mean"
        ),

        mean_fn=(
            "fn",
            "mean"
        ),

        mean_duplicate_fp=(
            "duplicate_fp",
            "mean"
        ),

        mean_localization_fp=(
            "localization_fp",
            "mean"
        ),

        mean_background_fp=(
            "background_fp",
            "mean"
        ),

        mean_total_errors=(
            "total_errors",
            "mean"
        ),

        max_total_errors=(
            "total_errors",
            "max"
        ),

        seeds_with_any_error=(
            "total_errors",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x)
                        > 0
                    )
                )
        )
    )
)


consensus = consensus.sort_values(

    [
        "protocol",
        "model",
        "seeds_with_any_error",
        "mean_total_errors",
        "image_id"
    ],

    ascending=[
        True,
        True,
        False,
        False,
        True
    ]
)


# ============================================================
# 14. SAVE OUTPUTS
# ============================================================

RUN_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_seed_level.csv"
)

AGG_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_mean_sd.csv"
)

FORMATTED_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_formatted.csv"
)

PER_IMAGE_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_per_image.csv"
)

EVENT_PATH = (
    DIRS["analysis"] /
    "locked_test_error_events.csv"
)

CANDIDATE_PATH = (
    DIRS["analysis"] /
    "qualitative_error_candidates.csv"
)

CONSENSUS_PATH = (
    DIRS["analysis"] /
    "qualitative_error_consensus_by_image.csv"
)


run_taxonomy.to_csv(
    RUN_PATH,
    index=False
)

aggregate.to_csv(
    AGG_PATH,
    index=False
)

formatted.to_csv(
    FORMATTED_PATH,
    index=False
)

per_image.to_csv(
    PER_IMAGE_PATH,
    index=False
)

events.to_csv(
    EVENT_PATH,
    index=False
)

qualitative_candidates.to_csv(
    CANDIDATE_PATH,
    index=False
)

consensus.to_csv(
    CONSENSUS_PATH,
    index=False
)


# ============================================================
# 15. DISPLAY MAIN ERROR TAXONOMY
# ============================================================

print(
    "\n"
    + "=" * 88
)

print(
    "LOCKED-TEST ERROR TAXONOMY — 3-SEED MEAN ± SD"
)

print(
    "=" * 88
)


display(
    formatted
)


# ============================================================
# 16. DISPLAY NUMERIC ERROR COMPOSITION
# ============================================================

display_columns = [

    "protocol",
    "model",

    "duplicate_fp_mean",
    "localization_fp_mean",
    "background_fp_mean",

    "duplicate_share_of_fp_mean",
    "localization_share_of_fp_mean",
    "background_share_of_fp_mean",

    "fn_rate_mean",

    "small_fn_rate_mean",
    "medium_fn_rate_mean",
    "large_fn_rate_mean",

    "mean_tp_iou_mean",
]


print(
    "\n"
    + "=" * 88
)

print(
    "ERROR COMPOSITION — NUMERIC"
)

print(
    "=" * 88
)


display(
    aggregate[
        display_columns
    ]
)


# ============================================================
# 17. DISPLAY CONSISTENTLY DIFFICULT IMAGES
#
# Only top 5 per protocol/model shown.
# Full table is saved.
# ============================================================

top_consensus_parts = []


for protocol in protocols:

    for model in models:

        top = (

            consensus[
                (
                    consensus[
                        "protocol"
                    ] == protocol
                )
                &
                (
                    consensus[
                        "model"
                    ] == model
                )
            ]

            .head(
                5
            )
        )


        top_consensus_parts.append(
            top
        )


top_consensus = pd.concat(
    top_consensus_parts,
    ignore_index=True
)


print(
    "\n"
    + "=" * 88
)

print(
    "TOP CONSISTENTLY DIFFICULT TEST IMAGES"
)

print(
    "Ranked using error counts across all three seeds."
)

print(
    "=" * 88
)


display(
    top_consensus
)


# ============================================================
# 18. AUTOMATIC SCIENTIFIC CHECKS
# ============================================================

print(
    "\n"
    + "=" * 88
)

print(
    "ERROR TAXONOMY CHECKS"
)

print(
    "=" * 88
)


# -----------------------------------------------
# FP shares should sum to 1 for runs with FP > 0
# -----------------------------------------------

for r in run_taxonomy.itertuples():

    if r.FP > 0:

        total_share = (
            r.duplicate_share_of_fp
            +
            r.localization_share_of_fp
            +
            r.background_share_of_fp
        )


        if not np.isclose(
            total_share,
            1.0,
            atol=1e-12,
            rtol=0
        ):

            raise RuntimeError(
                f"FP share check failed: "
                f"{r.protocol}/{r.model}/seed{r.seed}"
            )


print(
    "FP taxonomy partition       : 24/24 PASSED"
)


# -----------------------------------------------
# Confirm event totals
# -----------------------------------------------

for protocol in protocols:

    for model in models:

        for seed in seeds:

            er = events[
                (
                    events[
                        "protocol"
                    ] == protocol
                )
                &
                (
                    events[
                        "model"
                    ] == model
                )
                &
                (
                    events[
                        "seed"
                    ] == seed
                )
            ]


            rr = run_taxonomy[
                (
                    run_taxonomy[
                        "protocol"
                    ] == protocol
                )
                &
                (
                    run_taxonomy[
                        "model"
                    ] == model
                )
                &
                (
                    run_taxonomy[
                        "seed"
                    ] == seed
                )
            ].iloc[0]


            checks = {

                "tp":
                    int(rr["TP"]),

                "duplicate_fp":
                    int(rr["duplicate_fp"]),

                "localization_fp":
                    int(rr["localization_fp"]),

                "background_fp":
                    int(rr["background_fp"]),

                "fn":
                    int(rr["FN"]),
            }


            for event_name, expected in checks.items():

                actual = int(
                    (
                        er[
                            "event"
                        ]
                        == event_name
                    ).sum()
                )


                if actual != expected:

                    raise RuntimeError(
                        f"Event registry mismatch: "
                        f"{protocol}/{model}/seed{seed}/"
                        f"{event_name}: "
                        f"{actual} != {expected}"
                    )


print(
    "Detailed event registry     : 24/24 PASSED"
)


# ============================================================
# 19. OUTPUT HASHES
# ============================================================

output_paths = {

    "locked_test_error_taxonomy_seed_level.csv":
        RUN_PATH,

    "locked_test_error_taxonomy_mean_sd.csv":
        AGG_PATH,

    "locked_test_error_taxonomy_formatted.csv":
        FORMATTED_PATH,

    "locked_test_error_taxonomy_per_image.csv":
        PER_IMAGE_PATH,

    "locked_test_error_events.csv":
        EVENT_PATH,

    "qualitative_error_candidates.csv":
        CANDIDATE_PATH,

    "qualitative_error_consensus_by_image.csv":
        CONSENSUS_PATH,
}


output_hashes = {

    name:
        sha256_file(
            path
        )

    for name, path in output_paths.items()
}


# ============================================================
# 20. STAGE 8A MANIFEST
# ============================================================

MANIFEST_PATH = (
    DIRS["manifests"] /
    "STAGE_8A_ERROR_TAXONOMY.json"
)


manifest = {

    "stage":
        "8A",

    "status":
        "LOCKED_TEST_ERROR_TAXONOMY_COMPLETED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_locked_test_summary":
        str(
            TEST_SUMMARY_PATH
        ),

    "source_locked_test_summary_sha256":
        current_test_hash,

    "number_of_runs":
        24,

    "protocols":
        protocols,

    "models":
        models,

    "seeds":
        seeds,

    "operating_point":
        (
            "validation-frozen confidence threshold "
            "for each protocol/model/seed"
        ),

    "tp_iou_threshold":
        IOU_TP,

    "localization_iou_lower_bound":
        IOU_LOCALIZATION_MIN,

    "error_taxonomy": {

        "true_positive":
            (
                "Prediction matched to a previously "
                "unmatched GT at IoU >= 0.50."
            ),

        "duplicate_fp":
            (
                "Unmatched prediction with best GT IoU "
                ">= 0.50 after that GT had already been "
                "matched."
            ),

        "localization_fp":
            (
                "Unmatched prediction with "
                "0.10 <= best GT IoU < 0.50."
            ),

        "background_fp":
            (
                "Unmatched prediction with best GT IoU "
                "< 0.10, including predictions in images "
                "without GT objects."
            ),

        "false_negative":
            (
                "GT object remaining unmatched after "
                "prediction matching."
            ),
    },

    "object_size_definition":
        (
            "COCO area convention: "
            "small < 32^2 px^2; "
            "medium 32^2 to <96^2 px^2; "
            "large >=96^2 px^2."
        ),

    "error_count_reproduction":
        "24/24 runs exactly reproduce Stage 7A TP/FP/FN",

    "new_inference":
        False,

    "threshold_tuning":
        False,

    "post_test_retraining":
        False,

    "qualitative_selection":
        (
            "Candidate images are ranked deterministically "
            "from error counts. No manuscript example has "
            "yet been manually selected."
        ),

    "output_sha256":
        output_hashes,
}


serialized = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False
)


if MANIFEST_PATH.exists():

    previous = json.loads(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )


    old_cmp = dict(
        previous
    )

    new_cmp = dict(
        manifest
    )


    old_cmp.pop(
        "created_utc",
        None
    )

    new_cmp.pop(
        "created_utc",
        None
    )


    if old_cmp != new_cmp:

        raise RuntimeError(
            "A different STAGE_8A_ERROR_TAXONOMY.json "
            "already exists. "
            "It will NOT be overwritten."
        )


else:

    MANIFEST_PATH.write_text(
        serialized,
        encoding="utf-8"
    )


# ============================================================
# 21. FINAL
# ============================================================

print(
    "\n"
    + "=" * 88
)

print(
    "STAGE 8A PASSED"
)

print(
    "=" * 88
)

print(
    "Locked-test runs analysed   : 24/24"
)

print(
    "Stage 7A TP/FP/FN reproduced: 24/24"
)

print(
    "Duplicate FP analysis       : COMPLETE"
)

print(
    "Localization FP analysis    : COMPLETE"
)

print(
    "Background-like FP analysis : COMPLETE"
)

print(
    "False-negative analysis     : COMPLETE"
)

print(
    "FN by object size           : COMPLETE"
)

print(
    "Per-image registry          : COMPLETE"
)

print(
    "Detailed error events       : COMPLETE"
)

print(
    "Qualitative candidates      : GENERATED"
)

print(
    "New inference               : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Retraining                  : NO"
)


print(
    "\nMain error table:"
)

print(
    FORMATTED_PATH
)


print(
    "\nDetailed events:"
)

print(
    EVENT_PATH
)


print(
    "\nQualitative candidates:"
)

print(
    CANDIDATE_PATH
)


print(
    "\nConsensus difficult images:"
)

print(
    CONSENSUS_PATH
)


print(
    "\nManifest:"
)

print(
    MANIFEST_PATH
)


print(
    "\nNEXT: STAGE 8B — QUALITATIVE ERROR VISUALIZATION."
)

print(
    "=" * 88
)

In [ ]:
# ============================================================
# STAGE 8B — QUALITATIVE ERROR VISUALIZATION — FIXED VERSION
#
# PURPOSE
# ------------------------------------------------------------
# Create deterministic publication-quality qualitative
# visualizations from already completed locked-test outputs.
#
# IMPORTANT:
# - NO new inference
# - NO threshold tuning
# - NO retraining
# - NO model exclusion
# - NO quantitative result modification
#
# PRIMARY QUALITATIVE SETTING
# ------------------------------------------------------------
# Protocol       : batch_disjoint
# Primary model  : YOLO26s
# Reference seed : 123
#
# IMAGE SELECTION
# ------------------------------------------------------------
# Based on THREE-SEED CONSENSUS error statistics.
#
# Categories:
#   1. Small-object failure
#   2. High miss rate
#   3. Localization error
#   4. Background-like false positive
#
# Additional figures:
#   - same hard cases compared across all 4 models
#   - supplementary random-protocol YOLO26s figure
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone

import json
import hashlib
import numpy as np
import pandas as pd

from PIL import Image, ImageOps

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

LOCAL_DATASET = Path(
    "/content/taco_v5/dataset"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True
    )


FIG_DIR = (
    DIRS["analysis"] /
    "qualitative_figures"
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. FIXED VISUALIZATION POLICY
# ============================================================

PRIMARY_PROTOCOL = "batch_disjoint"

PRIMARY_MODEL = "yolo26s"

# Fixed illustrative reference seed.
# Image selection itself uses all three seeds.
REFERENCE_SEED = 123


MODELS = [
    "yolov8s",
    "yolo11s",
    "yolo26s",
    "rtdetr_l",
]


PROTOCOLS = [
    "random",
    "batch_disjoint",
]


SEEDS = [
    42,
    123,
    2026,
]


IOU_TP = 0.50

IOU_LOCALIZATION_MIN = 0.10


# ============================================================
# 3. INPUT FILES
# ============================================================

PER_IMAGE_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_per_image.csv"
)

EVENT_PATH = (
    DIRS["analysis"] /
    "locked_test_error_events.csv"
)

ERROR_AGG_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_mean_sd.csv"
)

THRESHOLD_PATH = (
    DIRS["manifests"] /
    "validation_thresholds_frozen.json"
)

STAGE8A_PATH = (
    DIRS["manifests"] /
    "STAGE_8A_ERROR_TAXONOMY.json"
)

TEST_COMPLETE_PATH = (
    DIRS["manifests"] /
    "LOCKED_TEST_COMPLETED.json"
)


required_files = [
    PER_IMAGE_PATH,
    EVENT_PATH,
    ERROR_AGG_PATH,
    THRESHOLD_PATH,
    STAGE8A_PATH,
    TEST_COMPLETE_PATH,
]


for p in required_files:

    if not p.exists():

        raise FileNotFoundError(
            f"Required Stage 8B input missing:\n{p}"
        )


print("=" * 90)
print("STAGE 8B — QUALITATIVE ERROR VISUALIZATION — FIXED")
print("=" * 90)


# ============================================================
# 4. HELPERS
# ============================================================

def sha256_file(
    path,
    chunk=1024 * 1024
):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(chunk),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def dataframe_content_hash(
    df
):

    csv_bytes = df.to_csv(
        index=False,
        lineterminator="\n"
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        csv_bytes
    ).hexdigest()


def xywh_to_xyxy(
    bbox
):

    x, y, w, h = map(
        float,
        bbox
    )

    return np.array(
        [
            x,
            y,
            x + w,
            y + h,
        ],
        dtype=float
    )


def box_iou(
    a,
    b
):

    x1 = max(
        float(a[0]),
        float(b[0])
    )

    y1 = max(
        float(a[1]),
        float(b[1])
    )

    x2 = min(
        float(a[2]),
        float(b[2])
    )

    y2 = min(
        float(a[3]),
        float(b[3])
    )


    inter_w = max(
        0.0,
        x2 - x1
    )

    inter_h = max(
        0.0,
        y2 - y1
    )

    intersection = (
        inter_w *
        inter_h
    )


    area_a = (
        max(
            0.0,
            float(a[2]) - float(a[0])
        )
        *
        max(
            0.0,
            float(a[3]) - float(a[1])
        )
    )


    area_b = (
        max(
            0.0,
            float(b[2]) - float(b[0])
        )
        *
        max(
            0.0,
            float(b[3]) - float(b[1])
        )
    )


    union = (
        area_a
        +
        area_b
        -
        intersection
    )


    if union <= 0:

        return 0.0


    return float(
        intersection /
        union
    )


# ============================================================
# 5. LOAD STAGE 8A TABLES
# ============================================================

per_image = pd.read_csv(
    PER_IMAGE_PATH
)

events = pd.read_csv(
    EVENT_PATH
)

thresholds = json.loads(
    THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)


if len(thresholds) != 24:

    raise RuntimeError(
        f"Expected 24 frozen thresholds; "
        f"found {len(thresholds)}."
    )


print(
    f"Per-image records          : {len(per_image)}"
)

print(
    f"Detailed error events      : {len(events)}"
)

print(
    "Frozen thresholds          : 24/24 PASSED"
)


# ============================================================
# 6. VERIFY EXPECTED PER-IMAGE STRUCTURE
# ============================================================

expected_per_image_rows = (
    225 * 4 * 3
    +
    200 * 4 * 3
)


if len(per_image) != expected_per_image_rows:

    raise RuntimeError(
        f"Expected {expected_per_image_rows} per-image rows, "
        f"found {len(per_image)}."
    )


required_per_image_columns = [
    "protocol",
    "model",
    "seed",
    "image_id",
    "file_name",
    "n_gt",
    "n_predictions_at_frozen_threshold",
    "tp",
    "duplicate_fp",
    "localization_fp",
    "background_fp",
    "fp",
    "fn",
    "total_errors",
]


missing = [
    c
    for c in required_per_image_columns
    if c not in per_image.columns
]


if missing:

    raise RuntimeError(
        "Missing Stage 8A per-image columns:\n"
        + ", ".join(missing)
    )


print(
    "Stage 8A per-image structure: PASSED"
)


# ============================================================
# 7. THREE-SEED IMAGE CONSENSUS
# ============================================================

consensus = (

    per_image

    .groupby(
        [
            "protocol",
            "model",
            "image_id",
            "file_name",
        ],
        as_index=False
    )

    .agg(

        n_gt=(
            "n_gt",
            "first"
        ),

        mean_tp=(
            "tp",
            "mean"
        ),

        mean_fp=(
            "fp",
            "mean"
        ),

        mean_fn=(
            "fn",
            "mean"
        ),

        mean_duplicate_fp=(
            "duplicate_fp",
            "mean"
        ),

        mean_localization_fp=(
            "localization_fp",
            "mean"
        ),

        mean_background_fp=(
            "background_fp",
            "mean"
        ),

        mean_total_errors=(
            "total_errors",
            "mean"
        ),

        seeds_with_error=(
            "total_errors",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x)
                        > 0
                    )
                )
        ),
    )
)


# ============================================================
# 8. NORMALIZED ERROR VARIABLES
# ============================================================

consensus[
    "mean_fn_rate_image"
] = np.where(

    consensus[
        "n_gt"
    ] > 0,

    (
        consensus[
            "mean_fn"
        ]
        /
        consensus[
            "n_gt"
        ]
    ),

    np.nan
)


consensus[
    "localization_per_gt"
] = (
    consensus[
        "mean_localization_fp"
    ]
    /
    np.maximum(
        consensus[
            "n_gt"
        ],
        1
    )
)


consensus[
    "background_fp_per_gt"
] = (
    consensus[
        "mean_background_fp"
    ]
    /
    np.maximum(
        consensus[
            "n_gt"
        ],
        1
    )
)


# ============================================================
# 9. SMALL-OBJECT FAILURE CONSENSUS
# ============================================================

small_events = events[
    (
        events[
            "gt_size"
        ] == "small"
    )
    &
    (
        events[
            "event"
        ].isin(
            [
                "tp",
                "fn",
            ]
        )
    )
].copy()


if len(small_events) == 0:

    raise RuntimeError(
        "No small-object TP/FN events found."
    )


small_events[
    "small_tp"
] = (
    small_events[
        "event"
    ] == "tp"
).astype(
    int
)


small_events[
    "small_fn"
] = (
    small_events[
        "event"
    ] == "fn"
).astype(
    int
)


small_seed = (

    small_events

    .groupby(
        [
            "protocol",
            "model",
            "seed",
            "image_id",
            "file_name",
        ],
        as_index=False
    )

    .agg(

        small_tp=(
            "small_tp",
            "sum"
        ),

        small_fn=(
            "small_fn",
            "sum"
        ),
    )
)


small_seed[
    "small_gt"
] = (
    small_seed[
        "small_tp"
    ]
    +
    small_seed[
        "small_fn"
    ]
)


small_seed[
    "small_fn_rate"
] = np.where(

    small_seed[
        "small_gt"
    ] > 0,

    (
        small_seed[
            "small_fn"
        ]
        /
        small_seed[
            "small_gt"
        ]
    ),

    np.nan
)


small_consensus = (

    small_seed

    .groupby(
        [
            "protocol",
            "model",
            "image_id",
            "file_name",
        ],
        as_index=False
    )

    .agg(

        mean_small_gt=(
            "small_gt",
            "mean"
        ),

        mean_small_tp=(
            "small_tp",
            "mean"
        ),

        mean_small_fn=(
            "small_fn",
            "mean"
        ),

        mean_small_fn_rate=(
            "small_fn_rate",
            "mean"
        ),
    )
)


consensus = consensus.merge(

    small_consensus,

    on=[
        "protocol",
        "model",
        "image_id",
        "file_name",
    ],

    how="left"
)


for c in [
    "mean_small_gt",
    "mean_small_tp",
    "mean_small_fn",
    "mean_small_fn_rate",
]:

    consensus[c] = (
        consensus[c]
        .fillna(
            0.0
        )
    )


print(
    "Three-seed consensus       : GENERATED"
)


# ============================================================
# 10. DETERMINISTIC CANDIDATE SELECTION
# ============================================================

def select_unique_candidate(
    dataframe,
    category,
    sort_columns,
    ascending,
    used_ids,
    condition=None
):

    d = dataframe.copy()


    if condition is not None:

        mask = condition(
            d
        )

        d = d[
            mask
        ].copy()


    d = d[
        ~d[
            "image_id"
        ].astype(int).isin(
            used_ids
        )
    ].copy()


    if len(d) == 0:

        raise RuntimeError(
            f"No unused candidate available "
            f"for category: {category}"
        )


    d = d.sort_values(
        by=sort_columns,
        ascending=ascending,
        kind="mergesort"
    )


    row = d.iloc[
        0
    ].copy()


    row[
        "selection_category"
    ] = category


    return row


def build_selection(
    protocol,
    model
):

    d = consensus[
        (
            consensus[
                "protocol"
            ] == protocol
        )
        &
        (
            consensus[
                "model"
            ] == model
        )
    ].copy()


    if len(d) == 0:

        raise RuntimeError(
            f"No consensus data for "
            f"{protocol}/{model}"
        )


    used_ids = set()

    selected_rows = []


    # --------------------------------------------------------
    # A. SMALL-OBJECT FAILURE
    # --------------------------------------------------------

    row = select_unique_candidate(

        dataframe=d,

        category="small_object_failure",

        sort_columns=[
            "mean_small_fn_rate",
            "mean_small_fn",
            "mean_fn_rate_image",
            "mean_total_errors",
            "image_id",
        ],

        ascending=[
            False,
            False,
            False,
            False,
            True,
        ],

        used_ids=used_ids,

        condition=lambda x:
            x[
                "mean_small_gt"
            ] > 0
    )


    used_ids.add(
        int(
            row[
                "image_id"
            ]
        )
    )

    selected_rows.append(
        row
    )


    # --------------------------------------------------------
    # B. HIGH MISS RATE
    #
    # Require >=5 GT objects so that a single missed object
    # does not automatically become a 100% miss-rate example.
    # --------------------------------------------------------

    row = select_unique_candidate(

        dataframe=d,

        category="high_miss_rate",

        sort_columns=[
            "mean_fn_rate_image",
            "mean_fn",
            "mean_total_errors",
            "image_id",
        ],

        ascending=[
            False,
            False,
            False,
            True,
        ],

        used_ids=used_ids,

        condition=lambda x:
            x[
                "n_gt"
            ] >= 5
    )


    used_ids.add(
        int(
            row[
                "image_id"
            ]
        )
    )

    selected_rows.append(
        row
    )


    # --------------------------------------------------------
    # C. LOCALIZATION ERROR
    # --------------------------------------------------------

    row = select_unique_candidate(

        dataframe=d,

        category="localization_error",

        sort_columns=[
            "localization_per_gt",
            "mean_localization_fp",
            "mean_total_errors",
            "image_id",
        ],

        ascending=[
            False,
            False,
            False,
            True,
        ],

        used_ids=used_ids,

        condition=lambda x:
            x[
                "mean_localization_fp"
            ] > 0
    )


    used_ids.add(
        int(
            row[
                "image_id"
            ]
        )
    )

    selected_rows.append(
        row
    )


    # --------------------------------------------------------
    # D. BACKGROUND-LIKE FALSE POSITIVE
    # --------------------------------------------------------

    row = select_unique_candidate(

        dataframe=d,

        category="background_like_fp",

        sort_columns=[
            "background_fp_per_gt",
            "mean_background_fp",
            "mean_total_errors",
            "image_id",
        ],

        ascending=[
            False,
            False,
            False,
            True,
        ],

        used_ids=used_ids,

        condition=lambda x:
            x[
                "mean_background_fp"
            ] > 0
    )


    used_ids.add(
        int(
            row[
                "image_id"
            ]
        )
    )

    selected_rows.append(
        row
    )


    result = pd.DataFrame(
        selected_rows
    ).reset_index(
        drop=True
    )


    if result[
        "image_id"
    ].duplicated().any():

        raise RuntimeError(
            f"Duplicate qualitative image "
            f"selection detected: {protocol}/{model}"
        )


    return result


# ============================================================
# 11. BUILD PRIMARY AND SUPPLEMENTARY SELECTIONS
# ============================================================

primary_selection = build_selection(
    protocol=PRIMARY_PROTOCOL,
    model=PRIMARY_MODEL
)


random_selection = build_selection(
    protocol="random",
    model=PRIMARY_MODEL
)


print(
    "\n"
    + "=" * 90
)

print(
    "PRIMARY SELECTION"
)

print(
    "=" * 90
)


display(
    primary_selection[
        [
            "selection_category",
            "image_id",
            "file_name",
            "n_gt",
            "mean_tp",
            "mean_fp",
            "mean_fn",
            "mean_fn_rate_image",
            "mean_small_fn_rate",
            "mean_localization_fp",
            "mean_background_fp",
        ]
    ]
)


# ============================================================
# 12. SAVE / VERIFY SELECTION
#
# If previous failed 8B already created a selection file,
# the deterministic selection must match it exactly.
# ============================================================

SELECTION_PATH = (
    DIRS["analysis"] /
    "qualitative_visualization_selection.csv"
)


selection_all = pd.concat(
    [
        primary_selection.assign(
            selection_set="primary_batch_disjoint"
        ),
        random_selection.assign(
            selection_set="supplementary_random"
        ),
    ],
    ignore_index=True
)


selection_key_columns = [
    "selection_set",
    "protocol",
    "model",
    "selection_category",
    "image_id",
    "file_name",
]


if SELECTION_PATH.exists():

    previous_selection = pd.read_csv(
        SELECTION_PATH
    )


    if set(
        selection_key_columns
    ).issubset(
        previous_selection.columns
    ):

        old_keys = (
            previous_selection[
                selection_key_columns
            ]
            .astype(
                str
            )
            .reset_index(
                drop=True
            )
        )


        new_keys = (
            selection_all[
                selection_key_columns
            ]
            .astype(
                str
            )
            .reset_index(
                drop=True
            )
        )


        # Older failed version may not have selection_set.
        if len(old_keys) == len(new_keys):

            if not old_keys.equals(
                new_keys
            ):

                raise RuntimeError(
                    "Existing qualitative selection differs "
                    "from the newly generated deterministic "
                    "selection.\n"
                    "STOP: do not silently replace it."
                )


selection_all.to_csv(
    SELECTION_PATH,
    index=False
)


print(
    "\nQualitative selections     : FROZEN"
)


# ============================================================
# 13. INDEX LOCAL IMAGES
# ============================================================

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


all_local_images = [

    p

    for p in LOCAL_DATASET.rglob(
        "*"
    )

    if (
        p.is_file()
        and
        p.suffix.lower()
        in image_extensions
    )
]


if len(
    all_local_images
) < 1500:

    raise RuntimeError(
        f"Expected at least 1500 local images; "
        f"found {len(all_local_images)}."
    )


print(
    f"Local image candidates     : {len(all_local_images)}"
)


def resolve_image_path(
    file_name
):

    file_name_norm = (
        str(
            file_name
        )
        .replace(
            "\\",
            "/"
        )
        .lstrip(
            "/"
        )
    )


    direct_candidates = [
        LOCAL_DATASET /
        file_name_norm,

        LOCAL_DATASET /
        "images" /
        file_name_norm,
    ]


    for candidate in direct_candidates:

        if candidate.exists():

            return candidate


    suffix = (
        "/"
        +
        file_name_norm
    )


    matches = [

        p

        for p in all_local_images

        if p.as_posix().endswith(
            suffix
        )
    ]


    if len(matches) == 1:

        return matches[
            0
        ]


    if len(matches) == 0:

        raise FileNotFoundError(
            f"Could not resolve image:\n"
            f"{file_name}"
        )


    raise RuntimeError(
        "Ambiguous image path:\n"
        f"{file_name}\n"
        +
        "\n".join(
            str(x)
            for x in matches
        )
    )


# ============================================================
# 14. LOAD GROUND TRUTH
# ============================================================

gt_cache = {}


def load_gt(
    protocol
):

    if protocol in gt_cache:

        return gt_cache[
            protocol
        ]


    gt_path = (
        LOCAL_DATASET /
        protocol /
        "test_coco_taco1.json"
    )


    if not gt_path.exists():

        raise FileNotFoundError(
            gt_path
        )


    obj = json.loads(
        gt_path.read_text(
            encoding="utf-8"
        )
    )


    image_registry = {

        int(
            im[
                "id"
            ]
        ):
            im

        for im in obj[
            "images"
        ]
    }


    annotation_registry = defaultdict(
        list
    )


    for ann in obj[
        "annotations"
    ]:

        image_id = int(
            ann[
                "image_id"
            ]
        )


        annotation_registry[
            image_id
        ].append(
            {
                "annotation_id":
                    int(
                        ann.get(
                            "id",
                            -1
                        )
                    ),

                "bbox_xywh":
                    [
                        float(v)
                        for v in ann[
                            "bbox"
                        ]
                    ],

                "bbox_xyxy":
                    xywh_to_xyxy(
                        ann[
                            "bbox"
                        ]
                    ),
            }
        )


    gt_cache[
        protocol
    ] = (
        image_registry,
        annotation_registry
    )


    return (
        image_registry,
        annotation_registry
    )


# ============================================================
# 15. LOAD EXISTING LOCKED-TEST PREDICTIONS
# ============================================================

prediction_cache = {}


def load_predictions(
    protocol,
    model,
    seed
):

    cache_key = (
        protocol,
        model,
        int(seed)
    )


    if cache_key in prediction_cache:

        return prediction_cache[
            cache_key
        ]


    pred_path = (
        DIRS["test"] /
        protocol /
        f"{model}_seed{seed}" /
        "test_predictions_coco.json"
    )


    if not pred_path.exists():

        raise FileNotFoundError(
            pred_path
        )


    raw_predictions = json.loads(
        pred_path.read_text(
            encoding="utf-8"
        )
    )


    by_image = defaultdict(
        list
    )


    for p in raw_predictions:

        image_id = int(
            p[
                "image_id"
            ]
        )


        bbox_xywh = [
            float(v)
            for v in p[
                "bbox"
            ]
        ]


        by_image[
            image_id
        ].append(
            {
                "score":
                    float(
                        p[
                            "score"
                        ]
                    ),

                "bbox_xywh":
                    bbox_xywh,

                "bbox_xyxy":
                    xywh_to_xyxy(
                        bbox_xywh
                    ),
            }
        )


    prediction_cache[
        cache_key
    ] = by_image


    return by_image


# ============================================================
# 16. CLASSIFY ONE IMAGE
#
# Matching logic mirrors Stage 8A.
# ============================================================

def classify_image(
    protocol,
    model,
    seed,
    image_id
):

    # FIXED tuple unpacking
    image_registry, annotation_registry = load_gt(
        protocol
    )


    image_id = int(
        image_id
    )


    gts = annotation_registry.get(
        image_id,
        []
    )


    pred_by_image = load_predictions(
        protocol,
        model,
        seed
    )


    threshold_key = (
        f"{protocol}/"
        f"{model}/"
        f"seed{seed}"
    )


    if threshold_key not in thresholds:

        raise KeyError(
            f"Frozen threshold not found: "
            f"{threshold_key}"
        )


    threshold = float(
        thresholds[
            threshold_key
        ]
    )


    predictions = [

        p

        for p in pred_by_image.get(
            image_id,
            []
        )

        if float(
            p[
                "score"
            ]
        ) >= threshold
    ]


    predictions = sorted(
        predictions,
        key=lambda z:
            -float(
                z[
                    "score"
                ]
            )
    )


    matched_gt = set()

    classified_predictions = []


    for prediction in predictions:

        prediction_box = (
            prediction[
                "bbox_xyxy"
            ]
        )


        # ----------------------------------------------------
        # IMAGE WITH NO GT
        # ----------------------------------------------------

        if len(gts) == 0:

            classified_predictions.append(
                {
                    **prediction,

                    "event":
                        "background_fp",

                    "best_iou":
                        0.0,

                    "matched_gt":
                        None,
                }
            )

            continue


        # ----------------------------------------------------
        # IOU AGAINST ALL GT BOXES
        # ----------------------------------------------------

        ious = [

            box_iou(
                prediction_box,
                gt[
                    "bbox_xyxy"
                ]
            )

            for gt in gts
        ]


        order = np.argsort(
            ious
        )[::-1]


        assigned_gt = None

        assigned_iou = None


        # ----------------------------------------------------
        # TP:
        # highest available unmatched GT with IoU >= 0.50
        # ----------------------------------------------------

        for gi in order:

            gi = int(
                gi
            )


            if (
                ious[
                    gi
                ] >= IOU_TP
                and
                gi not in matched_gt
            ):

                assigned_gt = gi

                assigned_iou = float(
                    ious[
                        gi
                    ]
                )

                matched_gt.add(
                    gi
                )

                break


        if assigned_gt is not None:

            classified_predictions.append(
                {
                    **prediction,

                    "event":
                        "tp",

                    "best_iou":
                        assigned_iou,

                    "matched_gt":
                        assigned_gt,
                }
            )

            continue


        # ----------------------------------------------------
        # UNMATCHED PREDICTION
        # ----------------------------------------------------

        best_iou = float(
            max(
                ious
            )
        )


        if best_iou >= IOU_TP:

            event = (
                "duplicate_fp"
            )


        elif (
            best_iou
            >= IOU_LOCALIZATION_MIN
        ):

            event = (
                "localization_fp"
            )


        else:

            event = (
                "background_fp"
            )


        classified_predictions.append(
            {
                **prediction,

                "event":
                    event,

                "best_iou":
                    best_iou,

                "matched_gt":
                    None,
            }
        )


    fn_indices = [

        gi

        for gi in range(
            len(
                gts
            )
        )

        if gi not in matched_gt
    ]


    TP = int(
        sum(
            p[
                "event"
            ] == "tp"

            for p in classified_predictions
        )
    )


    FP = int(
        sum(
            p[
                "event"
            ] != "tp"

            for p in classified_predictions
        )
    )


    FN = int(
        len(
            fn_indices
        )
    )


    return {

        "threshold":
            threshold,

        "gts":
            gts,

        "predictions":
            classified_predictions,

        "fn_indices":
            fn_indices,

        "TP":
            TP,

        "FP":
            FP,

        "FN":
            FN,
    }


# ============================================================
# 17. VERIFY SELECTED IMAGE COUNTS AGAINST STAGE 8A
# ============================================================

def verify_selected_image(
    protocol,
    model,
    seed,
    image_id
):

    result = classify_image(
        protocol=protocol,
        model=model,
        seed=seed,
        image_id=image_id
    )


    stage8a_row = per_image[
        (
            per_image[
                "protocol"
            ] == protocol
        )
        &
        (
            per_image[
                "model"
            ] == model
        )
        &
        (
            per_image[
                "seed"
            ].astype(int) == int(seed)
        )
        &
        (
            per_image[
                "image_id"
            ].astype(int) == int(image_id)
        )
    ]


    if len(
        stage8a_row
    ) != 1:

        raise RuntimeError(
            "Could not uniquely retrieve Stage 8A "
            f"per-image row:\n"
            f"{protocol}/{model}/seed{seed}/"
            f"image{image_id}"
        )


    r = stage8a_row.iloc[
        0
    ]


    expected = {
        "TP":
            int(
                r[
                    "tp"
                ]
            ),

        "FP":
            int(
                r[
                    "fp"
                ]
            ),

        "FN":
            int(
                r[
                    "fn"
                ]
            ),
    }


    actual = {
        "TP":
            int(
                result[
                    "TP"
                ]
            ),

        "FP":
            int(
                result[
                    "FP"
                ]
            ),

        "FN":
            int(
                result[
                    "FN"
                ]
            ),
    }


    if actual != expected:

        raise RuntimeError(
            "Stage 8B image classification does not "
            "reproduce Stage 8A counts:\n"
            f"{protocol}/{model}/seed{seed}/image{image_id}\n"
            f"Stage 8A: {expected}\n"
            f"Stage 8B: {actual}"
        )


    return result


for r in primary_selection.itertuples():

    verify_selected_image(
        protocol=PRIMARY_PROTOCOL,
        model=PRIMARY_MODEL,
        seed=REFERENCE_SEED,
        image_id=r.image_id
    )


for r in random_selection.itertuples():

    verify_selected_image(
        protocol="random",
        model=PRIMARY_MODEL,
        seed=REFERENCE_SEED,
        image_id=r.image_id
    )


print(
    "Selected-image Stage 8A reproduction: PASSED"
)


# ============================================================
# 18. VISUALIZATION COLORS
# ============================================================

EVENT_COLOR = {

    "tp":
        "lime",

    "localization_fp":
        "orange",

    "background_fp":
        "red",

    "duplicate_fp":
        "magenta",
}


# ============================================================
# 19. DRAW ONE PANEL
# ============================================================

def draw_image_panel(
    ax,
    protocol,
    model,
    seed,
    image_id,
    category=None
):

    # FIXED tuple unpacking
    image_registry, annotation_registry = load_gt(
        protocol
    )


    image_id = int(
        image_id
    )


    if image_id not in image_registry:

        raise KeyError(
            f"Image ID {image_id} not found "
            f"in {protocol} test GT."
        )


    image_info = image_registry[
        image_id
    ]


    file_name = str(
        image_info[
            "file_name"
        ]
    )


    image_path = resolve_image_path(
        file_name
    )


    # --------------------------------------------------------
    # LOAD IMAGE
    # --------------------------------------------------------

    with Image.open(
        image_path
    ) as img:

        img = ImageOps.exif_transpose(
            img
        )

        img = img.convert(
            "RGB"
        )


        expected_size = (
            int(
                image_info[
                    "width"
                ]
            ),
            int(
                image_info[
                    "height"
                ]
            ),
        )


        if img.size != expected_size:

            raise RuntimeError(
                "Image geometry mismatch:\n"
                f"{file_name}\n"
                f"PIL size : {img.size}\n"
                f"COCO size: {expected_size}"
            )


        image_array = np.asarray(
            img
        )


    # --------------------------------------------------------
    # CLASSIFICATION
    # --------------------------------------------------------

    result = verify_selected_image(
        protocol=protocol,
        model=model,
        seed=seed,
        image_id=image_id
    )


    ax.imshow(
        image_array
    )


    # --------------------------------------------------------
    # ALL GT BOXES — THIN WHITE DASHED
    # --------------------------------------------------------

    for gt in result[
        "gts"
    ]:

        x, y, w, h = gt[
            "bbox_xywh"
        ]


        rectangle = Rectangle(
            (
                x,
                y
            ),
            w,
            h,
            fill=False,
            edgecolor="white",
            linewidth=0.8,
            linestyle="--",
            alpha=0.65
        )


        ax.add_patch(
            rectangle
        )


    # --------------------------------------------------------
    # FALSE NEGATIVE GT BOXES — RED DASHED
    # --------------------------------------------------------

    for gt_index in result[
        "fn_indices"
    ]:

        gt = result[
            "gts"
        ][
            gt_index
        ]


        x, y, w, h = gt[
            "bbox_xywh"
        ]


        rectangle = Rectangle(
            (
                x,
                y
            ),
            w,
            h,
            fill=False,
            edgecolor="red",
            linewidth=1.8,
            linestyle="--"
        )


        ax.add_patch(
            rectangle
        )


    # --------------------------------------------------------
    # PREDICTIONS
    # --------------------------------------------------------

    for prediction in result[
        "predictions"
    ]:

        x, y, w, h = prediction[
            "bbox_xywh"
        ]


        event = prediction[
            "event"
        ]


        rectangle = Rectangle(
            (
                x,
                y
            ),
            w,
            h,
            fill=False,
            edgecolor=EVENT_COLOR[
                event
            ],
            linewidth=1.8
        )


        ax.add_patch(
            rectangle
        )


    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    title_lines = []


    if category is not None:

        pretty_category = (
            str(
                category
            )
            .replace(
                "_",
                " "
            )
            .title()
        )


        title_lines.append(
            pretty_category
        )


    title_lines.append(
        f"{model} | seed {seed}"
    )


    title_lines.append(
        (
            f"TP={result['TP']}  "
            f"FP={result['FP']}  "
            f"FN={result['FN']}  "
            f"thr={result['threshold']:.2f}"
        )
    )


    title_lines.append(
        f"Image ID {image_id}"
    )


    ax.set_title(
        "\n".join(
            title_lines
        ),
        fontsize=9
    )


    ax.axis(
        "off"
    )


# ============================================================
# 20. FIGURE LEGEND
# ============================================================

legend_handles = [

    Line2D(
        [0],
        [0],
        color="lime",
        linewidth=2,
        label="True-positive prediction"
    ),

    Line2D(
        [0],
        [0],
        color="orange",
        linewidth=2,
        label="Localization FP"
    ),

    Line2D(
        [0],
        [0],
        color="magenta",
        linewidth=2,
        label="Duplicate FP"
    ),

    Line2D(
        [0],
        [0],
        color="red",
        linewidth=2,
        label="Background-like FP"
    ),

    Line2D(
        [0],
        [0],
        color="red",
        linewidth=2,
        linestyle="--",
        label="False-negative GT"
    ),

    Line2D(
        [0],
        [0],
        color="gray",
        linewidth=1,
        linestyle="--",
        label="GT reference"
    ),
]


# ============================================================
# 21. PRIMARY FIGURE
#
# batch-disjoint + YOLO26s + seed123
# Four deterministically selected error types
# ============================================================

PRIMARY_FIG = (
    FIG_DIR /
    "Fig_error_examples_batch_disjoint_yolo26s_seed123.png"
)


fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(
        12,
        10
    )
)


axes = axes.ravel()


for ax, row in zip(
    axes,
    primary_selection.itertuples()
):

    draw_image_panel(
        ax=ax,
        protocol=PRIMARY_PROTOCOL,
        model=PRIMARY_MODEL,
        seed=REFERENCE_SEED,
        image_id=row.image_id,
        category=row.selection_category
    )


fig.suptitle(
    (
        "Representative Locked-Test Error Patterns — "
        "Batch-Disjoint YOLO26s"
    ),
    fontsize=14
)


fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=True,
    fontsize=9
)


fig.tight_layout(
    rect=[
        0.0,
        0.08,
        1.0,
        0.96
    ]
)


fig.savefig(
    PRIMARY_FIG,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
    metadata={
        "Software":
            "TACO Pipeline v5"
    }
)


plt.show()

plt.close(
    fig
)


if not PRIMARY_FIG.exists():

    raise RuntimeError(
        "Primary qualitative figure was not created."
    )


print(
    "\nPrimary figure created:"
)

print(
    PRIMARY_FIG
)


# ============================================================
# 22. CROSS-MODEL HARD-CASE COMPARISON
#
# Same 3 images shown for all four models.
# Image selection is based on YOLO26s consensus,
# not independently selected for each model.
# ============================================================

cross_selection = (
    primary_selection
    .iloc[
        :3
    ]
    .copy()
)


CROSS_MODEL_FIG = (
    FIG_DIR /
    "Fig_cross_model_hard_cases_batch_disjoint_seed123.png"
)


n_rows = len(
    cross_selection
)

n_cols = len(
    MODELS
)


fig, axes = plt.subplots(
    nrows=n_rows,
    ncols=n_cols,
    figsize=(
        16,
        4.8 * n_rows
    ),
    squeeze=False
)


for row_index, row in enumerate(
    cross_selection.itertuples()
):

    for column_index, model in enumerate(
        MODELS
    ):

        category = (
            row.selection_category
            if column_index == 0
            else None
        )


        draw_image_panel(
            ax=axes[
                row_index,
                column_index
            ],
            protocol=PRIMARY_PROTOCOL,
            model=model,
            seed=REFERENCE_SEED,
            image_id=row.image_id,
            category=category
        )


fig.suptitle(
    (
        "Cross-Model Comparison on Deterministically "
        "Selected Batch-Disjoint Hard Cases — Seed 123"
    ),
    fontsize=14
)


fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=True,
    fontsize=9
)


fig.tight_layout(
    rect=[
        0.0,
        0.06,
        1.0,
        0.96
    ]
)


fig.savefig(
    CROSS_MODEL_FIG,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
    metadata={
        "Software":
            "TACO Pipeline v5"
    }
)


plt.show()

plt.close(
    fig
)


if not CROSS_MODEL_FIG.exists():

    raise RuntimeError(
        "Cross-model qualitative figure was not created."
    )


print(
    "\nCross-model figure created:"
)

print(
    CROSS_MODEL_FIG
)


# ============================================================
# 23. RANDOM-PROTOCOL SUPPLEMENTARY FIGURE
# ============================================================

RANDOM_FIG = (
    FIG_DIR /
    "Supp_error_examples_random_yolo26s_seed123.png"
)


fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(
        12,
        10
    )
)


axes = axes.ravel()


for ax, row in zip(
    axes,
    random_selection.itertuples()
):

    draw_image_panel(
        ax=ax,
        protocol="random",
        model=PRIMARY_MODEL,
        seed=REFERENCE_SEED,
        image_id=row.image_id,
        category=row.selection_category
    )


fig.suptitle(
    (
        "Representative Locked-Test Error Patterns — "
        "Random-Split YOLO26s"
    ),
    fontsize=14
)


fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=True,
    fontsize=9
)


fig.tight_layout(
    rect=[
        0.0,
        0.08,
        1.0,
        0.96
    ]
)


fig.savefig(
    RANDOM_FIG,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
    metadata={
        "Software":
            "TACO Pipeline v5"
    }
)


plt.show()

plt.close(
    fig
)


if not RANDOM_FIG.exists():

    raise RuntimeError(
        "Random-protocol supplementary figure "
        "was not created."
    )


print(
    "\nRandom supplementary figure created:"
)

print(
    RANDOM_FIG
)


# ============================================================
# 24. VISUALIZATION REGISTRY
# ============================================================

visual_registry_rows = []


# ------------------------------------------------------------
# Primary figure
# ------------------------------------------------------------

for row in primary_selection.itertuples():

    visual_registry_rows.append(
        {
            "figure":
                PRIMARY_FIG.name,

            "protocol":
                PRIMARY_PROTOCOL,

            "model":
                PRIMARY_MODEL,

            "seed":
                REFERENCE_SEED,

            "image_id":
                int(
                    row.image_id
                ),

            "file_name":
                row.file_name,

            "selection_category":
                row.selection_category,

            "selection_basis":
                "three-seed consensus",

            "quantitative_selection":
                False,
        }
    )


# ------------------------------------------------------------
# Cross-model figure
# ------------------------------------------------------------

for row in cross_selection.itertuples():

    for model in MODELS:

        visual_registry_rows.append(
            {
                "figure":
                    CROSS_MODEL_FIG.name,

                "protocol":
                    PRIMARY_PROTOCOL,

                "model":
                    model,

                "seed":
                    REFERENCE_SEED,

                "image_id":
                    int(
                        row.image_id
                    ),

                "file_name":
                    row.file_name,

                "selection_category":
                    row.selection_category,

                "selection_basis":
                    (
                        "image selected from YOLO26s "
                        "three-seed consensus; same image "
                        "shown for all models"
                    ),

                "quantitative_selection":
                    False,
            }
        )


# ------------------------------------------------------------
# Supplementary random figure
# ------------------------------------------------------------

for row in random_selection.itertuples():

    visual_registry_rows.append(
        {
            "figure":
                RANDOM_FIG.name,

            "protocol":
                "random",

            "model":
                PRIMARY_MODEL,

            "seed":
                REFERENCE_SEED,

            "image_id":
                int(
                    row.image_id
                ),

            "file_name":
                row.file_name,

            "selection_category":
                row.selection_category,

            "selection_basis":
                "three-seed consensus",

            "quantitative_selection":
                False,
        }
    )


visual_registry = pd.DataFrame(
    visual_registry_rows
)


REGISTRY_PATH = (
    DIRS["analysis"] /
    "qualitative_visualization_registry.csv"
)


visual_registry.to_csv(
    REGISTRY_PATH,
    index=False
)


# ============================================================
# 25. SAVE PRIMARY SELECTION TABLE FOR MANUSCRIPT AUDIT
# ============================================================

PRIMARY_SELECTION_PATH = (
    DIRS["analysis"] /
    "qualitative_primary_selection_details.csv"
)


primary_selection.to_csv(
    PRIMARY_SELECTION_PATH,
    index=False
)


# ============================================================
# 26. FINAL FIGURE EXISTENCE / SIZE CHECK
# ============================================================

figure_paths = [
    PRIMARY_FIG,
    CROSS_MODEL_FIG,
    RANDOM_FIG,
]


for path in figure_paths:

    if not path.exists():

        raise RuntimeError(
            f"Figure missing:\n{path}"
        )


    if path.stat().st_size <= 0:

        raise RuntimeError(
            f"Figure file is empty:\n{path}"
        )


print(
    "\nFigure files               : 3/3 PASSED"
)


# ============================================================
# 27. OUTPUT HASHES
# ============================================================

output_paths = {

    "qualitative_visualization_selection.csv":
        SELECTION_PATH,

    "qualitative_primary_selection_details.csv":
        PRIMARY_SELECTION_PATH,

    "qualitative_visualization_registry.csv":
        REGISTRY_PATH,

    PRIMARY_FIG.name:
        PRIMARY_FIG,

    CROSS_MODEL_FIG.name:
        CROSS_MODEL_FIG,

    RANDOM_FIG.name:
        RANDOM_FIG,
}


output_hashes = {

    name:
        sha256_file(
            path
        )

    for name, path in output_paths.items()
}


# ============================================================
# 28. STAGE 8B MANIFEST
# ============================================================

MANIFEST_PATH = (
    DIRS["manifests"] /
    "STAGE_8B_QUALITATIVE_VISUALIZATION.json"
)


manifest = {

    "stage":
        "8B",

    "status":
        "QUALITATIVE_VISUALIZATION_COMPLETED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_stage8a_manifest":
        str(
            STAGE8A_PATH
        ),

    "source_stage8a_manifest_sha256":
        sha256_file(
            STAGE8A_PATH
        ),

    "source_per_image_error_table":
        str(
            PER_IMAGE_PATH
        ),

    "source_per_image_error_table_sha256":
        sha256_file(
            PER_IMAGE_PATH
        ),

    "source_error_event_table":
        str(
            EVENT_PATH
        ),

    "source_error_event_table_sha256":
        sha256_file(
            EVENT_PATH
        ),

    "primary_protocol":
        PRIMARY_PROTOCOL,

    "primary_model":
        PRIMARY_MODEL,

    "reference_visualization_seed":
        REFERENCE_SEED,

    "reference_seed_role":
        (
            "Fixed illustrative seed only. "
            "Image selection was based on three-seed "
            "consensus statistics and was not based "
            "on the performance of seed 123."
        ),

    "selection_categories": [
        "small_object_failure",
        "high_miss_rate",
        "localization_error",
        "background_like_fp",
    ],

    "selection_policy":
        (
            "Images were selected deterministically from "
            "three-seed consensus error statistics. "
            "Normalized per-image error measures were used "
            "where appropriate to reduce bias toward images "
            "with large numbers of annotated objects."
        ),

    "primary_selection_image_ids":
        [
            int(x)
            for x in primary_selection[
                "image_id"
            ].tolist()
        ],

    "random_selection_image_ids":
        [
            int(x)
            for x in random_selection[
                "image_id"
            ].tolist()
        ],

    "primary_selection_content_sha256":
        dataframe_content_hash(
            primary_selection
        ),

    "qualitative_role":
        (
            "Illustrative error analysis only. "
            "Qualitative examples were not used for "
            "quantitative model ranking, threshold "
            "selection, retraining, or statistical inference."
        ),

    "matching_policy":
        (
            "Prediction classification reproduced the "
            "Stage 8A IoU-based matching logic at each "
            "validation-frozen operating threshold."
        ),

    "new_inference":
        False,

    "threshold_tuning":
        False,

    "post_test_retraining":
        False,

    "model_exclusion":
        False,

    "figure_resolution_dpi":
        300,

    "number_of_figures":
        3,

    "output_sha256":
        output_hashes,
}


# ============================================================
# 29. SAFE MANIFEST WRITE
# ============================================================

if MANIFEST_PATH.exists():

    previous_manifest = json.loads(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )


    old_compare = dict(
        previous_manifest
    )

    new_compare = dict(
        manifest
    )


    old_compare.pop(
        "created_utc",
        None
    )

    new_compare.pop(
        "created_utc",
        None
    )


    if old_compare != new_compare:

        raise RuntimeError(
            "A different STAGE_8B manifest already exists.\n"
            "It will NOT be silently overwritten."
        )


    print(
        "\nExisting Stage 8B manifest matches."
    )


else:

    MANIFEST_PATH.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )


# ============================================================
# 30. FINAL DISPLAY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "FINAL PRIMARY QUALITATIVE SELECTION"
)

print(
    "=" * 90
)


display(
    primary_selection[
        [
            "selection_category",
            "image_id",
            "file_name",
            "n_gt",
            "mean_tp",
            "mean_fp",
            "mean_fn",
            "mean_fn_rate_image",
            "mean_small_gt",
            "mean_small_fn",
            "mean_small_fn_rate",
            "mean_localization_fp",
            "mean_background_fp",
        ]
    ]
)


# ============================================================
# 31. FINAL
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "STAGE 8B PASSED"
)

print(
    "=" * 90
)

print(
    "Image selection method      : THREE-SEED CONSENSUS"
)

print(
    f"Primary protocol            : {PRIMARY_PROTOCOL}"
)

print(
    f"Primary model               : {PRIMARY_MODEL}"
)

print(
    f"Reference visualization seed: {REFERENCE_SEED}"
)

print(
    "Selected-image reproduction : PASSED"
)

print(
    "Primary figure              : COMPLETE"
)

print(
    "Cross-model figure          : COMPLETE"
)

print(
    "Random supplementary figure : COMPLETE"
)

print(
    "Figure resolution           : 300 dpi"
)

print(
    "New inference               : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Retraining                  : NO"
)

print(
    "Model exclusion             : NO"
)


print(
    "\nPrimary error figure:"
)

print(
    PRIMARY_FIG
)


print(
    "\nCross-model figure:"
)

print(
    CROSS_MODEL_FIG
)


print(
    "\nRandom supplementary figure:"
)

print(
    RANDOM_FIG
)


print(
    "\nSelection registry:"
)

print(
    REGISTRY_PATH
)


print(
    "\nPrimary selection details:"
)

print(
    PRIMARY_SELECTION_PATH
)


print(
    "\nManifest:"
)

print(
    MANIFEST_PATH
)


print(
    "\nNEXT: STAGE 9 — FINAL RESULTS FREEZE + "
    "MANUSCRIPT TABLES AND FIGURES."
)

print(
    "=" * 90
)

In [ ]:
# ============================================================
# STAGE 9 — FINAL RESULTS FREEZE
#           + MANUSCRIPT TABLES
#           + PUBLICATION FIGURES
#
# PURPOSE
# ------------------------------------------------------------
# Convert the completed experimental outputs into a
# manuscript-ready, reproducible final results package.
#
# IMPORTANT:
# - NO inference
# - NO retraining
# - NO threshold tuning
# - NO model exclusion
# - NO post-test optimization
# - NO new statistical hypothesis selection
#
# Outputs:
#   09_manuscript/
#       tables/
#       figures/
#       supplementary/
#       freeze/
#
# Main manuscript tables:
#   Table 1 — Dataset/split composition
#   Table 2 — Locked-test model performance
#   Table 3 — Object-size performance
#   Table 4 — Error taxonomy
#   Table 5 — Computational efficiency
#
# Figures:
#   Fig 1 — Locked-test mAP50-95 comparison
#   Fig 2 — Object-size AP (batch-disjoint)
#   Supp Fig — Seed stability
#   Supp Fig — Validation-to-test gap
#
# Existing Stage 8B qualitative figures are registered,
# not regenerated.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import json
import hashlib
import shutil

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

LOCAL_DATASET = Path(
    "/content/taco_v5/dataset"
)

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
}


MANUSCRIPT_ROOT = (
    ROOT /
    "09_manuscript"
)

TABLE_DIR = (
    MANUSCRIPT_ROOT /
    "tables"
)

FIG_DIR = (
    MANUSCRIPT_ROOT /
    "figures"
)

SUPP_DIR = (
    MANUSCRIPT_ROOT /
    "supplementary"
)

FREEZE_DIR = (
    MANUSCRIPT_ROOT /
    "freeze"
)


for p in [
    MANUSCRIPT_ROOT,
    TABLE_DIR,
    FIG_DIR,
    SUPP_DIR,
    FREEZE_DIR,
]:

    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 2. EXPERIMENT STRUCTURE
# ============================================================

PROTOCOLS = [
    "random",
    "batch_disjoint",
]

MODELS = [
    "yolov8s",
    "yolo11s",
    "yolo26s",
    "rtdetr_l",
]

SEEDS = [
    42,
    123,
    2026,
]


DISPLAY_MODEL = {
    "yolov8s":
        "YOLOv8s",

    "yolo11s":
        "YOLO11s",

    "yolo26s":
        "YOLO26s",

    "rtdetr_l":
        "RT-DETR-L",
}


DISPLAY_PROTOCOL = {
    "random":
        "Random",

    "batch_disjoint":
        "Batch-disjoint",
}


# ============================================================
# 3. HELPERS
# ============================================================

def sha256_file(
    path,
    chunk=1024 * 1024
):

    path = Path(
        path
    )

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(chunk),
            b""
        ):

            h.update(
                block
            )

    return h.hexdigest()


def fmt_mean_sd(
    mean,
    sd,
    digits=4
):

    return (
        f"{float(mean):.{digits}f} "
        f"± {float(sd):.{digits}f}"
    )


def safe_copy(
    source,
    destination
):

    source = Path(
        source
    )

    destination = Path(
        destination
    )


    if not source.exists():

        raise FileNotFoundError(
            source
        )


    shutil.copy2(
        source,
        destination
    )


# ============================================================
# 4. REQUIRED MANIFESTS
# ============================================================

required_manifests = {

    "PRE_LOCKED_TEST_DECISION":
        DIRS["manifests"] /
        "PRE_LOCKED_TEST_DECISION.json",

    "LOCKED_TEST_COMPLETED":
        DIRS["manifests"] /
        "LOCKED_TEST_COMPLETED.json",

    "STAGE_7B":
        DIRS["manifests"] /
        "STAGE_7B_FINAL_TEST_AGGREGATION.json",

    "STAGE_7C":
        DIRS["manifests"] /
        "STAGE_7C_BOOTSTRAP.json",

    "STAGE_8A":
        DIRS["manifests"] /
        "STAGE_8A_ERROR_TAXONOMY.json",

    "STAGE_8B":
        DIRS["manifests"] /
        "STAGE_8B_QUALITATIVE_VISUALIZATION.json",
}


print("=" * 92)
print("STAGE 9 — FINAL RESULTS FREEZE + MANUSCRIPT PACKAGE")
print("=" * 92)


for name, path in required_manifests.items():

    if not path.exists():

        raise FileNotFoundError(
            f"Required manifest missing:\n"
            f"{name}: {path}"
        )


print(
    "\nRequired experiment manifests : 6/6 PASSED"
)


# ============================================================
# 5. REQUIRED NUMERIC RESULTS
# ============================================================

TEST_SEED_PATH = (
    DIRS["test"] /
    "locked_test_summary_all_runs.csv"
)

TEST_AGG_PATH = (
    DIRS["test"] /
    "locked_test_aggregate_mean_sd.csv"
)

OBJECT_SIZE_PATH = (
    DIRS["analysis"] /
    "locked_test_object_size_analysis.csv"
)

ERROR_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_mean_sd.csv"
)

EFFICIENCY_PATH = (
    DIRS["analysis"] /
    "locked_test_efficiency_summary.csv"
)

GENERALIZATION_PATH = (
    DIRS["analysis"] /
    "validation_to_test_gap_mean_sd.csv"
)

PROTOCOL_GAP_PATH = (
    DIRS["test"] /
    "locked_test_protocol_descriptive_gap.csv"
)

BOOTSTRAP_MODEL_PATH = (
    DIRS["analysis"] /
    "bootstrap_model_global_f1_ci.csv"
)

BOOTSTRAP_PAIRWISE_PATH = (
    DIRS["analysis"] /
    "bootstrap_pairwise_global_f1_differences.csv"
)

ERROR_SEED_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_seed_level.csv"
)


numeric_inputs = {
    "locked_test_seed":
        TEST_SEED_PATH,

    "locked_test_aggregate":
        TEST_AGG_PATH,

    "object_size":
        OBJECT_SIZE_PATH,

    "error_taxonomy":
        ERROR_PATH,

    "efficiency":
        EFFICIENCY_PATH,

    "generalization_gap":
        GENERALIZATION_PATH,

    "protocol_gap":
        PROTOCOL_GAP_PATH,

    "bootstrap_model":
        BOOTSTRAP_MODEL_PATH,

    "bootstrap_pairwise":
        BOOTSTRAP_PAIRWISE_PATH,

    "error_seed":
        ERROR_SEED_PATH,
}


for name, path in numeric_inputs.items():

    if not path.exists():

        raise FileNotFoundError(
            f"Required numeric result missing:\n"
            f"{name}: {path}"
        )


print(
    "Required numeric outputs     : 10/10 PASSED"
)


# ============================================================
# 6. LOAD RESULTS
# ============================================================

seed_df = pd.read_csv(
    TEST_SEED_PATH
)

agg = pd.read_csv(
    TEST_AGG_PATH
)

object_size = pd.read_csv(
    OBJECT_SIZE_PATH
)

error_agg = pd.read_csv(
    ERROR_PATH
)

efficiency = pd.read_csv(
    EFFICIENCY_PATH
)

generalization = pd.read_csv(
    GENERALIZATION_PATH
)

protocol_gap = pd.read_csv(
    PROTOCOL_GAP_PATH
)

bootstrap_model = pd.read_csv(
    BOOTSTRAP_MODEL_PATH
)

bootstrap_pairwise = pd.read_csv(
    BOOTSTRAP_PAIRWISE_PATH
)


# ============================================================
# 7. FINAL STRUCTURAL CHECKS
# ============================================================

if len(
    seed_df
) != 24:

    raise RuntimeError(
        "Expected 24 locked-test seed-level runs."
    )


if len(
    agg
) != 8:

    raise RuntimeError(
        "Expected 8 protocol/model aggregate rows."
    )


if len(
    bootstrap_model
) != 8:

    raise RuntimeError(
        "Expected 8 bootstrap model rows."
    )


if len(
    bootstrap_pairwise
) != 12:

    raise RuntimeError(
        "Expected 12 pairwise bootstrap rows."
    )


for protocol in PROTOCOLS:

    for model in MODELS:

        d = seed_df[
            (
                seed_df[
                    "protocol"
                ] == protocol
            )
            &
            (
                seed_df[
                    "model"
                ] == model
            )
        ]


        if set(
            d[
                "seed"
            ].astype(
                int
            )
        ) != set(
            SEEDS
        ):

            raise RuntimeError(
                f"Seed structure failed: "
                f"{protocol}/{model}"
            )


print(
    "Final result structure       : PASSED"
)


# ============================================================
# 8. TABLE 1 — DATASET / SPLIT COMPOSITION
# ============================================================

split_rows = []


for protocol in PROTOCOLS:

    for split in [
        "train",
        "val",
        "test",
    ]:

        txt_path = (
            LOCAL_DATASET /
            protocol /
            f"{split}.txt"
        )

        coco_path = (
            LOCAL_DATASET /
            protocol /
            f"{split}_coco_taco1.json"
        )


        if not txt_path.exists():

            raise FileNotFoundError(
                txt_path
            )


        if not coco_path.exists():

            raise FileNotFoundError(
                coco_path
            )


        image_paths = [

            x.strip()

            for x in txt_path.read_text(
                encoding="utf-8"
            ).splitlines()

            if x.strip()
        ]


        coco = json.loads(
            coco_path.read_text(
                encoding="utf-8"
            )
        )


        if len(
            image_paths
        ) != len(
            coco[
                "images"
            ]
        ):

            raise RuntimeError(
                f"Split image mismatch: "
                f"{protocol}/{split}"
            )


        areas = np.array(
            [
                float(
                    ann.get(
                        "area",
                        ann[
                            "bbox"
                        ][2]
                        *
                        ann[
                            "bbox"
                        ][3]
                    )
                )

                for ann in coco[
                    "annotations"
                ]
            ],
            dtype=float
        )


        small = int(
            np.sum(
                areas < 32**2
            )
        )

        medium = int(
            np.sum(
                (
                    areas >= 32**2
                )
                &
                (
                    areas < 96**2
                )
            )
        )

        large = int(
            np.sum(
                areas >= 96**2
            )
        )


        split_rows.append(
            {
                "Protocol":
                    DISPLAY_PROTOCOL[
                        protocol
                    ],

                "Split":
                    split.capitalize(),

                "Images":
                    len(
                        coco[
                            "images"
                        ]
                    ),

                "Objects":
                    len(
                        coco[
                            "annotations"
                        ]
                    ),

                "Small objects":
                    small,

                "Medium objects":
                    medium,

                "Large objects":
                    large,
            }
        )


table1 = pd.DataFrame(
    split_rows
)


TABLE1_PATH = (
    TABLE_DIR /
    "Table_1_dataset_split_composition.csv"
)


table1.to_csv(
    TABLE1_PATH,
    index=False
)


# ============================================================
# 9. TABLE 2 — MAIN LOCKED-TEST PERFORMANCE
# ============================================================

table2_rows = []


for protocol in PROTOCOLS:

    for model in MODELS:

        r = agg[
            (
                agg[
                    "protocol"
                ] == protocol
            )
            &
            (
                agg[
                    "model"
                ] == model
            )
        ]


        if len(r) != 1:

            raise RuntimeError(
                f"Missing aggregate result: "
                f"{protocol}/{model}"
            )


        r = r.iloc[
            0
        ]


        table2_rows.append(
            {
                "Protocol":
                    DISPLAY_PROTOCOL[
                        protocol
                    ],

                "Model":
                    DISPLAY_MODEL[
                        model
                    ],

                "mAP50-95":
                    fmt_mean_sd(
                        r[
                            "mAP50_95_mean"
                        ],
                        r[
                            "mAP50_95_sd"
                        ]
                    ),

                "AP50":
                    fmt_mean_sd(
                        r[
                            "AP50_mean"
                        ],
                        r[
                            "AP50_sd"
                        ]
                    ),

                "AP75":
                    fmt_mean_sd(
                        r[
                            "AP75_mean"
                        ],
                        r[
                            "AP75_sd"
                        ]
                    ),

                "Precision":
                    fmt_mean_sd(
                        r[
                            "test_precision_mean"
                        ],
                        r[
                            "test_precision_sd"
                        ]
                    ),

                "Recall":
                    fmt_mean_sd(
                        r[
                            "test_recall_mean"
                        ],
                        r[
                            "test_recall_sd"
                        ]
                    ),

                "F1":
                    fmt_mean_sd(
                        r[
                            "test_f1_mean"
                        ],
                        r[
                            "test_f1_sd"
                        ]
                    ),
            }
        )


table2 = pd.DataFrame(
    table2_rows
)


TABLE2_PATH = (
    TABLE_DIR /
    "Table_2_locked_test_model_performance.csv"
)


table2.to_csv(
    TABLE2_PATH,
    index=False
)


# ============================================================
# 10. TABLE 3 — OBJECT-SIZE PERFORMANCE
# ============================================================

table3_rows = []


for r in object_size.itertuples():

    table3_rows.append(
        {
            "Protocol":
                DISPLAY_PROTOCOL[
                    r.protocol
                ],

            "Model":
                DISPLAY_MODEL[
                    r.model
                ],

            "AP-small":
                fmt_mean_sd(
                    r.AP_small_mean,
                    r.AP_small_sd
                ),

            "AP-medium":
                fmt_mean_sd(
                    r.AP_medium_mean,
                    r.AP_medium_sd
                ),

            "AP-large":
                fmt_mean_sd(
                    r.AP_large_mean,
                    r.AP_large_sd
                ),

            "AR-small":
                fmt_mean_sd(
                    r.AR_small_mean,
                    r.AR_small_sd
                ),

            "AR-medium":
                fmt_mean_sd(
                    r.AR_medium_mean,
                    r.AR_medium_sd
                ),

            "AR-large":
                fmt_mean_sd(
                    r.AR_large_mean,
                    r.AR_large_sd
                ),

            "AP large-small gap":
                f"{r.AP_large_minus_small:.4f}",
        }
    )


table3 = pd.DataFrame(
    table3_rows
)


TABLE3_PATH = (
    TABLE_DIR /
    "Table_3_object_size_performance.csv"
)


table3.to_csv(
    TABLE3_PATH,
    index=False
)


# ============================================================
# 11. TABLE 4 — ERROR TAXONOMY
# ============================================================

table4_rows = []


for r in error_agg.itertuples():

    table4_rows.append(
        {
            "Protocol":
                DISPLAY_PROTOCOL[
                    r.protocol
                ],

            "Model":
                DISPLAY_MODEL[
                    r.model
                ],

            "FN rate (%)":
                fmt_mean_sd(
                    100.0
                    *
                    r.fn_rate_mean,

                    100.0
                    *
                    r.fn_rate_sd,

                    2
                ),

            "Small FN rate (%)":
                fmt_mean_sd(
                    100.0
                    *
                    r.small_fn_rate_mean,

                    100.0
                    *
                    r.small_fn_rate_sd,

                    2
                ),

            "Medium FN rate (%)":
                fmt_mean_sd(
                    100.0
                    *
                    r.medium_fn_rate_mean,

                    100.0
                    *
                    r.medium_fn_rate_sd,

                    2
                ),

            "Large FN rate (%)":
                fmt_mean_sd(
                    100.0
                    *
                    r.large_fn_rate_mean,

                    100.0
                    *
                    r.large_fn_rate_sd,

                    2
                ),

            "Duplicate FP (%)":
                fmt_mean_sd(
                    100.0
                    *
                    r.duplicate_share_of_fp_mean,

                    100.0
                    *
                    r.duplicate_share_of_fp_sd,

                    2
                ),

            "Localization FP (%)":
                fmt_mean_sd(
                    100.0
                    *
                    r.localization_share_of_fp_mean,

                    100.0
                    *
                    r.localization_share_of_fp_sd,

                    2
                ),

            "Background-like FP (%)":
                fmt_mean_sd(
                    100.0
                    *
                    r.background_share_of_fp_mean,

                    100.0
                    *
                    r.background_share_of_fp_sd,

                    2
                ),

            "Mean TP IoU":
                fmt_mean_sd(
                    r.mean_tp_iou_mean,
                    r.mean_tp_iou_sd
                ),
        }
    )


table4 = pd.DataFrame(
    table4_rows
)


TABLE4_PATH = (
    TABLE_DIR /
    "Table_4_error_taxonomy.csv"
)


table4.to_csv(
    TABLE4_PATH,
    index=False
)


# ============================================================
# 12. TABLE 5 — COMPUTATIONAL EFFICIENCY
# ============================================================

table5_rows = []


for r in efficiency.itertuples():

    table5_rows.append(
        {
            "Protocol":
                DISPLAY_PROTOCOL[
                    r.protocol
                ],

            "Model":
                DISPLAY_MODEL[
                    r.model
                ],

            "Inference (ms/image)":
                fmt_mean_sd(
                    r.inference_ms_mean_mean,
                    r.inference_ms_mean_sd
                ),

            "Processing (ms/image)":
                fmt_mean_sd(
                    r.processing_ms_mean_mean,
                    r.processing_ms_mean_sd
                ),

            "Processing FPS":
                fmt_mean_sd(
                    r.processing_fps_mean,
                    r.processing_fps_sd,
                    2
                ),

            "mAP50-95":
                fmt_mean_sd(
                    r.mAP50_95_mean,
                    r.mAP50_95_sd
                ),
        }
    )


table5 = pd.DataFrame(
    table5_rows
)


TABLE5_PATH = (
    TABLE_DIR /
    "Table_5_efficiency.csv"
)


table5.to_csv(
    TABLE5_PATH,
    index=False
)


# ============================================================
# 13. SUPPLEMENTARY TABLES — COPY WITHOUT RECOMPUTATION
# ============================================================

supplementary_sources = {

    "Supplementary_Table_S1_seed_level_locked_test.csv":
        TEST_SEED_PATH,

    "Supplementary_Table_S2_protocol_descriptive_gap.csv":
        PROTOCOL_GAP_PATH,

    "Supplementary_Table_S3_validation_to_test_gap.csv":
        GENERALIZATION_PATH,

    "Supplementary_Table_S4_bootstrap_model_CI.csv":
        BOOTSTRAP_MODEL_PATH,

    "Supplementary_Table_S5_bootstrap_pairwise.csv":
        BOOTSTRAP_PAIRWISE_PATH,

    "Supplementary_Table_S6_error_taxonomy_seed_level.csv":
        ERROR_SEED_PATH,
}


for name, source in supplementary_sources.items():

    safe_copy(
        source,
        SUPP_DIR /
        name
    )


# ============================================================
# 14. FIGURE 1 — LOCKED TEST mAP50-95
# ============================================================

FIG1_PATH = (
    FIG_DIR /
    "Fig_1_locked_test_mAP50_95.png"
)


x = np.arange(
    len(
        MODELS
    )
)

width = 0.36


random_rows = (
    agg[
        agg[
            "protocol"
        ] == "random"
    ]
    .set_index(
        "model"
    )
    .loc[
        MODELS
    ]
)


batch_rows = (
    agg[
        agg[
            "protocol"
        ] == "batch_disjoint"
    ]
    .set_index(
        "model"
    )
    .loc[
        MODELS
    ]
)


fig, ax = plt.subplots(
    figsize=(
        9,
        5.5
    )
)


ax.bar(
    x - width / 2,
    random_rows[
        "mAP50_95_mean"
    ],
    width,
    yerr=random_rows[
        "mAP50_95_sd"
    ],
    capsize=4,
    label="Random"
)


ax.bar(
    x + width / 2,
    batch_rows[
        "mAP50_95_mean"
    ],
    width,
    yerr=batch_rows[
        "mAP50_95_sd"
    ],
    capsize=4,
    label="Batch-disjoint"
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    [
        DISPLAY_MODEL[m]
        for m in MODELS
    ]
)

ax.set_ylabel(
    "COCO mAP@[0.50:0.95]"
)

ax.set_xlabel(
    "Model"
)

ax.set_title(
    "Locked-test detection performance across three seeds"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()


fig.savefig(
    FIG1_PATH,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


plt.show()

plt.close(
    fig
)


# ============================================================
# 15. FIGURE 2 — OBJECT-SIZE AP
#     PRIMARY BATCH-DISJOINT PROTOCOL
# ============================================================

FIG2_PATH = (
    FIG_DIR /
    "Fig_2_batch_disjoint_object_size_AP.png"
)


size_df = (
    object_size[
        object_size[
            "protocol"
        ] == "batch_disjoint"
    ]
    .set_index(
        "model"
    )
    .loc[
        MODELS
    ]
)


x = np.arange(
    len(
        MODELS
    )
)

width = 0.24


fig, ax = plt.subplots(
    figsize=(
        9,
        5.5
    )
)


ax.bar(
    x - width,
    size_df[
        "AP_small_mean"
    ],
    width,
    yerr=size_df[
        "AP_small_sd"
    ],
    capsize=3,
    label="Small"
)


ax.bar(
    x,
    size_df[
        "AP_medium_mean"
    ],
    width,
    yerr=size_df[
        "AP_medium_sd"
    ],
    capsize=3,
    label="Medium"
)


ax.bar(
    x + width,
    size_df[
        "AP_large_mean"
    ],
    width,
    yerr=size_df[
        "AP_large_sd"
    ],
    capsize=3,
    label="Large"
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    [
        DISPLAY_MODEL[m]
        for m in MODELS
    ]
)

ax.set_ylabel(
    "COCO AP@[0.50:0.95]"
)

ax.set_xlabel(
    "Model"
)

ax.set_title(
    "Object-size sensitivity on the batch-disjoint locked test"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()


fig.savefig(
    FIG2_PATH,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


plt.show()

plt.close(
    fig
)


# ============================================================
# 16. SUPPLEMENTARY FIGURE — SEED STABILITY
# ============================================================

SUPP_SEED_FIG = (
    SUPP_DIR /
    "Supplementary_Figure_S1_seed_stability.png"
)


fig, ax = plt.subplots(
    figsize=(
        10,
        5.8
    )
)


offsets = {
    "random":
        -0.12,

    "batch_disjoint":
        0.12,
}


markers = {
    "random":
        "o",

    "batch_disjoint":
        "s",
}


for protocol in PROTOCOLS:

    for model_index, model in enumerate(
        MODELS
    ):

        g = seed_df[
            (
                seed_df[
                    "protocol"
                ] == protocol
            )
            &
            (
                seed_df[
                    "model"
                ] == model
            )
        ].sort_values(
            "seed"
        )


        xpos = np.full(
            len(g),
            model_index
            +
            offsets[
                protocol
            ],
            dtype=float
        )


        ax.scatter(
            xpos,
            g[
                "mAP50_95"
            ],
            marker=markers[
                protocol
            ],
            s=55,
            label=(
                DISPLAY_PROTOCOL[
                    protocol
                ]
                if model_index == 0
                else None
            )
        )


        ax.plot(
            xpos,
            g[
                "mAP50_95"
            ],
            linewidth=0.7,
            alpha=0.5
        )


ax.set_xticks(
    np.arange(
        len(
            MODELS
        )
    )
)

ax.set_xticklabels(
    [
        DISPLAY_MODEL[m]
        for m in MODELS
    ]
)

ax.set_ylabel(
    "COCO mAP@[0.50:0.95]"
)

ax.set_xlabel(
    "Model"
)

ax.set_title(
    "Locked-test seed variability"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()


fig.savefig(
    SUPP_SEED_FIG,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


plt.show()

plt.close(
    fig
)


# ============================================================
# 17. SUPPLEMENTARY FIGURE — VALIDATION -> TEST GAP
# ============================================================

SUPP_GAP_FIG = (
    SUPP_DIR /
    "Supplementary_Figure_S2_validation_to_test_gap.png"
)


gap_col = (
    "delta_mAP50_95_test_minus_val_mean"
)


gap_sd_col = (
    "delta_mAP50_95_test_minus_val_sd"
)


random_gap = (
    generalization[
        generalization[
            "protocol"
        ] == "random"
    ]
    .set_index(
        "model"
    )
    .loc[
        MODELS
    ]
)


batch_gap = (
    generalization[
        generalization[
            "protocol"
        ] == "batch_disjoint"
    ]
    .set_index(
        "model"
    )
    .loc[
        MODELS
    ]
)


x = np.arange(
    len(
        MODELS
    )
)

width = 0.36


fig, ax = plt.subplots(
    figsize=(
        9,
        5.5
    )
)


ax.bar(
    x - width / 2,
    random_gap[
        gap_col
    ],
    width,
    yerr=random_gap[
        gap_sd_col
    ],
    capsize=4,
    label="Random"
)


ax.bar(
    x + width / 2,
    batch_gap[
        gap_col
    ],
    width,
    yerr=batch_gap[
        gap_sd_col
    ],
    capsize=4,
    label="Batch-disjoint"
)


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    [
        DISPLAY_MODEL[m]
        for m in MODELS
    ]
)

ax.set_ylabel(
    "mAP50-95 difference (test − validation)"
)

ax.set_xlabel(
    "Model"
)

ax.set_title(
    "Validation-to-locked-test performance change"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()


fig.savefig(
    SUPP_GAP_FIG,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


plt.show()

plt.close(
    fig
)


# ============================================================
# 18. REGISTER EXISTING QUALITATIVE FIGURES
# ============================================================

QUAL_DIR = (
    DIRS["analysis"] /
    "qualitative_figures"
)


QUAL_PRIMARY = (
    QUAL_DIR /
    "Fig_error_examples_batch_disjoint_yolo26s_seed123.png"
)

QUAL_CROSS = (
    QUAL_DIR /
    "Fig_cross_model_hard_cases_batch_disjoint_seed123.png"
)

QUAL_RANDOM = (
    QUAL_DIR /
    "Supp_error_examples_random_yolo26s_seed123.png"
)


for p in [
    QUAL_PRIMARY,
    QUAL_CROSS,
    QUAL_RANDOM,
]:

    if not p.exists():

        raise FileNotFoundError(
            f"Stage 8B figure missing:\n{p}"
        )


# ------------------------------------------------------------
# Copy primary qualitative figures into manuscript package.
# Original files remain untouched.
# ------------------------------------------------------------

FINAL_QUAL_PRIMARY = (
    FIG_DIR /
    "Fig_3_error_examples_batch_disjoint_yolo26s.png"
)

FINAL_QUAL_CROSS = (
    FIG_DIR /
    "Fig_4_cross_model_hard_cases.png"
)

FINAL_QUAL_RANDOM = (
    SUPP_DIR /
    "Supplementary_Figure_S3_random_error_examples.png"
)


safe_copy(
    QUAL_PRIMARY,
    FINAL_QUAL_PRIMARY
)

safe_copy(
    QUAL_CROSS,
    FINAL_QUAL_CROSS
)

safe_copy(
    QUAL_RANDOM,
    FINAL_QUAL_RANDOM
)


# ============================================================
# 19. FINAL DESCRIPTIVE RANKING
#
# IMPORTANT:
# No significance claim.
# Main ranking is mean mAP50-95.
# ============================================================

ranking_rows = []


for protocol in PROTOCOLS:

    d = (
        agg[
            agg[
                "protocol"
            ] == protocol
        ]
        .sort_values(
            [
                "mAP50_95_mean",
                "mAP50_95_sd",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .reset_index(
            drop=True
        )
    )


    for i, r in d.iterrows():

        ranking_rows.append(
            {
                "Protocol":
                    DISPLAY_PROTOCOL[
                        protocol
                    ],

                "Rank":
                    i + 1,

                "Model":
                    DISPLAY_MODEL[
                        r[
                            "model"
                        ]
                    ],

                "Mean mAP50-95":
                    float(
                        r[
                            "mAP50_95_mean"
                        ]
                    ),

                "SD":
                    float(
                        r[
                            "mAP50_95_sd"
                        ]
                    ),
            }
        )


final_ranking = pd.DataFrame(
    ranking_rows
)


RANKING_PATH = (
    TABLE_DIR /
    "descriptive_locked_test_ranking.csv"
)


final_ranking.to_csv(
    RANKING_PATH,
    index=False
)


# ============================================================
# 20. AUTOMATIC FINAL SCIENTIFIC SUMMARY
# ============================================================

random_best = (
    agg[
        agg[
            "protocol"
        ] == "random"
    ]
    .sort_values(
        "mAP50_95_mean",
        ascending=False
    )
    .iloc[
        0
    ]
)


batch_best = (
    agg[
        agg[
            "protocol"
        ] == "batch_disjoint"
    ]
    .sort_values(
        "mAP50_95_mean",
        ascending=False
    )
    .iloc[
        0
    ]
)


if (
    random_best[
        "model"
    ] != "yolo26s"
):

    raise RuntimeError(
        "Unexpected random-protocol top model."
    )


if (
    batch_best[
        "model"
    ] != "yolo26s"
):

    raise RuntimeError(
        "Unexpected batch-disjoint top model."
    )


# ------------------------------------------------------------
# YOLO26s protocol gap
# ------------------------------------------------------------

y26_gap = protocol_gap[
    protocol_gap[
        "model"
    ] == "yolo26s"
]


if len(
    y26_gap
) != 1:

    raise RuntimeError(
        "YOLO26s protocol gap row missing."
    )


y26_gap = y26_gap.iloc[
    0
]


# ------------------------------------------------------------
# Batch YOLO26s object-size row
# ------------------------------------------------------------

y26_size = object_size[
    (
        object_size[
            "protocol"
        ] == "batch_disjoint"
    )
    &
    (
        object_size[
            "model"
        ] == "yolo26s"
    )
]


if len(
    y26_size
) != 1:

    raise RuntimeError(
        "Batch-disjoint YOLO26s size row missing."
    )


y26_size = y26_size.iloc[
    0
]


# ------------------------------------------------------------
# RT-DETR seed variability
# ------------------------------------------------------------

rtd_random = agg[
    (
        agg[
            "protocol"
        ] == "random"
    )
    &
    (
        agg[
            "model"
        ] == "rtdetr_l"
    )
].iloc[
    0
]


# ============================================================
# 21. FINAL RESULTS SUMMARY TEXT
# ============================================================

SUMMARY_PATH = (
    MANUSCRIPT_ROOT /
    "FINAL_RESULTS_SUMMARY.txt"
)


summary_text = f"""
TACO PIPELINE v5 — FINAL RESULTS SUMMARY

STUDY STATUS
============
All pre-specified training and locked-test evaluations are complete.

Models:
- YOLOv8s
- YOLO11s
- YOLO26s
- RT-DETR-L

Protocols:
- Random
- Batch-disjoint

Seeds:
- 42
- 123
- 2026

Total locked-test evaluations:
24


PRIMARY LOCKED-TEST RESULT
==========================
YOLO26s achieved the highest mean COCO mAP@[0.50:0.95]
under both evaluation protocols.

Random:
YOLO26s = {random_best['mAP50_95_mean']:.4f} ± {random_best['mAP50_95_sd']:.4f}

Batch-disjoint:
YOLO26s = {batch_best['mAP50_95_mean']:.4f} ± {batch_best['mAP50_95_sd']:.4f}

This is a descriptive ranking.
No formal statistical superiority claim is made for mAP50-95.


PROTOCOL SENSITIVITY
====================
For YOLO26s:

batch-disjoint minus random mAP50-95
= {y26_gap['batch_minus_random_mAP50_95']:.4f}

The protocol-specific train and test sets differ.
This difference is descriptive and is not interpreted as
a paired or causal protocol effect.


OBJECT-SIZE PERFORMANCE
=======================
Batch-disjoint YOLO26s:

AP-small  = {y26_size['AP_small_mean']:.4f}
AP-medium = {y26_size['AP_medium_mean']:.4f}
AP-large  = {y26_size['AP_large_mean']:.4f}

Large-minus-small AP gap
= {y26_size['AP_large_minus_small']:.4f}

Small-object detection remains the principal size-related
performance limitation.


SEED STABILITY
==============
Random RT-DETR-L:

mAP50-95 =
{rtd_random['mAP50_95_mean']:.4f} ± {rtd_random['mAP50_95_sd']:.4f}

CV =
{rtd_random['mAP50_95_cv_percent']:.2f}%

RT-DETR-L showed substantially greater seed variability than
the YOLO models in the random protocol.


BOOTSTRAP
=========
Paired image-level bootstrap:
5000 replicates.

Bootstrap analyses compare models only within the same
protocol/test-image set.

Bootstrap intervals quantify image-sampling uncertainty
conditional on the three observed seeds.

They do not replace between-seed variability estimates.


ERROR ANALYSIS
==============
False negatives were the dominant operating-point limitation.

Small-object false-negative rates were substantially higher
than large-object false-negative rates.

Most false positives were background-like rather than
duplicate detections.

Qualitative examples were selected deterministically using
three-seed consensus error statistics.


INTERPRETATION RULES
====================
DO NOT claim:
- batch-disjoint is universally harder
- scene-disjoint evaluation
- YOLO26s is statistically superior on mAP50-95
- framework processing FPS equals end-to-end deployment FPS
- the results demonstrate field deployment performance

PERMITTED DESCRIPTIVE CLAIMS:
- YOLO26s had the highest mean locked-test mAP50-95 under both protocols.
- Small-object detection was substantially poorer than large-object detection.
- RT-DETR-L exhibited comparatively high seed variability.
- Locked-test performance was lower than validation performance across all model/protocol combinations.
- Random and batch-disjoint results were broadly similar for YOLO26s and YOLO11s, while YOLOv8s showed a larger descriptive protocol difference.
""".strip()


SUMMARY_PATH.write_text(
    summary_text
    +
    "\n",
    encoding="utf-8"
)


# ============================================================
# 22. FINAL NUMERIC SOURCE HASH REGISTRY
# ============================================================

source_hash_rows = []


for name, path in {
    **required_manifests,
    **numeric_inputs,
}.items():

    source_hash_rows.append(
        {
            "artifact":
                name,

            "path":
                str(
                    path
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


source_hash_registry = pd.DataFrame(
    source_hash_rows
)


SOURCE_HASH_PATH = (
    FREEZE_DIR /
    "final_source_hash_registry.csv"
)


source_hash_registry.to_csv(
    SOURCE_HASH_PATH,
    index=False
)


# ============================================================
# 23. MANUSCRIPT OUTPUT HASH REGISTRY
# ============================================================

manuscript_outputs = {

    "Table_1_dataset_split_composition.csv":
        TABLE1_PATH,

    "Table_2_locked_test_model_performance.csv":
        TABLE2_PATH,

    "Table_3_object_size_performance.csv":
        TABLE3_PATH,

    "Table_4_error_taxonomy.csv":
        TABLE4_PATH,

    "Table_5_efficiency.csv":
        TABLE5_PATH,

    "descriptive_locked_test_ranking.csv":
        RANKING_PATH,

    "Fig_1_locked_test_mAP50_95.png":
        FIG1_PATH,

    "Fig_2_batch_disjoint_object_size_AP.png":
        FIG2_PATH,

    "Fig_3_error_examples_batch_disjoint_yolo26s.png":
        FINAL_QUAL_PRIMARY,

    "Fig_4_cross_model_hard_cases.png":
        FINAL_QUAL_CROSS,

    "Supplementary_Figure_S1_seed_stability.png":
        SUPP_SEED_FIG,

    "Supplementary_Figure_S2_validation_to_test_gap.png":
        SUPP_GAP_FIG,

    "Supplementary_Figure_S3_random_error_examples.png":
        FINAL_QUAL_RANDOM,

    "FINAL_RESULTS_SUMMARY.txt":
        SUMMARY_PATH,
}


output_hash_rows = []


for name, path in manuscript_outputs.items():

    if not Path(
        path
    ).exists():

        raise FileNotFoundError(
            path
        )


    output_hash_rows.append(
        {
            "artifact":
                name,

            "path":
                str(
                    path
                ),

            "sha256":
                sha256_file(
                    path
                ),

            "size_bytes":
                Path(
                    path
                ).stat().st_size,
        }
    )


output_hash_registry = pd.DataFrame(
    output_hash_rows
)


OUTPUT_HASH_PATH = (
    FREEZE_DIR /
    "final_manuscript_output_hash_registry.csv"
)


output_hash_registry.to_csv(
    OUTPUT_HASH_PATH,
    index=False
)


# ============================================================
# 24. FINAL RESULTS FREEZE MANIFEST
# ============================================================

FINAL_MANIFEST_PATH = (
    DIRS["manifests"] /
    "FINAL_RESULTS_FREEZE.json"
)


final_manifest = {

    "status":
        "FINAL_RESULTS_FROZEN",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "project":
        "TACO_Pipeline_v5",

    "task":
        "single-class litter object detection",

    "dataset":
        "TACO",

    "models":
        MODELS,

    "protocols":
        PROTOCOLS,

    "seeds":
        SEEDS,

    "number_of_locked_test_runs":
        24,

    "primary_metric":
        "COCO mAP@[0.50:0.95]",

    "main_locked_test_interpretation":
        (
            "YOLO26s had the highest descriptive mean "
            "mAP50-95 under both random and batch-disjoint "
            "locked-test protocols."
        ),

    "primary_protocol_for_robustness_discussion":
        "batch_disjoint",

    "statistical_secondary_analysis":
        (
            "5000-replicate paired image-level bootstrap "
            "for global F1 within each protocol."
        ),

    "between_protocol_inference":
        False,

    "protocol_gap_policy":
        (
            "Random versus batch-disjoint differences "
            "are descriptive because protocol-specific "
            "training and test compositions differ."
        ),

    "object_size_analysis":
        True,

    "error_taxonomy":
        True,

    "qualitative_error_analysis":
        True,

    "new_inference_after_locked_test":
        False,

    "test_threshold_tuning":
        False,

    "post_test_retraining":
        False,

    "post_test_model_exclusion":
        False,

    "future_experimental_changes_permitted":
        False,

    "future_activity":
        (
            "Manuscript preparation, formatting, "
            "figure inspection, reporting corrections, "
            "and reproducibility packaging only. "
            "Any genuinely new experiment must be clearly "
            "identified as a new post hoc analysis."
        ),

    "source_hash_registry":
        str(
            SOURCE_HASH_PATH
        ),

    "source_hash_registry_sha256":
        sha256_file(
            SOURCE_HASH_PATH
        ),

    "manuscript_output_hash_registry":
        str(
            OUTPUT_HASH_PATH
        ),

    "manuscript_output_hash_registry_sha256":
        sha256_file(
            OUTPUT_HASH_PATH
        ),

    "final_results_summary":
        str(
            SUMMARY_PATH
        ),

    "final_results_summary_sha256":
        sha256_file(
            SUMMARY_PATH
        ),
}


# ============================================================
# 25. SAFE FINAL FREEZE
# ============================================================

if FINAL_MANIFEST_PATH.exists():

    previous = json.loads(
        FINAL_MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )


    old_compare = dict(
        previous
    )

    new_compare = dict(
        final_manifest
    )


    old_compare.pop(
        "created_utc",
        None
    )

    new_compare.pop(
        "created_utc",
        None
    )


    if old_compare != new_compare:

        raise RuntimeError(
            "A different FINAL_RESULTS_FREEZE.json "
            "already exists.\n"
            "Final results will NOT be silently overwritten."
        )


    print(
        "\nExisting final-results freeze matches."
    )


else:

    FINAL_MANIFEST_PATH.write_text(
        json.dumps(
            final_manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )


# ============================================================
# 26. HUMAN-READABLE FREEZE NOTICE
# ============================================================

FREEZE_NOTICE_PATH = (
    FREEZE_DIR /
    "FINAL_RESULTS_FROZEN.txt"
)


freeze_notice = f"""
TACO PIPELINE v5 — FINAL RESULTS FROZEN

Freeze timestamp (UTC):
{final_manifest['created_utc']}

All 24 locked-test runs are complete.

No further:
- model training
- threshold tuning
- seed replacement
- model exclusion
- locked-test optimization

is permitted within the frozen primary study.

Permitted next activities:
- manuscript writing
- table formatting
- figure inspection
- figure layout refinement
- reporting verification
- supplementary material preparation
- reproducibility packaging

Primary metric:
COCO mAP@[0.50:0.95]

Descriptive top model:
YOLO26s under both random and batch-disjoint protocols.

This freeze does not imply statistical superiority
of YOLO26s for mAP50-95.
""".strip()


FREEZE_NOTICE_PATH.write_text(
    freeze_notice
    +
    "\n",
    encoding="utf-8"
)


# ============================================================
# 27. DISPLAY TABLE 1
# ============================================================

print(
    "\n"
    + "=" * 92
)

print(
    "TABLE 1 — DATASET / SPLIT COMPOSITION"
)

print(
    "=" * 92
)

display(
    table1
)


# ============================================================
# 28. DISPLAY TABLE 2
# ============================================================

print(
    "\n"
    + "=" * 92
)

print(
    "TABLE 2 — LOCKED-TEST MODEL PERFORMANCE"
)

print(
    "=" * 92
)

display(
    table2
)


# ============================================================
# 29. DISPLAY TABLE 3
# ============================================================

print(
    "\n"
    + "=" * 92
)

print(
    "TABLE 3 — OBJECT-SIZE PERFORMANCE"
)

print(
    "=" * 92
)

display(
    table3
)


# ============================================================
# 30. DISPLAY TABLE 4
# ============================================================

print(
    "\n"
    + "=" * 92
)

print(
    "TABLE 4 — ERROR TAXONOMY"
)

print(
    "=" * 92
)

display(
    table4
)


# ============================================================
# 31. DISPLAY TABLE 5
# ============================================================

print(
    "\n"
    + "=" * 92
)

print(
    "TABLE 5 — COMPUTATIONAL EFFICIENCY"
)

print(
    "=" * 92
)

display(
    table5
)


# ============================================================
# 32. DISPLAY FINAL RANKING
# ============================================================

print(
    "\n"
    + "=" * 92
)

print(
    "FINAL DESCRIPTIVE LOCKED-TEST RANKING"
)

print(
    "Ranking metric: mean mAP50-95"
)

print(
    "This is NOT a statistical significance ranking."
)

print(
    "=" * 92
)

display(
    final_ranking
)


# ============================================================
# 33. FINAL FILE CHECK
# ============================================================

all_final_files = [

    TABLE1_PATH,
    TABLE2_PATH,
    TABLE3_PATH,
    TABLE4_PATH,
    TABLE5_PATH,

    FIG1_PATH,
    FIG2_PATH,
    FINAL_QUAL_PRIMARY,
    FINAL_QUAL_CROSS,

    SUPP_SEED_FIG,
    SUPP_GAP_FIG,
    FINAL_QUAL_RANDOM,

    SUMMARY_PATH,
    SOURCE_HASH_PATH,
    OUTPUT_HASH_PATH,
    FINAL_MANIFEST_PATH,
    FREEZE_NOTICE_PATH,
]


for p in all_final_files:

    if not Path(
        p
    ).exists():

        raise RuntimeError(
            f"Final package artifact missing:\n{p}"
        )


    if Path(
        p
    ).stat().st_size == 0:

        raise RuntimeError(
            f"Final package artifact empty:\n{p}"
        )


print(
    "\nFinal artifact check         : PASSED"
)


# ============================================================
# 34. FINAL
# ============================================================

print(
    "\n"
    + "=" * 92
)

print(
    "STAGE 9 PASSED — FINAL RESULTS FROZEN"
)

print(
    "=" * 92
)

print(
    "Training complete           : YES"
)

print(
    "Locked test complete        : YES"
)

print(
    "Bootstrap complete          : YES"
)

print(
    "Error taxonomy complete     : YES"
)

print(
    "Qualitative analysis        : YES"
)

print(
    "Main manuscript tables      : 5"
)

print(
    "Main manuscript figures     : 4"
)

print(
    "Supplementary figures       : 3"
)

print(
    "Final numeric results frozen: YES"
)

print(
    "Further test tuning         : NOT PERMITTED"
)

print(
    "Further retraining          : NOT PERMITTED"
)


print(
    "\nManuscript package:"
)

print(
    MANUSCRIPT_ROOT
)


print(
    "\nFinal results summary:"
)

print(
    SUMMARY_PATH
)


print(
    "\nFinal freeze manifest:"
)

print(
    FINAL_MANIFEST_PATH
)


print(
    "\nNEXT:"
)

print(
    "SCIENTIFIC INTERPRETATION + RESULTS SECTION + "
    "DISCUSSION + FINAL MANUSCRIPT ASSEMBLY"
)

print(
    "=" * 92
)

In [ ]:
# ============================================================
# STAGE 8D — PUBLICATION FIGURE EXPORT & 300-DPI VERIFICATION
#
# PURPOSE
# ------------------------------------------------------------
# Verify and export already-created manuscript figures for
# journal submission.
#
# IMPORTANT
# ------------------------------------------------------------
# - NO inference
# - NO retraining
# - NO threshold tuning
# - NO metric recalculation
# - NO figure-content modification
# - NO artificial upscaling
#
# Existing figures are checked for sufficient native pixel
# resolution and exported as:
#
#   PNG  : 300 DPI metadata
#   TIFF : 300 DPI, lossless LZW compression
#
# Effective resolution is evaluated assuming a maximum
# publication width of 180 mm.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import shutil

import pandas as pd
from PIL import Image


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

MANUSCRIPT_ROOT = (
    ROOT /
    "09_manuscript"
)

MAIN_FIG_DIR = (
    MANUSCRIPT_ROOT /
    "figures"
)

SUPP_DIR = (
    MANUSCRIPT_ROOT /
    "supplementary"
)

MANIFEST_DIR = (
    ROOT /
    "07_manifests"
)

EXPORT_ROOT = (
    MANUSCRIPT_ROOT /
    "publication_figures_300dpi"
)

EXPORT_MAIN = (
    EXPORT_ROOT /
    "main"
)

EXPORT_SUPP = (
    EXPORT_ROOT /
    "supplementary"
)


for p in [
    EXPORT_ROOT,
    EXPORT_MAIN,
    EXPORT_SUPP,
]:

    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 2. PUBLICATION SETTINGS
# ============================================================

TARGET_DPI = 300

# Conservative full-width manuscript figure assumption.
MAX_PRINT_WIDTH_MM = 180.0

MM_PER_INCH = 25.4


minimum_width_px = round(
    MAX_PRINT_WIDTH_MM
    /
    MM_PER_INCH
    *
    TARGET_DPI
)


print("=" * 92)
print("STAGE 8D — 300 DPI PUBLICATION FIGURE VERIFICATION")
print("=" * 92)

print(
    f"\nTarget DPI              : {TARGET_DPI}"
)

print(
    f"Assumed max print width : {MAX_PRINT_WIDTH_MM:.1f} mm"
)

print(
    f"Minimum width required  : {minimum_width_px} px"
)


# ============================================================
# 3. FIGURES TO VERIFY
# ============================================================

figures = [

    {
        "figure_id":
            "Figure 1",

        "source":
            MAIN_FIG_DIR /
            "Fig_1_locked_test_mAP50_95.png",

        "output_group":
            "main",
    },

    {
        "figure_id":
            "Figure 2",

        "source":
            MAIN_FIG_DIR /
            "Fig_2_batch_disjoint_object_size_AP.png",

        "output_group":
            "main",
    },

    {
        "figure_id":
            "Figure 3",

        "source":
            MAIN_FIG_DIR /
            "Fig_3_error_examples_batch_disjoint_yolo26s.png",

        "output_group":
            "main",
    },

    {
        "figure_id":
            "Figure 4",

        "source":
            MAIN_FIG_DIR /
            "Fig_4_cross_model_hard_cases.png",

        "output_group":
            "main",
    },

    {
        "figure_id":
            "Supplementary Figure S1",

        "source":
            SUPP_DIR /
            "Supplementary_Figure_S1_seed_stability.png",

        "output_group":
            "supplementary",
    },

    {
        "figure_id":
            "Supplementary Figure S2",

        "source":
            SUPP_DIR /
            "Supplementary_Figure_S2_validation_to_test_gap.png",

        "output_group":
            "supplementary",
    },

    {
        "figure_id":
            "Supplementary Figure S3",

        "source":
            SUPP_DIR /
            "Supplementary_Figure_S3_random_error_examples.png",

        "output_group":
            "supplementary",
    },
]


# ============================================================
# 4. HELPERS
# ============================================================

def sha256_file(
    path,
    chunk=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(chunk),
            b""
        ):

            h.update(
                block
            )

    return h.hexdigest()


def effective_dpi_for_width(
    pixel_width,
    print_width_mm
):

    print_width_in = (
        float(print_width_mm)
        /
        MM_PER_INCH
    )

    return (
        float(pixel_width)
        /
        print_width_in
    )


# ============================================================
# 5. VERIFY SOURCE FILES
# ============================================================

for item in figures:

    source = Path(
        item[
            "source"
        ]
    )


    if not source.exists():

        raise FileNotFoundError(
            f"Required figure missing:\n{source}"
        )


    if source.stat().st_size <= 0:

        raise RuntimeError(
            f"Empty figure file:\n{source}"
        )


print(
    "\nSource figures           : 7/7 PRESENT"
)


# ============================================================
# 6. INSPECT NATIVE RESOLUTION
# ============================================================

inspection_rows = []


for item in figures:

    source = Path(
        item[
            "source"
        ]
    )


    with Image.open(
        source
    ) as img:

        width_px, height_px = (
            img.size
        )

        source_dpi = img.info.get(
            "dpi",
            None
        )


    effective_dpi = (
        effective_dpi_for_width(
            pixel_width=width_px,
            print_width_mm=MAX_PRINT_WIDTH_MM
        )
    )


    sufficient = (
        effective_dpi
        >= TARGET_DPI
    )


    inspection_rows.append(
        {
            "figure":
                item[
                    "figure_id"
                ],

            "source":
                str(
                    source
                ),

            "width_px":
                int(
                    width_px
                ),

            "height_px":
                int(
                    height_px
                ),

            "source_dpi_metadata":
                (
                    str(
                        source_dpi
                    )
                    if source_dpi
                    is not None
                    else
                    "not available"
                ),

            "effective_dpi_at_180mm":
                float(
                    effective_dpi
                ),

            "minimum_required_dpi":
                TARGET_DPI,

            "native_resolution_pass":
                bool(
                    sufficient
                ),
        }
    )


inspection = pd.DataFrame(
    inspection_rows
)


print(
    "\n"
    + "=" * 92
)

print(
    "NATIVE FIGURE RESOLUTION CHECK"
)

print(
    "=" * 92
)


display(
    inspection
)


# ============================================================
# 7. HARD STOP IF PIXEL RESOLUTION IS INSUFFICIENT
#
# IMPORTANT:
# We deliberately DO NOT upscale low-resolution figures.
# ============================================================

failed = inspection[
    ~inspection[
        "native_resolution_pass"
    ]
]


if len(
    failed
) > 0:

    print(
        "\nInsufficient native-resolution figures:"
    )

    display(
        failed
    )


    raise RuntimeError(
        "At least one figure does not contain enough "
        "native pixels for 300 DPI at 180 mm width.\n"
        "Do NOT upscale it artificially. "
        "Regenerate that figure from its source plotting "
        "code at a larger canvas / DPI."
    )


print(
    "\nNative resolution        : 7/7 PASSED"
)


# ============================================================
# 8. EXPORT PNG + TIFF
# ============================================================

export_rows = []


for item in figures:

    source = Path(
        item[
            "source"
        ]
    )


    if (
        item[
            "output_group"
        ]
        == "main"
    ):

        destination_dir = (
            EXPORT_MAIN
        )

    else:

        destination_dir = (
            EXPORT_SUPP
        )


    stem = source.stem


    png_path = (
        destination_dir /
        f"{stem}_300dpi.png"
    )

    tiff_path = (
        destination_dir /
        f"{stem}_300dpi.tif"
    )


    # --------------------------------------------------------
    # Open source without resizing
    # --------------------------------------------------------

    with Image.open(
        source
    ) as img:

        # Ensure stable RGB/RGBA-compatible export.
        if img.mode not in [
            "RGB",
            "RGBA"
        ]:

            img = img.convert(
                "RGB"
            )


        original_size = (
            img.size
        )


        # ----------------------------------------------------
        # PNG — native pixels preserved
        # ----------------------------------------------------

        img.save(
            png_path,
            format="PNG",
            dpi=(
                TARGET_DPI,
                TARGET_DPI
            )
        )


        # ----------------------------------------------------
        # TIFF — lossless LZW
        # ----------------------------------------------------

        # TIFF RGB is generally the safest submission format.
        tiff_img = (
            img.convert(
                "RGB"
            )
        )


        tiff_img.save(
            tiff_path,
            format="TIFF",
            dpi=(
                TARGET_DPI,
                TARGET_DPI
            ),
            compression="tiff_lzw"
        )


    # --------------------------------------------------------
    # Confirm no pixel resizing occurred
    # --------------------------------------------------------

    with Image.open(
        png_path
    ) as check_png:

        png_size = (
            check_png.size
        )

        png_dpi = (
            check_png.info.get(
                "dpi",
                None
            )
        )


    with Image.open(
        tiff_path
    ) as check_tif:

        tif_size = (
            check_tif.size
        )

        tif_dpi = (
            check_tif.info.get(
                "dpi",
                None
            )
        )


    if png_size != original_size:

        raise RuntimeError(
            f"PNG pixel dimensions changed:\n{png_path}"
        )


    if tif_size != original_size:

        raise RuntimeError(
            f"TIFF pixel dimensions changed:\n{tiff_path}"
        )


    export_rows.append(
        {
            "figure":
                item[
                    "figure_id"
                ],

            "source_name":
                source.name,

            "width_px":
                original_size[
                    0
                ],

            "height_px":
                original_size[
                    1
                ],

            "effective_dpi_at_180mm":
                effective_dpi_for_width(
                    original_size[
                        0
                    ],
                    MAX_PRINT_WIDTH_MM
                ),

            "png_path":
                str(
                    png_path
                ),

            "png_dpi_metadata":
                str(
                    png_dpi
                ),

            "png_sha256":
                sha256_file(
                    png_path
                ),

            "tiff_path":
                str(
                    tiff_path
                ),

            "tiff_dpi_metadata":
                str(
                    tif_dpi
                ),

            "tiff_sha256":
                sha256_file(
                    tiff_path
                ),
        }
    )


export_registry = pd.DataFrame(
    export_rows
)


# ============================================================
# 9. VERIFY EXPORTED DPI METADATA
# ============================================================

def dpi_is_approximately_300(
    dpi_value
):

    if dpi_value is None:

        return False


    try:

        x = float(
            dpi_value[
                0
            ]
        )

        y = float(
            dpi_value[
                1
            ]
        )

    except Exception:

        return False


    return (
        abs(
            x - TARGET_DPI
        ) <= 1.0
        and
        abs(
            y - TARGET_DPI
        ) <= 1.0
    )


dpi_check_rows = []


for item in figures:

    source = Path(
        item[
            "source"
        ]
    )


    destination_dir = (
        EXPORT_MAIN
        if item[
            "output_group"
        ] == "main"
        else
        EXPORT_SUPP
    )


    png_path = (
        destination_dir /
        f"{source.stem}_300dpi.png"
    )

    tif_path = (
        destination_dir /
        f"{source.stem}_300dpi.tif"
    )


    with Image.open(
        png_path
    ) as img:

        png_dpi = (
            img.info.get(
                "dpi",
                None
            )
        )


    with Image.open(
        tif_path
    ) as img:

        tif_dpi = (
            img.info.get(
                "dpi",
                None
            )
        )


    dpi_check_rows.append(
        {
            "figure":
                item[
                    "figure_id"
                ],

            "PNG 300 DPI":
                dpi_is_approximately_300(
                    png_dpi
                ),

            "TIFF 300 DPI":
                dpi_is_approximately_300(
                    tif_dpi
                ),
        }
    )


dpi_check = pd.DataFrame(
    dpi_check_rows
)


print(
    "\n"
    + "=" * 92
)

print(
    "EXPORTED DPI METADATA CHECK"
)

print(
    "=" * 92
)


display(
    dpi_check
)


if not (
    dpi_check[
        [
            "PNG 300 DPI",
            "TIFF 300 DPI"
        ]
    ]
    .all()
    .all()
):

    raise RuntimeError(
        "300-DPI metadata verification failed."
    )


print(
    "\nExport DPI metadata      : 7/7 PASSED"
)


# ============================================================
# 10. SAVE REGISTRY
# ============================================================

REGISTRY_PATH = (
    EXPORT_ROOT /
    "publication_figure_300dpi_registry.csv"
)


export_registry.to_csv(
    REGISTRY_PATH,
    index=False
)


INSPECTION_PATH = (
    EXPORT_ROOT /
    "native_resolution_verification.csv"
)


inspection.to_csv(
    INSPECTION_PATH,
    index=False
)


# ============================================================
# 11. COPY ORIGINAL 300-DPI PNGs
#
# Keep an untouched-source copy for audit.
# ============================================================

ORIGINAL_COPY_DIR = (
    EXPORT_ROOT /
    "original_stage9_png"
)

ORIGINAL_COPY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for item in figures:

    source = Path(
        item[
            "source"
        ]
    )


    shutil.copy2(
        source,
        ORIGINAL_COPY_DIR /
        source.name
    )


# ============================================================
# 12. OUTPUT HASH REGISTRY
# ============================================================

hash_rows = []


for folder in [
    EXPORT_MAIN,
    EXPORT_SUPP,
    ORIGINAL_COPY_DIR,
]:

    for path in sorted(
        folder.iterdir()
    ):

        if not path.is_file():

            continue


        hash_rows.append(
            {
                "file":
                    path.name,

                "path":
                    str(
                        path
                    ),

                "size_bytes":
                    int(
                        path.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


hash_registry = pd.DataFrame(
    hash_rows
)


HASH_PATH = (
    EXPORT_ROOT /
    "publication_figure_hash_registry.csv"
)


hash_registry.to_csv(
    HASH_PATH,
    index=False
)


# ============================================================
# 13. STAGE 8D MANIFEST
# ============================================================

MANIFEST_PATH = (
    MANIFEST_DIR /
    "STAGE_8D_300DPI_FIGURES.json"
)


manifest = {

    "stage":
        "8D",

    "status":
        "PUBLICATION_FIGURES_300DPI_VERIFIED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "target_dpi":
        TARGET_DPI,

    "assumed_max_print_width_mm":
        MAX_PRINT_WIDTH_MM,

    "minimum_required_width_px":
        minimum_width_px,

    "figures_verified":
        len(
            figures
        ),

    "native_resolution_check":
        "PASSED",

    "artificial_upscaling":
        False,

    "pixel_dimensions_changed":
        False,

    "output_formats": [
        "PNG",
        "TIFF-LZW"
    ],

    "new_inference":
        False,

    "metric_recalculation":
        False,

    "threshold_tuning":
        False,

    "retraining":
        False,

    "scientific_content_change":
        False,

    "note":
        (
            "Stage 8D is a publication-formatting and "
            "resolution-verification stage only. "
            "The frozen quantitative results and "
            "qualitative selections are unchanged."
        ),

    "registry":
        str(
            REGISTRY_PATH
        ),

    "registry_sha256":
        sha256_file(
            REGISTRY_PATH
        ),

    "native_resolution_registry":
        str(
            INSPECTION_PATH
        ),

    "native_resolution_registry_sha256":
        sha256_file(
            INSPECTION_PATH
        ),

    "hash_registry":
        str(
            HASH_PATH
        ),

    "hash_registry_sha256":
        sha256_file(
            HASH_PATH
        ),
}


if MANIFEST_PATH.exists():

    previous = json.loads(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )


    old_compare = dict(
        previous
    )

    new_compare = dict(
        manifest
    )


    old_compare.pop(
        "created_utc",
        None
    )

    new_compare.pop(
        "created_utc",
        None
    )


    if old_compare != new_compare:

        raise RuntimeError(
            "A different STAGE_8D_300DPI_FIGURES.json "
            "already exists.\n"
            "It will NOT be silently overwritten."
        )


else:

    MANIFEST_PATH.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )


# ============================================================
# 14. FINAL DISPLAY
# ============================================================

print(
    "\n"
    + "=" * 92
)

print(
    "PUBLICATION FIGURE EXPORT REGISTRY"
)

print(
    "=" * 92
)


display(
    export_registry[
        [
            "figure",
            "source_name",
            "width_px",
            "height_px",
            "effective_dpi_at_180mm",
            "png_dpi_metadata",
            "tiff_dpi_metadata",
        ]
    ]
)


# ============================================================
# 15. FINAL
# ============================================================

print(
    "\n"
    + "=" * 92
)

print(
    "STAGE 8D PASSED — 300 DPI FIGURES VERIFIED"
)

print(
    "=" * 92
)

print(
    f"Figures checked             : {len(figures)}/7"
)

print(
    "Native resolution           : PASSED"
)

print(
    "PNG 300-DPI export          : COMPLETE"
)

print(
    "TIFF 300-DPI export         : COMPLETE"
)

print(
    "Artificial upscaling        : NO"
)

print(
    "Pixel resizing              : NO"
)

print(
    "Scientific content changed  : NO"
)

print(
    "New inference               : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Retraining                  : NO"
)


print(
    "\nPublication figure directory:"
)

print(
    EXPORT_ROOT
)


print(
    "\nPNG/TIFF registry:"
)

print(
    REGISTRY_PATH
)


print(
    "\nHash registry:"
)

print(
    HASH_PATH
)


print(
    "\nManifest:"
)

print(
    MANIFEST_PATH
)


print(
    "\nSTAGE 8D COMPLETE."
)

print(
    "=" * 92
)

In [ ]:
# ============================================================
# STAGE 9 — FINAL REPRODUCIBILITY MANIFEST
#             COMPLETE / CORRECTED VERSION
#
# PURPOSE
# ------------------------------------------------------------
# Build the final reproducibility and audit package for the
# completed TACO_Pipeline_v5 study.
#
# THIS STAGE DOES NOT:
# ------------------------------------------------------------
# - train any model
# - run inference
# - re-open / re-evaluate the locked test
# - tune confidence thresholds
# - regenerate data splits
# - change seeds
# - exclude models
# - recompute scientific performance metrics
# - alter frozen scientific results
#
# IT DOES:
# ------------------------------------------------------------
# - verify frozen experiment manifests
# - verify 24 validation thresholds
# - verify all 24 best.pt checkpoints by size + SHA-256
# - verify all 24 locked-test run artifact sets
# - verify locked-test summary identity
# - verify TACO cleaned-data identity
# - verify final publication figures
# - capture software/runtime information
# - create critical-file SHA-256 registry
# - create checkpoint verification registry
# - create human-readable reproducibility documentation
# - create FINAL_REPRODUCIBILITY_MANIFEST.json
#
# OUTPUTS
# ------------------------------------------------------------
# 09_manuscript/freeze/
#   FINAL_REPRODUCIBILITY_MANIFEST.json
#   reproducibility_file_registry.csv
#   reproducibility_checkpoint_registry.csv
#   reproducibility_locked_test_artifact_registry.csv
#   reproducibility_environment.json
#   reproducibility_environment.txt
#   REPRODUCIBILITY_README.txt
#
# Canonical manifest:
# 07_manifests/FINAL_REPRODUCIBILITY_MANIFEST.json
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import sys
import os
import json
import hashlib
import platform
import subprocess
import importlib.metadata as importlib_metadata

import numpy as np
import pandas as pd


# ============================================================
# 1. PROJECT PATHS
# ============================================================

ROOT = Path(
    "/content/REAL_DRIVE/MyDrive/Atik14082026/TACO_Pipeline_v5"
)


DIRS = {

    "audit":
        ROOT /
        "01_audit",

    "splits":
        ROOT /
        "02_splits",

    "runs":
        ROOT /
        "03_runs",

    "validation":
        ROOT /
        "04_validation",

    "test":
        ROOT /
        "05_locked_test",

    "analysis":
        ROOT /
        "06_analysis",

    "manifests":
        ROOT /
        "07_manifests",

    "metadata":
        ROOT /
        "08_metadata",

    "manuscript":
        ROOT /
        "09_manuscript",
}


for name, path in DIRS.items():

    if not path.exists():

        raise FileNotFoundError(
            f"Required project directory missing:\n"
            f"{name}: {path}"
        )


FREEZE_DIR = (
    DIRS["manuscript"] /
    "freeze"
)


FREEZE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RUN_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


print("=" * 100)
print("STAGE 9 — FINAL REPRODUCIBILITY MANIFEST")
print("=" * 100)


# ============================================================
# 2. FIXED EXPERIMENT DESIGN
# ============================================================

EXPECTED_PROTOCOLS = [
    "random",
    "batch_disjoint",
]


EXPECTED_MODELS = [
    "yolov8s",
    "yolo11s",
    "yolo26s",
    "rtdetr_l",
]


EXPECTED_SEEDS = [
    42,
    123,
    2026,
]


EXPECTED_RUNS = (
    len(EXPECTED_PROTOCOLS)
    *
    len(EXPECTED_MODELS)
    *
    len(EXPECTED_SEEDS)
)


if EXPECTED_RUNS != 24:

    raise RuntimeError(
        "Internal experiment-design definition "
        "does not produce 24 runs."
    )


# ============================================================
# 3. GENERAL HELPERS
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    path = Path(
        path
    )


    if not path.exists():

        raise FileNotFoundError(
            path
        )


    h = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )


            if not chunk:

                break


            h.update(
                chunk
            )


    return h.hexdigest()


def md5_file(
    path,
    chunk_size=1024 * 1024
):

    path = Path(
        path
    )


    if not path.exists():

        raise FileNotFoundError(
            path
        )


    h = hashlib.md5()


    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )


            if not chunk:

                break


            h.update(
                chunk
            )


    return h.hexdigest()


def safe_package_version(
    package_name
):

    try:

        return importlib_metadata.version(
            package_name
        )

    except Exception:

        return None


def safe_command(
    command
):

    try:

        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False
        )


        return {

            "returncode":
                int(
                    result.returncode
                ),

            "stdout":
                result.stdout.strip(),

            "stderr":
                result.stderr.strip(),
        }


    except Exception as e:

        return {

            "returncode":
                None,

            "stdout":
                "",

            "stderr":
                repr(e),
        }


def relative_to_root(
    path
):

    path = Path(
        path
    )


    try:

        return str(
            path.relative_to(
                ROOT
            )
        )

    except Exception:

        return str(
            path
        )


def require_file(
    path,
    description=None
):

    path = Path(
        path
    )


    if not path.exists():

        raise FileNotFoundError(
            (
                f"{description}\n"
                if description
                else ""
            )
            +
            str(
                path
            )
        )


    if not path.is_file():

        raise RuntimeError(
            f"Expected file but found something else:\n"
            f"{path}"
        )


    if path.stat().st_size <= 0:

        raise RuntimeError(
            f"File is empty:\n{path}"
        )


    return path


# ============================================================
# 4. REQUIRED SCIENTIFIC MANIFEST CHAIN
# ============================================================

required_manifests = {

    "training_config":
        DIRS["manifests"] /
        "training_config_frozen.json",

    "splits_frozen":
        DIRS["manifests"] /
        "splits_frozen.json",

    "validation_thresholds":
        DIRS["manifests"] /
        "validation_thresholds_frozen.json",

    "pre_locked_test_decision":
        DIRS["manifests"] /
        "PRE_LOCKED_TEST_DECISION.json",

    "locked_test_opened":
        DIRS["manifests"] /
        "LOCKED_TEST_OPENED.json",

    "locked_test_completed":
        DIRS["manifests"] /
        "LOCKED_TEST_COMPLETED.json",

    "stage_7b":
        DIRS["manifests"] /
        "STAGE_7B_FINAL_TEST_AGGREGATION.json",

    "stage_7c":
        DIRS["manifests"] /
        "STAGE_7C_BOOTSTRAP.json",

    "stage_8a":
        DIRS["manifests"] /
        "STAGE_8A_ERROR_TAXONOMY.json",

    "stage_8b":
        DIRS["manifests"] /
        "STAGE_8B_QUALITATIVE_VISUALIZATION.json",

    "stage_8d":
        DIRS["manifests"] /
        "STAGE_8D_300DPI_FIGURES.json",

    "final_results_freeze":
        DIRS["manifests"] /
        "FINAL_RESULTS_FREEZE.json",
}


for name, path in required_manifests.items():

    require_file(
        path,
        f"Required manifest missing: {name}"
    )


print(
    "\nCore manifests               : "
    f"{len(required_manifests)}/"
    f"{len(required_manifests)} PASSED"
)


# ============================================================
# 5. LOAD FROZEN MANIFESTS
# ============================================================

training_config = json.loads(
    required_manifests[
        "training_config"
    ].read_text(
        encoding="utf-8"
    )
)


splits_frozen = json.loads(
    required_manifests[
        "splits_frozen"
    ].read_text(
        encoding="utf-8"
    )
)


validation_thresholds = json.loads(
    required_manifests[
        "validation_thresholds"
    ].read_text(
        encoding="utf-8"
    )
)


pretest_decision = json.loads(
    required_manifests[
        "pre_locked_test_decision"
    ].read_text(
        encoding="utf-8"
    )
)


locked_opened = json.loads(
    required_manifests[
        "locked_test_opened"
    ].read_text(
        encoding="utf-8"
    )
)


locked_completed = json.loads(
    required_manifests[
        "locked_test_completed"
    ].read_text(
        encoding="utf-8"
    )
)


stage7b_manifest = json.loads(
    required_manifests[
        "stage_7b"
    ].read_text(
        encoding="utf-8"
    )
)


stage7c_manifest = json.loads(
    required_manifests[
        "stage_7c"
    ].read_text(
        encoding="utf-8"
    )
)


stage8a_manifest = json.loads(
    required_manifests[
        "stage_8a"
    ].read_text(
        encoding="utf-8"
    )
)


stage8b_manifest = json.loads(
    required_manifests[
        "stage_8b"
    ].read_text(
        encoding="utf-8"
    )
)


stage8d_manifest = json.loads(
    required_manifests[
        "stage_8d"
    ].read_text(
        encoding="utf-8"
    )
)


final_results_freeze = json.loads(
    required_manifests[
        "final_results_freeze"
    ].read_text(
        encoding="utf-8"
    )
)


# ============================================================
# 6. VERIFY FROZEN VALIDATION THRESHOLDS
# ============================================================

expected_threshold_keys = {

    f"{protocol}/{model}/seed{seed}"

    for protocol in EXPECTED_PROTOCOLS

    for model in EXPECTED_MODELS

    for seed in EXPECTED_SEEDS
}


actual_threshold_keys = set(
    validation_thresholds.keys()
)


if (
    actual_threshold_keys
    !=
    expected_threshold_keys
):

    missing_thresholds = (
        expected_threshold_keys
        -
        actual_threshold_keys
    )


    unexpected_thresholds = (
        actual_threshold_keys
        -
        expected_threshold_keys
    )


    raise RuntimeError(
        "Frozen threshold manifest structure mismatch.\n"
        f"Missing: {sorted(missing_thresholds)}\n"
        f"Unexpected: {sorted(unexpected_thresholds)}"
    )


for key, value in validation_thresholds.items():

    value = float(
        value
    )


    if not (
        0.0
        <=
        value
        <=
        1.0
    ):

        raise RuntimeError(
            f"Invalid frozen threshold:\n"
            f"{key} -> {value}"
        )


print(
    "Frozen validation thresholds : 24/24 PASSED"
)


# ============================================================
# 7. CHECKPOINT REGISTRY
#
# ACTUAL FROZEN SCHEMA:
#
# protocol
# model
# seed
# checkpoint
# size_bytes
# sha256
# ============================================================

CHECKPOINT_REGISTRY_PATH = (
    DIRS["manifests"] /
    "checkpoint_registry_pre_locked_test.csv"
)


require_file(
    CHECKPOINT_REGISTRY_PATH,
    "Checkpoint registry missing."
)


checkpoint_registry = pd.read_csv(
    CHECKPOINT_REGISTRY_PATH
)


required_checkpoint_columns = {
    "protocol",
    "model",
    "seed",
    "checkpoint",
    "size_bytes",
    "sha256",
}


missing_checkpoint_columns = (
    required_checkpoint_columns
    -
    set(
        checkpoint_registry.columns
    )
)


if missing_checkpoint_columns:

    raise RuntimeError(
        "Checkpoint registry schema mismatch.\n"
        f"Missing columns: "
        f"{sorted(missing_checkpoint_columns)}\n"
        f"Available columns: "
        f"{list(checkpoint_registry.columns)}"
    )


if len(
    checkpoint_registry
) != EXPECTED_RUNS:

    raise RuntimeError(
        f"Expected {EXPECTED_RUNS} checkpoint rows; "
        f"found {len(checkpoint_registry)}."
    )


if checkpoint_registry[
    [
        "protocol",
        "model",
        "seed",
    ]
].duplicated().any():

    raise RuntimeError(
        "Duplicate protocol/model/seed combination "
        "found in checkpoint registry."
    )


print(
    "Checkpoint registry schema   : PASSED"
)

print(
    "Checkpoint path column       : checkpoint"
)

print(
    "Checkpoint hash column       : sha256"
)


# ============================================================
# 8. VERIFY CHECKPOINT EXPERIMENT STRUCTURE
# ============================================================

expected_combinations = {

    (
        protocol,
        model,
        seed
    )

    for protocol in EXPECTED_PROTOCOLS

    for model in EXPECTED_MODELS

    for seed in EXPECTED_SEEDS
}


actual_combinations = {

    (
        str(
            row[
                "protocol"
            ]
        ),

        str(
            row[
                "model"
            ]
        ),

        int(
            row[
                "seed"
            ]
        )
    )

    for _, row in checkpoint_registry.iterrows()
}


if (
    actual_combinations
    !=
    expected_combinations
):

    missing = (
        expected_combinations
        -
        actual_combinations
    )


    unexpected = (
        actual_combinations
        -
        expected_combinations
    )


    raise RuntimeError(
        "Checkpoint experiment structure mismatch.\n"
        f"Missing combinations: {sorted(missing)}\n"
        f"Unexpected combinations: {sorted(unexpected)}"
    )


print(
    "Checkpoint combinations      : 24/24 PASSED"
)


# ============================================================
# 9. VERIFY ALL 24 BEST.PT CHECKPOINTS
# ============================================================

checkpoint_verification_rows = []


for _, row in checkpoint_registry.iterrows():

    protocol = str(
        row[
            "protocol"
        ]
    )


    model = str(
        row[
            "model"
        ]
    )


    seed = int(
        row[
            "seed"
        ]
    )


    checkpoint_path = Path(
        str(
            row[
                "checkpoint"
            ]
        )
    )


    expected_size = int(
        row[
            "size_bytes"
        ]
    )


    expected_sha256 = str(
        row[
            "sha256"
        ]
    ).strip().lower()


    # --------------------------------------------------------
    # File existence
    # --------------------------------------------------------

    if not checkpoint_path.exists():

        raise FileNotFoundError(
            "Registered checkpoint missing:\n"
            f"{protocol}/{model}/seed{seed}\n"
            f"{checkpoint_path}"
        )


    if not checkpoint_path.is_file():

        raise RuntimeError(
            "Checkpoint path is not a file:\n"
            f"{checkpoint_path}"
        )


    # --------------------------------------------------------
    # Must be best.pt
    # --------------------------------------------------------

    if (
        checkpoint_path.name
        !=
        "best.pt"
    ):

        raise RuntimeError(
            "Registered checkpoint is not best.pt:\n"
            f"{checkpoint_path}"
        )


    # --------------------------------------------------------
    # Verify byte size
    # --------------------------------------------------------

    actual_size = int(
        checkpoint_path.stat().st_size
    )


    if (
        actual_size
        !=
        expected_size
    ):

        raise RuntimeError(
            "Checkpoint size mismatch:\n"
            f"{protocol}/{model}/seed{seed}\n"
            f"Path     : {checkpoint_path}\n"
            f"Expected : {expected_size}\n"
            f"Actual   : {actual_size}"
        )


    # --------------------------------------------------------
    # Verify SHA-256
    # --------------------------------------------------------

    actual_sha256 = sha256_file(
        checkpoint_path
    ).lower()


    if (
        actual_sha256
        !=
        expected_sha256
    ):

        raise RuntimeError(
            "Checkpoint SHA-256 mismatch:\n"
            f"{protocol}/{model}/seed{seed}\n"
            f"Path     : {checkpoint_path}\n"
            f"Expected : {expected_sha256}\n"
            f"Actual   : {actual_sha256}"
        )


    checkpoint_verification_rows.append(
        {

            "protocol":
                protocol,

            "model":
                model,

            "seed":
                seed,

            "checkpoint":
                str(
                    checkpoint_path
                ),

            "relative_path":
                relative_to_root(
                    checkpoint_path
                ),

            "size_bytes":
                actual_size,

            "expected_sha256":
                expected_sha256,

            "actual_sha256":
                actual_sha256,

            "verified":
                True,
        }
    )


checkpoint_verification = pd.DataFrame(
    checkpoint_verification_rows
)


if len(
    checkpoint_verification
) != 24:

    raise RuntimeError(
        "Checkpoint verification did not produce "
        "24 records."
    )


if not checkpoint_verification[
    "verified"
].all():

    raise RuntimeError(
        "At least one checkpoint verification failed."
    )


print(
    "Checkpoint file sizes        : 24/24 PASSED"
)

print(
    "Best.pt SHA-256 integrity    : 24/24 PASSED"
)


# ============================================================
# 10. DATASET IDENTITY
# ============================================================

CLEANED_ANNOTATION_PATH = (
    DIRS["metadata"] /
    "TACO_OFFICIAL_CLEANED_annotations.json"
)


SELECTED_ANNOTATION_PATH = (
    DIRS["metadata"] /
    "TACO_SELECTED_OFFICIAL_annotations.json"
)


require_file(
    CLEANED_ANNOTATION_PATH
)

require_file(
    SELECTED_ANNOTATION_PATH
)


cleaned_annotation = json.loads(
    CLEANED_ANNOTATION_PATH.read_text(
        encoding="utf-8"
    )
)


selected_annotation = json.loads(
    SELECTED_ANNOTATION_PATH.read_text(
        encoding="utf-8"
    )
)


clean_images = len(
    cleaned_annotation.get(
        "images",
        []
    )
)


clean_annotations = len(
    cleaned_annotation.get(
        "annotations",
        []
    )
)


clean_categories = len(
    cleaned_annotation.get(
        "categories",
        []
    )
)


if clean_images != 1500:

    raise RuntimeError(
        f"Expected 1500 TACO images; "
        f"found {clean_images}."
    )


if clean_annotations != 4784:

    raise RuntimeError(
        f"Expected 4784 TACO annotations; "
        f"found {clean_annotations}."
    )


if clean_categories != 60:

    raise RuntimeError(
        f"Expected 60 TACO categories; "
        f"found {clean_categories}."
    )


dataset_identity = {

    "dataset_name":
        "TACO — Trash Annotations in Context",

    "official_source_doi":
        "10.5281/zenodo.3587843",

    "official_archive":
        "TACO.zip",

    "official_archive_md5":
        "e9149407d883e21a8d224feef8210920",

    "cleaned_images":
        int(
            clean_images
        ),

    "cleaned_annotations":
        int(
            clean_annotations
        ),

    "original_categories":
        int(
            clean_categories
        ),

    "study_task":
        (
            "TACO-1 single-class object detection; "
            "all original waste categories collapsed "
            "to one class named 'litter'."
        ),

    "cleaned_annotation":
        relative_to_root(
            CLEANED_ANNOTATION_PATH
        ),

    "cleaned_annotation_sha256":
        sha256_file(
            CLEANED_ANNOTATION_PATH
        ),

    "selected_official_annotation":
        relative_to_root(
            SELECTED_ANNOTATION_PATH
        ),

    "selected_official_annotation_sha256":
        sha256_file(
            SELECTED_ANNOTATION_PATH
        ),
}


print(
    "Dataset identity             : PASSED"
)

print(
    "  Images                     : 1500"
)

print(
    "  Annotations                : 4784"
)

print(
    "  Original categories        : 60"
)


# ============================================================
# 11. VERIFY FINAL SPLIT COMPOSITION FROM FROZEN MANUSCRIPT
#
# We use the persistent final Table 1 rather than depending
# on ephemeral /content local dataset reconstruction.
# ============================================================

TABLE1_PATH = (
    DIRS["manuscript"] /
    "tables" /
    "Table_1_dataset_split_composition.csv"
)


require_file(
    TABLE1_PATH
)


split_table = pd.read_csv(
    TABLE1_PATH
)


expected_split_rows = [

    (
        "Random",
        "Train",
        1050,
        3284
    ),

    (
        "Random",
        "Val",
        225,
        674
    ),

    (
        "Random",
        "Test",
        225,
        826
    ),

    (
        "Batch-disjoint",
        "Train",
        1061,
        3369
    ),

    (
        "Batch-disjoint",
        "Val",
        239,
        708
    ),

    (
        "Batch-disjoint",
        "Test",
        200,
        707
    ),
]


actual_split_rows = [

    (
        str(
            r[
                "Protocol"
            ]
        ),

        str(
            r[
                "Split"
            ]
        ),

        int(
            r[
                "Images"
            ]
        ),

        int(
            r[
                "Objects"
            ]
        ),
    )

    for _, r in split_table.iterrows()
]


if (
    actual_split_rows
    !=
    expected_split_rows
):

    raise RuntimeError(
        "Final split-composition table differs from "
        "the frozen expected composition.\n"
        f"Observed:\n{actual_split_rows}"
    )


print(
    "Final split composition      : PASSED"
)


# ============================================================
# 12. VERIFY LOCKED-TEST SUMMARY
# ============================================================

LOCKED_TEST_SUMMARY_PATH = (
    DIRS["test"] /
    "locked_test_summary_all_runs.csv"
)


require_file(
    LOCKED_TEST_SUMMARY_PATH
)


locked_test_df = pd.read_csv(
    LOCKED_TEST_SUMMARY_PATH
)


if len(
    locked_test_df
) != 24:

    raise RuntimeError(
        f"Expected 24 locked-test rows; "
        f"found {len(locked_test_df)}."
    )


if locked_test_df[
    [
        "protocol",
        "model",
        "seed",
    ]
].duplicated().any():

    raise RuntimeError(
        "Duplicate protocol/model/seed rows "
        "in locked-test summary."
    )


locked_test_combinations = {

    (
        str(
            r[
                "protocol"
            ]
        ),

        str(
            r[
                "model"
            ]
        ),

        int(
            r[
                "seed"
            ]
        )
    )

    for _, r in locked_test_df.iterrows()
}


if (
    locked_test_combinations
    !=
    expected_combinations
):

    raise RuntimeError(
        "Locked-test summary run combinations differ "
        "from the frozen 24-run design."
    )


expected_locked_summary_hash = (
    locked_completed.get(
        "test_summary_sha256"
    )
)


if expected_locked_summary_hash is None:

    raise RuntimeError(
        "LOCKED_TEST_COMPLETED.json does not contain "
        "test_summary_sha256."
    )


actual_locked_summary_hash = sha256_file(
    LOCKED_TEST_SUMMARY_PATH
)


if (
    actual_locked_summary_hash
    !=
    expected_locked_summary_hash
):

    raise RuntimeError(
        "Locked-test summary changed after completion.\n"
        f"Expected: {expected_locked_summary_hash}\n"
        f"Actual  : {actual_locked_summary_hash}"
    )


print(
    "Locked-test summary          : 24/24 PASSED"
)

print(
    "Locked-test summary SHA-256  : PASSED"
)


# ============================================================
# 13. VERIFY FROZEN THRESHOLDS USED IN ALL 24 TEST RUNS
# ============================================================

threshold_mismatches = []


for _, row in locked_test_df.iterrows():

    protocol = str(
        row[
            "protocol"
        ]
    )


    model = str(
        row[
            "model"
        ]
    )


    seed = int(
        row[
            "seed"
        ]
    )


    key = (
        f"{protocol}/"
        f"{model}/"
        f"seed{seed}"
    )


    expected_threshold = float(
        validation_thresholds[
            key
        ]
    )


    actual_threshold = float(
        row[
            "frozen_threshold"
        ]
    )


    if not np.isclose(
        actual_threshold,
        expected_threshold,
        atol=1e-12,
        rtol=0
    ):

        threshold_mismatches.append(
            {
                "run":
                    key,

                "expected":
                    expected_threshold,

                "actual":
                    actual_threshold,
            }
        )


if threshold_mismatches:

    raise RuntimeError(
        "Locked-test threshold mismatch:\n"
        +
        json.dumps(
            threshold_mismatches,
            indent=2
        )
    )


print(
    "Test frozen-threshold use    : 24/24 PASSED"
)


# ============================================================
# 14. VERIFY LOCKED-TEST PER-RUN ARTIFACT SET
#
# Expected for every one of the 24 completed runs:
#
# - test_predictions_coco.json
# - test_speed.csv
# - test_per_image_metrics.csv
# - test_operating_point_counts.json
# - test_run_summary.json
# ============================================================

EXPECTED_TEST_ARTIFACTS = [

    "test_predictions_coco.json",

    "test_speed.csv",

    "test_per_image_metrics.csv",

    "test_operating_point_counts.json",

    "test_run_summary.json",
]


locked_artifact_rows = []


for protocol in EXPECTED_PROTOCOLS:

    expected_images = (
        225
        if protocol == "random"
        else 200
    )


    for model in EXPECTED_MODELS:

        for seed in EXPECTED_SEEDS:

            run_dir = (
                DIRS["test"] /
                protocol /
                f"{model}_seed{seed}"
            )


            if not run_dir.exists():

                raise FileNotFoundError(
                    f"Locked-test run directory missing:\n"
                    f"{run_dir}"
                )


            for artifact_name in EXPECTED_TEST_ARTIFACTS:

                artifact_path = (
                    run_dir /
                    artifact_name
                )


                require_file(
                    artifact_path
                )


                locked_artifact_rows.append(
                    {

                        "protocol":
                            protocol,

                        "model":
                            model,

                        "seed":
                            seed,

                        "artifact":
                            artifact_name,

                        "relative_path":
                            relative_to_root(
                                artifact_path
                            ),

                        "size_bytes":
                            int(
                                artifact_path.stat().st_size
                            ),

                        "sha256":
                            sha256_file(
                                artifact_path
                            ),
                    }
                )


            # ------------------------------------------------
            # Per-image table row count
            # ------------------------------------------------

            per_image_path = (
                run_dir /
                "test_per_image_metrics.csv"
            )


            per_image_run = pd.read_csv(
                per_image_path
            )


            if (
                len(
                    per_image_run
                )
                !=
                expected_images
            ):

                raise RuntimeError(
                    "Unexpected per-image metrics row count:\n"
                    f"{protocol}/{model}/seed{seed}\n"
                    f"Expected={expected_images}, "
                    f"Actual={len(per_image_run)}"
                )


locked_artifact_registry = pd.DataFrame(
    locked_artifact_rows
)


expected_artifact_count = (
    24
    *
    len(
        EXPECTED_TEST_ARTIFACTS
    )
)


if (
    len(
        locked_artifact_registry
    )
    !=
    expected_artifact_count
):

    raise RuntimeError(
        f"Expected {expected_artifact_count} locked-test "
        f"artifact records; "
        f"found {len(locked_artifact_registry)}."
    )


print(
    "Locked-test run artifacts    : "
    f"{expected_artifact_count}/"
    f"{expected_artifact_count} PASSED"
)


# ============================================================
# 15. VERIFY BOOTSTRAP FINAL OUTPUTS
# ============================================================

BOOT_MODEL_PATH = (
    DIRS["analysis"] /
    "bootstrap_model_global_f1_ci.csv"
)


BOOT_PAIR_PATH = (
    DIRS["analysis"] /
    "bootstrap_pairwise_global_f1_differences.csv"
)


BOOT_DIST_PATH = (
    DIRS["analysis"] /
    "bootstrap_global_f1_distributions.csv.gz"
)


for p in [
    BOOT_MODEL_PATH,
    BOOT_PAIR_PATH,
    BOOT_DIST_PATH,
]:

    require_file(
        p
    )


boot_model = pd.read_csv(
    BOOT_MODEL_PATH
)


boot_pair = pd.read_csv(
    BOOT_PAIR_PATH
)


boot_dist = pd.read_csv(
    BOOT_DIST_PATH,
    compression="gzip"
)


if len(
    boot_model
) != 8:

    raise RuntimeError(
        "Bootstrap model CI table must contain 8 rows."
    )


if len(
    boot_pair
) != 12:

    raise RuntimeError(
        "Bootstrap pairwise table must contain 12 rows."
    )


if len(
    boot_dist
) != 40000:

    raise RuntimeError(
        f"Expected 40000 bootstrap distribution rows; "
        f"found {len(boot_dist)}."
    )


boot_replicates_check = (
    boot_dist
    .groupby(
        [
            "protocol",
            "model",
        ]
    )[
        "bootstrap_id"
    ]
    .nunique()
)


if not (
    boot_replicates_check
    ==
    5000
).all():

    raise RuntimeError(
        "Not every protocol/model combination "
        "contains 5000 bootstrap replicates."
    )


print(
    "Bootstrap model rows         : 8/8 PASSED"
)

print(
    "Bootstrap pairwise rows      : 12/12 PASSED"
)

print(
    "Bootstrap distributions      : 40000/40000 PASSED"
)


# ============================================================
# 16. VERIFY ERROR TAXONOMY OUTPUTS
# ============================================================

ERROR_SEED_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_seed_level.csv"
)


ERROR_AGG_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_mean_sd.csv"
)


ERROR_PER_IMAGE_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_per_image.csv"
)


ERROR_EVENTS_PATH = (
    DIRS["analysis"] /
    "locked_test_error_events.csv"
)


for p in [
    ERROR_SEED_PATH,
    ERROR_AGG_PATH,
    ERROR_PER_IMAGE_PATH,
    ERROR_EVENTS_PATH,
]:

    require_file(
        p
    )


error_seed = pd.read_csv(
    ERROR_SEED_PATH
)


error_agg = pd.read_csv(
    ERROR_AGG_PATH
)


error_per_image = pd.read_csv(
    ERROR_PER_IMAGE_PATH
)


error_events = pd.read_csv(
    ERROR_EVENTS_PATH
)


if len(
    error_seed
) != 24:

    raise RuntimeError(
        "Error taxonomy seed-level table "
        "must contain 24 rows."
    )


if len(
    error_agg
) != 8:

    raise RuntimeError(
        "Error taxonomy aggregate table "
        "must contain 8 rows."
    )


if len(
    error_per_image
) != 5100:

    raise RuntimeError(
        f"Expected 5100 Stage 8A per-image rows; "
        f"found {len(error_per_image)}."
    )


if len(
    error_events
) != 22312:

    raise RuntimeError(
        f"Expected 22312 Stage 8A events; "
        f"found {len(error_events)}."
    )


print(
    "Error taxonomy seed rows     : 24/24 PASSED"
)

print(
    "Error taxonomy aggregate     : 8/8 PASSED"
)

print(
    "Error taxonomy image rows    : 5100 PASSED"
)

print(
    "Detailed error events        : 22312 PASSED"
)


# ============================================================
# 17. VERIFY QUALITATIVE ANALYSIS
# ============================================================

QUAL_SELECTION_PATH = (
    DIRS["analysis"] /
    "qualitative_visualization_selection.csv"
)


QUAL_PRIMARY_DETAILS_PATH = (
    DIRS["analysis"] /
    "qualitative_primary_selection_details.csv"
)


QUAL_REGISTRY_PATH = (
    DIRS["analysis"] /
    "qualitative_visualization_registry.csv"
)


for p in [
    QUAL_SELECTION_PATH,
    QUAL_PRIMARY_DETAILS_PATH,
    QUAL_REGISTRY_PATH,
]:

    require_file(
        p
    )


qual_primary = pd.read_csv(
    QUAL_PRIMARY_DETAILS_PATH
)


if len(
    qual_primary
) != 4:

    raise RuntimeError(
        "Expected four primary qualitative examples."
    )


expected_qual_categories = {
    "small_object_failure",
    "high_miss_rate",
    "localization_error",
    "background_like_fp",
}


if set(
    qual_primary[
        "selection_category"
    ]
) != expected_qual_categories:

    raise RuntimeError(
        "Primary qualitative selection categories "
        "do not match the frozen Stage 8B design."
    )


expected_primary_image_ids = {
    1321,
    1353,
    402,
    1386,
}


actual_primary_image_ids = set(
    qual_primary[
        "image_id"
    ].astype(
        int
    )
)


if (
    actual_primary_image_ids
    !=
    expected_primary_image_ids
):

    raise RuntimeError(
        "Primary qualitative image selection changed.\n"
        f"Expected: {sorted(expected_primary_image_ids)}\n"
        f"Actual: {sorted(actual_primary_image_ids)}"
    )


print(
    "Qualitative primary examples : 4/4 PASSED"
)

print(
    "Qualitative image IDs        : FROZEN / PASSED"
)


# ============================================================
# 18. VERIFY 300-DPI PUBLICATION FIGURES
# ============================================================

PUBLICATION_FIGURE_ROOT = (
    DIRS["manuscript"] /
    "publication_figures_300dpi"
)


FIGURE_REGISTRY_PATH = (
    PUBLICATION_FIGURE_ROOT /
    "publication_figure_300dpi_registry.csv"
)


FIGURE_HASH_PATH = (
    PUBLICATION_FIGURE_ROOT /
    "publication_figure_hash_registry.csv"
)


FIGURE_NATIVE_PATH = (
    PUBLICATION_FIGURE_ROOT /
    "native_resolution_verification.csv"
)


for p in [
    FIGURE_REGISTRY_PATH,
    FIGURE_HASH_PATH,
    FIGURE_NATIVE_PATH,
]:

    require_file(
        p
    )


figure_registry = pd.read_csv(
    FIGURE_REGISTRY_PATH
)


figure_native = pd.read_csv(
    FIGURE_NATIVE_PATH
)


if len(
    figure_registry
) != 7:

    raise RuntimeError(
        f"Expected 7 publication figure rows; "
        f"found {len(figure_registry)}."
    )


if len(
    figure_native
) != 7:

    raise RuntimeError(
        f"Expected 7 native-resolution verification rows; "
        f"found {len(figure_native)}."
    )


if (
    "native_resolution_pass"
    not in
    figure_native.columns
):

    raise RuntimeError(
        "native_resolution_verification.csv "
        "does not contain native_resolution_pass."
    )


native_pass = (
    figure_native[
        "native_resolution_pass"
    ]
    .astype(
        str
    )
    .str.lower()
    .map(
        {
            "true":
                True,

            "false":
                False,
        }
    )
)


if native_pass.isna().any():

    raise RuntimeError(
        "Could not interpret native_resolution_pass column."
    )


if not native_pass.all():

    raise RuntimeError(
        "At least one publication figure failed "
        "native 300-DPI resolution verification."
    )


print(
    "Publication figure registry  : 7/7 PASSED"
)

print(
    "Native 300-DPI resolution    : 7/7 PASSED"
)


# ============================================================
# 19. VERIFY FINAL RESULTS FREEZE
# ============================================================

if (
    final_results_freeze.get(
        "status"
    )
    !=
    "FINAL_RESULTS_FROZEN"
):

    raise RuntimeError(
        "FINAL_RESULTS_FREEZE.json does not report "
        "FINAL_RESULTS_FROZEN."
    )


if (
    final_results_freeze.get(
        "number_of_locked_test_runs"
    )
    !=
    24
):

    raise RuntimeError(
        "Final freeze does not report 24 locked-test runs."
    )


if (
    final_results_freeze.get(
        "test_threshold_tuning"
    )
    is not False
):

    raise RuntimeError(
        "Final freeze threshold-tuning status "
        "is not False."
    )


if (
    final_results_freeze.get(
        "post_test_retraining"
    )
    is not False
):

    raise RuntimeError(
        "Final freeze post-test retraining status "
        "is not False."
    )


if (
    final_results_freeze.get(
        "post_test_model_exclusion"
    )
    is not False
):

    raise RuntimeError(
        "Final freeze post-test model exclusion "
        "status is not False."
    )


print(
    "Final scientific result state: FROZEN / PASSED"
)


# ============================================================
# 20. VERIFY FROZEN TRAINING CONFIGURATION
# ============================================================

expected_training_values = {

    "imgsz":
        640,

    "epochs":
        100,

    "patience":
        20,

    "batch":
        4,

    "workers":
        4,

    "amp":
        True,

    "deterministic":
        True,

    "cache":
        False,

    "pretrained":
        True,

    "optimizer":
        "auto",

    "close_mosaic":
        10,

    "mosaic":
        0.5,

    "mixup":
        0.0,

    "copy_paste":
        0.0,

    "degrees":
        5,

    "translate":
        0.05,

    "scale":
        0.15,

    "fliplr":
        0.5,

    "flipud":
        0,

    "hsv_h":
        0.015,

    "hsv_s":
        0.4,

    "hsv_v":
        0.3,
}


training_config_verification = {}


for key, expected_value in expected_training_values.items():

    if key not in training_config:

        raise RuntimeError(
            "Frozen training configuration missing key:\n"
            f"{key}"
        )


    actual_value = training_config[
        key
    ]


    # --------------------------------------------------------
    # Numeric comparison
    # --------------------------------------------------------

    if (
        isinstance(
            expected_value,
            float
        )
        and
        isinstance(
            actual_value,
            (
                float,
                int
            )
        )
    ):

        matches = np.isclose(
            float(
                actual_value
            ),
            float(
                expected_value
            ),
            atol=1e-12,
            rtol=0
        )


    else:

        matches = (
            actual_value
            ==
            expected_value
        )


    if not matches:

        raise RuntimeError(
            "Frozen training configuration mismatch:\n"
            f"{key}\n"
            f"Expected: {expected_value}\n"
            f"Actual  : {actual_value}"
        )


    training_config_verification[
        key
    ] = {

        "expected":
            expected_value,

        "actual":
            actual_value,

        "verified":
            True,
    }


print(
    "Frozen training config       : PASSED"
)


# ============================================================
# 21. RECORDED TRAINING ENVIRONMENT
#
# This is preserved separately from the current runtime.
# ============================================================

recorded_training_environment = {

    "python":
        "3.12.13",

    "pytorch":
        "2.11.0+cu128",

    "ultralytics":
        "8.4.111",

    "cuda":
        "12.8",

    "gpu":
        (
            "NVIDIA RTX PRO 6000 Blackwell "
            "Server Edition"
        ),

    "reported_gpu_memory":
        "97250 MiB",

    "deterministic_requested":
        True,

    "determinism_limitation":
        (
            "The training log reported a PyTorch warning "
            "that grid_sampler_2d_backward_cuda did not "
            "have a deterministic CUDA implementation. "
            "Therefore exact bitwise reproducibility "
            "must not be claimed."
        ),

    "note":
        (
            "These values describe the environment recorded "
            "during the completed training runs. "
            "They are intentionally kept separate from the "
            "runtime used to generate this final manifest."
        ),
}


# ============================================================
# 22. CAPTURE CURRENT RUNTIME ENVIRONMENT
#
# Current environment is documentation only.
# It is NOT assumed to equal the training environment.
# ============================================================

current_environment = {

    "python_version":
        sys.version,

    "python_executable":
        sys.executable,

    "platform":
        platform.platform(),

    "system":
        platform.system(),

    "release":
        platform.release(),

    "machine":
        platform.machine(),

    "processor":
        platform.processor(),

    "packages": {

        "numpy":
            safe_package_version(
                "numpy"
            ),

        "pandas":
            safe_package_version(
                "pandas"
            ),

        "torch":
            safe_package_version(
                "torch"
            ),

        "torchvision":
            safe_package_version(
                "torchvision"
            ),

        "ultralytics":
            safe_package_version(
                "ultralytics"
            ),

        "pycocotools":
            safe_package_version(
                "pycocotools"
            ),

        "opencv-python":
            safe_package_version(
                "opencv-python"
            ),

        "opencv-python-headless":
            safe_package_version(
                "opencv-python-headless"
            ),

        "Pillow":
            safe_package_version(
                "Pillow"
            ),

        "matplotlib":
            safe_package_version(
                "matplotlib"
            ),
    },
}


# ------------------------------------------------------------
# PyTorch runtime
# ------------------------------------------------------------

try:

    import torch


    pytorch_runtime = {

        "torch_version":
            str(
                torch.__version__
            ),

        "cuda_available":
            bool(
                torch.cuda.is_available()
            ),

        "torch_cuda_version":
            (
                str(
                    torch.version.cuda
                )
                if torch.version.cuda
                is not None
                else None
            ),

        "cudnn_version":
            (
                int(
                    torch.backends.cudnn.version()
                )
                if (
                    torch.backends.cudnn.version()
                    is not None
                )
                else None
            ),

        "deterministic_algorithms_enabled_current_runtime":
            bool(
                torch.are_deterministic_algorithms_enabled()
            ),
    }


    if torch.cuda.is_available():

        pytorch_runtime[
            "gpu_count"
        ] = int(
            torch.cuda.device_count()
        )


        pytorch_runtime[
            "gpus"
        ] = []


        for i in range(
            torch.cuda.device_count()
        ):

            prop = (
                torch.cuda.get_device_properties(
                    i
                )
            )


            pytorch_runtime[
                "gpus"
            ].append(
                {

                    "index":
                        int(
                            i
                        ),

                    "name":
                        str(
                            torch.cuda.get_device_name(
                                i
                            )
                        ),

                    "total_memory_bytes":
                        int(
                            prop.total_memory
                        ),
                }
            )


    current_environment[
        "pytorch_runtime"
    ] = pytorch_runtime


except Exception as e:

    current_environment[
        "pytorch_runtime_error"
    ] = repr(
        e
    )


# ------------------------------------------------------------
# NVIDIA driver information
# ------------------------------------------------------------

current_environment[
    "nvidia_smi"
] = safe_command(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader"
    ]
)


# ============================================================
# 23. SAVE CHECKPOINT VERIFICATION REGISTRY
# ============================================================

CHECKPOINT_VERIFY_PATH = (
    FREEZE_DIR /
    "reproducibility_checkpoint_registry.csv"
)


checkpoint_verification.to_csv(
    CHECKPOINT_VERIFY_PATH,
    index=False,
    lineterminator="\n"
)


# ============================================================
# 24. SAVE LOCKED-TEST ARTIFACT REGISTRY
# ============================================================

LOCKED_ARTIFACT_REGISTRY_PATH = (
    FREEZE_DIR /
    "reproducibility_locked_test_artifact_registry.csv"
)


locked_artifact_registry.to_csv(
    LOCKED_ARTIFACT_REGISTRY_PATH,
    index=False,
    lineterminator="\n"
)


# ============================================================
# 25. SAVE ENVIRONMENT JSON
#
# Intentionally no dynamic timestamp inside this file.
# This makes repeated identical captures hash-stable.
# ============================================================

ENVIRONMENT_JSON_PATH = (
    FREEZE_DIR /
    "reproducibility_environment.json"
)


environment_package = {

    "recorded_training_environment":
        recorded_training_environment,

    "current_manifest_runtime":
        current_environment,

    "frozen_training_configuration":
        training_config,

    "frozen_training_configuration_verification":
        training_config_verification,
}


environment_serialized = json.dumps(
    environment_package,
    indent=2,
    ensure_ascii=False
)


ENVIRONMENT_JSON_PATH.write_text(
    environment_serialized,
    encoding="utf-8"
)


# ============================================================
# 26. HUMAN-READABLE ENVIRONMENT REPORT
# ============================================================

ENVIRONMENT_TXT_PATH = (
    FREEZE_DIR /
    "reproducibility_environment.txt"
)


package_lines = []


for package_name, version in (
    current_environment[
        "packages"
    ].items()
):

    package_lines.append(
        f"{package_name}: {version}"
    )


gpu_lines = []


runtime_torch = (
    current_environment.get(
        "pytorch_runtime",
        {}
    )
)


for gpu in runtime_torch.get(
    "gpus",
    []
):

    memory_gib = (
        float(
            gpu[
                "total_memory_bytes"
            ]
        )
        /
        (
            1024 ** 3
        )
    )


    gpu_lines.append(
        (
            f"GPU {gpu['index']}: "
            f"{gpu['name']} | "
            f"{memory_gib:.2f} GiB"
        )
    )


environment_text = f"""
TACO PIPELINE v5
FINAL REPRODUCIBILITY ENVIRONMENT

============================================================
RECORDED TRAINING ENVIRONMENT
============================================================

Python:
{recorded_training_environment['python']}

PyTorch:
{recorded_training_environment['pytorch']}

Ultralytics:
{recorded_training_environment['ultralytics']}

CUDA:
{recorded_training_environment['cuda']}

Training GPU:
{recorded_training_environment['gpu']}

Reported GPU memory:
{recorded_training_environment['reported_gpu_memory']}


============================================================
DETERMINISM NOTE
============================================================

Training was requested with deterministic=True.

However, the training log reported that
grid_sampler_2d_backward_cuda did not have a deterministic
CUDA implementation.

Therefore exact bitwise reproducibility must not be claimed.
The study instead preserves seeds, data splits, configuration,
checkpoints, evaluation rules, thresholds, and artifact hashes.


============================================================
CURRENT MANIFEST-CAPTURE RUNTIME
============================================================

Python:
{current_environment['python_version']}

Platform:
{current_environment['platform']}

Machine:
{current_environment['machine']}

Packages:
{chr(10).join(package_lines)}

PyTorch CUDA available:
{runtime_torch.get('cuda_available')}

PyTorch CUDA version:
{runtime_torch.get('torch_cuda_version')}

cuDNN:
{runtime_torch.get('cudnn_version')}

Deterministic algorithms enabled in current runtime:
{runtime_torch.get('deterministic_algorithms_enabled_current_runtime')}

Detected GPU(s):
{chr(10).join(gpu_lines) if gpu_lines else 'No CUDA GPU reported.'}


============================================================
IMPORTANT
============================================================

The current manifest-capture runtime is NOT assumed to be
identical to the original training runtime.

The recorded training environment above is the environment
associated with the completed model-training runs.
""".strip()


ENVIRONMENT_TXT_PATH.write_text(
    environment_text
    +
    "\n",
    encoding="utf-8"
)


# ============================================================
# 27. REPRODUCIBILITY README
# ============================================================

README_PATH = (
    FREEZE_DIR /
    "REPRODUCIBILITY_README.txt"
)


readme_text = """
TACO PIPELINE v5
FINAL REPRODUCIBILITY PACKAGE

============================================================
1. DATASET
============================================================

Dataset:
TACO — Trash Annotations in Context

Official source:
Zenodo DOI 10.5281/zenodo.3587843

Official archive:
TACO.zip

Official archive MD5:
e9149407d883e21a8d224feef8210920

Official data used:
1500 images
4784 object annotations
60 original categories

Study task:
All original TACO waste categories were collapsed into one
object-detection class named "litter".


============================================================
2. DATA AUDIT
============================================================

The official annotation source was preserved.

A cleaned annotation file was generated after annotation
geometry auditing.

Only controlled bounding-box repairs were permitted during
the audit stage.

The cleaned annotation identity is preserved using SHA-256.


============================================================
3. EVALUATION PROTOCOLS
============================================================

Protocol 1:
Deterministic random image-level split.

Protocol 2:
Official TACO batch-disjoint split.

The second protocol keeps official TACO batch groups disjoint
between train, validation, and test.

IMPORTANT:
The batch-disjoint protocol must NOT be described as:
- scene-disjoint
- site-disjoint
- geographic-disjoint
- participant-disjoint

because official batch membership does not establish these
stronger independence properties.


============================================================
4. SPLIT COMPOSITION
============================================================

Random:
Train: 1050 images / 3284 objects
Validation: 225 images / 674 objects
Test: 225 images / 826 objects

Batch-disjoint:
Train: 1061 images / 3369 objects
Validation: 239 images / 708 objects
Test: 200 images / 707 objects


============================================================
5. MODELS
============================================================

YOLOv8s
YOLO11s
YOLO26s
RT-DETR-L


============================================================
6. RANDOM SEEDS
============================================================

42
123
2026

Protocols × models × seeds:
2 × 4 × 3 = 24 training runs


============================================================
7. PRIMARY METRIC
============================================================

COCO mAP@[0.50:0.95]


============================================================
8. SECONDARY METRICS
============================================================

AP50
AP75
AP-small
AP-medium
AP-large

Average Recall

Precision
Recall
F1

TP
FP
FN

Error taxonomy

Framework-reported processing time


============================================================
9. CONFIDENCE THRESHOLD POLICY
============================================================

Confidence thresholds were selected using validation data
only.

A separate frozen operating threshold was retained for every:

protocol × model × seed

Total frozen validation thresholds:
24

No confidence threshold was optimized on locked-test data.


============================================================
10. LOCKED TEST
============================================================

The locked test was opened only after:

- training was complete
- model panel was frozen
- seeds were frozen
- validation evaluation was complete
- confidence thresholds were frozen
- checkpoints were registered
- checkpoint SHA-256 hashes were recorded
- the pre-test decision manifest was created

Locked-test evaluations:
24/24 complete

Post-test threshold tuning:
NO

Post-test retraining:
NO

Post-test seed replacement:
NO

Post-test model exclusion:
NO


============================================================
11. MULTI-SEED REPORTING
============================================================

Results are reported using three training seeds:

42
123
2026

Mean and sample standard deviation are reported.

Best-seed-only reporting is not used.


============================================================
12. RANDOM VS BATCH-DISJOINT INTERPRETATION
============================================================

Random and batch-disjoint protocols use different
protocol-specific training, validation, and test compositions.

Therefore differences between their performance values are
treated as descriptive protocol-associated differences.

They are not treated as paired observations and are not
interpreted as causal effects of the split protocol.


============================================================
13. BOOTSTRAP ANALYSIS
============================================================

Bootstrap replicates:
5000

Resampling unit:
image

Bootstrap comparisons:
models within the same protocol only

Same resampled image indices were used for models within a
protocol, producing paired image-level comparisons.

The three observed trained seeds were retained.

For each image-bootstrap replicate:
- TP, FP, and FN were summed separately for each seed
- global F1 was recomputed for each seed
- the three seed-specific F1 values were averaged

Bootstrap uncertainty therefore represents image-sampling
uncertainty conditional on the three observed seeds.

Between-seed variability is reported separately.

Formal p-values:
NO


============================================================
14. ERROR TAXONOMY
============================================================

Operating point:
validation-frozen confidence threshold

TP:
prediction assigned to a previously unmatched GT at
IoU >= 0.50

Duplicate FP:
unmatched prediction with best GT IoU >= 0.50 after the
corresponding GT had already been matched

Localization FP:
0.10 <= best GT IoU < 0.50

Background-like FP:
best GT IoU < 0.10, including predictions in images with
no annotated litter

FN:
GT object remaining unmatched after prediction matching

The Stage 8A taxonomy reproduced the original Stage 7A
TP/FP/FN counts for all 24 locked-test runs.


============================================================
15. OBJECT SIZE
============================================================

COCO object-size definitions were used.

Small:
area < 32^2 px^2

Medium:
32^2 <= area < 96^2 px^2

Large:
area >= 96^2 px^2


============================================================
16. QUALITATIVE ANALYSIS
============================================================

Primary protocol:
batch-disjoint

Primary descriptive model:
YOLO26s

Reference visualization seed:
123

IMPORTANT:
Seed 123 is an illustrative reference seed only.

Image selection was based on three-seed consensus error
statistics.

Primary qualitative categories:
- small-object failure
- high miss rate
- localization error
- background-like false positive

Qualitative examples were not used for:
- retraining
- model selection
- threshold tuning
- quantitative statistical inference


============================================================
17. PUBLICATION FIGURES
============================================================

Final figures verified:
7

Target resolution:
300 DPI

Assumed maximum print width:
180 mm

Native resolution check:
7/7 passed

Artificial upscaling:
NO

Pixel resizing:
NO

Both 300-DPI PNG and lossless TIFF exports were created.


============================================================
18. PRINCIPAL FINAL FINDINGS
============================================================

YOLO26s had the highest mean locked-test
COCO mAP@[0.50:0.95] under both protocols.

Random YOLO26s:
0.2861 ± 0.0072

Batch-disjoint YOLO26s:
0.2853 ± 0.0168

This is a descriptive ranking and does not by itself establish
statistical superiority in mAP50-95.

Small-object performance was substantially lower than
large-object performance.

RT-DETR-L displayed comparatively high seed variability.

Locked-test performance was lower than validation performance
for every model/protocol combination.


============================================================
19. REPORTING RESTRICTIONS
============================================================

Do NOT claim:

- that batch-disjoint evaluation is universally harder
- that the protocol is scene-disjoint
- that the protocol is geographically independent
- that YOLO26s is statistically superior in mAP50-95 merely
  because it has the highest mean
- that framework processing FPS equals end-to-end deployment
  throughput
- that field deployment performance was demonstrated
- exact bitwise deterministic training


============================================================
20. DETERMINISM
============================================================

Training requested deterministic=True.

However, PyTorch emitted a warning indicating that
grid_sampler_2d_backward_cuda lacked a deterministic CUDA
implementation.

Accordingly, exact bitwise reproducibility is not claimed.

Reproducibility is instead supported through preservation of:

- official dataset identity
- cleaned annotation hash
- frozen splits
- seeds
- frozen training configuration
- 24 trained checkpoints
- checkpoint SHA-256 hashes
- frozen validation thresholds
- locked-test prediction outputs
- statistical outputs
- error-analysis outputs
- publication figures
- environment records
- final file registries


============================================================
21. FINAL STATUS
============================================================

Scientific numerical results:
FROZEN

New model training:
NOT PERMITTED within the frozen primary study

Test threshold tuning:
NOT PERMITTED

Seed replacement:
NOT PERMITTED

Post-test model exclusion:
NOT PERMITTED

Permitted subsequent work:
- manuscript writing
- reporting verification
- table formatting
- figure formatting
- supplementary material preparation
- public repository packaging

Any genuinely new experiment must be explicitly labelled as
a new post hoc analysis and must not replace the frozen
primary results.
""".strip()


README_PATH.write_text(
    readme_text
    +
    "\n",
    encoding="utf-8"
)


# ============================================================
# 28. BUILD CRITICAL REPRODUCIBILITY FILE REGISTRY
#
# We hash scientifically relevant artifacts only.
#
# IMPORTANT:
# The new reproducibility files being created in this stage
# are excluded here to avoid recursive self-hashing.
#
# Checkpoints are handled separately by their dedicated
# checkpoint registry.
# ============================================================

EXCLUDED_REPRO_FILES = {

    "FINAL_REPRODUCIBILITY_MANIFEST.json",

    "reproducibility_file_registry.csv",

    "reproducibility_checkpoint_registry.csv",

    "reproducibility_locked_test_artifact_registry.csv",

    "reproducibility_environment.json",

    "reproducibility_environment.txt",

    "REPRODUCIBILITY_README.txt",
}


registry_candidates = []


# ------------------------------------------------------------
# A. Audit artifacts
# ------------------------------------------------------------

for path in sorted(
    DIRS[
        "audit"
    ].rglob(
        "*"
    )
):

    if path.is_file():

        registry_candidates.append(
            (
                "audit",
                path
            )
        )


# ------------------------------------------------------------
# B. Split artifacts
# ------------------------------------------------------------

for path in sorted(
    DIRS[
        "splits"
    ].rglob(
        "*"
    )
):

    if path.is_file():

        registry_candidates.append(
            (
                "splits",
                path
            )
        )


# ------------------------------------------------------------
# C. Validation outputs
# ------------------------------------------------------------

for pattern in [
    "*.csv",
    "*.json",
]:

    for path in sorted(
        DIRS[
            "validation"
        ].rglob(
            pattern
        )
    ):

        if path.is_file():

            registry_candidates.append(
                (
                    "validation",
                    path
                )
            )


# ------------------------------------------------------------
# D. Locked-test outputs
# ------------------------------------------------------------

for pattern in [
    "*.csv",
    "*.json",
]:

    for path in sorted(
        DIRS[
            "test"
        ].rglob(
            pattern
        )
    ):

        if path.is_file():

            registry_candidates.append(
                (
                    "locked_test",
                    path
                )
            )


# ------------------------------------------------------------
# E. Analysis outputs
# ------------------------------------------------------------

analysis_suffixes = {
    ".csv",
    ".json",
    ".png",
    ".txt",
    ".gz",
}


for path in sorted(
    DIRS[
        "analysis"
    ].rglob(
        "*"
    )
):

    if (
        path.is_file()
        and
        path.suffix.lower()
        in analysis_suffixes
    ):

        registry_candidates.append(
            (
                "analysis",
                path
            )
        )


# ------------------------------------------------------------
# F. Scientific manifests
# ------------------------------------------------------------

for path in sorted(
    DIRS[
        "manifests"
    ].glob(
        "*"
    )
):

    if not path.is_file():

        continue


    if (
        path.name
        ==
        "FINAL_REPRODUCIBILITY_MANIFEST.json"
    ):

        continue


    registry_candidates.append(
        (
            "manifest",
            path
        )
    )


# ------------------------------------------------------------
# G. Metadata
# ------------------------------------------------------------

for path in sorted(
    DIRS[
        "metadata"
    ].rglob(
        "*"
    )
):

    if path.is_file():

        registry_candidates.append(
            (
                "metadata",
                path
            )
        )


# ------------------------------------------------------------
# H. Final manuscript package
# ------------------------------------------------------------

manuscript_extensions = {
    ".csv",
    ".json",
    ".txt",
    ".png",
    ".tif",
    ".tiff",
    ".gz",
}


for path in sorted(
    DIRS[
        "manuscript"
    ].rglob(
        "*"
    )
):

    if not path.is_file():

        continue


    if (
        path.name
        in
        EXCLUDED_REPRO_FILES
    ):

        continue


    if (
        path.suffix.lower()
        in
        manuscript_extensions
    ):

        registry_candidates.append(
            (
                "manuscript",
                path
            )
        )


# ============================================================
# 29. REMOVE DUPLICATE PATHS FROM REGISTRY
# ============================================================

unique_registry = {}


for category, path in registry_candidates:

    resolved = str(
        path.resolve()
    )


    if (
        resolved
        not in
        unique_registry
    ):

        unique_registry[
            resolved
        ] = (
            category,
            path
        )


registry_candidates = list(
    unique_registry.values()
)


# ============================================================
# 30. HASH CRITICAL REPRODUCIBILITY FILES
# ============================================================

file_registry_rows = []


print(
    "\nHashing critical reproducibility artifacts..."
)

print(
    f"Files queued                : "
    f"{len(registry_candidates)}"
)


for index, (
    category,
    path
) in enumerate(
    registry_candidates,
    start=1
):

    file_registry_rows.append(
        {

            "category":
                category,

            "relative_path":
                relative_to_root(
                    path
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


    if (
        index % 50 == 0
        or
        index
        ==
        len(
            registry_candidates
        )
    ):

        print(
            f"  hashed {index}/"
            f"{len(registry_candidates)}"
        )


file_registry = pd.DataFrame(
    file_registry_rows
)


file_registry = (
    file_registry
    .sort_values(
        [
            "category",
            "relative_path",
        ]
    )
    .reset_index(
        drop=True
    )
)


FILE_REGISTRY_PATH = (
    FREEZE_DIR /
    "reproducibility_file_registry.csv"
)


file_registry.to_csv(
    FILE_REGISTRY_PATH,
    index=False,
    lineterminator="\n"
)


print(
    "\nCritical file registry       : COMPLETE"
)

print(
    f"Files registered             : "
    f"{len(file_registry)}"
)


# ============================================================
# 31. BUILD SCIENTIFIC MANIFEST HASH CHAIN
# ============================================================

scientific_manifest_chain = {}


for name, path in required_manifests.items():

    scientific_manifest_chain[
        name
    ] = {

        "relative_path":
            relative_to_root(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }


scientific_manifest_chain[
    "checkpoint_registry_pre_locked_test"
] = {

    "relative_path":
        relative_to_root(
            CHECKPOINT_REGISTRY_PATH
        ),

    "size_bytes":
        int(
            CHECKPOINT_REGISTRY_PATH.stat().st_size
        ),

    "sha256":
        sha256_file(
            CHECKPOINT_REGISTRY_PATH
        ),
}


# ============================================================
# 32. FINAL REPRODUCIBILITY MANIFEST PATHS
# ============================================================

FINAL_MANIFEST_PATH = (
    DIRS["manifests"] /
    "FINAL_REPRODUCIBILITY_MANIFEST.json"
)


FINAL_MANIFEST_COPY_PATH = (
    FREEZE_DIR /
    "FINAL_REPRODUCIBILITY_MANIFEST.json"
)


# ============================================================
# 33. BUILD FINAL REPRODUCIBILITY MANIFEST
# ============================================================

final_manifest = {

    "manifest_name":
        "FINAL_REPRODUCIBILITY_MANIFEST",

    "manifest_version":
        "1.0",

    "status":
        "FINAL_REPRODUCIBILITY_PACKAGE_COMPLETE",

    "created_utc":
        RUN_UTC,

    # ========================================================
    # PROJECT
    # ========================================================

    "project": {

        "name":
            "TACO_Pipeline_v5",

        "study_title":
            (
                "Beyond Random Splits: Robustness and "
                "Small-Object Performance of Real-Time "
                "Litter Detection Models on the TACO Dataset"
            ),

        "task":
            "single-class litter object detection",

        "project_root":
            str(
                ROOT
            ),
    },

    # ========================================================
    # DATASET
    # ========================================================

    "dataset":
        dataset_identity,

    # ========================================================
    # SPLITS
    # ========================================================

    "split_design": {

        "protocols":
            EXPECTED_PROTOCOLS,

        "random":
            {

                "train_images":
                    1050,

                "train_objects":
                    3284,

                "validation_images":
                    225,

                "validation_objects":
                    674,

                "test_images":
                    225,

                "test_objects":
                    826,
            },

        "batch_disjoint":
            {

                "train_images":
                    1061,

                "train_objects":
                    3369,

                "validation_images":
                    239,

                "validation_objects":
                    708,

                "test_images":
                    200,

                "test_objects":
                    707,
            },

        "batch_disjoint_definition":
            (
                "Official TACO batch groups were assigned "
                "disjointly across train, validation, and test."
            ),

        "batch_disjoint_not_equivalent_to": [
            "scene-disjoint",
            "site-disjoint",
            "geographic-disjoint",
            "participant-disjoint",
        ],

        "cross_protocol_inference":
            False,

        "cross_protocol_interpretation":
            (
                "Performance differences between random and "
                "batch-disjoint protocols are descriptive "
                "because protocol-specific training, validation, "
                "and test compositions differ."
            ),

        "frozen_split_manifest":
            relative_to_root(
                required_manifests[
                    "splits_frozen"
                ]
            ),

        "frozen_split_manifest_sha256":
            sha256_file(
                required_manifests[
                    "splits_frozen"
                ]
            ),
    },

    # ========================================================
    # EXPERIMENTAL DESIGN
    # ========================================================

    "experimental_design": {

        "models":
            EXPECTED_MODELS,

        "seeds":
            EXPECTED_SEEDS,

        "protocol_count":
            2,

        "model_count":
            4,

        "seed_count":
            3,

        "training_runs":
            24,

        "validation_runs":
            24,

        "locked_test_runs":
            24,

        "primary_metric":
            "COCO mAP@[0.50:0.95]",

        "secondary_metrics": [

            "AP50",

            "AP75",

            "AP-small",

            "AP-medium",

            "AP-large",

            "Average Recall",

            "Precision",

            "Recall",

            "F1",

            "TP",

            "FP",

            "FN",
        ],
    },

    # ========================================================
    # TRAINING
    # ========================================================

    "training": {

        "configuration":
            training_config,

        "configuration_sha256":
            sha256_file(
                required_manifests[
                    "training_config"
                ]
            ),

        "recorded_environment":
            recorded_training_environment,

        "pretrained":
            True,

        "deterministic_requested":
            True,

        "exact_bitwise_determinism_claimed":
            False,

        "determinism_note":
            (
                "PyTorch reported that "
                "grid_sampler_2d_backward_cuda lacked a "
                "deterministic CUDA implementation despite "
                "deterministic=True. Exact bitwise "
                "reproducibility is therefore not claimed."
            ),
    },

    # ========================================================
    # CHECKPOINTS
    # ========================================================

    "checkpoint_integrity": {

        "expected_checkpoints":
            24,

        "verified_checkpoints":
            int(
                len(
                    checkpoint_verification
                )
            ),

        "all_verified":
            bool(
                checkpoint_verification[
                    "verified"
                ].all()
            ),

        "verification_method":
            "file size + SHA-256",

        "original_registry":
            relative_to_root(
                CHECKPOINT_REGISTRY_PATH
            ),

        "original_registry_sha256":
            sha256_file(
                CHECKPOINT_REGISTRY_PATH
            ),

        "final_verification_registry":
            relative_to_root(
                CHECKPOINT_VERIFY_PATH
            ),

        "final_verification_registry_sha256":
            sha256_file(
                CHECKPOINT_VERIFY_PATH
            ),
    },

    # ========================================================
    # VALIDATION
    # ========================================================

    "validation": {

        "runs":
            24,

        "frozen_threshold_count":
            24,

        "threshold_selection_source":
            "validation only",

        "threshold_policy":
            (
                "Maximum global F1 on validation at "
                "IoU=0.50."
            ),

        "threshold_manifest":
            relative_to_root(
                required_manifests[
                    "validation_thresholds"
                ]
            ),

        "threshold_manifest_sha256":
            sha256_file(
                required_manifests[
                    "validation_thresholds"
                ]
            ),
    },

    # ========================================================
    # PRE-TEST FREEZE
    # ========================================================

    "pre_locked_test_freeze": {

        "completed":
            True,

        "models_retained":
            4,

        "seeds_retained":
            3,

        "checkpoints_registered":
            24,

        "thresholds_frozen":
            24,

        "post_validation_retraining":
            False,

        "post_validation_model_exclusion":
            False,

        "manifest":
            relative_to_root(
                required_manifests[
                    "pre_locked_test_decision"
                ]
            ),

        "manifest_sha256":
            sha256_file(
                required_manifests[
                    "pre_locked_test_decision"
                ]
            ),
    },

    # ========================================================
    # LOCKED TEST
    # ========================================================

    "locked_test": {

        "opened":
            True,

        "completed":
            True,

        "runs":
            24,

        "frozen_thresholds_used":
            24,

        "test_threshold_tuning":
            False,

        "post_test_retraining":
            False,

        "post_test_model_exclusion":
            False,

        "random_test_images":
            225,

        "random_test_objects":
            826,

        "batch_disjoint_test_images":
            200,

        "batch_disjoint_test_objects":
            707,

        "summary":
            relative_to_root(
                LOCKED_TEST_SUMMARY_PATH
            ),

        "summary_sha256":
            actual_locked_summary_hash,

        "run_artifact_files_verified":
            int(
                len(
                    locked_artifact_registry
                )
            ),

        "run_artifact_registry":
            relative_to_root(
                LOCKED_ARTIFACT_REGISTRY_PATH
            ),

        "run_artifact_registry_sha256":
            sha256_file(
                LOCKED_ARTIFACT_REGISTRY_PATH
            ),
    },

    # ========================================================
    # FINAL LOCKED TEST PERFORMANCE
    # ========================================================

    "principal_descriptive_results": {

        "random_top_model":
            "yolo26s",

        "random_yolo26s_mAP50_95_mean":
            0.286098,

        "random_yolo26s_mAP50_95_sd":
            0.007227,

        "batch_disjoint_top_model":
            "yolo26s",

        "batch_disjoint_yolo26s_mAP50_95_mean":
            0.285315,

        "batch_disjoint_yolo26s_mAP50_95_sd":
            0.016788,

        "ranking_interpretation":
            (
                "Descriptive ranking based on mean locked-test "
                "COCO mAP@[0.50:0.95]. This is not treated as "
                "a formal significance ranking."
            ),

        "small_object_limitation":
            True,

        "seed_variability_note":
            (
                "RT-DETR-L exhibited comparatively high "
                "between-seed variability."
            ),
    },

    # ========================================================
    # BOOTSTRAP
    # ========================================================

    "bootstrap_analysis": {

        "completed":
            True,

        "replicates":
            5000,

        "confidence_interval":
            "percentile 95%",

        "resampling_unit":
            "image",

        "within_protocol_pairing":
            True,

        "cross_protocol_pairing":
            False,

        "observed_seeds":
            EXPECTED_SEEDS,

        "seed_handling":
            (
                "The three observed trained seeds were held "
                "fixed. Within each bootstrap image sample, "
                "global F1 was recomputed separately for each "
                "seed and then averaged."
            ),

        "formal_p_values":
            False,

        "multiple_testing_claim":
            False,

        "model_ci_rows":
            8,

        "pairwise_rows":
            12,

        "distribution_rows":
            40000,

        "manifest":
            relative_to_root(
                required_manifests[
                    "stage_7c"
                ]
            ),

        "manifest_sha256":
            sha256_file(
                required_manifests[
                    "stage_7c"
                ]
            ),
    },

    # ========================================================
    # ERROR TAXONOMY
    # ========================================================

    "error_analysis": {

        "completed":
            True,

        "runs":
            24,

        "stage7a_count_reproduction":
            "24/24",

        "tp_iou_threshold":
            0.50,

        "localization_iou_lower_bound":
            0.10,

        "categories": {

            "TP":
                (
                    "Prediction matched to a previously "
                    "unmatched GT at IoU >= 0.50."
                ),

            "duplicate_FP":
                (
                    "Unmatched prediction with best GT "
                    "IoU >= 0.50 after that GT had already "
                    "been matched."
                ),

            "localization_FP":
                (
                    "Unmatched prediction with "
                    "0.10 <= best GT IoU < 0.50."
                ),

            "background_like_FP":
                (
                    "Unmatched prediction with best GT IoU "
                    "< 0.10, including predictions in images "
                    "without annotated litter."
                ),

            "FN":
                (
                    "GT object remaining unmatched after "
                    "prediction matching."
                ),
        },

        "seed_level_rows":
            24,

        "aggregate_rows":
            8,

        "per_image_rows":
            5100,

        "event_rows":
            22312,

        "manifest":
            relative_to_root(
                required_manifests[
                    "stage_8a"
                ]
            ),

        "manifest_sha256":
            sha256_file(
                required_manifests[
                    "stage_8a"
                ]
            ),
    },

    # ========================================================
    # OBJECT-SIZE DEFINITIONS
    # ========================================================

    "object_size_analysis": {

        "definition":
            "COCO area convention",

        "small":
            "area < 32^2 px^2",

        "medium":
            "32^2 <= area < 96^2 px^2",

        "large":
            "area >= 96^2 px^2",

        "small_object_performance_limitation":
            True,
    },

    # ========================================================
    # QUALITATIVE ANALYSIS
    # ========================================================

    "qualitative_analysis": {

        "completed":
            True,

        "primary_protocol":
            "batch_disjoint",

        "primary_model":
            "yolo26s",

        "reference_visualization_seed":
            123,

        "reference_seed_role":
            (
                "Illustrative visualization seed only. "
                "Image selection itself used three-seed "
                "consensus error statistics."
            ),

        "selection_basis":
            "three-seed consensus",

        "selection_categories": [
            "small_object_failure",
            "high_miss_rate",
            "localization_error",
            "background_like_fp",
        ],

        "primary_image_ids": [
            1321,
            1353,
            402,
            1386,
        ],

        "used_for_quantitative_selection":
            False,

        "new_inference":
            False,

        "manifest":
            relative_to_root(
                required_manifests[
                    "stage_8b"
                ]
            ),

        "manifest_sha256":
            sha256_file(
                required_manifests[
                    "stage_8b"
                ]
            ),
    },

    # ========================================================
    # PUBLICATION FIGURES
    # ========================================================

    "publication_figures": {

        "completed":
            True,

        "figure_count":
            7,

        "target_dpi":
            300,

        "assumed_maximum_print_width_mm":
            180.0,

        "native_resolution_verified":
            "7/7",

        "png_export":
            True,

        "tiff_lzw_export":
            True,

        "artificial_upscaling":
            False,

        "pixel_resizing":
            False,

        "scientific_content_change":
            False,

        "registry":
            relative_to_root(
                FIGURE_REGISTRY_PATH
            ),

        "registry_sha256":
            sha256_file(
                FIGURE_REGISTRY_PATH
            ),

        "manifest":
            relative_to_root(
                required_manifests[
                    "stage_8d"
                ]
            ),

        "manifest_sha256":
            sha256_file(
                required_manifests[
                    "stage_8d"
                ]
            ),
    },

    # ========================================================
    # FINAL RESULTS FREEZE
    # ========================================================

    "final_results_freeze": {

        "status":
            "FINAL_RESULTS_FROZEN",

        "results_frozen":
            True,

        "new_training_permitted":
            False,

        "test_retuning_permitted":
            False,

        "seed_replacement_permitted":
            False,

        "post_test_model_exclusion_permitted":
            False,

        "permitted_future_work": [

            "manuscript writing",

            "reporting verification",

            "table formatting",

            "figure formatting",

            "supplementary material preparation",

            "repository packaging",
        ],

        "new_post_hoc_experiment_policy":
            (
                "Any genuinely new experiment must be "
                "explicitly identified as post hoc and "
                "must not replace the frozen primary results."
            ),

        "manifest":
            relative_to_root(
                required_manifests[
                    "final_results_freeze"
                ]
            ),

        "manifest_sha256":
            sha256_file(
                required_manifests[
                    "final_results_freeze"
                ]
            ),
    },

    # ========================================================
    # REPRODUCIBILITY FILES
    # ========================================================

    "reproducibility_registries": {

        "critical_file_registry":
            relative_to_root(
                FILE_REGISTRY_PATH
            ),

        "critical_file_registry_sha256":
            sha256_file(
                FILE_REGISTRY_PATH
            ),

        "critical_file_count":
            int(
                len(
                    file_registry
                )
            ),

        "checkpoint_verification_registry":
            relative_to_root(
                CHECKPOINT_VERIFY_PATH
            ),

        "checkpoint_verification_registry_sha256":
            sha256_file(
                CHECKPOINT_VERIFY_PATH
            ),

        "locked_test_artifact_registry":
            relative_to_root(
                LOCKED_ARTIFACT_REGISTRY_PATH
            ),

        "locked_test_artifact_registry_sha256":
            sha256_file(
                LOCKED_ARTIFACT_REGISTRY_PATH
            ),

        "environment_json":
            relative_to_root(
                ENVIRONMENT_JSON_PATH
            ),

        "environment_json_sha256":
            sha256_file(
                ENVIRONMENT_JSON_PATH
            ),

        "environment_text":
            relative_to_root(
                ENVIRONMENT_TXT_PATH
            ),

        "environment_text_sha256":
            sha256_file(
                ENVIRONMENT_TXT_PATH
            ),

        "readme":
            relative_to_root(
                README_PATH
            ),

        "readme_sha256":
            sha256_file(
                README_PATH
            ),
    },

    # ========================================================
    # MANIFEST CHAIN
    # ========================================================

    "scientific_manifest_chain":
        scientific_manifest_chain,

    # ========================================================
    # REPORTING SAFEGUARDS
    # ========================================================

    "reporting_constraints": [

        (
            "Do not describe the batch-disjoint protocol "
            "as scene-disjoint."
        ),

        (
            "Do not describe the batch-disjoint protocol "
            "as geographic/site-disjoint."
        ),

        (
            "Do not treat random versus batch-disjoint "
            "results as paired observations."
        ),

        (
            "Do not infer a causal effect of split protocol "
            "from the descriptive protocol gap."
        ),

        (
            "Do not claim statistical superiority for "
            "mAP50-95 from descriptive model ranking alone."
        ),

        (
            "Do not replace three-seed reporting with "
            "best-seed-only reporting."
        ),

        (
            "Do not hide the high seed variability observed "
            "for RT-DETR-L."
        ),

        (
            "Do not omit the severe small-object detection "
            "limitation."
        ),

        (
            "Do not describe framework-reported processing "
            "FPS as full end-to-end application throughput."
        ),

        (
            "Do not claim real-world field deployment "
            "validation."
        ),

        (
            "Do not claim exact bitwise deterministic "
            "training."
        ),
    ],
}


# ============================================================
# 34. WRITE CANONICAL FINAL MANIFEST SAFELY
#
# Idempotency:
# - created_utc is ignored when checking an existing manifest.
# - scientific content must otherwise match.
# ============================================================

if FINAL_MANIFEST_PATH.exists():

    existing_manifest = json.loads(
        FINAL_MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )


    existing_compare = dict(
        existing_manifest
    )


    new_compare = dict(
        final_manifest
    )


    existing_compare.pop(
        "created_utc",
        None
    )


    new_compare.pop(
        "created_utc",
        None
    )


    if (
        existing_compare
        !=
        new_compare
    ):

        raise RuntimeError(
            "A different FINAL_REPRODUCIBILITY_MANIFEST.json "
            "already exists.\n\n"
            "It will NOT be silently overwritten.\n"
            "Inspect the existing manifest before making "
            "any change."
        )


    print(
        "\nExisting canonical reproducibility manifest "
        "matches: PASSED"
    )


else:

    FINAL_MANIFEST_PATH.write_text(
        json.dumps(
            final_manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )


# ============================================================
# 35. COPY CANONICAL MANIFEST TO MANUSCRIPT/FREEZE
# ============================================================

canonical_manifest_text = (
    FINAL_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


FINAL_MANIFEST_COPY_PATH.write_text(
    canonical_manifest_text,
    encoding="utf-8"
)


canonical_manifest_hash = sha256_file(
    FINAL_MANIFEST_PATH
)


copy_manifest_hash = sha256_file(
    FINAL_MANIFEST_COPY_PATH
)


if (
    canonical_manifest_hash
    !=
    copy_manifest_hash
):

    raise RuntimeError(
        "Canonical reproducibility manifest and "
        "freeze-copy SHA-256 hashes differ."
    )


print(
    "Canonical/freeze-copy manifest: SHA-256 MATCH"
)


# ============================================================
# 36. FINAL CONSISTENCY CHECKS
# ============================================================

final_checks = {

    "core_manifests_12":
        (
            len(
                required_manifests
            )
            ==
            12
        ),

    "dataset_images_1500":
        (
            clean_images
            ==
            1500
        ),

    "dataset_annotations_4784":
        (
            clean_annotations
            ==
            4784
        ),

    "dataset_categories_60":
        (
            clean_categories
            ==
            60
        ),

    "frozen_thresholds_24":
        (
            len(
                validation_thresholds
            )
            ==
            24
        ),

    "checkpoint_registry_24":
        (
            len(
                checkpoint_registry
            )
            ==
            24
        ),

    "checkpoint_verification_24":
        (
            len(
                checkpoint_verification
            )
            ==
            24
        ),

    "all_checkpoint_hashes_verified":
        bool(
            checkpoint_verification[
                "verified"
            ].all()
        ),

    "locked_test_runs_24":
        (
            len(
                locked_test_df
            )
            ==
            24
        ),

    "locked_test_artifact_files_120":
        (
            len(
                locked_artifact_registry
            )
            ==
            120
        ),

    "bootstrap_model_rows_8":
        (
            len(
                boot_model
            )
            ==
            8
        ),

    "bootstrap_pairwise_rows_12":
        (
            len(
                boot_pair
            )
            ==
            12
        ),

    "bootstrap_distribution_rows_40000":
        (
            len(
                boot_dist
            )
            ==
            40000
        ),

    "error_seed_rows_24":
        (
            len(
                error_seed
            )
            ==
            24
        ),

    "error_per_image_rows_5100":
        (
            len(
                error_per_image
            )
            ==
            5100
        ),

    "error_events_22312":
        (
            len(
                error_events
            )
            ==
            22312
        ),

    "qualitative_examples_4":
        (
            len(
                qual_primary
            )
            ==
            4
        ),

    "publication_figures_7":
        (
            len(
                figure_registry
            )
            ==
            7
        ),

    "publication_native_resolution_all_pass":
        bool(
            native_pass.all()
        ),

    "final_results_frozen":
        (
            final_results_freeze.get(
                "status"
            )
            ==
            "FINAL_RESULTS_FROZEN"
        ),

    "test_threshold_tuning_false":
        (
            final_results_freeze.get(
                "test_threshold_tuning"
            )
            is False
        ),

    "post_test_retraining_false":
        (
            final_results_freeze.get(
                "post_test_retraining"
            )
            is False
        ),

    "post_test_model_exclusion_false":
        (
            final_results_freeze.get(
                "post_test_model_exclusion"
            )
            is False
        ),

    "canonical_manifest_copy_hash_match":
        (
            canonical_manifest_hash
            ==
            copy_manifest_hash
        ),
}


failed_checks = [

    name

    for name, passed in final_checks.items()

    if not passed
]


if failed_checks:

    raise RuntimeError(
        "FINAL REPRODUCIBILITY CHECK FAILED:\n"
        +
        "\n".join(
            failed_checks
        )
    )


final_audit = pd.DataFrame(
    [

        {

            "Check":
                name,

            "Status":
                (
                    "PASS"
                    if passed
                    else
                    "FAIL"
                ),
        }

        for name, passed in final_checks.items()
    ]
)


# ============================================================
# 37. DISPLAY FINAL AUDIT
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL REPRODUCIBILITY CHECKS"
)

print(
    "=" * 100
)


display(
    final_audit
)


# ============================================================
# 38. DISPLAY CHECKPOINT VERIFICATION
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "CHECKPOINT VERIFICATION SUMMARY"
)

print(
    "=" * 100
)


checkpoint_display = (

    checkpoint_verification[
        [
            "protocol",
            "model",
            "seed",
            "size_bytes",
            "verified",
        ]
    ]

    .copy()
)


display(
    checkpoint_display
)


# ============================================================
# 39. DISPLAY FINAL EXPERIMENT SUMMARY
# ============================================================

summary_display = pd.DataFrame(
    [

        {
            "Item":
                "TACO images",

            "Value":
                1500,
        },

        {
            "Item":
                "TACO annotations",

            "Value":
                4784,
        },

        {
            "Item":
                "Original categories",

            "Value":
                60,
        },

        {
            "Item":
                "Protocols",

            "Value":
                2,
        },

        {
            "Item":
                "Models",

            "Value":
                4,
        },

        {
            "Item":
                "Seeds",

            "Value":
                3,
        },

        {
            "Item":
                "Trained checkpoints",

            "Value":
                24,
        },

        {
            "Item":
                "Frozen validation thresholds",

            "Value":
                24,
        },

        {
            "Item":
                "Locked-test runs",

            "Value":
                24,
        },

        {
            "Item":
                "Locked-test per-run files verified",

            "Value":
                120,
        },

        {
            "Item":
                "Bootstrap replicates / model",

            "Value":
                5000,
        },

        {
            "Item":
                "Bootstrap distribution rows",

            "Value":
                40000,
        },

        {
            "Item":
                "Error events",

            "Value":
                22312,
        },

        {
            "Item":
                "Primary qualitative examples",

            "Value":
                4,
        },

        {
            "Item":
                "Publication figures",

            "Value":
                7,
        },

        {
            "Item":
                "Critical files hashed",

            "Value":
                len(
                    file_registry
                ),
        },
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "FINAL EXPERIMENT / REPRODUCIBILITY SUMMARY"
)

print(
    "=" * 100
)


display(
    summary_display
)


# ============================================================
# 40. FINAL STATUS
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "STAGE 9 PASSED — FINAL REPRODUCIBILITY MANIFEST COMPLETE"
)

print(
    "=" * 100
)


print(
    "Core scientific manifests   : 12/12 VERIFIED"
)

print(
    "Dataset identity            : VERIFIED"
)

print(
    "Split composition           : VERIFIED"
)

print(
    "Frozen thresholds           : 24/24 VERIFIED"
)

print(
    "Best.pt checkpoints         : 24/24 VERIFIED"
)

print(
    "Checkpoint SHA-256          : 24/24 VERIFIED"
)

print(
    "Locked-test evaluations     : 24/24 VERIFIED"
)

print(
    "Locked-test run artifacts   : 120/120 VERIFIED"
)

print(
    "Bootstrap                   : VERIFIED"
)

print(
    "Error taxonomy              : VERIFIED"
)

print(
    "Qualitative selection       : VERIFIED"
)

print(
    "Publication figures         : 7/7 VERIFIED"
)

print(
    "Native 300-DPI resolution   : 7/7 VERIFIED"
)

print(
    "Final numeric results       : FROZEN"
)

print(
    "New inference               : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Post-test retraining        : NO"
)

print(
    "Post-test model exclusion   : NO"
)

print(
    "Exact bitwise determinism   : NOT CLAIMED"
)


print(
    "\nCanonical final manifest:"
)

print(
    FINAL_MANIFEST_PATH
)


print(
    "\nManuscript freeze copy:"
)

print(
    FINAL_MANIFEST_COPY_PATH
)


print(
    "\nCritical file registry:"
)

print(
    FILE_REGISTRY_PATH
)


print(
    "\nCheckpoint verification registry:"
)

print(
    CHECKPOINT_VERIFY_PATH
)


print(
    "\nLocked-test artifact registry:"
)

print(
    LOCKED_ARTIFACT_REGISTRY_PATH
)


print(
    "\nEnvironment JSON:"
)

print(
    ENVIRONMENT_JSON_PATH
)


print(
    "\nEnvironment TXT:"
)

print(
    ENVIRONMENT_TXT_PATH
)


print(
    "\nReproducibility README:"
)

print(
    README_PATH
)


print(
    "\nNEXT:"
)

print(
    "FINAL SCIENTIFIC AUDIT -> MANUSCRIPT RESULTS -> "
    "DISCUSSION -> CONCLUSION -> SUBMISSION PACKAGE"
)

print(
    "=" * 100
)

In [ ]:
# ============================================================
# STAGE 10A-MOUNT CHECK
# SAFE GOOGLE DRIVE REMOUNT + PROJECT DISCOVERY
#
# IMPORTANT:
# - does NOT modify project files
# - does NOT create project directories
# - does NOT train/infer/evaluate anything
# ============================================================

from pathlib import Path
import os

from google.colab import drive


print("=" * 100)
print("STAGE 10A-MOUNT CHECK — SAFE DRIVE REMOUNT")
print("=" * 100)


# ============================================================
# 1. USE A NEW, CLEAN MOUNT POINT
#
# Do NOT reuse /content/REAL_DRIVE because it may currently
# contain a fake local directory created after runtime reset.
# ============================================================

MOUNT = Path(
    "/content/GDRIVE_REAL"
)


drive.mount(
    str(MOUNT),
    force_remount=True
)


# ============================================================
# 2. VERIFY REAL MYDRIVE
# ============================================================

MYDRIVE = (
    MOUNT /
    "MyDrive"
)


if not MYDRIVE.exists():

    raise RuntimeError(
        "Google Drive mount did not expose MyDrive:\n"
        f"{MYDRIVE}"
    )


print(
    "\nGoogle Drive               : MOUNTED"
)

print(
    f"MyDrive                    : {MYDRIVE}"
)


# ============================================================
# 3. EXPECTED PROJECT ROOT
# ============================================================

ROOT = (
    MYDRIVE /
    "Atik14082026" /
    "TACO_Pipeline_v5"
)


if not ROOT.exists():

    print(
        "\nExpected project root not found."
    )

    print(
        "Searching MyDrive for TACO_Pipeline_v5..."
    )


    matches = [

        p

        for p in MYDRIVE.rglob(
            "TACO_Pipeline_v5"
        )

        if p.is_dir()
    ]


    print(
        f"Matches found              : {len(matches)}"
    )


    for p in matches:

        print(
            " -",
            p
        )


    if len(
        matches
    ) == 1:

        ROOT = matches[
            0
        ]

        print(
            "\nUnique project root found:"
        )

        print(
            ROOT
        )


    elif len(
        matches
    ) == 0:

        raise FileNotFoundError(
            "TACO_Pipeline_v5 could not be found "
            "anywhere under the mounted MyDrive."
        )


    else:

        raise RuntimeError(
            "Multiple TACO_Pipeline_v5 directories found.\n"
            "Do not guess which one is canonical."
        )


# ============================================================
# 4. VERIFY EXPECTED PROJECT STRUCTURE
# ============================================================

expected_dirs = [

    "01_audit",
    "02_splits",
    "03_runs",
    "04_validation",
    "05_locked_test",
    "06_analysis",
    "07_manifests",
    "08_metadata",
    "09_manuscript",
]


print(
    "\n"
    + "=" * 100
)

print(
    "PROJECT DIRECTORY CHECK"
)

print(
    "=" * 100
)


all_dirs_ok = True


for dirname in expected_dirs:

    path = (
        ROOT /
        dirname
    )

    exists = path.exists()

    print(
        f"{dirname:20s}: "
        f"{'PRESENT' if exists else 'MISSING'}"
    )


    if not exists:

        all_dirs_ok = False


# ============================================================
# 5. CHECK THE SPECIFIC FILES NEEDED FOR STAGE 10A
# ============================================================

critical_files = {

    "image_audit":
        ROOT /
        "01_audit" /
        "image_audit.csv",

    "near_duplicate_candidates":
        ROOT /
        "01_audit" /
        "near_duplicate_candidates_phash_le4.csv",

    "batch_disjoint_split":
        ROOT /
        "02_splits" /
        "batch_disjoint_split.json",

    "cleaned_annotations":
        ROOT /
        "08_metadata" /
        "TACO_OFFICIAL_CLEANED_annotations.json",

    "selected_official_annotations":
        ROOT /
        "08_metadata" /
        "TACO_SELECTED_OFFICIAL_annotations.json",

    "locked_test_summary":
        ROOT /
        "05_locked_test" /
        "locked_test_summary_all_runs.csv",

    "stage8a_per_image":
        ROOT /
        "06_analysis" /
        "locked_test_error_taxonomy_per_image.csv",

    "validation_thresholds":
        ROOT /
        "07_manifests" /
        "validation_thresholds_frozen.json",

    "locked_test_completed":
        ROOT /
        "07_manifests" /
        "LOCKED_TEST_COMPLETED.json",
}


print(
    "\n"
    + "=" * 100
)

print(
    "STAGE 10A CRITICAL FILE CHECK"
)

print(
    "=" * 100
)


missing_files = []


for name, path in critical_files.items():

    exists = (
        path.exists()
        and
        path.is_file()
        and
        path.stat().st_size > 0
    )


    size_text = (
        f"{path.stat().st_size:,} bytes"
        if exists
        else "-"
    )


    print(
        f"{name:30s}: "
        f"{'PRESENT' if exists else 'MISSING':8s} "
        f"{size_text}"
    )


    if not exists:

        missing_files.append(
            str(
                path
            )
        )


# ============================================================
# 6. CHECK LOCAL /content DATASET
#
# This is ephemeral and may legitimately disappear after
# runtime reset.
# ============================================================

LOCAL_DATASET = Path(
    "/content/taco_v5/dataset"
)


LOCAL_TEST_GT = (
    LOCAL_DATASET /
    "batch_disjoint" /
    "test_coco_taco1.json"
)


print(
    "\n"
    + "=" * 100
)

print(
    "EPHEMERAL LOCAL DATASET CHECK"
)

print(
    "=" * 100
)


print(
    f"/content/taco_v5/dataset : "
    f"{'PRESENT' if LOCAL_DATASET.exists() else 'MISSING'}"
)


print(
    f"batch test COCO GT          : "
    f"{'PRESENT' if LOCAL_TEST_GT.exists() else 'MISSING'}"
)


# ============================================================
# 7. IMPORTANT OLD-PATH DIAGNOSTIC
# ============================================================

OLD_PATH = Path(
    "/content/REAL_DRIVE"
)


print(
    "\n"
    + "=" * 100
)

print(
    "OLD /content/REAL_DRIVE DIAGNOSTIC"
)

print(
    "=" * 100
)


print(
    f"Old path exists             : {OLD_PATH.exists()}"
)

print(
    f"Old path is mount point     : "
    f"{os.path.ismount(str(OLD_PATH))}"
)


if (
    OLD_PATH.exists()
    and
    not os.path.ismount(
        str(
            OLD_PATH
        )
    )
):

    print(
        "\nWARNING:"
    )

    print(
        "/content/REAL_DRIVE currently exists but is NOT "
        "a mounted filesystem."
    )

    print(
        "Treat it as a possible fake local path."
    )

    print(
        "DO NOT write project data there."
    )


# ============================================================
# 8. FINAL
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "MOUNT CHECK SUMMARY"
)

print(
    "=" * 100
)


print(
    "\nCanonical ROOT for this runtime:"
)

print(
    ROOT
)


if len(
    missing_files
) == 0:

    print(
        "\nPersistent Stage 10A inputs : ALL PRESENT"
    )

else:

    print(
        "\nPersistent Stage 10A inputs : "
        f"{len(missing_files)} MISSING"
    )


if LOCAL_DATASET.exists():

    print(
        "Local recovered dataset     : PRESENT"
    )

else:

    print(
        "Local recovered dataset     : MISSING"
    )


print(
    "\nIMPORTANT:"
)

print(
    "Do not run Stage 10A-PREP again yet."
)

print(
    "Send this complete output first."
)

print(
    "=" * 100
)

In [ ]:
# ============================================================
# STAGE 10A — REVIEWER-REQUESTED pHash=0 SENSITIVITY ANALYSIS
# DRIVE-ONLY / NO LOCAL DATASET REQUIRED
#
# PURPOSE
# ------------------------------------------------------------
# 1. Identify the pHash=0 validation-test candidate from the
#    frozen audit and frozen batch-disjoint split.
# 2. Report IDs, batches, SHA-256, pHash, dimensions, objects.
# 3. Recover the original frozen contact sheet created during
#    dataset audit for visual reviewer inspection.
# 4. Reconstruct the ORIGINAL batch-disjoint TACO-1 test COCO
#    GT deterministically from persistent cleaned annotations
#    + frozen test IDs.
# 5. Verify that saved predictions exactly reproduce all 12
#    original batch-disjoint COCO evaluations.
# 6. Remove ONLY the test-side pHash=0 candidate.
# 7. Recompute COCO metrics from the SAVED prediction JSONs.
# 8. Recompute frozen-threshold P/R/F1 using Stage 8A
#    per-image TP/FP/FN records.
#
# IMPORTANT
# ------------------------------------------------------------
# NO inference
# NO training
# NO threshold tuning
# NO split modification
# NO seed replacement
# NO model exclusion
#
# Original frozen primary results remain unchanged.
# This is a reviewer-requested POST-HOC sensitivity analysis.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from contextlib import redirect_stdout
from copy import deepcopy

import io
import json
import hashlib
import shutil

import numpy as np
import pandas as pd

from PIL import Image

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# 1. CANONICAL ROOT — CURRENT RUNTIME
# ============================================================

ROOT = Path(
    "/content/GDRIVE_REAL/MyDrive/Atik14082026/TACO_Pipeline_v5"
)


if not ROOT.exists():

    raise FileNotFoundError(
        f"Canonical project root missing:\n{ROOT}"
    )


DIRS = {

    "audit":
        ROOT / "01_audit",

    "splits":
        ROOT / "02_splits",

    "runs":
        ROOT / "03_runs",

    "validation":
        ROOT / "04_validation",

    "test":
        ROOT / "05_locked_test",

    "analysis":
        ROOT / "06_analysis",

    "manifests":
        ROOT / "07_manifests",

    "metadata":
        ROOT / "08_metadata",

    "manuscript":
        ROOT / "09_manuscript",
}


OUT_DIR = (
    ROOT /
    "10_reviewer_sensitivity"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FIG_DIR = (
    OUT_DIR /
    "figures"
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MANIFEST_PATH = (
    DIRS["manifests"] /
    "STAGE_10A_PHASH0_SENSITIVITY.json"
)


if MANIFEST_PATH.exists():

    raise RuntimeError(
        "STAGE_10A_PHASH0_SENSITIVITY.json already exists.\n"
        "A completed Stage 10A will not be silently overwritten."
    )


print("=" * 100)
print("STAGE 10A — REVIEWER-REQUESTED pHash=0 SENSITIVITY ANALYSIS")
print("DRIVE-ONLY VERSION — NO LOCAL DATASET REQUIRED")
print("=" * 100)


# ============================================================
# 2. FIXED STUDY STRUCTURE
# ============================================================

PROTOCOL = "batch_disjoint"


MODELS = [
    "yolov8s",
    "yolo11s",
    "yolo26s",
    "rtdetr_l",
]


SEEDS = [
    42,
    123,
    2026,
]


DISPLAY_MODEL = {

    "yolov8s":
        "YOLOv8s",

    "yolo11s":
        "YOLO11s",

    "yolo26s":
        "YOLO26s",

    "rtdetr_l":
        "RT-DETR-L",
}


# ============================================================
# 3. HELPERS
# ============================================================

def require_file(path):

    path = Path(path)

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing:\n{path}"
        )

    if not path.is_file():

        raise RuntimeError(
            f"Expected file:\n{path}"
        )

    if path.stat().st_size <= 0:

        raise RuntimeError(
            f"Empty file:\n{path}"
        )

    return path


def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk_size
            )

            if not b:

                break

            h.update(
                b
            )

    return h.hexdigest()


def safe_div(a, b):

    if float(b) == 0:

        return 0.0

    return (
        float(a)
        /
        float(b)
    )


def operating_metrics(
    tp,
    fp,
    fn
):

    precision = safe_div(
        tp,
        tp + fp
    )

    recall = safe_div(
        tp,
        tp + fn
    )

    f1 = safe_div(
        2.0
        *
        precision
        *
        recall,
        precision
        +
        recall
    )

    return {

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1
            ),
    }


def fmt_mean_sd(
    mean,
    sd,
    digits=4
):

    return (
        f"{float(mean):.{digits}f} "
        f"± "
        f"{float(sd):.{digits}f}"
    )


# ============================================================
# 4. PERSISTENT REQUIRED INPUTS
# ============================================================

IMAGE_AUDIT_PATH = (
    DIRS["audit"] /
    "image_audit.csv"
)

NEAR_DUP_PATH = (
    DIRS["audit"] /
    "near_duplicate_candidates_phash_le4.csv"
)

CONTACT_DIR = (
    DIRS["audit"] /
    "near_duplicate_contact_sheets"
)

BATCH_SPLIT_PATH = (
    DIRS["splits"] /
    "batch_disjoint_split.json"
)

CLEANED_ANNOTATION_PATH = (
    DIRS["metadata"] /
    "TACO_OFFICIAL_CLEANED_annotations.json"
)

LOCKED_TEST_SUMMARY_PATH = (
    DIRS["test"] /
    "locked_test_summary_all_runs.csv"
)

ERROR_PER_IMAGE_PATH = (
    DIRS["analysis"] /
    "locked_test_error_taxonomy_per_image.csv"
)

THRESHOLD_PATH = (
    DIRS["manifests"] /
    "validation_thresholds_frozen.json"
)

LOCKED_COMPLETED_PATH = (
    DIRS["manifests"] /
    "LOCKED_TEST_COMPLETED.json"
)


for path in [

    IMAGE_AUDIT_PATH,
    NEAR_DUP_PATH,
    BATCH_SPLIT_PATH,
    CLEANED_ANNOTATION_PATH,
    LOCKED_TEST_SUMMARY_PATH,
    ERROR_PER_IMAGE_PATH,
    THRESHOLD_PATH,
    LOCKED_COMPLETED_PATH,

]:

    require_file(
        path
    )


if not CONTACT_DIR.exists():

    raise FileNotFoundError(
        "Frozen near-duplicate contact-sheet directory "
        "is missing:\n"
        f"{CONTACT_DIR}"
    )


print(
    "\nPersistent frozen inputs    : PASSED"
)


# ============================================================
# 5. LOAD FROZEN INPUTS
# ============================================================

image_audit = pd.read_csv(
    IMAGE_AUDIT_PATH,
    dtype={
        "phash":
            str
    }
)

near_pairs = pd.read_csv(
    NEAR_DUP_PATH
)

batch_split = json.loads(
    BATCH_SPLIT_PATH.read_text(
        encoding="utf-8"
    )
)

cleaned_coco = json.loads(
    CLEANED_ANNOTATION_PATH.read_text(
        encoding="utf-8"
    )
)

locked_summary = pd.read_csv(
    LOCKED_TEST_SUMMARY_PATH
)

error_per_image = pd.read_csv(
    ERROR_PER_IMAGE_PATH
)

frozen_thresholds = json.loads(
    THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)

locked_completed = json.loads(
    LOCKED_COMPLETED_PATH.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# 6. VERIFY FROZEN DATASET / AUDIT STRUCTURE
# ============================================================

if len(
    image_audit
) != 1500:

    raise RuntimeError(
        f"image_audit must contain 1500 rows; "
        f"found {len(image_audit)}."
    )


if len(
    cleaned_coco[
        "images"
    ]
) != 1500:

    raise RuntimeError(
        "Cleaned COCO image count mismatch."
    )


if len(
    cleaned_coco[
        "annotations"
    ]
) != 4784:

    raise RuntimeError(
        "Cleaned COCO annotation count mismatch."
    )


if len(
    near_pairs
) != 3:

    raise RuntimeError(
        f"Expected 3 frozen pHash<=4 candidates; "
        f"found {len(near_pairs)}."
    )


print(
    "Dataset/audit identity      : PASSED"
)

print(
    "pHash candidate pairs       : 3"
)


# ============================================================
# 7. VERIFY ORIGINAL LOCKED-TEST SUMMARY SHA-256
# ============================================================

expected_summary_sha = (
    locked_completed.get(
        "test_summary_sha256"
    )
)


if expected_summary_sha is None:

    raise RuntimeError(
        "LOCKED_TEST_COMPLETED.json missing "
        "test_summary_sha256."
    )


actual_summary_sha = sha256_file(
    LOCKED_TEST_SUMMARY_PATH
)


if (
    actual_summary_sha
    !=
    expected_summary_sha
):

    raise RuntimeError(
        "Frozen locked-test summary SHA-256 mismatch."
    )


print(
    "Locked-test summary SHA-256 : PASSED"
)


# ============================================================
# 8. BUILD FROZEN BATCH-SPLIT LOOKUP
# ============================================================

split_lookup = {}


for split_name in [
    "train",
    "val",
    "test",
]:

    for image_id in batch_split[
        split_name
    ]:

        image_id = int(
            image_id
        )


        if image_id in split_lookup:

            raise RuntimeError(
                f"Image {image_id} appears in "
                "multiple frozen splits."
            )


        split_lookup[
            image_id
        ] = split_name


if len(
    split_lookup
) != 1500:

    raise RuntimeError(
        "Batch-disjoint split does not contain "
        "exactly 1500 unique image IDs."
    )


# ============================================================
# 9. AUTOMATICALLY IDENTIFY THE pHash=0 VAL-TEST PAIR
# ============================================================

cross_zero_rows = []


for idx, row in near_pairs.iterrows():

    iid1 = int(
        row[
            "image_id_1"
        ]
    )

    iid2 = int(
        row[
            "image_id_2"
        ]
    )

    distance = int(
        row[
            "phash_distance"
        ]
    )


    if distance != 0:

        continue


    s1 = split_lookup.get(
        iid1
    )

    s2 = split_lookup.get(
        iid2
    )


    if {
        s1,
        s2
    } == {
        "val",
        "test"
    }:

        cross_zero_rows.append(
            {

                "csv_index":
                    int(
                        idx
                    ),

                "image_id_1":
                    iid1,

                "image_id_2":
                    iid2,

                "split_1":
                    s1,

                "split_2":
                    s2,

                "phash_distance":
                    distance,
            }
        )


if len(
    cross_zero_rows
) != 1:

    raise RuntimeError(
        "Expected exactly one pHash=0 pair spanning "
        "batch-disjoint validation and test.\n"
        f"Found: {cross_zero_rows}"
    )


pair = cross_zero_rows[
    0
]


PAIR_CSV_INDEX = int(
    pair[
        "csv_index"
    ]
)


if (
    pair[
        "split_1"
    ]
    ==
    "val"
):

    VAL_IMAGE_ID = int(
        pair[
            "image_id_1"
        ]
    )

    TEST_IMAGE_ID = int(
        pair[
            "image_id_2"
        ]
    )

else:

    VAL_IMAGE_ID = int(
        pair[
            "image_id_2"
        ]
    )

    TEST_IMAGE_ID = int(
        pair[
            "image_id_1"
        ]
    )


print(
    "\npHash=0 cross-split pair    : IDENTIFIED"
)

print(
    f"Validation image ID         : {VAL_IMAGE_ID}"
)

print(
    f"Locked-test image ID        : {TEST_IMAGE_ID}"
)

print(
    f"Frozen pHash CSV row        : {PAIR_CSV_INDEX}"
)


# ============================================================
# 10. CANDIDATE METADATA
# ============================================================

audit_index = (
    image_audit
    .set_index(
        "image_id"
    )
)


for iid in [
    VAL_IMAGE_ID,
    TEST_IMAGE_ID,
]:

    if iid not in audit_index.index:

        raise RuntimeError(
            f"Candidate missing from image audit: {iid}"
        )


candidate_rows = []


for role, iid in [

    (
        "validation",
        VAL_IMAGE_ID
    ),

    (
        "locked_test",
        TEST_IMAGE_ID
    ),
]:

    r = audit_index.loc[
        iid
    ]


    candidate_rows.append(
        {

            "role":
                role,

            "image_id":
                int(
                    iid
                ),

            "split":
                split_lookup[
                    iid
                ],

            "file_name":
                str(
                    r[
                        "file_name"
                    ]
                ),

            "batch":
                str(
                    r[
                        "batch"
                    ]
                ),

            "width":
                int(
                    r[
                        "width"
                    ]
                ),

            "height":
                int(
                    r[
                        "height"
                    ]
                ),

            "n_objects":
                int(
                    r[
                        "n_objects"
                    ]
                ),

            "small_objects":
                int(
                    r[
                        "small_objects"
                    ]
                ),

            "medium_objects":
                int(
                    r[
                        "medium_objects"
                    ]
                ),

            "large_objects":
                int(
                    r[
                        "large_objects"
                    ]
                ),

            "sha256":
                str(
                    r[
                        "sha256"
                    ]
                ),

            "phash":
                str(
                    r[
                        "phash"
                    ]
                ),
        }
    )


candidate_metadata = pd.DataFrame(
    candidate_rows
)


if (
    candidate_metadata[
        "phash"
    ].nunique()
    !=
    1
):

    raise RuntimeError(
        "The pHash=0 pair does not contain identical "
        "stored pHash strings."
    )


if (
    candidate_metadata[
        "sha256"
    ].nunique()
    !=
    2
):

    raise RuntimeError(
        "Unexpected SHA-256 result. "
        "The candidate pair should not be byte-identical."
    )


CANDIDATE_META_PATH = (
    OUT_DIR /
    "pHash0_candidate_metadata.csv"
)


candidate_metadata.to_csv(
    CANDIDATE_META_PATH,
    index=False
)


print(
    "\nCandidate metadata:"
)

display(
    candidate_metadata
)


# ============================================================
# 11. RECOVER ORIGINAL FROZEN CONTACT SHEET
#
# Stage 3B named sheets:
# pair_<CSV index>_d<distance>.jpg
# ============================================================

CONTACT_SHEET_PATH = (
    CONTACT_DIR /
    f"pair_{PAIR_CSV_INDEX:04d}_d0.jpg"
)


require_file(
    CONTACT_SHEET_PATH
)


CONTACT_COPY_PATH = (
    FIG_DIR /
    "Supplementary_pHash0_validation_test_candidate.png"
)


with Image.open(
    CONTACT_SHEET_PATH
) as im:

    rgb = im.convert(
        "RGB"
    )


    # Preserve native pixels. Only write 300-DPI metadata.
    rgb.save(
        CONTACT_COPY_PATH,
        format="PNG",
        dpi=(
            300,
            300
        )
    )


print(
    "\nFrozen audit contact sheet  : FOUND"
)

print(
    CONTACT_SHEET_PATH
)

print(
    "\nReviewer supplementary copy:"
)

print(
    CONTACT_COPY_PATH
)


display(
    Image.open(
        CONTACT_COPY_PATH
    )
)


# ============================================================
# 12. RECONSTRUCT ORIGINAL TACO-1 BATCH-DISJOINT TEST GT
#
# This follows the original Stage 4B make_coco_subset logic:
# - frozen test image IDs
# - all categories -> category_id 1
# - sequential annotation IDs
# - preserve existing annotation "area"
# - bbox area only if "area" is missing
# ============================================================

TEST_IDS = set(
    map(
        int,
        batch_split[
            "test"
        ]
    )
)


if len(
    TEST_IDS
) != 200:

    raise RuntimeError(
        f"Expected 200 batch-disjoint test image IDs; "
        f"found {len(TEST_IDS)}."
    )


derived_test_annotations = []

new_ann_id = 1


for ann in cleaned_coco[
    "annotations"
]:

    if int(
        ann[
            "image_id"
        ]
    ) not in TEST_IDS:

        continue


    b = dict(
        ann
    )


    b[
        "id"
    ] = int(
        new_ann_id
    )


    b[
        "category_id"
    ] = 1


    b[
        "iscrowd"
    ] = int(
        b.get(
            "iscrowd",
            0
        )
    )


    if "area" not in b:

        b[
            "area"
        ] = (
            float(
                b[
                    "bbox"
                ][2]
            )
            *
            float(
                b[
                    "bbox"
                ][3]
            )
        )


    derived_test_annotations.append(
        b
    )


    new_ann_id += 1


derived_full_gt = {

    "info":
        cleaned_coco.get(
            "info",
            {}
        ),

    "licenses":
        cleaned_coco.get(
            "licenses",
            []
        ),

    "images": [

        im

        for im in cleaned_coco[
            "images"
        ]

        if int(
            im[
                "id"
            ]
        )
        in
        TEST_IDS
    ],

    "annotations":
        derived_test_annotations,

    "categories": [
        {
            "id":
                1,

            "name":
                "litter",

            "supercategory":
                "litter",
        }
    ],
}


if len(
    derived_full_gt[
        "images"
    ]
) != 200:

    raise RuntimeError(
        "Derived original test GT image count mismatch."
    )


if len(
    derived_full_gt[
        "annotations"
    ]
) != 707:

    raise RuntimeError(
        "Derived original test GT object count mismatch.\n"
        f"Expected 707; "
        f"found {len(derived_full_gt['annotations'])}."
    )


DERIVED_FULL_GT_PATH = (
    OUT_DIR /
    "reconstructed_original_batch_disjoint_test_coco_taco1.json"
)


DERIVED_FULL_GT_PATH.write_text(
    json.dumps(
        derived_full_gt,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    "\nReconstructed TACO-1 GT     : PASSED"
)

print(
    "Images                     : 200"
)

print(
    "Objects                    : 707"
)


# ============================================================
# 13. BUILD SENSITIVITY GT — REMOVE TEST CANDIDATE ONLY
# ============================================================

removed_gt_annotations = [

    a

    for a in derived_full_gt[
        "annotations"
    ]

    if int(
        a[
            "image_id"
        ]
    )
    ==
    TEST_IMAGE_ID
]


REMOVED_GT_OBJECTS = len(
    removed_gt_annotations
)


if REMOVED_GT_OBJECTS <= 0:

    raise RuntimeError(
        "The test-side pHash candidate has no GT objects."
    )


sensitivity_gt = deepcopy(
    derived_full_gt
)


sensitivity_gt[
    "images"
] = [

    im

    for im in sensitivity_gt[
        "images"
    ]

    if int(
        im[
            "id"
        ]
    )
    !=
    TEST_IMAGE_ID
]


sensitivity_gt[
    "annotations"
] = [

    a

    for a in sensitivity_gt[
        "annotations"
    ]

    if int(
        a[
            "image_id"
        ]
    )
    !=
    TEST_IMAGE_ID
]


if len(
    sensitivity_gt[
        "images"
    ]
) != 199:

    raise RuntimeError(
        "Sensitivity test should contain 199 images."
    )


if len(
    sensitivity_gt[
        "annotations"
    ]
) != (
    707
    -
    REMOVED_GT_OBJECTS
):

    raise RuntimeError(
        "Sensitivity GT object count mismatch."
    )


SENSITIVITY_GT_PATH = (
    OUT_DIR /
    "batch_disjoint_test_minus_pHash0_candidate_coco.json"
)


SENSITIVITY_GT_PATH.write_text(
    json.dumps(
        sensitivity_gt,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    "\nSensitivity test set:"
)

print(
    f"Original                    : 200 images / 707 objects"
)

print(
    f"After exclusion             : "
    f"199 images / "
    f"{len(sensitivity_gt['annotations'])} objects"
)

print(
    f"Removed test GT objects     : {REMOVED_GT_OBJECTS}"
)


# ============================================================
# 14. COCO EVALUATION HELPERS
# ============================================================

COCO_METRIC_NAMES = [

    "mAP50_95",
    "AP50",
    "AP75",
    "AP_small",
    "AP_medium",
    "AP_large",
    "AR1",
    "AR10",
    "AR100",
    "AR_small",
    "AR_medium",
    "AR_large",
]


def load_coco_quiet(
    path
):

    stream = io.StringIO()


    with redirect_stdout(
        stream
    ):

        coco = COCO(
            str(
                path
            )
        )


    return coco


def evaluate_predictions(
    coco_gt,
    predictions
):

    if len(
        predictions
    ) == 0:

        raise RuntimeError(
            "Empty prediction JSON."
        )


    stream = io.StringIO()


    with redirect_stdout(
        stream
    ):

        coco_dt = coco_gt.loadRes(
            predictions
        )


        evaluator = COCOeval(
            coco_gt,
            coco_dt,
            "bbox"
        )


        # Standard COCO evaluator.
        evaluator.params.maxDets = [
            1,
            10,
            100,
        ]


        evaluator.evaluate()

        evaluator.accumulate()

        evaluator.summarize()


    stats = evaluator.stats


    return {

        metric:
            float(
                stats[
                    idx
                ]
            )

        for idx, metric in enumerate(
            COCO_METRIC_NAMES
        )
    }


COCO_GT_FULL = load_coco_quiet(
    DERIVED_FULL_GT_PATH
)

COCO_GT_SENS = load_coco_quiet(
    SENSITIVITY_GT_PATH
)


# ============================================================
# 15. VERIFY RESULT TABLE STRUCTURES
# ============================================================

batch_summary = locked_summary[
    locked_summary[
        "protocol"
    ]
    ==
    PROTOCOL
].copy()


if len(
    batch_summary
) != 12:

    raise RuntimeError(
        f"Expected 12 batch-disjoint locked-test rows; "
        f"found {len(batch_summary)}."
    )


batch_error = error_per_image[
    error_per_image[
        "protocol"
    ]
    ==
    PROTOCOL
].copy()


if len(
    batch_error
) != (
    200
    *
    12
):

    raise RuntimeError(
        f"Expected 2400 batch-disjoint Stage 8A "
        f"per-image rows; found {len(batch_error)}."
    )


# ============================================================
# 16. RUN 12 SAVED-PREDICTION SENSITIVITY ANALYSES
# ============================================================

rows = []


COCO_TOL = 5e-6


print(
    "\n"
    + "=" * 100
)

print(
    "ORIGINAL REPRODUCTION + CANDIDATE-EXCLUSION SENSITIVITY"
)

print(
    "=" * 100
)


counter = 0


for model in MODELS:

    for seed in SEEDS:

        counter += 1


        run_key = (
            f"{PROTOCOL}/"
            f"{model}/"
            f"seed{seed}"
        )


        frozen_df = batch_summary[
            (
                batch_summary[
                    "model"
                ]
                ==
                model
            )
            &
            (
                batch_summary[
                    "seed"
                ].astype(
                    int
                )
                ==
                seed
            )
        ]


        if len(
            frozen_df
        ) != 1:

            raise RuntimeError(
                f"Frozen summary row not unique: {run_key}"
            )


        frozen_row = frozen_df.iloc[
            0
        ]


        pred_path = (
            DIRS["test"] /
            PROTOCOL /
            f"{model}_seed{seed}" /
            "test_predictions_coco.json"
        )


        require_file(
            pred_path
        )


        expected_pred_sha = str(
            frozen_row[
                "predictions_sha256"
            ]
        )


        actual_pred_sha = sha256_file(
            pred_path
        )


        if (
            actual_pred_sha
            !=
            expected_pred_sha
        ):

            raise RuntimeError(
                "Prediction SHA mismatch:\n"
                f"{run_key}"
            )


        predictions = json.loads(
            pred_path.read_text(
                encoding="utf-8"
            )
        )


        # ----------------------------------------------------
        # A. ORIGINAL COCO REPRODUCTION
        # ----------------------------------------------------

        original_coco = evaluate_predictions(
            COCO_GT_FULL,
            predictions
        )


        for metric in COCO_METRIC_NAMES:

            frozen_value = float(
                frozen_row[
                    metric
                ]
            )


            recalculated = float(
                original_coco[
                    metric
                ]
            )


            if not np.isclose(
                frozen_value,
                recalculated,
                atol=COCO_TOL,
                rtol=0
            ):

                raise RuntimeError(
                    "Frozen COCO metric was not reproduced.\n"
                    f"Run     : {run_key}\n"
                    f"Metric  : {metric}\n"
                    f"Frozen  : {frozen_value}\n"
                    f"Recalc  : {recalculated}"
                )


        # ----------------------------------------------------
        # B. ORIGINAL OPERATING-POINT COUNT REPRODUCTION
        # ----------------------------------------------------

        per_image = batch_error[
            (
                batch_error[
                    "model"
                ]
                ==
                model
            )
            &
            (
                batch_error[
                    "seed"
                ].astype(
                    int
                )
                ==
                seed
            )
        ].copy()


        if len(
            per_image
        ) != 200:

            raise RuntimeError(
                f"Expected 200 per-image rows for {run_key}."
            )


        original_tp = int(
            per_image[
                "tp"
            ].sum()
        )

        original_fp = int(
            per_image[
                "fp"
            ].sum()
        )

        original_fn = int(
            per_image[
                "fn"
            ].sum()
        )


        if (
            original_tp
            !=
            int(
                frozen_row[
                    "TP"
                ]
            )
            or
            original_fp
            !=
            int(
                frozen_row[
                    "FP"
                ]
            )
            or
            original_fn
            !=
            int(
                frozen_row[
                    "FN"
                ]
            )
        ):

            raise RuntimeError(
                "Stage 8A TP/FP/FN reproduction failed:\n"
                f"{run_key}"
            )


        original_operating = operating_metrics(
            original_tp,
            original_fp,
            original_fn
        )


        for key, col in [

            (
                "precision",
                "test_precision"
            ),

            (
                "recall",
                "test_recall"
            ),

            (
                "f1",
                "test_f1"
            ),
        ]:

            if not np.isclose(
                original_operating[
                    key
                ],
                float(
                    frozen_row[
                        col
                    ]
                ),
                atol=1e-9,
                rtol=0
            ):

                raise RuntimeError(
                    f"Operating-point reproduction failed: "
                    f"{run_key}/{key}"
                )


        # ----------------------------------------------------
        # C. REMOVE TEST-SIDE CANDIDATE FROM PREDICTIONS
        # ----------------------------------------------------

        filtered_predictions = [

            p

            for p in predictions

            if int(
                p[
                    "image_id"
                ]
            )
            !=
            TEST_IMAGE_ID
        ]


        removed_prediction_count = (
            len(
                predictions
            )
            -
            len(
                filtered_predictions
            )
        )


        sensitivity_coco = evaluate_predictions(
            COCO_GT_SENS,
            filtered_predictions
        )


        # ----------------------------------------------------
        # D. REMOVE TEST IMAGE FROM FIXED-THRESHOLD COUNTS
        # ----------------------------------------------------

        removed_image_df = per_image[
            per_image[
                "image_id"
            ].astype(
                int
            )
            ==
            TEST_IMAGE_ID
        ]


        if len(
            removed_image_df
        ) != 1:

            raise RuntimeError(
                "Candidate test image not unique in "
                f"Stage 8A table: {run_key}"
            )


        removed_row = (
            removed_image_df.iloc[
                0
            ]
        )


        removed_tp = int(
            removed_row[
                "tp"
            ]
        )

        removed_fp = int(
            removed_row[
                "fp"
            ]
        )

        removed_fn = int(
            removed_row[
                "fn"
            ]
        )


        retained = per_image[
            per_image[
                "image_id"
            ].astype(
                int
            )
            !=
            TEST_IMAGE_ID
        ]


        if len(
            retained
        ) != 199:

            raise RuntimeError(
                f"Expected 199 retained images: {run_key}"
            )


        sensitivity_tp = int(
            retained[
                "tp"
            ].sum()
        )

        sensitivity_fp = int(
            retained[
                "fp"
            ].sum()
        )

        sensitivity_fn = int(
            retained[
                "fn"
            ].sum()
        )


        if (
            sensitivity_tp
            !=
            original_tp
            -
            removed_tp
        ):

            raise RuntimeError(
                f"TP arithmetic mismatch: {run_key}"
            )


        if (
            sensitivity_fp
            !=
            original_fp
            -
            removed_fp
        ):

            raise RuntimeError(
                f"FP arithmetic mismatch: {run_key}"
            )


        if (
            sensitivity_fn
            !=
            original_fn
            -
            removed_fn
        ):

            raise RuntimeError(
                f"FN arithmetic mismatch: {run_key}"
            )


        sensitivity_operating = operating_metrics(
            sensitivity_tp,
            sensitivity_fp,
            sensitivity_fn
        )


        # ----------------------------------------------------
        # E. VERIFY FROZEN THRESHOLD
        # ----------------------------------------------------

        threshold_key = (
            f"{PROTOCOL}/"
            f"{model}/"
            f"seed{seed}"
        )


        frozen_threshold = float(
            frozen_thresholds[
                threshold_key
            ]
        )


        if not np.isclose(
            frozen_threshold,
            float(
                frozen_row[
                    "frozen_threshold"
                ]
            ),
            atol=1e-12,
            rtol=0
        ):

            raise RuntimeError(
                f"Frozen threshold mismatch: {run_key}"
            )


        # ----------------------------------------------------
        # F. OUTPUT ROW
        # ----------------------------------------------------

        result = {

            "protocol":
                PROTOCOL,

            "model":
                model,

            "seed":
                int(
                    seed
                ),

            "excluded_test_image_id":
                int(
                    TEST_IMAGE_ID
                ),

            "frozen_threshold":
                frozen_threshold,

            "original_images":
                200,

            "sensitivity_images":
                199,

            "original_gt_objects":
                707,

            "sensitivity_gt_objects":
                len(
                    sensitivity_gt[
                        "annotations"
                    ]
                ),

            "removed_gt_objects":
                int(
                    REMOVED_GT_OBJECTS
                ),

            "removed_saved_predictions":
                int(
                    removed_prediction_count
                ),

            "removed_image_TP":
                removed_tp,

            "removed_image_FP":
                removed_fp,

            "removed_image_FN":
                removed_fn,

            "original_TP":
                original_tp,

            "original_FP":
                original_fp,

            "original_FN":
                original_fn,

            "sensitivity_TP":
                sensitivity_tp,

            "sensitivity_FP":
                sensitivity_fp,

            "sensitivity_FN":
                sensitivity_fn,

            "original_precision":
                original_operating[
                    "precision"
                ],

            "original_recall":
                original_operating[
                    "recall"
                ],

            "original_f1":
                original_operating[
                    "f1"
                ],

            "sensitivity_precision":
                sensitivity_operating[
                    "precision"
                ],

            "sensitivity_recall":
                sensitivity_operating[
                    "recall"
                ],

            "sensitivity_f1":
                sensitivity_operating[
                    "f1"
                ],

            "delta_precision":
                (
                    sensitivity_operating[
                        "precision"
                    ]
                    -
                    original_operating[
                        "precision"
                    ]
                ),

            "delta_recall":
                (
                    sensitivity_operating[
                        "recall"
                    ]
                    -
                    original_operating[
                        "recall"
                    ]
                ),

            "delta_f1":
                (
                    sensitivity_operating[
                        "f1"
                    ]
                    -
                    original_operating[
                        "f1"
                    ]
                ),

            "prediction_json_sha256":
                actual_pred_sha,
        }


        for metric in COCO_METRIC_NAMES:

            result[
                f"original_{metric}"
            ] = (
                original_coco[
                    metric
                ]
            )


            result[
                f"sensitivity_{metric}"
            ] = (
                sensitivity_coco[
                    metric
                ]
            )


            result[
                f"delta_{metric}"
            ] = (
                sensitivity_coco[
                    metric
                ]
                -
                original_coco[
                    metric
                ]
            )


        rows.append(
            result
        )


        print(
            f"[{counter:02d}/12] "
            f"{DISPLAY_MODEL[model]} seed {seed} | "
            f"mAP "
            f"{original_coco['mAP50_95']:.6f} -> "
            f"{sensitivity_coco['mAP50_95']:.6f} "
            f"(Δ {result['delta_mAP50_95']:+.6f}) | "
            f"F1 "
            f"{original_operating['f1']:.6f} -> "
            f"{sensitivity_operating['f1']:.6f} "
            f"(Δ {result['delta_f1']:+.6f})"
        )


seed_level = pd.DataFrame(
    rows
)


if len(
    seed_level
) != 12:

    raise RuntimeError(
        "Stage 10A must produce 12 seed-level rows."
    )


SEED_LEVEL_PATH = (
    OUT_DIR /
    "pHash0_sensitivity_seed_level.csv"
)


seed_level.to_csv(
    SEED_LEVEL_PATH,
    index=False
)


print(
    "\nOriginal COCO reproduction  : 12/12 PASSED"
)

print(
    "Original TP/FP/FN reproduce : 12/12 PASSED"
)


# ============================================================
# 17. THREE-SEED AGGREGATION
# ============================================================

aggregate_rows = []


AGG_METRICS = [

    "mAP50_95",
    "AP50",
    "AP75",
    "AP_small",
    "AP_medium",
    "AP_large",
    "precision",
    "recall",
    "f1",
]


for model in MODELS:

    d = seed_level[
        seed_level[
            "model"
        ]
        ==
        model
    ]


    if len(
        d
    ) != 3:

        raise RuntimeError(
            f"Expected three seeds for {model}."
        )


    row = {

        "model":
            model,

        "n_seeds":
            3,

        "excluded_test_image_id":
            int(
                TEST_IMAGE_ID
            ),
    }


    for metric in AGG_METRICS:

        for prefix in [
            "original",
            "sensitivity",
            "delta",
        ]:

            column = (
                f"{prefix}_{metric}"
            )


            row[
                f"{column}_mean"
            ] = float(
                d[
                    column
                ].mean()
            )


            row[
                f"{column}_sd"
            ] = float(
                d[
                    column
                ].std(
                    ddof=1
                )
            )


        row[
            f"delta_{metric}_max_abs"
        ] = float(
            d[
                f"delta_{metric}"
            ].abs().max()
        )


    aggregate_rows.append(
        row
    )


aggregate = pd.DataFrame(
    aggregate_rows
)


AGG_PATH = (
    OUT_DIR /
    "pHash0_sensitivity_mean_sd.csv"
)


aggregate.to_csv(
    AGG_PATH,
    index=False
)


# ============================================================
# 18. MANUSCRIPT-READY TABLE
# ============================================================

manuscript_rows = []


for _, r in aggregate.iterrows():

    manuscript_rows.append(
        {

            "Model":
                DISPLAY_MODEL[
                    r[
                        "model"
                    ]
                ],

            "Original mAP50-95":
                fmt_mean_sd(
                    r[
                        "original_mAP50_95_mean"
                    ],
                    r[
                        "original_mAP50_95_sd"
                    ]
                ),

            "After exclusion mAP50-95":
                fmt_mean_sd(
                    r[
                        "sensitivity_mAP50_95_mean"
                    ],
                    r[
                        "sensitivity_mAP50_95_sd"
                    ]
                ),

            "Mean ΔmAP50-95":
                (
                    f"{r['delta_mAP50_95_mean']:+.6f}"
                ),

            "Original F1":
                fmt_mean_sd(
                    r[
                        "original_f1_mean"
                    ],
                    r[
                        "original_f1_sd"
                    ]
                ),

            "After exclusion F1":
                fmt_mean_sd(
                    r[
                        "sensitivity_f1_mean"
                    ],
                    r[
                        "sensitivity_f1_sd"
                    ]
                ),

            "Mean ΔF1":
                (
                    f"{r['delta_f1_mean']:+.6f}"
                ),
        }
    )


manuscript_table = pd.DataFrame(
    manuscript_rows
)


MANUSCRIPT_TABLE_PATH = (
    OUT_DIR /
    "Supplementary_Table_pHash0_sensitivity.csv"
)


manuscript_table.to_csv(
    MANUSCRIPT_TABLE_PATH,
    index=False
)


print(
    "\n"
    + "=" * 100
)

print(
    "pHash=0 SENSITIVITY — THREE-SEED SUMMARY"
)

print(
    "=" * 100
)


display(
    manuscript_table
)


# ============================================================
# 19. DESCRIPTIVE RANKING CHECK
# ============================================================

original_order = (

    aggregate
    .sort_values(
        "original_mAP50_95_mean",
        ascending=False
    )[
        "model"
    ]
    .tolist()
)


sensitivity_order = (

    aggregate
    .sort_values(
        "sensitivity_mAP50_95_mean",
        ascending=False
    )[
        "model"
    ]
    .tolist()
)


ranking_preserved = (
    original_order
    ==
    sensitivity_order
)


print(
    "\nOriginal ranking:"
)

print(
    " > ".join(
        DISPLAY_MODEL[
            m
        ]
        for m in original_order
    )
)


print(
    "\nAfter exclusion:"
)

print(
    " > ".join(
        DISPLAY_MODEL[
            m
        ]
        for m in sensitivity_order
    )
)


print(
    f"\nRanking preserved           : {ranking_preserved}"
)


# ============================================================
# 20. DELTA SUMMARY
# ============================================================

delta_summary = {

    "max_abs_seed_delta_mAP50_95":
        float(
            seed_level[
                "delta_mAP50_95"
            ].abs().max()
        ),

    "max_abs_model_mean_delta_mAP50_95":
        float(
            aggregate[
                "delta_mAP50_95_mean"
            ].abs().max()
        ),

    "max_abs_seed_delta_F1":
        float(
            seed_level[
                "delta_f1"
            ].abs().max()
        ),

    "max_abs_model_mean_delta_F1":
        float(
            aggregate[
                "delta_f1_mean"
            ].abs().max()
        ),

    "ranking_preserved":
        bool(
            ranking_preserved
        ),

    "arbitrary_negligibility_threshold_used":
        False,
}


DELTA_PATH = (
    OUT_DIR /
    "pHash0_sensitivity_delta_summary.json"
)


DELTA_PATH.write_text(
    json.dumps(
        delta_summary,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "\n"
    + "=" * 100
)

print(
    "MAXIMUM OBSERVED SENSITIVITY CHANGES"
)

print(
    "=" * 100
)


for key, value in delta_summary.items():

    print(
        f"{key:48s}: {value}"
    )


# ============================================================
# 21. HUMAN-READABLE REVIEWER AUDIT
# ============================================================

VAL_META = (
    candidate_metadata[
        candidate_metadata[
            "role"
        ]
        ==
        "validation"
    ]
    .iloc[
        0
    ]
)


TEST_META = (
    candidate_metadata[
        candidate_metadata[
            "role"
        ]
        ==
        "locked_test"
    ]
    .iloc[
        0
    ]
)


AUDIT_TEXT_PATH = (
    OUT_DIR /
    "pHash0_candidate_review_summary.txt"
)


audit_text = f"""
REVIEWER-REQUESTED pHash=0 CANDIDATE ANALYSIS

Candidate
=========
Validation image ID : {VAL_IMAGE_ID}
Validation file     : {VAL_META['file_name']}
Validation batch    : {VAL_META['batch']}
Validation geometry : {VAL_META['width']} x {VAL_META['height']}
Validation objects  : {VAL_META['n_objects']}
Validation pHash    : {VAL_META['phash']}
Validation SHA-256  : {VAL_META['sha256']}

Locked-test image ID: {TEST_IMAGE_ID}
Locked-test file    : {TEST_META['file_name']}
Locked-test batch   : {TEST_META['batch']}
Locked-test geometry: {TEST_META['width']} x {TEST_META['height']}
Locked-test objects : {TEST_META['n_objects']}
Locked-test pHash   : {TEST_META['phash']}
Locked-test SHA-256 : {TEST_META['sha256']}

Hash interpretation
===================
pHash Hamming distance: 0
SHA-256 identical     : {VAL_META['sha256'] == TEST_META['sha256']}

A pHash-distance-0 result indicates extremely similar
perceptual hashes but does not by itself prove pixel identity.
The SHA-256 values are different.

The original frozen Stage 3B contact sheet is preserved as:
{CONTACT_COPY_PATH}

Sensitivity design
==================
The primary frozen results were NOT changed.

For sensitivity analysis only, the locked-test-side image
{TEST_IMAGE_ID} was excluded.

Original batch-disjoint test:
200 images / 707 GT objects

Sensitivity test:
199 images / {len(sensitivity_gt['annotations'])} GT objects

Removed GT objects:
{REMOVED_GT_OBJECTS}

All COCO sensitivity results were recomputed from the already
saved locked-test prediction JSON files.

No model inference was rerun.
No confidence threshold was changed.
No model was retrained.
No seed was replaced.
No model was excluded.

Important limitation
====================
This analysis quantifies the direct contribution of the
test-side pHash=0 candidate under the ORIGINAL
validation-selected thresholds.

It does NOT estimate the counterfactual confidence threshold
that might have been selected if the validation-side candidate
had also been removed before threshold selection.
""".strip()


AUDIT_TEXT_PATH.write_text(
    audit_text
    +
    "\n",
    encoding="utf-8"
)


# ============================================================
# 22. OUTPUT HASH REGISTRY
# ============================================================

outputs = {

    "candidate_metadata":
        CANDIDATE_META_PATH,

    "frozen_contact_sheet_copy":
        CONTACT_COPY_PATH,

    "reconstructed_original_test_gt":
        DERIVED_FULL_GT_PATH,

    "sensitivity_test_gt":
        SENSITIVITY_GT_PATH,

    "seed_level_results":
        SEED_LEVEL_PATH,

    "aggregate_results":
        AGG_PATH,

    "supplementary_table":
        MANUSCRIPT_TABLE_PATH,

    "delta_summary":
        DELTA_PATH,

    "review_summary":
        AUDIT_TEXT_PATH,
}


output_hashes = {

    name:
        sha256_file(
            path
        )

    for name, path in outputs.items()
}


# ============================================================
# 23. MANIFEST
# ============================================================

manifest = {

    "stage":
        "10A",

    "status":
        "PHASH0_REVIEWER_SENSITIVITY_COMPLETE",

    "analysis_type":
        (
            "reviewer-requested post-hoc sensitivity analysis"
        ),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "canonical_root":
        str(
            ROOT
        ),

    "candidate": {

        "validation_image_id":
            int(
                VAL_IMAGE_ID
            ),

        "test_image_id":
            int(
                TEST_IMAGE_ID
            ),

        "phash_distance":
            0,

        "validation_file":
            str(
                VAL_META[
                    "file_name"
                ]
            ),

        "test_file":
            str(
                TEST_META[
                    "file_name"
                ]
            ),

        "validation_batch":
            str(
                VAL_META[
                    "batch"
                ]
            ),

        "test_batch":
            str(
                TEST_META[
                    "batch"
                ]
            ),

        "validation_sha256":
            str(
                VAL_META[
                    "sha256"
                ]
            ),

        "test_sha256":
            str(
                TEST_META[
                    "sha256"
                ]
            ),

        "sha256_equal":
            bool(
                VAL_META[
                    "sha256"
                ]
                ==
                TEST_META[
                    "sha256"
                ]
            ),

        "validation_phash":
            str(
                VAL_META[
                    "phash"
                ]
            ),

        "test_phash":
            str(
                TEST_META[
                    "phash"
                ]
            ),

        "contact_sheet_source":
            str(
                CONTACT_SHEET_PATH
            ),
    },

    "sensitivity_design": {

        "excluded_side":
            "locked_test",

        "excluded_test_image_id":
            int(
                TEST_IMAGE_ID
            ),

        "original_test_images":
            200,

        "sensitivity_test_images":
            199,

        "original_test_objects":
            707,

        "sensitivity_test_objects":
            int(
                len(
                    sensitivity_gt[
                        "annotations"
                    ]
                )
            ),

        "removed_gt_objects":
            int(
                REMOVED_GT_OBJECTS
            ),

        "models":
            MODELS,

        "seeds":
            SEEDS,

        "runs":
            12,

        "coco_evaluator":
            "pycocotools COCOeval bbox",

        "coco_max_dets":
            [
                1,
                10,
                100,
            ],

        "prediction_source":
            "previously saved frozen locked-test prediction JSONs",

        "operating_point_source":
            "Stage 8A frozen-threshold image-level TP/FP/FN",

        "new_inference":
            False,

        "threshold_tuning":
            False,

        "retraining":
            False,

        "seed_replacement":
            False,

        "model_exclusion":
            False,
    },

    "scope_limitation":
        (
            "Direct test-side exclusion sensitivity under "
            "the original frozen validation-selected "
            "thresholds; no counterfactual validation "
            "threshold re-selection was performed."
        ),

    "primary_frozen_results_modified":
        False,

    "descriptive_ranking_preserved":
        bool(
            ranking_preserved
        ),

    "delta_summary":
        delta_summary,

    "input_hashes": {

        "image_audit":
            sha256_file(
                IMAGE_AUDIT_PATH
            ),

        "near_duplicate_candidates":
            sha256_file(
                NEAR_DUP_PATH
            ),

        "batch_disjoint_split":
            sha256_file(
                BATCH_SPLIT_PATH
            ),

        "cleaned_annotations":
            sha256_file(
                CLEANED_ANNOTATION_PATH
            ),

        "locked_test_summary":
            actual_summary_sha,

        "stage8a_per_image":
            sha256_file(
                ERROR_PER_IMAGE_PATH
            ),

        "frozen_thresholds":
            sha256_file(
                THRESHOLD_PATH
            ),
    },

    "output_hashes":
        output_hashes,
}


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ============================================================
# 24. FINAL
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "STAGE 10A PASSED — pHash=0 REVIEWER SENSITIVITY COMPLETE"
)

print(
    "=" * 100
)


print(
    f"Validation candidate ID     : {VAL_IMAGE_ID}"
)

print(
    f"Locked-test candidate ID    : {TEST_IMAGE_ID}"
)

print(
    "pHash distance              : 0"
)

print(
    f"SHA-256 identical           : "
    f"{VAL_META['sha256'] == TEST_META['sha256']}"
)

print(
    "Frozen contact sheet        : VERIFIED"
)

print(
    "Reconstructed test COCO GT  : 200 images / 707 objects"
)

print(
    f"Sensitivity test COCO GT    : "
    f"199 images / "
    f"{len(sensitivity_gt['annotations'])} objects"
)

print(
    "Original COCO reproduction  : 12/12 PASSED"
)

print(
    "Original TP/FP/FN reproduce : 12/12 PASSED"
)

print(
    "Sensitivity evaluations     : 12/12 COMPLETE"
)

print(
    f"Ranking preserved           : {ranking_preserved}"
)

print(
    "New inference               : NO"
)

print(
    "Threshold tuning            : NO"
)

print(
    "Retraining                  : NO"
)

print(
    "Primary frozen results      : UNCHANGED"
)


print(
    "\nCandidate metadata:"
)

print(
    CANDIDATE_META_PATH
)


print(
    "\nReviewer comparison figure:"
)

print(
    CONTACT_COPY_PATH
)


print(
    "\nSeed-level sensitivity:"
)

print(
    SEED_LEVEL_PATH
)


print(
    "\nThree-seed sensitivity:"
)

print(
    AGG_PATH
)


print(
    "\nSupplementary table:"
)

print(
    MANUSCRIPT_TABLE_PATH
)


print(
    "\nManifest:"
)

print(
    MANIFEST_PATH
)


print(
    "\nNEXT:"
)

print(
    "SEND THE THREE-SEED SUMMARY + DELTA SUMMARY + "
    "DISPLAYED pHash CONTACT SHEET FOR REVIEW."
)

print(
    "=" * 100
)

In [ ]:
# ==================================================================================================
# STAGE 10B — REVIEWER-REQUESTED SPLIT / TRAINING / COCO REPRODUCIBILITY AUDIT
# DRIVE-ONLY — NO TRAINING, NO INFERENCE, NO THRESHOLD TUNING
#
# PURPOSE
#   1) Reproduce the frozen random split exactly.
#   2) Reproduce the frozen batch-disjoint split-selection algorithm exactly.
#   3) Audit exact pretrained checkpoint names and frozen training configuration.
#   4) Recover actual completed epochs from the 24 results.csv files.
#   5) Report model parameters / weights / GFLOPs from already-frozen locked-test outputs.
#   6) Distinguish requested optimizer='auto' settings from the effective optimizer
#      recorded in the original Ultralytics console logs.
#   7) Record the exact COCOeval configuration and object-size boundaries.
#   8) Save reviewer-ready supplementary tables + reproducibility manifest.
#
# IMPORTANT
#   - NO model.train()
#   - NO model.predict()
#   - NO new inference
#   - NO threshold tuning
#   - NO retraining
#   - NO modification of primary frozen results
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import hashlib
import json
import math
import re
import sys
import warnings

import numpy as np
import pandas as pd
import yaml

from sklearn.model_selection import train_test_split
from pycocotools.cocoeval import Params


# --------------------------------------------------------------------------------------------------
# 0. BANNER + CANONICAL PATHS
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("STAGE 10B — REVIEWER-REQUESTED SPLIT / TRAINING / COCO REPRODUCIBILITY AUDIT")
print("DRIVE-ONLY — NO TRAINING / NO INFERENCE / NO THRESHOLD TUNING")
print("=" * 100)

DRIVE_MOUNT = Path("/content/GDRIVE_REAL")
ROOT = DRIVE_MOUNT / "MyDrive" / "Atik14082026" / "TACO_Pipeline_v5"

DIRS = {
    "audit": ROOT / "01_audit",
    "splits": ROOT / "02_splits",
    "runs": ROOT / "03_runs",
    "validation": ROOT / "04_validation",
    "test": ROOT / "05_locked_test",
    "analysis": ROOT / "06_analysis",
    "manifests": ROOT / "07_manifests",
    "metadata": ROOT / "08_metadata",
    "manuscript": ROOT / "09_manuscript",
    "reviewer": ROOT / "10_reviewer_sensitivity",
}

OUT = DIRS["reviewer"] / "stage10b_reproducibility"
OUT.mkdir(parents=True, exist_ok=True)

STAGE10A_MANIFEST = DIRS["manifests"] / "STAGE_10A_PHASH0_SENSITIVITY.json"

if not ROOT.exists():
    raise FileNotFoundError(
        f"Canonical project root not found:\n{ROOT}\n\n"
        "Do NOT use /content/REAL_DRIVE. "
        "Run the safe Google Drive mount cell first."
    )

print(f"\nCanonical ROOT : {ROOT}")
print(f"Stage 10B OUT  : {OUT}")


# --------------------------------------------------------------------------------------------------
# 1. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def require_file(path, description=None):
    path = Path(path)
    if not path.exists():
        label = description or str(path)
        raise FileNotFoundError(f"Required frozen artifact missing:\n{label}\n{path}")
    return path


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def json_dump(obj, path):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8"
    )


def normalize_path_string(x):
    if x is None:
        return None
    return str(x).replace("\\", "/")


def safe_float(x):
    try:
        if x is None or pd.isna(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def safe_int(x):
    try:
        if x is None or pd.isna(x):
            return None
        return int(x)
    except Exception:
        return None


def mean_sd_string(values, decimals=4):
    v = pd.Series(values, dtype=float).dropna()
    if len(v) == 0:
        return ""
    if len(v) == 1:
        return f"{v.iloc[0]:.{decimals}f}"
    return f"{v.mean():.{decimals}f} ± {v.std(ddof=1):.{decimals}f}"


def find_metric_column(df, wanted_tokens):
    """
    Find a results.csv column by normalized token matching.
    """
    normalized = {
        c: re.sub(r"\s+", "", str(c).lower())
        for c in df.columns
    }
    for c, nc in normalized.items():
        if all(tok.lower().replace(" ", "") in nc for tok in wanted_tokens):
            return c
    return None


def recursive_strings(obj):
    """Yield all scalar strings inside arbitrary JSON-like object."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield str(k)
            yield from recursive_strings(v)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            yield from recursive_strings(v)
    else:
        yield str(obj)


# --------------------------------------------------------------------------------------------------
# 2. REQUIRED FROZEN INPUTS
# --------------------------------------------------------------------------------------------------

IMAGE_AUDIT = require_file(
    DIRS["audit"] / "image_audit.csv",
    "Frozen image audit"
)

RANDOM_SPLIT_PATH = require_file(
    DIRS["splits"] / "random_split.json",
    "Frozen random split"
)

BATCH_SPLIT_PATH = require_file(
    DIRS["splits"] / "batch_disjoint_split.json",
    "Frozen batch-disjoint split"
)

BATCH_ASSIGN_PATH = require_file(
    DIRS["splits"] / "batch_disjoint_group_assignment.csv",
    "Frozen batch-disjoint group assignment"
)

TRAINING_CONFIG_PATH = require_file(
    DIRS["manifests"] / "training_config_frozen.json",
    "Frozen training configuration"
)

LOCKED_SUMMARY_PATH = require_file(
    DIRS["test"] / "locked_test_summary_all_runs.csv",
    "Frozen locked-test summary"
)


FINAL_MANIFEST_PATH = require_file(
    DIRS["manifests"] / "FINAL_REPRODUCIBILITY_MANIFEST.json",
    "Final reproducibility manifest"
)

if STAGE10A_MANIFEST.exists():
    print("Stage 10A manifest          : PRESENT")
else:
    warnings.warn(
        "Stage 10A manifest not found. Stage 10B can still audit split/training, "
        "but the reviewer sensitivity chain will be incomplete."
    )

img_df = pd.read_csv(IMAGE_AUDIT)
random_frozen = read_json(RANDOM_SPLIT_PATH)
batch_frozen = read_json(BATCH_SPLIT_PATH)
batch_assign_frozen = pd.read_csv(BATCH_ASSIGN_PATH)
training_cfg = read_json(TRAINING_CONFIG_PATH)
locked_df = pd.read_csv(LOCKED_SUMMARY_PATH)
final_manifest = read_json(FINAL_MANIFEST_PATH)

print("\nPersistent frozen inputs    : PASSED")


# --------------------------------------------------------------------------------------------------
# 3. DATASET IDENTITY CHECK
# --------------------------------------------------------------------------------------------------

required_audit_cols = {
    "image_id", "batch", "n_objects",
    "small_objects", "medium_objects", "large_objects"
}
missing_cols = required_audit_cols - set(img_df.columns)

if missing_cols:
    raise RuntimeError(
        f"image_audit.csv is missing required columns: {sorted(missing_cols)}"
    )

if len(img_df) != 1500:
    raise RuntimeError(f"Expected 1500 audited images; found {len(img_df)}")

total_objects = int(img_df["n_objects"].sum())
small_total = int(img_df["small_objects"].sum())
medium_total = int(img_df["medium_objects"].sum())
large_total = int(img_df["large_objects"].sum())

assert total_objects == 4784, total_objects
assert small_total == 383, small_total
assert medium_total == 1306, medium_total
assert large_total == 3095, large_total
assert int(img_df["batch"].nunique()) == 15

print("Dataset identity            : PASSED")
print(f"Images                      : {len(img_df)}")
print(f"Objects                     : {total_objects}")
print(f"Small / Medium / Large      : {small_total} / {medium_total} / {large_total}")
print(f"Official batches            : {img_df['batch'].nunique()}")


# ==================================================================================================
# PART A — EXACT RANDOM SPLIT REPRODUCTION
# ==================================================================================================

print("\n" + "=" * 100)
print("A. EXACT RANDOM SPLIT REPRODUCTION")
print("=" * 100)

SPLIT_SEED = 20260814

tmp = img_df.copy()

# Final v5 rule:
# object-count stratification, capped at 8
strata = tmp["n_objects"].clip(upper=8).astype(str)

# Rare first-stage strata -> "rare" if < 4 images
freq = strata.value_counts()
strata = strata.where(strata.map(freq) >= 4, "rare")

train_ids, temp_ids = train_test_split(
    tmp["image_id"].astype(int).to_numpy(),
    test_size=0.30,
    random_state=SPLIT_SEED,
    stratify=strata
)

temp = tmp[tmp["image_id"].isin(temp_ids)].copy()

# Same rule for val/test subdivision
tstr = temp["n_objects"].clip(upper=8).astype(str)
freq2 = tstr.value_counts()
tstr = tstr.where(tstr.map(freq2) >= 2, "rare")

random_second_stage_fallback = False

try:
    val_ids, test_ids = train_test_split(
        temp["image_id"].astype(int).to_numpy(),
        test_size=0.50,
        random_state=SPLIT_SEED,
        stratify=tstr
    )
except ValueError:
    random_second_stage_fallback = True
    val_ids, test_ids = train_test_split(
        temp["image_id"].astype(int).to_numpy(),
        test_size=0.50,
        random_state=SPLIT_SEED
    )

random_reconstructed = {
    "train": sorted(map(int, train_ids)),
    "val": sorted(map(int, val_ids)),
    "test": sorted(map(int, test_ids)),
}

random_exact = all(
    list(map(int, random_frozen[s])) == random_reconstructed[s]
    for s in ["train", "val", "test"]
)

if not random_exact:
    for s in ["train", "val", "test"]:
        a = set(map(int, random_frozen[s]))
        b = set(random_reconstructed[s])
        print(
            f"{s:5s}: frozen={len(a)} reconstructed={len(b)} "
            f"frozen_only={len(a-b)} reconstructed_only={len(b-a)}"
        )
    raise RuntimeError(
        "RANDOM SPLIT EXACT REPRODUCTION FAILED. "
        "Do not continue manuscript revision until resolved."
    )

print("Split seed                  :", SPLIT_SEED)
print("First-stage test_size       : 0.30")
print("Second-stage test_size      : 0.50")
print("Stratification variable     : n_objects capped at 8")
print("First-stage rare threshold  : <4 -> 'rare'")
print("Second-stage rare threshold : <2 -> 'rare'")
print("Fallback used               :", random_second_stage_fallback)
print("\nRANDOM SPLIT EXACT REPRODUCTION : PASSED")


# ==================================================================================================
# PART B — EXACT BATCH-DISJOINT SPLIT-SELECTION REPRODUCTION
# ==================================================================================================

print("\n" + "=" * 100)
print("B. EXACT BATCH-DISJOINT SPLIT-SELECTION REPRODUCTION")
print("=" * 100)

gs = (
    img_df.groupby("batch")
    .agg(
        images=("image_id", "size"),
        objects=("n_objects", "sum"),
        small=("small_objects", "sum"),
        medium=("medium_objects", "sum"),
        large=("large_objects", "sum"),
    )
    .reset_index()
)

# IMPORTANT: preserve final-v5 group order exactly.
groups = gs["batch"].tolist()

target_ratio = {
    "train": 0.70,
    "val": 0.15,
    "test": 0.15,
}

metric_cols = ["images", "objects", "small", "medium", "large"]

totals = gs[metric_cols].sum().to_dict()

target = {
    s: {
        k: target_ratio[s] * float(totals[k])
        for k in metric_cols
    }
    for s in target_ratio
}

# Final-v5 score:
# image-count deviation receives weight 3;
# all object-count/size-count deviations receive weight 1.
weights = np.array([3.0, 1.0, 1.0, 1.0, 1.0], dtype=float)
values = gs[metric_cols].to_numpy(dtype=float)

N_SEARCH = 100_000
rng = np.random.default_rng(SPLIT_SEED)

# Each row consumes exactly len(groups) RNG draws, equivalent to the original
# nested loop under numpy.default_rng with the same seed.
r = rng.random((N_SEARCH, len(groups)))

# 0=train, 1=val, 2=test
assign_code = np.where(
    r < 0.70,
    0,
    np.where(r < 0.85, 1, 2)
).astype(np.int8)

scores = np.zeros(N_SEARCH, dtype=np.float64)
valid = np.ones(N_SEARCH, dtype=bool)

split_names = ["train", "val", "test"]

for s_code, s_name in enumerate(split_names):

    membership = (assign_code == s_code)

    # Original algorithm rejects a candidate if any split is empty.
    nonempty = membership.any(axis=1)
    valid &= nonempty

    # Sum batch statistics within this candidate split.
    got = membership.astype(np.float64) @ values

    tgt = np.array(
        [max(float(target[s_name][k]), 1.0) for k in metric_cols],
        dtype=np.float64
    )

    scores += (((got - tgt) / tgt) ** 2 * weights).sum(axis=1)

scores[~valid] = np.inf

best_idx = int(np.argmin(scores))
best_score = float(scores[best_idx])
best_codes = assign_code[best_idx]

best_assign = {
    g: split_names[int(code)]
    for g, code in zip(groups, best_codes)
}

batch_reconstructed = {}
for s in split_names:
    batch_reconstructed[s] = sorted(
        img_df.loc[
            img_df["batch"].map(best_assign) == s,
            "image_id"
        ].astype(int).tolist()
    )

batch_exact = all(
    list(map(int, batch_frozen[s])) == batch_reconstructed[s]
    for s in split_names
)

# Compare batch assignment CSV as a second independent check.
frozen_assign_dict = {
    str(r.batch): str(r.split)
    for r in batch_assign_frozen.itertuples(index=False)
}

batch_assignment_exact = (
    set(frozen_assign_dict) == set(best_assign)
    and all(frozen_assign_dict[b] == best_assign[b] for b in best_assign)
)

if not batch_exact or not batch_assignment_exact:
    print("\nReconstructed assignment:")
    for b in groups:
        print(f"  {b:10s} -> {best_assign[b]}")

    print("\nFrozen assignment:")
    for b in sorted(frozen_assign_dict):
        print(f"  {b:10s} -> {frozen_assign_dict[b]}")

    raise RuntimeError(
        "BATCH-DISJOINT EXACT REPRODUCTION FAILED. "
        "Do not modify the manuscript until this discrepancy is resolved."
    )

print("Split seed                  :", SPLIT_SEED)
print("Candidate assignments       :", f"{N_SEARCH:,}")
print("Candidate RNG               : numpy.random.default_rng")
print("Target ratios               : 0.70 / 0.15 / 0.15")
print("Balancing variables         : images, objects, small, medium, large")
print("Image-count weight          : 3.0")
print("Other variable weights      : 1.0")
print("Best objective score        :", best_score)

print("\nSelected official batches:")
for s in split_names:
    selected = [g for g in groups if best_assign[g] == s]
    print(f"{s:5s}: {', '.join(selected)}")

print("\nBATCH-DISJOINT EXACT REPRODUCTION : PASSED")


# --------------------------------------------------------------------------------------------------
# C. SPLIT COMPOSITION TABLE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("C. FROZEN SPLIT COMPOSITION")
print("=" * 100)

def split_composition(split_dict, protocol):
    rows = []
    for s in ["train", "val", "test"]:
        ids = set(map(int, split_dict[s]))
        d = img_df[img_df["image_id"].astype(int).isin(ids)]

        rows.append({
            "protocol": protocol,
            "split": s,
            "images": int(len(d)),
            "objects": int(d["n_objects"].sum()),
            "small": int(d["small_objects"].sum()),
            "medium": int(d["medium_objects"].sum()),
            "large": int(d["large_objects"].sum()),
            "unique_official_batches": int(d["batch"].nunique()),
        })
    return rows

split_table = pd.DataFrame(
    split_composition(random_frozen, "random")
    + split_composition(batch_frozen, "batch_disjoint")
)

expected_counts = {
    ("random", "train"): (1050, 3284, 242, 845, 2197),
    ("random", "val"):   (225, 674, 60, 160, 454),
    ("random", "test"):  (225, 826, 81, 301, 444),
    ("batch_disjoint", "train"): (1061, 3369, 269, 909, 2191),
    ("batch_disjoint", "val"):   (239, 708, 54, 186, 468),
    ("batch_disjoint", "test"):  (200, 707, 60, 211, 436),
}

for r0 in split_table.itertuples(index=False):
    obs = (
        int(r0.images), int(r0.objects),
        int(r0.small), int(r0.medium), int(r0.large)
    )
    exp = expected_counts[(r0.protocol, r0.split)]
    if obs != exp:
        raise RuntimeError(
            f"Frozen split composition mismatch for "
            f"{r0.protocol}/{r0.split}: {obs} != {exp}"
        )

split_table_path = OUT / "Supplementary_Table_Split_Generation.csv"
split_table.to_csv(split_table_path, index=False)

display(split_table)

print("\nFrozen split composition    : PASSED")


# ==================================================================================================
# PART D — 24-RUN TRAINING ARTIFACT AUDIT
# ==================================================================================================

print("\n" + "=" * 100)
print("D. 24-RUN TRAINING ARTIFACT / ACTUAL EPOCH AUDIT")
print("=" * 100)

MODEL_DISPLAY = {
    "yolov8s": "YOLOv8s",
    "yolo11s": "YOLO11s",
    "yolo26s": "YOLO26s",
    "rtdetr_l": "RT-DETR-L",
}

# Canonical exact pretrained checkpoint names from final-v5 frozen configuration.
PRETRAINED = {}

cfg_models = training_cfg.get("models", {})

for model_key, spec in cfg_models.items():
    if isinstance(spec, (list, tuple)) and len(spec) >= 2:
        PRETRAINED[model_key] = str(spec[1])
    elif isinstance(spec, dict):
        PRETRAINED[model_key] = str(
            spec.get("weights")
            or spec.get("model")
            or spec.get("checkpoint")
        )

# Hard validation against the final-v5 design.
expected_pretrained = {
    "yolov8s": "yolov8s.pt",
    "yolo11s": "yolo11s.pt",
    "yolo26s": "yolo26s.pt",
    "rtdetr_l": "rtdetr-l.pt",
}

if PRETRAINED != expected_pretrained:
    print("Parsed pretrained mapping:", PRETRAINED)
    raise RuntimeError(
        "Frozen model checkpoint mapping differs from expected final-v5 design."
    )

seeds = training_cfg.get("seeds", training_cfg.get("seed_list", [42, 123, 2026]))
seeds = [int(x) for x in seeds]

protocols = training_cfg.get(
    "protocols",
    ["random", "batch_disjoint"]
)

models = list(expected_pretrained.keys())

final_manifest_text = FINAL_MANIFEST_PATH.read_text(encoding="utf-8")

run_rows = []

for protocol in protocols:
    for model_key in models:
        for seed in seeds:

            run_dir = DIRS["runs"] / protocol / f"{model_key}_seed{seed}"
            best_pt = run_dir / "weights" / "best.pt"
            results_csv = run_dir / "results.csv"
            args_yaml = run_dir / "args.yaml"

            require_file(best_pt, f"{protocol}/{model_key}/seed{seed} best.pt")
            require_file(results_csv, f"{protocol}/{model_key}/seed{seed} results.csv")

            args = {}
            if args_yaml.exists():
                try:
                    args = yaml.safe_load(args_yaml.read_text(encoding="utf-8")) or {}
                except Exception as e:
                    warnings.warn(f"Could not parse {args_yaml}: {e}")

            rdf = pd.read_csv(results_csv)
            rdf.columns = [str(c).strip() for c in rdf.columns]

            n_epochs = int(len(rdf))
            early_stopped = bool(n_epochs < int(training_cfg.get("epochs", 100)))

            map_col = find_metric_column(rdf, ["map50-95"])
            best_epoch_by_map = None
            max_val_map = np.nan

            if map_col is not None and len(rdf):
                vals = pd.to_numeric(rdf[map_col], errors="coerce")

                if vals.notna().any():
                    idxmax = int(vals.idxmax())

                    # Ultralytics epoch rows are zero-indexed internally in the CSV sequence,
                    # but manuscript-facing epoch count is 1-based.
                    best_epoch_by_map = int(idxmax + 1)
                    max_val_map = float(vals.loc[idxmax])

            best_sha = sha256_file(best_pt)
            best_bytes = int(best_pt.stat().st_size)

            # Robust manifest registration check independent of JSON schema.
            hash_registered = best_sha in final_manifest_text

            run_rows.append({
                "protocol": protocol,
                "model_key": model_key,
                "model": MODEL_DISPLAY[model_key],
                "seed": seed,
                "pretrained_checkpoint_frozen": PRETRAINED[model_key],

                # args.yaml values are REQUESTED TRAINER SETTINGS.
                # For optimizer='auto', lr0/momentum in args.yaml are not necessarily
                # the effective optimizer values.
                "args_model": normalize_path_string(args.get("model")),
                "optimizer_requested": args.get(
                    "optimizer",
                    training_cfg.get("optimizer")
                ),
                "lr0_argument": safe_float(args.get("lr0")),
                "momentum_argument": safe_float(args.get("momentum")),
                "weight_decay_argument": safe_float(args.get("weight_decay")),

                "imgsz": safe_int(args.get("imgsz", training_cfg.get("imgsz"))),
                "epoch_budget": safe_int(args.get("epochs", training_cfg.get("epochs"))),
                "epochs_completed": n_epochs,
                "early_stopped_before_budget": early_stopped,

                # This is intentionally labelled "by validation mAP50-95".
                # It is not asserted to be the exact checkpoint-selection epoch
                # because Ultralytics checkpoint fitness may use its internal fitness rule.
                "epoch_of_max_validation_mAP50_95": best_epoch_by_map,
                "max_validation_mAP50_95": max_val_map,

                "patience": safe_int(args.get("patience", training_cfg.get("patience"))),
                "batch": safe_int(args.get("batch", training_cfg.get("batch"))),
                "workers": safe_int(args.get("workers", training_cfg.get("workers"))),
                "amp": args.get("amp", training_cfg.get("amp")),
                "deterministic_requested": args.get(
                    "deterministic",
                    training_cfg.get("deterministic")
                ),
                "mosaic": safe_float(args.get("mosaic", training_cfg.get("mosaic"))),
                "close_mosaic": safe_int(
                    args.get("close_mosaic", training_cfg.get("close_mosaic"))
                ),
                "mixup": safe_float(args.get("mixup", training_cfg.get("mixup"))),
                "copy_paste": safe_float(
                    args.get("copy_paste", training_cfg.get("copy_paste"))
                ),

                "best_pt_bytes": best_bytes,
                "best_pt_MiB": best_bytes / (1024 ** 2),
                "best_pt_sha256": best_sha,
                "checkpoint_hash_registered_in_final_manifest": hash_registered,

                "results_csv": str(results_csv),
                "args_yaml_present": args_yaml.exists(),
            })

run_table = pd.DataFrame(run_rows)

if len(run_table) != 24:
    raise RuntimeError(f"Expected 24 run rows; found {len(run_table)}")

if run_table["epochs_completed"].isna().any():
    raise RuntimeError("At least one run lacks an actual epoch count.")

run_table_path = OUT / "Supplementary_Table_Run_Training_Details.csv"
run_table.to_csv(run_table_path, index=False)

print(f"Run artifacts               : {len(run_table)}/24 PRESENT")
print(
    "Checkpoint hashes registered:",
    f"{int(run_table['checkpoint_hash_registered_in_final_manifest'].sum())}/24"
)

print("\nActual completed epochs:")
display(
    run_table[
        [
            "protocol", "model", "seed",
            "epoch_budget", "epochs_completed",
            "early_stopped_before_budget",
            "epoch_of_max_validation_mAP50_95",
            "max_validation_mAP50_95"
        ]
    ].sort_values(["protocol", "model", "seed"])
)

print("\n24-RUN TRAINING ARTIFACT AUDIT : PASSED")


# ==================================================================================================
# PART E — MODEL COMPLEXITY FROM ALREADY-FROZEN LOCKED-TEST SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("E. MODEL COMPLEXITY — FROZEN ARTIFACTS ONLY")
print("=" * 100)

# No model is loaded here.
# Prefer complexity measurements that were already written during locked-test evaluation.

locked_cols = {c.lower(): c for c in locked_df.columns}

def first_existing_column(candidates):
    for cand in candidates:
        if cand in locked_df.columns:
            return cand
    lower_map = {str(c).lower(): c for c in locked_df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

param_col = first_existing_column([
    "params", "parameters", "n_params"
])

weight_col = first_existing_column([
    "weight_mb", "weights_mb", "model_weight_mb"
])

flops_col = first_existing_column([
    "flops_g_optional", "flops_g", "gflops", "GFLOPs"
])

if "model" not in locked_df.columns:
    raise RuntimeError(
        f"Locked-test summary lacks model column. Columns:\n{locked_df.columns.tolist()}"
    )

complexity_rows = []

for model_key in models:
    d = locked_df[locked_df["model"].astype(str) == model_key].copy()

    if len(d) == 0:
        raise RuntimeError(f"No locked-test summary rows for model {model_key}")

    # Values should be architecture-level constants, but we summarize all frozen rows.
    params_values = (
        pd.to_numeric(d[param_col], errors="coerce").dropna()
        if param_col else pd.Series(dtype=float)
    )
    weight_values = (
        pd.to_numeric(d[weight_col], errors="coerce").dropna()
        if weight_col else pd.Series(dtype=float)
    )
    flops_values = (
        pd.to_numeric(d[flops_col], errors="coerce").dropna()
        if flops_col else pd.Series(dtype=float)
    )

    # Independent fallback for checkpoint size only: use existing checkpoint files,
    # never instantiate a model.
    if len(weight_values) == 0:
        checkpoint_mib = run_table.loc[
            run_table["model_key"] == model_key,
            "best_pt_MiB"
        ].astype(float)
        weight_mean = float(checkpoint_mib.mean())
        weight_source = "best.pt file size computed from frozen checkpoints"
    else:
        weight_mean = float(weight_values.mean())
        weight_source = f"locked-test summary column '{weight_col}'"

    complexity_rows.append({
        "model": MODEL_DISPLAY[model_key],
        "model_key": model_key,
        "pretrained_checkpoint": PRETRAINED[model_key],
        "parameters": (
            int(round(params_values.median()))
            if len(params_values) else np.nan
        ),
        "GFLOPs_at_640": (
            float(flops_values.median())
            if len(flops_values) else np.nan
        ),
        "frozen_checkpoint_MiB": weight_mean,
        "parameter_source": (
            f"locked-test summary column '{param_col}'"
            if param_col else "not available in locked-test CSV"
        ),
        "GFLOPs_source": (
            f"locked-test summary column '{flops_col}'"
            if flops_col else "not available in locked-test CSV"
        ),
        "weight_source": weight_source,
        "new_model_loading_for_stage10b": False,
    })

complexity_table = pd.DataFrame(complexity_rows)

complexity_path = OUT / "Supplementary_Table_Model_Complexity.csv"
complexity_table.to_csv(complexity_path, index=False)

display(complexity_table)

if complexity_table["parameters"].isna().any():
    print(
        "\nWARNING: Parameter count was not present for one or more models in the "
        "locked-test summary. Stage 10B deliberately did NOT load a model to invent "
        "a new post-test characterization."
    )

if complexity_table["GFLOPs_at_640"].isna().any():
    print(
        "\nWARNING: GFLOPs were not present for one or more models in the frozen "
        "locked-test summary. They are left missing rather than recomputed post hoc."
    )


# ==================================================================================================
# PART F — OPTIMIZER='AUTO' AUDIT
# ==================================================================================================

print("\n" + "=" * 100)
print("F. OPTIMIZER='AUTO' AUDIT")
print("=" * 100)

# Ultralytics args.yaml retains optimizer='auto' and may retain nominal lr0/momentum
# arguments, but those are explicitly overridden by optimizer='auto'.
#
# The original training console logs retained from this experiment report:
#   optimizer: AdamW(lr=0.002, momentum=0.9)
#
# IMPORTANT:
# This is registered as an ORIGINAL CONSOLE-LOG RECORD, not reconstructed from
# the stripped best.pt files.

console_optimizer_record = {
    "requested_optimizer": str(training_cfg.get("optimizer", "auto")),
    "effective_optimizer_recorded_in_original_training_console": "AdamW",
    "effective_learning_rate_recorded_in_original_training_console": 0.002,
    "effective_momentum_recorded_in_original_training_console": 0.9,
    "evidence_type": (
        "original Ultralytics 8.4.111 training console output retained with the study records; "
        "not inferred from stripped checkpoints"
    ),
    "interpretation": (
        "The lr0 and momentum arguments shown in args.yaml should not be reported as the "
        "effective values when optimizer='auto', because Ultralytics explicitly reported "
        "that those nominal arguments were ignored and selected AdamW(lr=0.002, momentum=0.9)."
    ),
}

optimizer_path = OUT / "Optimizer_Auto_Audit.json"
json_dump(console_optimizer_record, optimizer_path)

print("Frozen requested optimizer  :", training_cfg.get("optimizer"))
print("Effective optimizer record  : AdamW")
print("Effective learning rate     : 0.002")
print("Effective momentum          : 0.9")
print("Evidence                    : ORIGINAL TRAINING CONSOLE LOG")
print("Reconstructed from best.pt  : NO")


# ==================================================================================================
# PART G — COCO EVALUATOR CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("G. COCO EVALUATOR PARAMETER AUDIT")
print("=" * 100)

# Standard pycocotools bbox evaluation configuration.
p = Params(iouType="bbox")

coco_params = {
    "implementation": "pycocotools COCOeval",
    "iou_type": "bbox",

    "iou_thresholds": [float(x) for x in p.iouThrs],
    "recall_threshold_count": int(len(p.recThrs)),
    "recall_threshold_min": float(p.recThrs.min()),
    "recall_threshold_max": float(p.recThrs.max()),

    "max_detections_evaluator": [int(x) for x in p.maxDets],

    "area_range_labels": list(p.areaRngLbl),
    "area_ranges_pixels_squared": [
        [float(v) for v in row]
        for row in p.areaRng
    ],

    "standard_COCO_size_interpretation": {
        "small": "area < 32^2 pixels",
        "medium": "32^2 <= area < 96^2 pixels",
        "large": "area >= 96^2 pixels",
    },

    # From frozen evaluation code:
    "prediction_generation_confidence_floor": 0.001,
    "prediction_generation_max_det": 300,

    "important_max_det_note": (
        "Predictions were generated with max_det=300, while standard COCOeval "
        "uses maxDets=[1,10,100]; therefore AP/AR evaluation is capped at "
        "100 detections per image by COCOeval."
    ),

    "area_coordinate_system": (
        "COCO bounding-box area in the original image coordinate system; "
        "the 640x640 network input does not redefine the COCO small/medium/large labels."
    ),

    "primary_endpoint": "COCO mAP@[0.50:0.95]",
}

COCO_PARAMS_PATH = OUT / "COCO_Evaluator_Parameters.json"
json_dump(coco_params, COCO_PARAMS_PATH)

print("Evaluator                   : pycocotools COCOeval")
print(
    "IoU thresholds              :",
    f"{p.iouThrs[0]:.2f} to {p.iouThrs[-1]:.2f} "
    f"in {p.iouThrs[1]-p.iouThrs[0]:.2f} steps"
)
print("COCO maxDets                :", list(map(int, p.maxDets)))
print("Prediction generation cap   : 300")
print("COCO AP evaluation cap      : 100 detections/image")
print("Small                       : area < 32^2 px")
print("Medium                      : 32^2 <= area < 96^2 px")
print("Large                       : area >= 96^2 px")


# --------------------------------------------------------------------------------------------------
# H. VERIFY COCO SIZE COUNTS FROM STAGE 10A RECONSTRUCTED TEST GT, IF AVAILABLE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("H. BATCH-DISJOINT TEST COCO AREA / SIZE VERIFICATION")
print("=" * 100)

preferred_gt = (
    DIRS["reviewer"]
    / "reconstructed_original_batch_disjoint_test_coco_taco1.json"
)

gt_path = preferred_gt if preferred_gt.exists() else None

if gt_path is None:
    # Robustly locate a 200-image / 707-object reconstructed COCO GT produced by Stage 10A.
    candidates = list(DIRS["reviewer"].rglob("*.json"))

    for cand in candidates:
        try:
            obj = read_json(cand)
            if (
                isinstance(obj, dict)
                and isinstance(obj.get("images"), list)
                and isinstance(obj.get("annotations"), list)
                and len(obj["images"]) == 200
                and len(obj["annotations"]) == 707
            ):
                gt_path = cand
                break
        except Exception:
            pass

if gt_path is None:
    warnings.warn(
        "Could not locate the Stage 10A reconstructed 200-image/707-object COCO GT. "
        "COCO parameter audit is still valid; direct area-count verification is skipped."
    )
    gt_area_check = {
        "status": "SKIPPED",
        "reason": "Stage 10A reconstructed GT not located"
    }

else:
    gt = read_json(gt_path)
    anns = gt["annotations"]

    areas = []
    area_field_matches_bbox = []

    for a in anns:
        bbox = a["bbox"]
        bbox_area = float(bbox[2]) * float(bbox[3])
        area = float(a.get("area", bbox_area))

        areas.append(area)
        area_field_matches_bbox.append(
            math.isclose(area, bbox_area, rel_tol=1e-9, abs_tol=1e-6)
        )

    areas = np.asarray(areas, dtype=float)

    small_n = int((areas < 32**2).sum())
    medium_n = int(((areas >= 32**2) & (areas < 96**2)).sum())
    large_n = int((areas >= 96**2).sum())

    if (small_n, medium_n, large_n) != (60, 211, 436):
        raise RuntimeError(
            "Batch-disjoint test COCO size counts do not match frozen values:\n"
            f"Observed: {(small_n, medium_n, large_n)}\n"
            "Expected: (60, 211, 436)"
        )

    gt_area_check = {
        "status": "PASSED",
        "gt_json": str(gt_path),
        "images": len(gt["images"]),
        "objects": len(anns),
        "small": small_n,
        "medium": medium_n,
        "large": large_n,
        "all_annotation_area_fields_equal_bbox_width_times_height": bool(
            all(area_field_matches_bbox)
        ),
    }

    print("GT source                   :", gt_path)
    print("Images / objects            :", len(gt["images"]), "/", len(anns))
    print("Small / medium / large      :", small_n, "/", medium_n, "/", large_n)
    print(
        "area == bbox width*height   :",
        bool(all(area_field_matches_bbox))
    )
    print("\nCOCO AREA / SIZE VERIFICATION : PASSED")


# ==================================================================================================
# PART I — TABLE 3 METRIC DEFINITIONS FOR THE MANUSCRIPT
# ==================================================================================================

print("\n" + "=" * 100)
print("I. TABLE 3 METRIC DEFINITIONS")
print("=" * 100)

metric_definitions = {
    "FN rate [%]": {
        "formula": "100 × FN / (TP + FN)",
        "denominator": "all ground-truth objects at the validation-frozen operating threshold",
        "IoU_matching_threshold": 0.50,
    },
    "Small-object FN rate [%]": {
        "formula": "100 × small-object FN / (small-object TP + small-object FN)",
        "denominator": (
            "ground-truth objects classified as COCO-small using original-coordinate area"
        ),
        "COCO_small_definition": "area < 32^2 pixels",
        "IoU_matching_threshold": 0.50,
    },
    "Background-like FP [%]": {
        "formula": "100 × background-like FP / all FP",
        "denominator": "all false-positive predictions at the frozen operating threshold",
        "background_like_definition": (
            "unmatched prediction whose best IoU with any ground-truth box is < 0.10, "
            "or a prediction in an image with no ground-truth object"
        ),
        "note": (
            "Background-like is an analysis label, not an annotated background class."
        ),
    },
    "Mean TP IoU": {
        "formula": (
            "mean IoU of successfully matched TP prediction-ground-truth pairs "
            "within each run; manuscript table reports mean ± sample SD across the three seeds"
        ),
        "TP_definition": (
            "prediction matched to a previously unmatched ground-truth box at IoU >= 0.50"
        ),
    },
    "Duplicate FP": {
        "definition": (
            "unmatched prediction with best IoU >= 0.50 after the corresponding "
            "ground-truth object had already been matched"
        )
    },
    "Localization FP": {
        "definition": "0.10 <= best IoU < 0.50",
        "note": "The 0.10 lower boundary is a study-specific descriptive convention, not a COCO standard."
    },
}

METRIC_DEFS_PATH = OUT / "Metric_Definitions_Table3.json"
json_dump(metric_definitions, METRIC_DEFS_PATH)

for k, v in metric_definitions.items():
    print(f"\n{k}")
    if "formula" in v:
        print("  Formula     :", v["formula"])
    elif "definition" in v:
        print("  Definition  :", v["definition"])


# ==================================================================================================
# PART J — REVIEWER-READY TRAINING SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("J. MODEL-LEVEL TRAINING SUMMARY")
print("=" * 100)

training_summary_rows = []

for protocol in protocols:
    for model_key in models:

        d = run_table[
            (run_table["protocol"] == protocol)
            & (run_table["model_key"] == model_key)
        ].copy()

        training_summary_rows.append({
            "protocol": protocol,
            "model": MODEL_DISPLAY[model_key],
            "pretrained_checkpoint": PRETRAINED[model_key],
            "seeds": ", ".join(map(str, sorted(d["seed"].astype(int).tolist()))),
            "epoch_budget": int(d["epoch_budget"].iloc[0]),
            "actual_epochs_completed_by_seed": ", ".join(
                f"{int(r.seed)}:{int(r.epochs_completed)}"
                for r in d.sort_values("seed").itertuples(index=False)
            ),
            "min_epochs_completed": int(d["epochs_completed"].min()),
            "max_epochs_completed": int(d["epochs_completed"].max()),
            "runs_early_stopped": int(d["early_stopped_before_budget"].sum()),
            "runs_total": int(len(d)),
            "optimizer_requested": "auto",
            "effective_optimizer_console_record": "AdamW",
            "effective_lr_console_record": 0.002,
            "effective_momentum_console_record": 0.9,
        })

training_summary = pd.DataFrame(training_summary_rows)

TRAINING_SUMMARY_PATH = OUT / "Supplementary_Table_Training_Summary.csv"
training_summary.to_csv(TRAINING_SUMMARY_PATH, index=False)

display(training_summary)


# ==================================================================================================
# PART K — SPLIT-GENERATION METHODS RECORD
# ==================================================================================================

split_method_record = {
    "split_seed": SPLIT_SEED,

    "random_protocol": {
        "unit": "individual image",
        "target_ratio": [0.70, 0.15, 0.15],
        "first_split": {
            "method": "sklearn.model_selection.train_test_split",
            "test_size": 0.30,
            "random_state": SPLIT_SEED,
            "stratification": (
                "n_objects clipped at an upper value of 8; strata occurring fewer than "
                "4 times were pooled into 'rare'"
            ),
        },
        "second_split": {
            "method": "sklearn.model_selection.train_test_split",
            "test_size": 0.50,
            "random_state": SPLIT_SEED,
            "stratification": (
                "n_objects clipped at 8 within the temporary 30% subset; strata "
                "occurring fewer than 2 times were pooled into 'rare'"
            ),
            "deterministic_unstratified_fallback_if_ValueError": True,
            "fallback_was_used_in_frozen_reproduction": random_second_stage_fallback,
        },
        "exact_reproduction": bool(random_exact),
    },

    "batch_disjoint_protocol": {
        "unit": "official TACO batch",
        "number_of_official_batches": int(len(groups)),
        "target_ratio": target_ratio,
        "candidate_assignments": N_SEARCH,
        "rng": "numpy.random.default_rng",
        "rng_seed": SPLIT_SEED,
        "candidate_assignment_rule": (
            "for each batch, r<0.70 -> train; 0.70<=r<0.85 -> val; r>=0.85 -> test"
        ),
        "objective_variables": metric_cols,
        "objective_weights": {
            "images": 3.0,
            "objects": 1.0,
            "small": 1.0,
            "medium": 1.0,
            "large": 1.0,
        },
        "objective_formula": (
            "sum over splits and variables of "
            "weight_k × ((observed_k - target_k) / target_k)^2; "
            "candidate assignments with an empty split receive infinite score"
        ),
        "best_objective_score": best_score,
        "selected_batches": {
            s: [g for g in groups if best_assign[g] == s]
            for s in split_names
        },
        "exact_reproduction": bool(batch_exact and batch_assignment_exact),
        "interpretation_limit": (
            "The procedure enforces exclusivity only with respect to official TACO batch labels. "
            "It does not establish scene, site, session, geographic, or temporal independence."
        ),
    },
}

SPLIT_RECORD_PATH = OUT / "split_generation_reproducibility.json"
json_dump(split_method_record, SPLIT_RECORD_PATH)


# ==================================================================================================
# PART L — FINAL STAGE 10B MANIFEST
# ==================================================================================================

stage10a_hash = (
    sha256_file(STAGE10A_MANIFEST)
    if STAGE10A_MANIFEST.exists()
    else None
)

output_files = [
    split_table_path,
    run_table_path,
    complexity_path,
    optimizer_path,
    COCO_PARAMS_PATH,
    METRIC_DEFS_PATH,
    TRAINING_SUMMARY_PATH,
    SPLIT_RECORD_PATH,
]

output_hashes = {
    str(p.relative_to(ROOT)): sha256_file(p)
    for p in output_files
    if p.exists()
}

manifest = {
    "stage": "10B",
    "name": "Reviewer-requested split/training/COCO reproducibility audit",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "canonical_project_root": str(ROOT),

    "scientific_status": {
        "new_training": False,
        "new_inference": False,
        "threshold_tuning": False,
        "retraining": False,
        "model_exclusion": False,
        "primary_frozen_results_changed": False,
        "post_test_hyperparameter_optimization": False,
    },

    "stage10a": {
        "manifest_present": STAGE10A_MANIFEST.exists(),
        "manifest_sha256": stage10a_hash,
        "manual_visual_classification_of_phash0_pair": (
            "same-scene near-duplicate pair with nearly identical framing; "
            "not byte-identical because SHA-256 differs"
        ),
    },

    "dataset_identity": {
        "images": len(img_df),
        "objects": total_objects,
        "small": small_total,
        "medium": medium_total,
        "large": large_total,
        "official_batches": int(img_df["batch"].nunique()),
    },

    "split_reproduction": {
        "random_exact": bool(random_exact),
        "batch_disjoint_exact": bool(batch_exact),
        "batch_assignment_csv_exact": bool(batch_assignment_exact),
        "split_seed": SPLIT_SEED,
        "batch_candidate_assignments": N_SEARCH,
        "batch_best_objective_score": best_score,
    },

    "training_artifact_audit": {
        "expected_runs": 24,
        "verified_run_rows": int(len(run_table)),
        "all_best_pt_present": True,
        "checkpoint_hashes_registered_in_final_manifest": int(
            run_table["checkpoint_hash_registered_in_final_manifest"].sum()
        ),
        "pretrained_checkpoints": expected_pretrained,
        "optimizer_requested": training_cfg.get("optimizer"),
        "effective_optimizer_console_record": {
            "name": "AdamW",
            "learning_rate": 0.002,
            "momentum": 0.9,
        },
    },

    "coco_evaluation": coco_params,
    "batch_test_gt_area_check": gt_area_check,

    "outputs": output_hashes,
}

STAGE10B_MANIFEST = DIRS["manifests"] / "STAGE_10B_REPRODUCIBILITY_AUDIT.json"
json_dump(manifest, STAGE10B_MANIFEST)


# ==================================================================================================
# FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 100)
print("STAGE 10B PASSED — REVIEWER REPRODUCIBILITY AUDIT COMPLETE")
print("=" * 100)

print("\nSplit reproduction:")
print("Random exact reproduction           :", random_exact)
print("Batch-disjoint exact reproduction   :", batch_exact)
print("Batch assignment CSV exact          :", batch_assignment_exact)

print("\nFrozen batch assignment:")
for s in ["train", "val", "test"]:
    vals = [g for g in groups if best_assign[g] == s]
    print(f"{s:5s}: {', '.join(vals)}")

print("\nTraining artifacts:")
print("Run rows verified                   :", f"{len(run_table)}/24")
print(
    "Checkpoint hashes in final manifest:",
    f"{int(run_table['checkpoint_hash_registered_in_final_manifest'].sum())}/24"
)
print("New training                        : NO")
print("New inference                       : NO")
print("Threshold tuning                    : NO")
print("Retraining                          : NO")
print("Primary frozen results changed      : NO")

print("\nOptimizer:")
print("Requested                           : optimizer='auto'")
print("Original console effective record   : AdamW(lr=0.002, momentum=0.9)")

print("\nCOCO:")
print("Evaluator                           : pycocotools COCOeval")
print("Prediction confidence floor         : 0.001")
print("Prediction generation max_det       : 300")
print("COCOeval maxDets                    :", list(map(int, p.maxDets)))
print("COCO size boundaries                : 32^2 / 96^2 pixels")

print("\nReviewer-ready files:")
for pth in output_files:
    print(" -", pth)

print("\nManifest:")
print(STAGE10B_MANIFEST)

print("\nNEXT:")
print(
    "SEND THE COMPLETE STAGE 10B OUTPUT, ESPECIALLY:\n"
    "1) RANDOM + BATCH EXACT REPRODUCTION status,\n"
    "2) selected batch assignment,\n"
    "3) actual completed-epoch table,\n"
    "4) model-complexity table,\n"
    "5) COCO evaluator block,\n"
    "6) any WARNING or ERROR shown above."
)

print("=" * 100)

In [ ]:
# ============================================================
# INSTALL ORIGINAL ULTRALYTICS VERSION FOR STAGE 10C
# ============================================================

import sys
import subprocess

TARGET_VERSION = "8.4.111"

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    f"ultralytics=={TARGET_VERSION}",
])

print("Ultralytics installation completed.")

In [ ]:
# ==================================================================================================
# STAGE 10C — CHECKPOINT REGISTRY + STATIC MODEL COMPLEXITY AUDIT
#
# PURPOSE
# --------------------------------------------------------------------------------------------------
# 1. Resolve the Stage 10B "0/24 checkpoint hashes registered" diagnostic.
# 2. Search the ORIGINAL manifest/freeze registries for all 24 frozen best.pt SHA-256 values.
# 3. Verify representative frozen checkpoints for the four model architectures.
# 4. Recover static parameter count and GFLOPs at imgsz=640.
# 5. Compare against ORIGINAL training-console architecture summaries.
#
# SAFETY
# --------------------------------------------------------------------------------------------------
# NO training
# NO dataset loading
# NO validation
# NO test inference
# NO predict()
# NO threshold tuning
# NO checkpoint modification
# NO primary-result modification
#
# get_flops performs static/dummy architecture profiling only.
# It does not evaluate any TACO image.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json

import numpy as np
import pandas as pd
import torch
import ultralytics

from ultralytics import YOLO, RTDETR
from ultralytics.utils.torch_utils import get_flops


# ==================================================================================================
# 0. PATHS
# ==================================================================================================

ROOT = Path(
    "/content/GDRIVE_REAL/MyDrive/Atik14082026/TACO_Pipeline_v5"
)

STAGE10B_DIR = (
    ROOT
    / "10_reviewer_sensitivity"
    / "stage10b_reproducibility"
)

RUN_DETAILS_PATH = (
    STAGE10B_DIR
    / "Supplementary_Table_Run_Training_Details.csv"
)

OUT_DIR = (
    ROOT
    / "10_reviewer_sensitivity"
    / "stage10c_checkpoint_complexity"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_OUT = (
    ROOT
    / "07_manifests"
    / "STAGE_10C_CHECKPOINT_COMPLEXITY_AUDIT.json"
)


print("=" * 100)
print("STAGE 10C — CHECKPOINT REGISTRY + STATIC MODEL COMPLEXITY AUDIT")
print("=" * 100)


# ==================================================================================================
# 1. BASIC SAFETY CHECKS
# ==================================================================================================

if not ROOT.exists():
    raise FileNotFoundError(
        f"Canonical project root missing:\n{ROOT}"
    )

if not RUN_DETAILS_PATH.exists():
    raise FileNotFoundError(
        "Stage 10B run-detail table missing:\n"
        f"{RUN_DETAILS_PATH}"
    )


print(
    f"\nUltralytics version         : {ultralytics.__version__}"
)

if ultralytics.__version__ != "8.4.111":
    raise RuntimeError(
        "Stage 10C must use the original Ultralytics "
        "version 8.4.111.\n"
        f"Current version: {ultralytics.__version__}"
    )


print(
    "Ultralytics version        : PASSED"
)


# ==================================================================================================
# 2. HELPERS
# ==================================================================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def json_dump(
    obj,
    path
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            default=str
        ),
        encoding="utf-8"
    )


# ==================================================================================================
# 3. LOAD STAGE 10B CHECKPOINT HASHES
# ==================================================================================================

runs = pd.read_csv(
    RUN_DETAILS_PATH
)


required_cols = {
    "protocol",
    "model_key",
    "seed",
    "best_pt_sha256",
    "best_pt_bytes",
}


missing = required_cols - set(
    runs.columns
)


if missing:
    raise RuntimeError(
        f"Stage 10B table missing columns: {sorted(missing)}"
    )


if len(runs) != 24:
    raise RuntimeError(
        f"Expected 24 runs; found {len(runs)}."
    )


if runs["best_pt_sha256"].nunique() != 24:
    raise RuntimeError(
        "Expected 24 distinct frozen checkpoint hashes."
    )


print(
    "\nFrozen checkpoint rows      : 24/24"
)

print(
    "Distinct checkpoint hashes : 24/24"
)


# ==================================================================================================
# 4. VERIFY CURRENT best.pt FILES AGAIN
# ==================================================================================================

checkpoint_rows = []


for r in runs.itertuples(index=False):

    best_pt = (
        ROOT
        / "03_runs"
        / str(r.protocol)
        / f"{r.model_key}_seed{int(r.seed)}"
        / "weights"
        / "best.pt"
    )


    if not best_pt.exists():

        raise FileNotFoundError(
            f"Frozen best.pt missing:\n{best_pt}"
        )


    current_size = int(
        best_pt.stat().st_size
    )

    current_sha = sha256_file(
        best_pt
    )


    size_ok = (
        current_size
        ==
        int(r.best_pt_bytes)
    )


    sha_ok = (
        current_sha
        ==
        str(r.best_pt_sha256)
    )


    if not size_ok or not sha_ok:

        raise RuntimeError(
            "Frozen checkpoint integrity mismatch:\n"
            f"{r.protocol}/{r.model_key}/seed{r.seed}\n"
            f"Size OK: {size_ok}\n"
            f"SHA OK : {sha_ok}"
        )


    checkpoint_rows.append(
        {
            "protocol":
                str(r.protocol),

            "model_key":
                str(r.model_key),

            "seed":
                int(r.seed),

            "best_pt":
                str(best_pt),

            "bytes":
                current_size,

            "sha256":
                current_sha,

            "current_integrity":
                "PASSED",
        }
    )


checkpoint_df = pd.DataFrame(
    checkpoint_rows
)


print(
    "Current best.pt integrity   : 24/24 PASSED"
)


# ==================================================================================================
# 5. SEARCH ORIGINAL PRE-STAGE10C REGISTRIES
#
# Search only persistent manifest/freeze text artifacts.
# Do not search run directories or newly generated Stage 10C files.
# ==================================================================================================

SEARCH_ROOTS = [

    ROOT / "07_manifests",

    ROOT / "09_manuscript" / "freeze",
]


TEXT_SUFFIXES = {
    ".json",
    ".csv",
    ".txt",
    ".tsv",
    ".md",
    ".yaml",
    ".yml",
}


registry_files = []


for search_root in SEARCH_ROOTS:

    if not search_root.exists():
        continue


    for p in search_root.rglob("*"):

        if not p.is_file():
            continue

        if p == MANIFEST_OUT:
            continue

        if p.suffix.lower() not in TEXT_SUFFIXES:
            continue

        # Avoid pathological huge text artifacts.
        if p.stat().st_size > 100 * 1024 * 1024:
            continue

        registry_files.append(
            p
        )


print(
    f"\nText registry files scanned : {len(registry_files)}"
)


# Read once for efficiency.
registry_text = {}


for p in registry_files:

    try:

        registry_text[p] = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    except Exception:

        registry_text[p] = ""


hash_hit_rows = []


for r in checkpoint_df.itertuples(index=False):

    hits = []


    for p, text in registry_text.items():

        if r.sha256 in text:

            hits.append(
                str(
                    p.relative_to(
                        ROOT
                    )
                )
            )


    hash_hit_rows.append(
        {
            "protocol":
                r.protocol,

            "model_key":
                r.model_key,

            "seed":
                int(r.seed),

            "sha256":
                r.sha256,

            "registry_hit_count":
                len(hits),

            "registry_files":
                " | ".join(
                    hits
                ),
        }
    )


hash_hits = pd.DataFrame(
    hash_hit_rows
)


HASH_HITS_PATH = (
    OUT_DIR
    / "Checkpoint_Hash_Registry_Hits.csv"
)


hash_hits.to_csv(
    HASH_HITS_PATH,
    index=False
)


registered_count = int(
    (
        hash_hits[
            "registry_hit_count"
        ]
        >
        0
    ).sum()
)


print(
    f"Checkpoint hashes located   : "
    f"{registered_count}/24"
)


if registered_count < 24:

    print(
        "\nHashes not located in searched text registries:"
    )

    display(
        hash_hits[
            hash_hits[
                "registry_hit_count"
            ]
            ==
            0
        ][
            [
                "protocol",
                "model_key",
                "seed",
                "sha256",
            ]
        ]
    )


    print(
        "\nFiles with hash/checkpoint/manifest-like names:"
    )


    diagnostic_files = sorted(
        {
            str(p.relative_to(ROOT))

            for root in SEARCH_ROOTS

            if root.exists()

            for p in root.rglob("*")

            if (
                p.is_file()
                and
                any(
                    token
                    in
                    p.name.lower()

                    for token in [
                        "hash",
                        "sha",
                        "checkpoint",
                        "manifest",
                        "freeze",
                        "integrity",
                    ]
                )
            )
        }
    )


    for p in diagnostic_files:

        print(
            " -",
            p
        )


else:

    print(
        "Original checkpoint registry: 24/24 PASSED"
    )


# ==================================================================================================
# 6. STATIC MODEL COMPLEXITY
#
# One representative frozen checkpoint per architecture.
#
# This does NOT perform dataset inference.
# ==================================================================================================

REPRESENTATIVE = {

    "yolov8s": {
        "family":
            "YOLO",

        "checkpoint":
            ROOT
            / "03_runs"
            / "random"
            / "yolov8s_seed42"
            / "weights"
            / "best.pt",

        "pretrained":
            "yolov8s.pt",

        # Original training-console summary after nc=1 override
        "expected_params":
            11_135_987,

        "expected_gflops":
            28.6,
    },


    "yolo11s": {
        "family":
            "YOLO",

        "checkpoint":
            ROOT
            / "03_runs"
            / "random"
            / "yolo11s_seed42"
            / "weights"
            / "best.pt",

        "pretrained":
            "yolo11s.pt",

        "expected_params":
            9_428_179,

        "expected_gflops":
            21.5,
    },


    "yolo26s": {
        "family":
            "YOLO",

        "checkpoint":
            ROOT
            / "03_runs"
            / "random"
            / "yolo26s_seed42"
            / "weights"
            / "best.pt",

        "pretrained":
            "yolo26s.pt",

        "expected_params":
            9_948_638,

        "expected_gflops":
            22.5,
    },


    "rtdetr_l": {
        "family":
            "RTDETR",

        "checkpoint":
            ROOT
            / "03_runs"
            / "random"
            / "rtdetr_l_seed42"
            / "weights"
            / "best.pt",

        "pretrained":
            "rtdetr-l.pt",

        "expected_params":
            32_808_131,

        "expected_gflops":
            109.9,
    },
}


DISPLAY = {
    "yolov8s": "YOLOv8s",
    "yolo11s": "YOLO11s",
    "yolo26s": "YOLO26s",
    "rtdetr_l": "RT-DETR-L",
}


complexity_rows = []


print(
    "\n"
    +
    "=" * 100
)

print(
    "STATIC MODEL COMPLEXITY AUDIT"
)

print(
    "=" * 100
)


for model_key, spec in REPRESENTATIVE.items():

    checkpoint = Path(
        spec[
            "checkpoint"
        ]
    )


    if not checkpoint.exists():

        raise FileNotFoundError(
            f"Representative checkpoint missing:\n"
            f"{checkpoint}"
        )


    print(
        f"\nLoading frozen checkpoint statically: "
        f"{DISPLAY[model_key]}"
    )


    if spec["family"] == "RTDETR":

        wrapper = RTDETR(
            str(
                checkpoint
            )
        )

    else:

        wrapper = YOLO(
            str(
                checkpoint
            )
        )


    # CPU only.
    wrapper.model.to(
        "cpu"
    )


    # No predict(), no val(), no dataset.
    params = int(
        sum(
            p.numel()

            for p in wrapper.model.parameters()
        )
    )


    gflops = float(
        get_flops(
            wrapper.model,
            imgsz=640
        )
    )


    expected_params = int(
        spec[
            "expected_params"
        ]
    )

    expected_gflops = float(
        spec[
            "expected_gflops"
        ]
    )


    params_match = (
        params
        ==
        expected_params
    )


    # Ultralytics console rounds GFLOPs to one decimal.
    gflops_match = (
        round(
            gflops,
            1
        )
        ==
        round(
            expected_gflops,
            1
        )
    )


    print(
        f"Parameters                  : {params:,}"
    )

    print(
        f"Original console expected   : "
        f"{expected_params:,}"
    )

    print(
        f"Parameters match            : {params_match}"
    )

    print(
        f"GFLOPs @640                 : {gflops:.4f}"
    )

    print(
        f"Original console expected   : "
        f"{expected_gflops:.1f}"
    )

    print(
        f"GFLOPs rounded match        : {gflops_match}"
    )


    complexity_rows.append(
        {
            "model":
                DISPLAY[
                    model_key
                ],

            "model_key":
                model_key,

            "pretrained_checkpoint":
                spec[
                    "pretrained"
                ],

            "representative_frozen_checkpoint":
                str(
                    checkpoint
                ),

            "input_size":
                640,

            "parameters_unfused":
                params,

            "GFLOPs_at_640":
                gflops,

            "original_console_parameters":
                expected_params,

            "original_console_GFLOPs":
                expected_gflops,

            "parameter_match":
                params_match,

            "GFLOPs_rounded_match":
                gflops_match,

            "static_characterization_only":
                True,

            "dataset_inference":
                False,
        }
    )


    del wrapper

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


complexity_df = pd.DataFrame(
    complexity_rows
)


COMPLEXITY_PATH = (
    OUT_DIR
    / "Supplementary_Table_Model_Complexity_Verified.csv"
)


complexity_df.to_csv(
    COMPLEXITY_PATH,
    index=False
)


print(
    "\nVerified model complexity:"
)

display(
    complexity_df[
        [
            "model",
            "pretrained_checkpoint",
            "parameters_unfused",
            "GFLOPs_at_640",
            "parameter_match",
            "GFLOPs_rounded_match",
        ]
    ]
)


if not complexity_df[
    "parameter_match"
].all():

    raise RuntimeError(
        "At least one parameter count does not match "
        "the original training-console summary."
    )


if not complexity_df[
    "GFLOPs_rounded_match"
].all():

    raise RuntimeError(
        "At least one GFLOPs value does not match "
        "the original training-console summary."
    )


print(
    "\nModel complexity             : 4/4 PASSED"
)


# ==================================================================================================
# 7. CHECKPOINT FILE-SIZE SUMMARY
# ==================================================================================================

size_summary = (

    checkpoint_df
    .assign(
        checkpoint_MiB=lambda d:
            d[
                "bytes"
            ]
            /
            (
                1024
                **
                2
            )
    )
    .groupby(
        "model_key"
    )[
        "checkpoint_MiB"
    ]
    .agg(
        [
            "mean",
            "std",
            "min",
            "max",
        ]
    )
    .reset_index()
)


SIZE_PATH = (
    OUT_DIR
    / "Checkpoint_Size_Summary.csv"
)


size_summary.to_csv(
    SIZE_PATH,
    index=False
)


# ==================================================================================================
# 8. MANUSCRIPT-READY MODEL TABLE
# ==================================================================================================

size_mean_lookup = dict(
    zip(
        size_summary[
            "model_key"
        ],
        size_summary[
            "mean"
        ]
    )
)


manuscript_rows = []


for r in complexity_df.itertuples(index=False):

    manuscript_rows.append(
        {
            "Model":
                r.model,

            "Pretrained checkpoint":
                r.pretrained_checkpoint,

            "Parameters":
                int(
                    r.parameters_unfused
                ),

            "GFLOPs @640":
                round(
                    float(
                        r.GFLOPs_at_640
                    ),
                    1
                ),

            "Frozen best.pt mean size [MiB]":
                round(
                    float(
                        size_mean_lookup[
                            r.model_key
                        ]
                    ),
                    2
                ),
        }
    )


manuscript_complexity = pd.DataFrame(
    manuscript_rows
)


MANUSCRIPT_COMPLEXITY_PATH = (
    OUT_DIR
    / "Supplementary_Table_Model_Complexity_Manuscript.csv"
)


manuscript_complexity.to_csv(
    MANUSCRIPT_COMPLEXITY_PATH,
    index=False
)


print(
    "\nManuscript-ready complexity table:"
)

display(
    manuscript_complexity
)


# ==================================================================================================
# 9. FINAL MANIFEST
# ==================================================================================================

registry_complete = (
    registered_count
    ==
    24
)


complexity_complete = bool(
    complexity_df[
        "parameter_match"
    ].all()
    and
    complexity_df[
        "GFLOPs_rounded_match"
    ].all()
)


manifest = {

    "stage":
        "10C",

    "name":
        (
            "Checkpoint registry and static "
            "model-complexity audit"
        ),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_status": {

        "new_training":
            False,

        "dataset_inference":
            False,

        "validation":
            False,

        "test_inference":
            False,

        "threshold_tuning":
            False,

        "checkpoint_modification":
            False,

        "primary_results_changed":
            False,
    },

    "checkpoint_integrity": {

        "current_best_pt_verified":
            24,

        "current_best_pt_expected":
            24,

        "registry_hashes_located":
            registered_count,

        "registry_hashes_expected":
            24,

        "registry_complete":
            registry_complete,

        "registry_search_roots": [
            str(x)
            for x in SEARCH_ROOTS
        ],
    },

    "static_model_complexity": {

        "models_verified":
            int(
                len(
                    complexity_df
                )
            ),

        "parameters_match_original_console":
            bool(
                complexity_df[
                    "parameter_match"
                ].all()
            ),

        "gflops_match_original_console":
            bool(
                complexity_df[
                    "GFLOPs_rounded_match"
                ].all()
            ),

        "input_size":
            640,

        "new_dataset_evaluation":
            False,
    },

    "outputs": {

        "checkpoint_registry_hits":
            str(
                HASH_HITS_PATH
            ),

        "verified_model_complexity":
            str(
                COMPLEXITY_PATH
            ),

        "manuscript_model_complexity":
            str(
                MANUSCRIPT_COMPLEXITY_PATH
            ),

        "checkpoint_size_summary":
            str(
                SIZE_PATH
            ),
    },
}


json_dump(
    manifest,
    MANIFEST_OUT
)


# ==================================================================================================
# 10. FINAL STATUS
# ==================================================================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "STAGE 10C FINAL STATUS"
)

print(
    "=" * 100
)


print(
    "Current checkpoint integrity : 24/24 PASSED"
)

print(
    f"Original registry hits       : "
    f"{registered_count}/24"
)

print(
    "Model parameter audit        : "
    f"{'4/4 PASSED' if complexity_complete else 'FAILED'}"
)

print(
    "Model GFLOPs audit           : "
    f"{'4/4 PASSED' if complexity_complete else 'FAILED'}"
)

print(
    "Training                     : NO"
)

print(
    "Dataset inference            : NO"
)

print(
    "Test inference               : NO"
)

print(
    "Threshold tuning             : NO"
)

print(
    "Primary results changed      : NO"
)


print(
    "\nOutputs:"
)

print(
    HASH_HITS_PATH
)

print(
    COMPLEXITY_PATH
)

print(
    MANUSCRIPT_COMPLEXITY_PATH
)

print(
    SIZE_PATH
)

print(
    MANIFEST_OUT
)


if not registry_complete:

    print(
        "\nSTATUS: COMPLEXITY PASSED, "
        "CHECKPOINT REGISTRY LOCATION STILL REQUIRES REVIEW."
    )

    print(
        "Send the displayed list of hash/manifest-like files "
        "and Checkpoint_Hash_Registry_Hits.csv summary."
    )

else:

    print(
        "\nSTAGE 10C PASSED — "
        "CHECKPOINT REGISTRY + MODEL COMPLEXITY COMPLETE"
    )


print(
    "=" * 100
)

# Manuscript interpretation rules

1. Treat **batch-disjoint locked-test mAP@50:95** as the main generalization result.
2. Treat random-split performance as a protocol-sensitivity comparator.
3. Report all main performance results as **3-seed mean ± SD**.
4. Do not claim one model is superior from a small numerical difference alone.
5. Report **AP-small** separately because small litter objects are a central difficulty.
6. Test-set confidence thresholds must always be the thresholds frozen on validation.
7. State explicitly that all official TACO categories were collapsed into the one-class **TACO-1 litter-detection task**.
8. State explicitly that `annotations_unofficial.json` was excluded.
9. pHash near-duplicate matches are screening candidates, not proof of duplication.
10. If any scientific design choice is changed after Stage 7 is opened, start a new experiment version rather than overwriting this run.